In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# @title
# TASK B — CELL 1: Targeted data and notebook inventory

from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

DATA_DIR = BASE_DIR / "data"
NOTEBOOK_DIR = BASE_DIR / "notebooks"

print("=" * 78)
print("DIRECTORY CHECK")
print("=" * 78)
print("Base directory:", BASE_DIR)
print("Exists:", BASE_DIR.exists())
print()
print("Data directory:", DATA_DIR)
print("Exists:", DATA_DIR.exists())
print()
print("Notebook directory:", NOTEBOOK_DIR)
print("Exists:", NOTEBOOK_DIR.exists())

# ---------------------------------------------------------------------
# 1. List notebooks and Python scripts
# ---------------------------------------------------------------------

notebook_files = sorted(
    list(NOTEBOOK_DIR.rglob("*.ipynb"))
    + list(NOTEBOOK_DIR.rglob("*.py"))
)

print()
print("=" * 78)
print("AVAILABLE NOTEBOOKS AND SCRIPTS")
print("=" * 78)
print("Files found:", len(notebook_files))

for path in notebook_files:
    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{size_mb:8.3f} MB  "
        f"{path.relative_to(BASE_DIR)}"
    )

# ---------------------------------------------------------------------
# 2. Identify likely Task B notebooks by filename and content
# ---------------------------------------------------------------------

notebook_keyword_rows = []

keywords = [
    "task b",
    "taskb",
    "definition",
    "generate",
    "generation",
    "t5",
    "bart",
    "gpt",
    "bertscore",
    "rouge",
    "bleu",
]

for path in notebook_files:
    try:
        if path.suffix.casefold() == ".ipynb":
            with open(
                path,
                "r",
                encoding="utf-8",
            ) as file:
                notebook = json.load(file)

            source_text = "\n".join(
                "".join(
                    cell.get(
                        "source",
                        [],
                    )
                )
                for cell in notebook.get(
                    "cells",
                    [],
                )
            )

            n_cells = len(
                notebook.get(
                    "cells",
                    [],
                )
            )

        else:
            source_text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )

            n_cells = None

        source_casefolded = (
            source_text.casefold()
        )

        matched_keywords = [
            keyword
            for keyword in keywords
            if keyword in source_casefolded
        ]

        notebook_keyword_rows.append(
            {
                "file": str(
                    path.relative_to(
                        BASE_DIR
                    )
                ),
                "suffix": path.suffix,
                "cells": n_cells,
                "matched_keywords": (
                    " | ".join(
                        matched_keywords
                    )
                ),
                "likely_task_b": bool(
                    matched_keywords
                ),
            }
        )

    except Exception as error:
        notebook_keyword_rows.append(
            {
                "file": str(
                    path.relative_to(
                        BASE_DIR
                    )
                ),
                "suffix": path.suffix,
                "cells": None,
                "matched_keywords": (
                    f"ERROR: {error!r}"
                ),
                "likely_task_b": False,
            }
        )

notebook_audit = pd.DataFrame(
    notebook_keyword_rows
)

print()
print("=" * 78)
print("NOTEBOOK CONTENT AUDIT")
print("=" * 78)

if notebook_audit.empty:
    print("No notebooks or scripts found.")
else:
    print(
        notebook_audit.to_string(
            index=False
        )
    )

# ---------------------------------------------------------------------
# 3. List potentially relevant Task B data files
# ---------------------------------------------------------------------

supported_extensions = {
    ".csv",
    ".tsv",
    ".xlsx",
    ".xls",
    ".json",
    ".jsonl",
    ".txt",
}

all_data_files = sorted(
    path
    for path in DATA_DIR.rglob("*")
    if (
        path.is_file()
        and path.suffix.casefold()
        in supported_extensions
    )
)

taskb_data_files = [
    path
    for path in all_data_files
    if any(
        keyword in str(path).casefold()
        for keyword in [
            "taskb",
            "task_b",
            "task b",
            "definition",
        ]
    )
]

print()
print("=" * 78)
print("LIKELY TASK B DATA FILES")
print("=" * 78)
print(
    "Likely Task B files:",
    len(taskb_data_files),
)
print(
    "All supported files under data:",
    len(all_data_files),
)

files_to_show = (
    taskb_data_files
    if taskb_data_files
    else all_data_files
)

for path in files_to_show:
    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{size_mb:8.3f} MB  "
        f"{path.relative_to(BASE_DIR)}"
    )

# ---------------------------------------------------------------------
# 4. Inspect schemas of likely Task B tabular files
# ---------------------------------------------------------------------

tabular_extensions = {
    ".csv",
    ".tsv",
    ".xlsx",
    ".xls",
    ".json",
    ".jsonl",
}

tabular_files = [
    path
    for path in files_to_show
    if path.suffix.casefold()
    in tabular_extensions
]

print()
print("=" * 78)
print("TASK B TABULAR SCHEMAS")
print("=" * 78)

for path in tabular_files:
    print()
    print("-" * 78)
    print(
        "FILE:",
        path.relative_to(BASE_DIR),
    )
    print("-" * 78)

    try:
        suffix = path.suffix.casefold()

        if suffix == ".csv":
            table = pd.read_csv(path)

        elif suffix == ".tsv":
            table = pd.read_csv(
                path,
                sep="\t",
            )

        elif suffix in {
            ".xlsx",
            ".xls",
        }:
            workbook = pd.ExcelFile(path)

            print(
                "Sheets:",
                workbook.sheet_names,
            )

            for sheet in workbook.sheet_names:
                preview = pd.read_excel(
                    path,
                    sheet_name=sheet,
                    nrows=5,
                )

                print()
                print("Sheet:", sheet)
                print(
                    "Columns:",
                    list(preview.columns),
                )
                print(
                    preview.to_string(
                        index=False
                    )
                )

            continue

        elif suffix == ".jsonl":
            table = pd.read_json(
                path,
                lines=True,
            )

        else:
            table = pd.read_json(path)

        print("Rows:", len(table))
        print(
            "Columns:",
            list(table.columns),
        )

        print()
        print("Preview:")
        print(
            table.head().to_string(
                index=False
            )
        )

        print()
        print("Missing values:")
        print(
            table.isna()
            .sum()
            .to_string()
        )

        print()
        print("Unique values:")
        for column in table.columns:
            print(
                f"  {column}: "
                f"{table[column].nunique(dropna=True)}"
            )

    except Exception as error:
        print(
            "Could not inspect:",
            repr(error),
        )

# ---------------------------------------------------------------------
# 5. Preview likely Task B text files
# ---------------------------------------------------------------------

text_files = [
    path
    for path in files_to_show
    if path.suffix.casefold()
    == ".txt"
]

print()
print("=" * 78)
print("TASK B TEXT FILES")
print("=" * 78)
print("Text files:", len(text_files))

for path in text_files[:10]:
    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    print()
    print(
        "FILE:",
        path.relative_to(BASE_DIR),
    )
    print("Characters:", len(text))
    print(
        "Preview:",
        repr(text[:300]),
    )

print()
print("Task B Cell 1 completed successfully.")

In [ ]:
# TASK B — RESTORE LOCKED ARTIFACTS
# Run after mounting Drive and defining BASE_DIR.

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Authoritative directories
# ------------------------------------------------------------

TASKB_RESULTS_DIR = (
    BASE_DIR
    / "journal_experiments/taskB_holdout/results"
)

TASKB_SPLIT_DIR = (
    BASE_DIR
    / "journal_experiments/taskB_holdout/splits"
)


# ------------------------------------------------------------
# 2. Authoritative artifact paths
# ------------------------------------------------------------

GOLD_LINKS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_unique_gold_links.csv"
)

MENTION_INPUTS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_mention_inputs.csv"
)

DEFINITIONS_CANONICAL_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_reference_definitions.csv"
)

DOCUMENT_METADATA_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_document_metadata_before_split.csv"
)

DOCUMENT_SPLIT_PATH = (
    TASKB_SPLIT_DIR
    / "taskB_document_holdout_grouped_70_15_15.csv"
)

GOLD_LINKS_SPLIT_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_gold_links_with_split.csv"
)

MENTION_INPUTS_SPLIT_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_mention_inputs_with_split.csv"
)

SPLIT_METADATA_PATH = (
    TASKB_SPLIT_DIR
    / "taskB_document_holdout_metadata.json"
)

REPAIR_METADATA_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_concept_label_repairs.json"
)


# ------------------------------------------------------------
# 3. Confirm that all locked artifacts exist
# ------------------------------------------------------------

required_paths = [
    GOLD_LINKS_PATH,
    MENTION_INPUTS_PATH,
    DEFINITIONS_CANONICAL_PATH,
    DOCUMENT_METADATA_PATH,
    DOCUMENT_SPLIT_PATH,
    GOLD_LINKS_SPLIT_PATH,
    MENTION_INPUTS_SPLIT_PATH,
    SPLIT_METADATA_PATH,
    REPAIR_METADATA_PATH,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing locked Task B artifacts:\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


# ------------------------------------------------------------
# 4. Reload authoritative tables
# ------------------------------------------------------------

taskb_gold_links = pd.read_csv(
    GOLD_LINKS_PATH,
    dtype={
        "num_article": str,
    },
)

taskb_mention_inputs = pd.read_csv(
    MENTION_INPUTS_PATH,
    dtype={
        "num_article": str,
    },
)

taskb_reference_definitions = pd.read_csv(
    DEFINITIONS_CANONICAL_PATH
)

document_metadata = pd.read_csv(
    DOCUMENT_METADATA_PATH,
    dtype={
        "num_article": str,
    },
)

taskb_document_split = pd.read_csv(
    DOCUMENT_SPLIT_PATH,
    dtype={
        "num_article": str,
    },
)

taskb_gold_links_split = pd.read_csv(
    GOLD_LINKS_SPLIT_PATH,
    dtype={
        "num_article": str,
    },
)

taskb_mention_inputs_split = pd.read_csv(
    MENTION_INPUTS_SPLIT_PATH,
    dtype={
        "num_article": str,
    },
)

split_metadata = json.loads(
    SPLIT_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)

repair_metadata = json.loads(
    REPAIR_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# 5. Restore helper used by later cells
# ------------------------------------------------------------

def normalize_concept(value):
    value = re.sub(
        r"\s+",
        " ",
        str(value).strip(),
    )

    if (
        value.startswith("<")
        and value.endswith(">")
    ):
        value = value[1:-1].strip()

    return value.casefold()


# ------------------------------------------------------------
# 6. Integrity checks
# ------------------------------------------------------------

assert len(taskb_gold_links) == 869
assert len(taskb_mention_inputs) == 790
assert len(taskb_reference_definitions) == 315
assert len(taskb_document_split) == 37
assert len(taskb_gold_links_split) == 869
assert len(taskb_mention_inputs_split) == 790

assert (
    taskb_gold_links_split["split"]
    .notna()
    .all()
)

assert (
    taskb_mention_inputs_split["split"]
    .notna()
    .all()
)

assert set(
    taskb_gold_links_split["split"]
) == {
    "train",
    "development",
    "test",
}

assert (
    taskb_document_split[
        "document_key"
    ].nunique()
    == 37
)


# ------------------------------------------------------------
# 7. Verify split counts
# ------------------------------------------------------------

expected_document_counts = {
    ("mental_health", "train"): 16,
    ("mental_health", "development"): 3,
    ("mental_health", "test"): 4,
    ("parkinson", "train"): 10,
    ("parkinson", "development"): 2,
    ("parkinson", "test"): 2,
}

actual_document_counts = (
    taskb_document_split.groupby(
        ["domain", "split"]
    )
    .size()
    .to_dict()
)

assert (
    actual_document_counts
    == expected_document_counts
), actual_document_counts


# ------------------------------------------------------------
# 8. Report restored state
# ------------------------------------------------------------

print("=" * 78)
print("LOCKED TASK B STATE RESTORED")
print("=" * 78)

print(
    "Selected split seed:",
    split_metadata["selected_seed"],
)

print(
    "Authoritative semantic links:",
    len(taskb_gold_links_split),
)

print(
    "Authoritative mention inputs:",
    len(taskb_mention_inputs_split),
)

print(
    "Canonical concepts/definitions:",
    len(taskb_reference_definitions),
)

print("\nDocuments by split:")
print(
    taskb_document_split.groupby(
        ["domain", "split"]
    ).size()
)

print("\nGold links by split:")
print(
    taskb_gold_links_split.groupby(
        ["domain", "split"]
    ).size()
)

print("\nMention inputs by split:")
print(
    taskb_mention_inputs_split.groupby(
        ["domain", "split"]
    ).size()
)

print("\nRestoration completed successfully.")

In [ ]:
# @title
# TASK B — CELL 2
# Gold-resource and leakage audit before choosing the holdout design

from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

TASKB_TRAIN_CSV_DIR = BASE_DIR / "data/taskB/training/csv"
TASKB_RESULTS_DIR = BASE_DIR / "journal_experiments/taskB_holdout/results"
TASKB_SPLIT_DIR = BASE_DIR / "journal_experiments/taskB_holdout/splits"

TASKB_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TASKB_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

PA_MENTIONS_PATH = (
    TASKB_TRAIN_CSV_DIR / "taskB_Mentions_parkinson_train.csv"
)
MH_MENTIONS_PATH = (
    TASKB_TRAIN_CSV_DIR / "taskB_Mentions_mental_health_train.csv"
)
DEFINITIONS_PATH = (
    TASKB_TRAIN_CSV_DIR / "taskB_Definitions_train.csv"
)

pa = pd.read_csv(PA_MENTIONS_PATH)
mh = pd.read_csv(MH_MENTIONS_PATH)
definitions = pd.read_csv(DEFINITIONS_PATH)

pa["domain"] = "parkinson"
mh["domain"] = "mental_health"

mentions = pd.concat([pa, mh], ignore_index=True)

definitions = definitions.rename(
    columns={
        "<Concept>": "concept",
        "Intensional definition": "definition",
    }
)

required_mention_columns = {
    "domain",
    "num_article",
    "doi",
    "mention",
    "concept",
}
required_definition_columns = {"concept", "definition"}

missing_mentions = required_mention_columns - set(mentions.columns)
missing_definitions = required_definition_columns - set(definitions.columns)

if missing_mentions:
    raise ValueError(
        f"Missing mention columns: {sorted(missing_mentions)}"
    )

if missing_definitions:
    raise ValueError(
        f"Missing definition columns: {sorted(missing_definitions)}"
    )


def normalize_space(value):
    return re.sub(r"\s+", " ", str(value).strip())


def normalize_concept(value):
    value = normalize_space(value)

    if value.startswith("<") and value.endswith(">"):
        value = value[1:-1].strip()

    return value.casefold()


def normalize_mention(value):
    return normalize_space(value).casefold()


def normalize_doi(value):
    value = normalize_space(value).lower()
    value = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", value)
    value = re.sub(r"^doi:\s*", "", value)
    return value.rstrip("/")


mentions["num_article"] = mentions["num_article"].astype(str)
mentions["doi"] = mentions["doi"].fillna("").map(normalize_space)
mentions["doi_normalized"] = mentions["doi"].map(normalize_doi)

mentions["mention"] = mentions["mention"].map(normalize_space)
mentions["mention_normalized"] = mentions["mention"].map(
    normalize_mention
)

mentions["concept"] = mentions["concept"].map(normalize_space)
mentions["concept_normalized"] = mentions["concept"].map(
    normalize_concept
)

definitions["concept"] = definitions["concept"].map(normalize_space)
definitions["concept_normalized"] = definitions["concept"].map(
    normalize_concept
)
definitions["definition"] = definitions["definition"].map(
    normalize_space
)
definitions["definition_normalized"] = definitions["definition"].str.casefold()

mentions["document_key"] = (
    mentions["domain"]
    + "::"
    + mentions["num_article"]
)

# Definition duplication and concept linkage
definition_counts = (
    definitions.groupby("concept_normalized")
    .size()
    .rename("definition_rows")
)

definition_text_counts = (
    definitions.groupby("definition_normalized")
    .size()
    .rename("concepts_sharing_definition")
)

mention_concepts = set(mentions["concept_normalized"])
defined_concepts = set(definitions["concept_normalized"])

missing_definitions_for_mentions = sorted(
    mention_concepts - defined_concepts
)
definitions_without_mentions = sorted(
    defined_concepts - mention_concepts
)

# Mention ambiguity
mention_ambiguity = (
    mentions.groupby("mention_normalized")
    .agg(
        rows=("concept_normalized", "size"),
        concepts=("concept_normalized", "nunique"),
        domains=("domain", "nunique"),
        documents=("document_key", "nunique"),
    )
    .reset_index()
)

ambiguous_mentions = mention_ambiguity[
    mention_ambiguity["concepts"] > 1
].copy()

# Concept recurrence across documents and domains
concept_recurrence = (
    mentions.groupby("concept_normalized")
    .agg(
        rows=("mention", "size"),
        unique_mentions=("mention_normalized", "nunique"),
        documents=("document_key", "nunique"),
        domains=("domain", "nunique"),
        doi_groups=("doi_normalized", "nunique"),
    )
    .reset_index()
)

concept_recurrence["single_document"] = (
    concept_recurrence["documents"] == 1
)
concept_recurrence["cross_domain"] = (
    concept_recurrence["domains"] > 1
)

# DOI duplication
doi_groups = (
    mentions[
        ["document_key", "domain", "num_article", "doi_normalized"]
    ]
    .drop_duplicates()
    .groupby("doi_normalized")
    .agg(
        documents=("document_key", "nunique"),
        domains=("domain", "nunique"),
        document_keys=(
            "document_key",
            lambda x: " | ".join(sorted(set(x))),
        ),
    )
    .reset_index()
)

doi_groups = doi_groups[
    doi_groups["doi_normalized"] != ""
].copy()

repeated_dois = doi_groups[
    doi_groups["documents"] > 1
].copy()

# Exact duplicate rows
duplicate_gold_rows = mentions.duplicated(
    subset=[
        "domain",
        "num_article",
        "doi_normalized",
        "mention",
        "concept",
    ],
    keep=False,
)

print("=" * 78)
print("TASK B GOLD RESOURCE AUDIT")
print("=" * 78)

print("\nRows and inventories:")
print(f"Mention–concept rows: {len(mentions)}")
print(f"Documents: {mentions['document_key'].nunique()}")
print(f"Normalized mentions: {mentions['mention_normalized'].nunique()}")
print(f"Linked concepts: {len(mention_concepts)}")
print(f"Reference definitions: {len(definitions)}")
print(f"Defined concepts: {len(defined_concepts)}")

print("\nRows by domain:")
print(mentions.groupby("domain").size())

print("\nDocuments by domain:")
print(
    mentions[
        ["domain", "document_key"]
    ]
    .drop_duplicates()
    .groupby("domain")
    .size()
)

print("\nDefinition linkage:")
print(
    "Mention concepts without a definition:",
    len(missing_definitions_for_mentions),
)
print(
    "Defined concepts absent from mention annotations:",
    len(definitions_without_mentions),
)
print(
    "Concepts with multiple definition rows:",
    int((definition_counts > 1).sum()),
)
print(
    "Definition texts shared by multiple concepts:",
    int((definition_text_counts > 1).sum()),
)

print("\nMention ambiguity:")
print(
    "Normalized mentions linked to multiple concepts:",
    len(ambiguous_mentions),
)
print(
    "Rows involving ambiguous normalized mentions:",
    int(
        mentions["mention_normalized"]
        .isin(ambiguous_mentions["mention_normalized"])
        .sum()
    ),
)

print("\nConcept recurrence:")
print(
    "Concepts occurring in only one document:",
    int(concept_recurrence["single_document"].sum()),
)
print(
    "Concepts occurring in at least two documents:",
    int((concept_recurrence["documents"] >= 2).sum()),
)
print(
    "Concepts occurring in at least three documents:",
    int((concept_recurrence["documents"] >= 3).sum()),
)
print(
    "Concepts shared across domains:",
    int(concept_recurrence["cross_domain"].sum()),
)

print("\nPotential document leakage:")
print("Repeated DOI groups:", len(repeated_dois))
print(
    "Exact duplicate annotation rows:",
    int(duplicate_gold_rows.sum()),
)

print("\nConcept document-count distribution:")
print(
    concept_recurrence["documents"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nTwenty most recurrent concepts:")
display(
    concept_recurrence.sort_values(
        ["documents", "rows"],
        ascending=False,
    ).head(20)
)

if len(ambiguous_mentions):
    print("\nFirst twenty ambiguous normalized mentions:")
    display(
        ambiguous_mentions.sort_values(
            ["concepts", "rows"],
            ascending=False,
        ).head(20)
    )

if len(repeated_dois):
    print("\nRepeated DOI groups:")
    display(repeated_dois)

# Save authoritative normalized tables and audit outputs
mentions.to_csv(
    TASKB_RESULTS_DIR / "taskB_normalized_mention_concept_rows.csv",
    index=False,
)
definitions.to_csv(
    TASKB_RESULTS_DIR / "taskB_normalized_reference_definitions.csv",
    index=False,
)
concept_recurrence.to_csv(
    TASKB_RESULTS_DIR / "taskB_concept_recurrence_audit.csv",
    index=False,
)
ambiguous_mentions.to_csv(
    TASKB_RESULTS_DIR / "taskB_ambiguous_mentions_audit.csv",
    index=False,
)
repeated_dois.to_csv(
    TASKB_RESULTS_DIR / "taskB_repeated_doi_audit.csv",
    index=False,
)

print("\nSaved audit files to:")
print(TASKB_RESULTS_DIR)
print("\nCell 2 completed successfully.")

In [ ]:
# @title
# TASK B — CELL 3
# Diagnose definition mismatches, repeated annotation rows,
# multi-concept mentions, and literal occurrence alignment

from pathlib import Path
import pandas as pd
import numpy as np
import re


TASKB_TRAIN_ABS_DIR = (
    BASE_DIR / "data/taskB/training/abstracts"
)


def read_taskb_abstract(domain, num_article):
    prefix = "pa_" if domain == "parkinson" else "mh_"

    path = (
        TASKB_TRAIN_ABS_DIR
        / domain
        / f"{prefix}{num_article}.txt"
    )

    return path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).replace("\r\n", "\n")


def boundary_pattern(surface):
    """
    Match a literal surface form while preventing matches
    inside a larger alphanumeric string.

    Punctuation internal to the mention is retained literally.
    """
    escaped = re.escape(str(surface))

    return re.compile(
        rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])",
        flags=re.IGNORECASE,
    )


# ------------------------------------------------------------
# 1. Exact concept-definition mismatches
# ------------------------------------------------------------

mention_concept_display = (
    mentions[
        ["concept_normalized", "concept"]
    ]
    .drop_duplicates()
    .sort_values(
        ["concept_normalized", "concept"]
    )
)

definition_concept_display = (
    definitions[
        ["concept_normalized", "concept"]
    ]
    .drop_duplicates()
    .sort_values(
        ["concept_normalized", "concept"]
    )
)

missing_definition_table = (
    mention_concept_display[
        mention_concept_display["concept_normalized"].isin(
            missing_definitions_for_mentions
        )
    ]
    .merge(
        concept_recurrence,
        on="concept_normalized",
        how="left",
    )
)

unused_definition_table = (
    definition_concept_display[
        definition_concept_display["concept_normalized"].isin(
            definitions_without_mentions
        )
    ]
    .merge(
        definitions[
            [
                "concept_normalized",
                "definition",
            ]
        ],
        on="concept_normalized",
        how="left",
    )
)


# ------------------------------------------------------------
# 2. Annotation multiplicity at document–mention level
# ------------------------------------------------------------

document_mention_groups = (
    mentions.groupby(
        [
            "domain",
            "num_article",
            "doi_normalized",
            "document_key",
            "mention_normalized",
        ],
        as_index=False,
    )
    .agg(
        displayed_mentions=(
            "mention",
            lambda x: " | ".join(
                sorted(set(x))
            ),
        ),
        annotation_rows=("concept", "size"),
        unique_concepts=(
            "concept_normalized",
            "nunique",
        ),
        concepts=(
            "concept",
            lambda x: " | ".join(
                sorted(set(x))
            ),
        ),
    )
)

document_mention_groups["multi_concept"] = (
    document_mention_groups["unique_concepts"] > 1
)


# ------------------------------------------------------------
# 3. Exact duplicate annotation rows
# ------------------------------------------------------------

duplicate_key = [
    "domain",
    "num_article",
    "doi_normalized",
    "mention",
    "concept",
]

duplicate_annotation_groups = (
    mentions.groupby(
        duplicate_key,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "annotation_row_count"})
)

duplicate_annotation_groups = (
    duplicate_annotation_groups[
        duplicate_annotation_groups[
            "annotation_row_count"
        ] > 1
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. Literal occurrence audit
# ------------------------------------------------------------

abstract_cache = {}

for domain, num_article in (
    mentions[
        ["domain", "num_article"]
    ]
    .drop_duplicates()
    .itertuples(index=False, name=None)
):
    abstract_cache[(domain, str(num_article))] = (
        read_taskb_abstract(
            domain,
            str(num_article),
        )
    )


def count_boundary_occurrences(row):
    text = abstract_cache[
        (
            row["domain"],
            str(row["num_article"]),
        )
    ]

    displayed_surface = (
        str(row["displayed_mentions"])
        .split(" | ")[0]
    )

    return len(
        list(
            boundary_pattern(
                displayed_surface
            ).finditer(text)
        )
    )


document_mention_groups[
    "boundary_valid_text_occurrences"
] = document_mention_groups.apply(
    count_boundary_occurrences,
    axis=1,
)

document_mention_groups[
    "has_literal_occurrence"
] = (
    document_mention_groups[
        "boundary_valid_text_occurrences"
    ] > 0
)

document_mention_groups[
    "annotations_exceed_occurrences"
] = (
    document_mention_groups[
        "annotation_rows"
    ]
    >
    document_mention_groups[
        "boundary_valid_text_occurrences"
    ]
)

# Annotation rows can exceed occurrences legitimately when
# one surface form is linked to multiple concepts. Therefore
# compare the maximum repetition of any one concept as well.

per_concept_multiplicity = (
    mentions.groupby(
        [
            "domain",
            "num_article",
            "doi_normalized",
            "document_key",
            "mention_normalized",
            "concept_normalized",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "rows_for_same_concept"
        }
    )
)

per_concept_multiplicity = (
    per_concept_multiplicity.merge(
        document_mention_groups[
            [
                "document_key",
                "mention_normalized",
                "boundary_valid_text_occurrences",
            ]
        ],
        on=[
            "document_key",
            "mention_normalized",
        ],
        how="left",
    )
)

per_concept_multiplicity[
    "same_concept_rows_exceed_occurrences"
] = (
    per_concept_multiplicity[
        "rows_for_same_concept"
    ]
    >
    per_concept_multiplicity[
        "boundary_valid_text_occurrences"
    ]
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("=" * 78)
print("DEFINITION LINKAGE")
print("=" * 78)

print(
    "\nMention-linked concepts without definitions:",
    len(missing_definition_table),
)
display(missing_definition_table)

print(
    "\nDefinitions without mention annotations:",
    len(unused_definition_table),
)
display(unused_definition_table)


print("\n" + "=" * 78)
print("DOCUMENT–MENTION ANNOTATION STRUCTURE")
print("=" * 78)

print(
    "Unique document–mention groups:",
    len(document_mention_groups),
)
print(
    "Groups linked to multiple concepts:",
    int(
        document_mention_groups[
            "multi_concept"
        ].sum()
    ),
)
print(
    "Groups with no boundary-valid literal occurrence:",
    int(
        (~document_mention_groups[
            "has_literal_occurrence"
        ]).sum()
    ),
)
print(
    "Groups where annotation rows exceed text occurrences:",
    int(
        document_mention_groups[
            "annotations_exceed_occurrences"
        ].sum()
    ),
)
print(
    "Document–mention–concept groups where repetitions "
    "exceed literal occurrences:",
    int(
        per_concept_multiplicity[
            "same_concept_rows_exceed_occurrences"
        ].sum()
    ),
)


print("\n" + "=" * 78)
print("EXACT DUPLICATE ANNOTATION GROUPS")
print("=" * 78)

print(
    "Duplicate groups:",
    len(duplicate_annotation_groups),
)
print(
    "Rows represented by duplicate groups:",
    int(
        duplicate_annotation_groups[
            "annotation_row_count"
        ].sum()
    )
    if len(duplicate_annotation_groups)
    else 0,
)

display(duplicate_annotation_groups)


print("\n" + "=" * 78)
print("FIRST MULTI-CONCEPT DOCUMENT–MENTION GROUPS")
print("=" * 78)

display(
    document_mention_groups[
        document_mention_groups[
            "multi_concept"
        ]
    ]
    .sort_values(
        [
            "unique_concepts",
            "annotation_rows",
        ],
        ascending=False,
    )
    .head(30)
)


print("\n" + "=" * 78)
print("FIRST LITERAL-ALIGNMENT FAILURES")
print("=" * 78)

display(
    document_mention_groups[
        ~document_mention_groups[
            "has_literal_occurrence"
        ]
    ].head(30)
)


print("\n" + "=" * 78)
print("POTENTIALLY IMPOSSIBLE MULTIPLICITIES")
print("=" * 78)

display(
    per_concept_multiplicity[
        per_concept_multiplicity[
            "same_concept_rows_exceed_occurrences"
        ]
    ].head(30)
)


# ------------------------------------------------------------
# 6. Save diagnostics
# ------------------------------------------------------------

missing_definition_table.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_concepts_missing_definitions.csv",
    index=False,
)

unused_definition_table.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_definitions_without_mentions.csv",
    index=False,
)

document_mention_groups.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_document_mention_structure.csv",
    index=False,
)

duplicate_annotation_groups.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_duplicate_annotation_groups.csv",
    index=False,
)

per_concept_multiplicity.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_document_mention_concept_multiplicity.csv",
    index=False,
)

print("\nSaved Cell 3 diagnostics to:")
print(TASKB_RESULTS_DIR)
print("\nCell 3 completed successfully.")

In [ ]:
# @title
# TASK B — CELL 4
# Diagnose and propose repairs for concept-definition linkage.
# This cell does not modify the source files.

from difflib import SequenceMatcher
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Candidate similarity function
# ------------------------------------------------------------

def concept_similarity(left, right):
    return SequenceMatcher(
        None,
        normalize_concept(left),
        normalize_concept(right),
    ).ratio()


defined_concept_rows = (
    definitions[
        [
            "concept",
            "concept_normalized",
            "definition",
        ]
    ]
    .drop_duplicates("concept_normalized")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. Find nearest defined concepts for every unmatched label
# ------------------------------------------------------------

nearest_rows = []

for missing_concept in missing_definition_table[
    "concept"
].tolist():

    scored = defined_concept_rows.copy()

    scored["similarity"] = scored[
        "concept"
    ].map(
        lambda candidate: concept_similarity(
            missing_concept,
            candidate,
        )
    )

    scored = scored.sort_values(
        "similarity",
        ascending=False,
    ).head(10)

    for rank, row in enumerate(
        scored.itertuples(index=False),
        start=1,
    ):
        nearest_rows.append(
            {
                "unmatched_concept": missing_concept,
                "rank": rank,
                "candidate_concept": row.concept,
                "similarity": row.similarity,
                "candidate_definition": row.definition,
            }
        )

nearest_candidates = pd.DataFrame(nearest_rows)


# ------------------------------------------------------------
# 3. Display candidate matches
# ------------------------------------------------------------

print("=" * 78)
print("NEAREST DEFINED CONCEPTS FOR UNMATCHED LABELS")
print("=" * 78)

for missing_concept in nearest_candidates[
    "unmatched_concept"
].drop_duplicates():

    print(f"\nUNMATCHED: {missing_concept}")

    display(
        nearest_candidates[
            nearest_candidates[
                "unmatched_concept"
            ] == missing_concept
        ][
            [
                "rank",
                "candidate_concept",
                "similarity",
                "candidate_definition",
            ]
        ]
    )


# ------------------------------------------------------------
# 4. Explicit conservative repair proposal
# ------------------------------------------------------------

proposed_repairs = pd.DataFrame(
    [
        {
            "source_concept": "<Biophysiological mechanism>",
            "target_concept": "<Biophysiogical mechanism>",
            "reason": (
                "Near-identical labels; apparent spelling "
                "difference between annotation and definition inventory."
            ),
            "repair_status": "proposed",
        },
        {
            "source_concept": "<Diseease>",
            "target_concept": "<Disease>",
            "reason": (
                "Apparent typographical error in the annotation label."
            ),
            "repair_status": "proposed",
        },
        {
            "source_concept": "<Microbe>",
            "target_concept": "<Microorganism>",
            "reason": (
                "Likely synonymous inventory mapping; requires confirmation "
                "from nearest-candidate output before adoption."
            ),
            "repair_status": "review",
        },
        {
            "source_concept": (
                "<Placebo-controlled double-blind design>"
            ),
            "target_concept": (
                "<Placebo-controlled double-blinded design>"
            ),
            "reason": (
                "Near-identical morphological variant between "
                "annotation and definition inventory."
            ),
            "repair_status": "proposed",
        },
        {
            "source_concept": "<Psychiatric diseases>",
            "target_concept": "<Psychiatric disease>",
            "reason": (
                "Plural versus singular inventory variant."
            ),
            "repair_status": "proposed",
        },
    ]
)

print("\n" + "=" * 78)
print("CONSERVATIVE REPAIR PROPOSAL")
print("=" * 78)

display(proposed_repairs)


# ------------------------------------------------------------
# 5. Validate proposed targets
# ------------------------------------------------------------

defined_original_labels = set(
    definitions["concept"]
)

proposed_repairs[
    "target_exists_in_definition_inventory"
] = proposed_repairs[
    "target_concept"
].isin(defined_original_labels)

print("\nTarget validation:")
display(
    proposed_repairs[
        [
            "source_concept",
            "target_concept",
            "repair_status",
            "target_exists_in_definition_inventory",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Build the provisional unique-link evaluation table
#    without yet applying concept repairs
# ------------------------------------------------------------

evaluation_key = [
    "domain",
    "num_article",
    "doi_normalized",
    "document_key",
    "mention_normalized",
    "concept_normalized",
]

taskb_unique_links_provisional = (
    mentions.sort_values(
        evaluation_key
    )
    .drop_duplicates(
        subset=evaluation_key,
        keep="first",
    )
    .copy()
)

removed_exact_repeated_links = (
    len(mentions)
    - len(taskb_unique_links_provisional)
)

print("\n" + "=" * 78)
print("PROVISIONAL AUTHORITATIVE EVALUATION TABLE")
print("=" * 78)

print("Original annotation rows:", len(mentions))
print(
    "Unique document–mention–concept links:",
    len(taskb_unique_links_provisional),
)
print(
    "Exact repeated links removed:",
    removed_exact_repeated_links,
)

print("\nUnique links by domain:")
print(
    taskb_unique_links_provisional
    .groupby("domain")
    .size()
)

print("\nUnique document–mention inputs by domain:")
print(
    taskb_unique_links_provisional[
        [
            "domain",
            "document_key",
            "mention_normalized",
        ]
    ]
    .drop_duplicates()
    .groupby("domain")
    .size()
)

print("\nMulti-label document–mention inputs:")
provisional_label_cardinality = (
    taskb_unique_links_provisional
    .groupby(
        [
            "domain",
            "document_key",
            "mention_normalized",
        ]
    )["concept_normalized"]
    .nunique()
)

print(
    provisional_label_cardinality
    .value_counts()
    .sort_index()
    .rename_axis("gold_concepts_per_mention")
    .to_frame("mention_inputs")
)


# ------------------------------------------------------------
# 7. Save audit artifacts
# ------------------------------------------------------------

nearest_candidates.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_unmatched_concept_nearest_candidates.csv",
    index=False,
)

proposed_repairs.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_concept_definition_repair_proposal.csv",
    index=False,
)

taskb_unique_links_provisional.to_csv(
    TASKB_RESULTS_DIR
    / "taskB_unique_links_provisional.csv",
    index=False,
)

print("\nSaved Cell 4 outputs to:")
print(TASKB_RESULTS_DIR)
print("\nCell 4 completed successfully.")

In [ ]:
# @title
# TASK B — CELL 5
# Apply audited concept-label repairs and lock the authoritative
# unique document–mention–concept evaluation table.

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Frozen, explicitly audited repair map
# ------------------------------------------------------------

CONCEPT_REPAIR_MAP = {
    normalize_concept(
        "<Biophysiological mechanism>"
    ): "<Biophysiogical mechanism>",

    normalize_concept(
        "<Diseease>"
    ): "<Disease>",

    normalize_concept(
        "<Microbe>"
    ): "<Microorganism>",

    normalize_concept(
        "<Placebo-controlled double-blind design>"
    ): "<Placebo-controlled double-blinded design>",

    normalize_concept(
        "<Psychiatric diseases>"
    ): "<Psychiatric disease>",
}


# ------------------------------------------------------------
# 2. Apply repairs without altering the source-label columns
# ------------------------------------------------------------

taskb_links = mentions.copy()

taskb_links["concept_original"] = taskb_links["concept"]
taskb_links["concept_normalized_original"] = (
    taskb_links["concept_normalized"]
)

taskb_links["concept_canonical"] = taskb_links.apply(
    lambda row: CONCEPT_REPAIR_MAP.get(
        row["concept_normalized"],
        row["concept"],
    ),
    axis=1,
)

taskb_links["concept_normalized_canonical"] = (
    taskb_links["concept_canonical"].map(
        normalize_concept
    )
)

taskb_links["concept_label_repaired"] = (
    taskb_links["concept_normalized_original"]
    !=
    taskb_links["concept_normalized_canonical"]
)


# ------------------------------------------------------------
# 3. Remove only exact repeated semantic links
#
# A unique input is:
# domain + document + normalized mention
#
# A unique gold link additionally contains the canonical concept.
# ------------------------------------------------------------

AUTHORITATIVE_LINK_KEY = [
    "domain",
    "num_article",
    "doi_normalized",
    "document_key",
    "mention_normalized",
    "concept_normalized_canonical",
]

taskb_gold_links = (
    taskb_links.sort_values(
        AUTHORITATIVE_LINK_KEY
    )
    .drop_duplicates(
        subset=AUTHORITATIVE_LINK_KEY,
        keep="first",
    )
    .reset_index(drop=True)
)

taskb_gold_links["gold_link_id"] = [
    f"taskb_link_{i:04d}"
    for i in range(len(taskb_gold_links))
]


# ------------------------------------------------------------
# 4. Canonical definition table
# ------------------------------------------------------------

taskb_reference_definitions = (
    definitions[
        [
            "concept",
            "concept_normalized",
            "definition",
            "definition_normalized",
        ]
    ]
    .rename(
        columns={
            "concept": "concept_canonical",
            "concept_normalized":
                "concept_normalized_canonical",
        }
    )
    .drop_duplicates(
        subset="concept_normalized_canonical",
        keep="first",
    )
    .reset_index(drop=True)
)

definition_lookup = dict(
    zip(
        taskb_reference_definitions[
            "concept_normalized_canonical"
        ],
        taskb_reference_definitions[
            "definition"
        ],
    )
)

taskb_gold_links["reference_definition"] = (
    taskb_gold_links[
        "concept_normalized_canonical"
    ].map(definition_lookup)
)


# ------------------------------------------------------------
# 5. Integrity checks
# ------------------------------------------------------------

missing_definition_mask = (
    taskb_gold_links[
        "reference_definition"
    ].isna()
)

linked_canonical_concepts = set(
    taskb_gold_links[
        "concept_normalized_canonical"
    ]
)

definition_concepts = set(
    taskb_reference_definitions[
        "concept_normalized_canonical"
    ]
)

canonical_without_definition = sorted(
    linked_canonical_concepts
    - definition_concepts
)

definitions_without_canonical_link = sorted(
    definition_concepts
    - linked_canonical_concepts
)

assert not missing_definition_mask.any(), (
    "Some canonical links still lack reference definitions."
)

assert len(canonical_without_definition) == 0


# ------------------------------------------------------------
# 6. Construct authoritative mention inputs
# ------------------------------------------------------------

MENTION_INPUT_KEY = [
    "domain",
    "num_article",
    "doi_normalized",
    "document_key",
    "mention_normalized",
]

taskb_mention_inputs = (
    taskb_gold_links.groupby(
        MENTION_INPUT_KEY,
        as_index=False,
    )
    .agg(
        mention=(
            "mention",
            "first",
        ),
        gold_concept_count=(
            "concept_normalized_canonical",
            "nunique",
        ),
        gold_concepts=(
            "concept_canonical",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
        gold_concepts_normalized=(
            "concept_normalized_canonical",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
    )
)

taskb_mention_inputs["is_multilabel"] = (
    taskb_mention_inputs[
        "gold_concept_count"
    ] > 1
)

taskb_mention_inputs["mention_word_count"] = (
    taskb_mention_inputs["mention"]
    .str.split()
    .str.len()
)


# ------------------------------------------------------------
# 7. Document-level metadata for split selection
# ------------------------------------------------------------

document_metadata = (
    taskb_mention_inputs.groupby(
        [
            "domain",
            "num_article",
            "doi_normalized",
            "document_key",
        ],
        as_index=False,
    )
    .agg(
        mention_inputs=(
            "mention_normalized",
            "size",
        ),
        multilabel_inputs=(
            "is_multilabel",
            "sum",
        ),
        mean_labels_per_mention=(
            "gold_concept_count",
            "mean",
        ),
        mean_mention_words=(
            "mention_word_count",
            "mean",
        ),
        max_mention_words=(
            "mention_word_count",
            "max",
        ),
    )
)

links_per_document = (
    taskb_gold_links.groupby(
        "document_key"
    )
    .size()
    .rename("gold_links")
)

concepts_per_document = (
    taskb_gold_links.groupby(
        "document_key"
    )[
        "concept_normalized_canonical"
    ]
    .nunique()
    .rename("unique_concepts")
)

document_metadata = (
    document_metadata
    .merge(
        links_per_document,
        on="document_key",
        how="left",
    )
    .merge(
        concepts_per_document,
        on="document_key",
        how="left",
    )
)

document_metadata["multilabel_ratio"] = (
    document_metadata["multilabel_inputs"]
    /
    document_metadata["mention_inputs"]
)


# ------------------------------------------------------------
# 8. Report authoritative counts
# ------------------------------------------------------------

print("=" * 78)
print("AUTHORITATIVE TASK B GOLD RESOURCES")
print("=" * 78)

print("Original annotation rows:", len(mentions))
print(
    "Rows receiving a canonical-label repair:",
    int(taskb_links["concept_label_repaired"].sum()),
)
print(
    "Authoritative unique semantic links:",
    len(taskb_gold_links),
)
print(
    "Unique mention inputs:",
    len(taskb_mention_inputs),
)
print(
    "Canonical linked concepts:",
    len(linked_canonical_concepts),
)
print(
    "Reference-definition concepts:",
    len(definition_concepts),
)
print(
    "Linked concepts without definitions:",
    len(canonical_without_definition),
)
print(
    "Definitions without linked concepts:",
    len(definitions_without_canonical_link),
)

print("\nSemantic links by domain:")
print(
    taskb_gold_links.groupby("domain").size()
)

print("\nMention inputs by domain:")
print(
    taskb_mention_inputs.groupby("domain").size()
)

print("\nLabel cardinality:")
print(
    taskb_mention_inputs[
        "gold_concept_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("gold_concepts_per_mention")
    .to_frame("mention_inputs")
)

print("\nRepaired rows:")
display(
    taskb_gold_links[
        taskb_gold_links[
            "concept_label_repaired"
        ]
    ][
        [
            "domain",
            "num_article",
            "mention",
            "concept_original",
            "concept_canonical",
            "reference_definition",
        ]
    ]
)

print("\nDocument metadata summary:")
display(
    document_metadata.groupby("domain")[
        [
            "mention_inputs",
            "gold_links",
            "unique_concepts",
            "multilabel_ratio",
            "mean_labels_per_mention",
            "mean_mention_words",
            "max_mention_words",
        ]
    ]
    .agg(
        ["count", "mean", "std", "min", "max"]
    )
    .round(4)
)


# ------------------------------------------------------------
# 9. Save locked resources
# ------------------------------------------------------------

GOLD_LINKS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_unique_gold_links.csv"
)

MENTION_INPUTS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_mention_inputs.csv"
)

DEFINITIONS_CANONICAL_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_reference_definitions.csv"
)

DOCUMENT_METADATA_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_document_metadata_before_split.csv"
)

REPAIR_METADATA_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_concept_label_repairs.json"
)

taskb_gold_links.to_csv(
    GOLD_LINKS_PATH,
    index=False,
)

taskb_mention_inputs.to_csv(
    MENTION_INPUTS_PATH,
    index=False,
)

taskb_reference_definitions.to_csv(
    DEFINITIONS_CANONICAL_PATH,
    index=False,
)

document_metadata.to_csv(
    DOCUMENT_METADATA_PATH,
    index=False,
)

repair_metadata = {
    "evaluation_unit":
        "unique_document_mention_concept_link",
    "original_annotation_rows":
        int(len(mentions)),
    "authoritative_unique_links":
        int(len(taskb_gold_links)),
    "unique_mention_inputs":
        int(len(taskb_mention_inputs)),
    "canonical_concepts":
        int(len(linked_canonical_concepts)),
    "reference_definitions":
        int(len(definition_concepts)),
    "repairs": [
        {
            "source_normalized": source,
            "target_label": target,
            "target_normalized":
                normalize_concept(target),
        }
        for source, target
        in CONCEPT_REPAIR_MAP.items()
    ],
}

REPAIR_METADATA_PATH.write_text(
    json.dumps(
        repair_metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nSaved authoritative gold links to:")
print(GOLD_LINKS_PATH)

print("\nSaved authoritative mention inputs to:")
print(MENTION_INPUTS_PATH)

print("\nSaved canonical definitions to:")
print(DEFINITIONS_CANONICAL_PATH)

print("\nSaved document metadata to:")
print(DOCUMENT_METADATA_PATH)

print("\nSaved repair metadata to:")
print(REPAIR_METADATA_PATH)

print("\nCell 5 completed successfully.")

In [ ]:
# @title
# TASK B — CELL 6
# Select a domain-stratified, document-grouped 70/15/15 split
# using gold metadata only. No model outputs are involved.

import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Fixed document counts
# ------------------------------------------------------------

DOMAIN_SPLIT_COUNTS = {
    "mental_health": {
        "train": 16,
        "development": 3,
        "test": 4,
    },
    "parkinson": {
        "train": 10,
        "development": 2,
        "test": 2,
    },
}

SPLIT_FEATURES = [
    "mention_inputs",
    "gold_links",
    "unique_concepts",
    "multilabel_ratio",
    "mean_labels_per_mention",
    "mean_mention_words",
    "max_mention_words",
]


# ------------------------------------------------------------
# 2. Validate requested counts
# ------------------------------------------------------------

for domain, counts in DOMAIN_SPLIT_COUNTS.items():
    available = int(
        (document_metadata["domain"] == domain).sum()
    )
    requested = sum(counts.values())

    if available != requested:
        raise ValueError(
            f"{domain}: available={available}, "
            f"requested={requested}"
        )


# ------------------------------------------------------------
# 3. Standardization statistics
# ------------------------------------------------------------

feature_reference = {}

for domain, group in document_metadata.groupby("domain"):
    feature_reference[domain] = {}

    for feature in SPLIT_FEATURES:
        mean = float(group[feature].mean())
        std = float(group[feature].std(ddof=0))

        if not np.isfinite(std) or std < 1e-12:
            std = 1.0

        feature_reference[domain][feature] = {
            "mean": mean,
            "std": std,
        }


# ------------------------------------------------------------
# 4. Generate one candidate split
# ------------------------------------------------------------

def make_candidate_split(seed):
    rng = np.random.default_rng(seed)
    pieces = []

    for domain, counts in DOMAIN_SPLIT_COUNTS.items():
        domain_docs = (
            document_metadata[
                document_metadata["domain"] == domain
            ]
            .copy()
            .reset_index(drop=True)
        )

        order = rng.permutation(len(domain_docs))
        domain_docs = domain_docs.iloc[order].copy()

        n_train = counts["train"]
        n_dev = counts["development"]

        split_labels = (
            ["train"] * n_train
            + ["development"] * n_dev
            + ["test"] * counts["test"]
        )

        domain_docs["split"] = split_labels
        pieces.append(domain_docs)

    return pd.concat(
        pieces,
        ignore_index=True,
    )


# ------------------------------------------------------------
# 5. Metadata-balance component
# ------------------------------------------------------------

def metadata_balance(candidate):
    differences = []

    for domain, domain_rows in candidate.groupby("domain"):
        for split, split_rows in domain_rows.groupby("split"):
            for feature in SPLIT_FEATURES:
                reference = feature_reference[domain][feature]

                standardized_difference = (
                    float(split_rows[feature].mean())
                    - reference["mean"]
                ) / reference["std"]

                differences.append(
                    abs(standardized_difference)
                )

    return {
        "mean_abs_standardized_difference":
            float(np.mean(differences)),
        "max_abs_standardized_difference":
            float(np.max(differences)),
    }


# ------------------------------------------------------------
# 6. Concept-coverage component
# ------------------------------------------------------------

def concept_coverage(candidate):
    split_lookup = dict(
        zip(
            candidate["document_key"],
            candidate["split"],
        )
    )

    links = taskb_gold_links.copy()
    links["candidate_split"] = links[
        "document_key"
    ].map(split_lookup)

    train_concepts = set(
        links.loc[
            links["candidate_split"] == "train",
            "concept_normalized_canonical",
        ]
    )

    result = {}

    for split in ["development", "test"]:
        subset = links[
            links["candidate_split"] == split
        ]

        seen_mask = subset[
            "concept_normalized_canonical"
        ].isin(train_concepts)

        split_concepts = set(
            subset["concept_normalized_canonical"]
        )

        seen_unique_concepts = (
            split_concepts & train_concepts
        )

        result[f"{split}_seen_link_ratio"] = (
            float(seen_mask.mean())
            if len(subset)
            else np.nan
        )

        result[f"{split}_seen_concept_ratio"] = (
            len(seen_unique_concepts)
            / len(split_concepts)
            if split_concepts
            else np.nan
        )

        result[f"{split}_unseen_links"] = int(
            (~seen_mask).sum()
        )

        result[f"{split}_unseen_concepts"] = int(
            len(split_concepts - train_concepts)
        )

    result["seen_link_ratio_difference"] = abs(
        result["development_seen_link_ratio"]
        - result["test_seen_link_ratio"]
    )

    result["seen_concept_ratio_difference"] = abs(
        result["development_seen_concept_ratio"]
        - result["test_seen_concept_ratio"]
    )

    return result


# ------------------------------------------------------------
# 7. Candidate objective
#
# Metadata balance is primary.
# The two coverage terms prevent development and test from
# having radically different seen/unseen difficulty.
# ------------------------------------------------------------

def score_candidate(candidate):
    balance = metadata_balance(candidate)
    coverage = concept_coverage(candidate)

    objective = (
        balance[
            "mean_abs_standardized_difference"
        ]
        + 0.50
        * balance[
            "max_abs_standardized_difference"
        ]
        + 0.50
        * coverage[
            "seen_link_ratio_difference"
        ]
        + 0.25
        * coverage[
            "seen_concept_ratio_difference"
        ]
    )

    return {
        **balance,
        **coverage,
        "objective": float(objective),
    }


# ------------------------------------------------------------
# 8. Search reproducible candidate seeds
# ------------------------------------------------------------

N_SPLIT_SEEDS = 50000

best_seed = None
best_candidate = None
best_score = None
search_rows = []

for seed in range(N_SPLIT_SEEDS):
    candidate = make_candidate_split(seed)
    score = score_candidate(candidate)

    search_rows.append(
        {
            "seed": seed,
            **score,
        }
    )

    if (
        best_score is None
        or score["objective"] < best_score["objective"]
    ):
        best_seed = seed
        best_candidate = candidate.copy()
        best_score = score.copy()

split_search = pd.DataFrame(search_rows).sort_values(
    "objective"
).reset_index(drop=True)

taskb_document_split = (
    best_candidate[
        [
            "domain",
            "num_article",
            "doi_normalized",
            "document_key",
            "split",
        ]
        + SPLIT_FEATURES
    ]
    .sort_values(
        ["domain", "split", "num_article"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Attach split to authoritative resources
# ------------------------------------------------------------

split_lookup = dict(
    zip(
        taskb_document_split["document_key"],
        taskb_document_split["split"],
    )
)

taskb_gold_links_split = taskb_gold_links.copy()
taskb_gold_links_split["split"] = (
    taskb_gold_links_split[
        "document_key"
    ].map(split_lookup)
)

taskb_mention_inputs_split = taskb_mention_inputs.copy()
taskb_mention_inputs_split["split"] = (
    taskb_mention_inputs_split[
        "document_key"
    ].map(split_lookup)
)

assert taskb_gold_links_split["split"].notna().all()
assert taskb_mention_inputs_split["split"].notna().all()


# ------------------------------------------------------------
# 10. Final seen/unseen labels relative to training only
# ------------------------------------------------------------

training_concepts = set(
    taskb_gold_links_split.loc[
        taskb_gold_links_split["split"] == "train",
        "concept_normalized_canonical",
    ]
)

taskb_gold_links_split["concept_seen_in_train"] = (
    taskb_gold_links_split[
        "concept_normalized_canonical"
    ].isin(training_concepts)
)

taskb_gold_links_split["concept_familiarity"] = np.where(
    taskb_gold_links_split["concept_seen_in_train"],
    "seen",
    "unseen",
)

# Mention familiarity:
# all_seen = every gold concept appeared in training
# mixed = some appeared and some did not
# all_unseen = none appeared in training

mention_familiarity = (
    taskb_gold_links_split.groupby(
        [
            "domain",
            "document_key",
            "mention_normalized",
        ]
    )["concept_seen_in_train"]
    .agg(["sum", "count"])
    .reset_index()
)

mention_familiarity["mention_familiarity"] = np.select(
    [
        mention_familiarity["sum"]
        == mention_familiarity["count"],

        mention_familiarity["sum"] == 0,
    ],
    [
        "all_seen",
        "all_unseen",
    ],
    default="mixed",
)

taskb_mention_inputs_split = (
    taskb_mention_inputs_split.merge(
        mention_familiarity[
            [
                "domain",
                "document_key",
                "mention_normalized",
                "mention_familiarity",
            ]
        ],
        on=[
            "domain",
            "document_key",
            "mention_normalized",
        ],
        how="left",
    )
)


# ------------------------------------------------------------
# 11. Balance table
# ------------------------------------------------------------

balance_rows = []

for domain, domain_rows in taskb_document_split.groupby(
    "domain"
):
    for split, split_rows in domain_rows.groupby("split"):
        for feature in SPLIT_FEATURES:
            ref = feature_reference[domain][feature]

            split_mean = float(
                split_rows[feature].mean()
            )

            standardized_difference = (
                split_mean - ref["mean"]
            ) / ref["std"]

            balance_rows.append(
                {
                    "domain": domain,
                    "split": split,
                    "feature": feature,
                    "global_mean": ref["mean"],
                    "split_mean": split_mean,
                    "standardized_difference":
                        standardized_difference,
                    "absolute_difference":
                        abs(standardized_difference),
                }
            )

split_balance = pd.DataFrame(balance_rows)


# ------------------------------------------------------------
# 12. Report selected split
# ------------------------------------------------------------

print("=" * 78)
print("SELECTED TASK B DOCUMENT HOLDOUT")
print("=" * 78)

print("Selected metadata-balance seed:", best_seed)
print("Selection objective:", best_score["objective"])

print("\nSelection components:")
for key, value in best_score.items():
    print(f"{key}: {value}")

print("\nDocument counts:")
print(
    taskb_document_split.groupby(
        ["domain", "split"]
    ).size()
)

print("\nMention-input counts:")
print(
    taskb_mention_inputs_split.groupby(
        ["domain", "split"]
    ).size()
)

print("\nGold-link counts:")
print(
    taskb_gold_links_split.groupby(
        ["domain", "split"]
    ).size()
)

print("\nConcept familiarity of development/test links:")
print(
    taskb_gold_links_split[
        taskb_gold_links_split["split"].isin(
            ["development", "test"]
        )
    ]
    .groupby(
        ["domain", "split", "concept_familiarity"]
    )
    .size()
)

print("\nMention familiarity:")
print(
    taskb_mention_inputs_split[
        taskb_mention_inputs_split["split"].isin(
            ["development", "test"]
        )
    ]
    .groupby(
        ["domain", "split", "mention_familiarity"]
    )
    .size()
)

print("\nMean document characteristics:")
display(
    taskb_document_split.groupby(
        ["domain", "split"]
    )[SPLIT_FEATURES]
    .mean()
    .round(4)
)

print(
    "\nLargest absolute standardized difference:",
    split_balance[
        "absolute_difference"
    ].max(),
)

print("\nFifteen largest balance differences:")
display(
    split_balance.sort_values(
        "absolute_difference",
        ascending=False,
    ).head(15)
)

print("\nTop ten candidate seeds:")
display(split_search.head(10))


# ------------------------------------------------------------
# 13. Save authoritative split artifacts
# ------------------------------------------------------------

DOCUMENT_SPLIT_PATH = (
    TASKB_SPLIT_DIR
    / "taskB_document_holdout_grouped_70_15_15.csv"
)

GOLD_LINKS_SPLIT_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_gold_links_with_split.csv"
)

MENTION_INPUTS_SPLIT_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_authoritative_mention_inputs_with_split.csv"
)

SPLIT_BALANCE_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_document_holdout_balance.csv"
)

SPLIT_SEARCH_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_document_holdout_seed_search.csv"
)

SPLIT_METADATA_PATH = (
    TASKB_SPLIT_DIR
    / "taskB_document_holdout_metadata.json"
)

taskb_document_split.to_csv(
    DOCUMENT_SPLIT_PATH,
    index=False,
)

taskb_gold_links_split.to_csv(
    GOLD_LINKS_SPLIT_PATH,
    index=False,
)

taskb_mention_inputs_split.to_csv(
    MENTION_INPUTS_SPLIT_PATH,
    index=False,
)

split_balance.to_csv(
    SPLIT_BALANCE_PATH,
    index=False,
)

split_search.to_csv(
    SPLIT_SEARCH_PATH,
    index=False,
)

split_metadata = {
    "protocol": "domain_stratified_document_grouped_holdout",
    "split_ratio_nominal": {
        "train": 0.70,
        "development": 0.15,
        "test": 0.15,
    },
    "domain_split_counts": DOMAIN_SPLIT_COUNTS,
    "candidate_seeds_searched": N_SPLIT_SEEDS,
    "selected_seed": int(best_seed),
    "selection_score": {
        key: (
            int(value)
            if isinstance(value, (np.integer,))
            else float(value)
        )
        for key, value in best_score.items()
    },
    "selection_used_model_predictions": False,
    "selection_used_test_performance": False,
    "evaluation_unit":
        "unique_document_mention_concept_link",
    "mention_input_unit":
        "unique_document_normalized_mention",
    "canonical_concepts": int(
        len(
            set(
                taskb_gold_links_split[
                    "concept_normalized_canonical"
                ]
            )
        )
    ),
}

SPLIT_METADATA_PATH.write_text(
    json.dumps(
        split_metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nAuthoritative document split saved to:")
print(DOCUMENT_SPLIT_PATH)

print("\nSplit gold links saved to:")
print(GOLD_LINKS_SPLIT_PATH)

print("\nSplit mention inputs saved to:")
print(MENTION_INPUTS_SPLIT_PATH)

print("\nSplit metadata saved to:")
print(SPLIT_METADATA_PATH)

print("\nCell 6 completed successfully.")

In [ ]:
# TASK B — CELL 7
# Training-only exact normalized mention lookup baseline.
# Evaluated on development only.

import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Build mention-level gold sets directly from canonical links
# ------------------------------------------------------------

MENTION_GROUP_KEY = [
    "domain",
    "document_key",
    "num_article",
    "doi_normalized",
    "mention_normalized",
]

mention_gold = (
    taskb_gold_links_split.groupby(
        MENTION_GROUP_KEY,
        as_index=False,
    )
    .agg(
        mention=("mention", "first"),
        split=("split", "first"),
        gold_concepts=(
            "concept_normalized_canonical",
            lambda values: frozenset(values),
        ),
        gold_concept_labels=(
            "concept_canonical",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
    )
)

mention_gold["gold_concept_count"] = (
    mention_gold["gold_concepts"].map(len)
)


# ------------------------------------------------------------
# 2. Construct training-only mention dictionary
# ------------------------------------------------------------

training_links = taskb_gold_links_split[
    taskb_gold_links_split["split"] == "train"
].copy()

training_mention_dictionary = (
    training_links.groupby(
        "mention_normalized"
    )["concept_normalized_canonical"]
    .agg(
        lambda values: frozenset(values)
    )
    .to_dict()
)

training_concepts = set(
    training_links[
        "concept_normalized_canonical"
    ]
)

training_mentions = set(
    training_mention_dictionary
)


# ------------------------------------------------------------
# 3. Development predictions
# ------------------------------------------------------------

development_mentions = (
    mention_gold[
        mention_gold["split"] == "development"
    ]
    .copy()
    .reset_index(drop=True)
)

development_mentions["predicted_concepts"] = (
    development_mentions[
        "mention_normalized"
    ].map(
        lambda mention: (
            training_mention_dictionary.get(
                mention,
                frozenset(),
            )
        )
    )
)

development_mentions["mention_seen_in_train"] = (
    development_mentions[
        "mention_normalized"
    ].isin(training_mentions)
)

development_mentions["seen_gold_count"] = (
    development_mentions[
        "gold_concepts"
    ].map(
        lambda concepts: len(
            set(concepts) & training_concepts
        )
    )
)

development_mentions["concept_familiarity"] = (
    development_mentions.apply(
        lambda row: (
            "all_seen"
            if row["seen_gold_count"]
            == row["gold_concept_count"]
            else (
                "all_unseen"
                if row["seen_gold_count"] == 0
                else "mixed"
            )
        ),
        axis=1,
    )
)


# ------------------------------------------------------------
# 4. Set-valued evaluation
# ------------------------------------------------------------

def evaluate_set_predictions(frame):
    total_tp = 0
    total_fp = 0
    total_fn = 0
    exact_matches = []
    jaccards = []
    prediction_coverages = []

    for row in frame.itertuples(index=False):
        gold = set(row.gold_concepts)
        predicted = set(row.predicted_concepts)

        tp = len(gold & predicted)
        fp = len(predicted - gold)
        fn = len(gold - predicted)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        exact_matches.append(
            gold == predicted
        )

        union = gold | predicted
        jaccards.append(
            len(gold & predicted) / len(union)
            if union
            else 1.0
        )

        prediction_coverages.append(
            len(predicted) > 0
        )

    precision = (
        total_tp / (total_tp + total_fp)
        if total_tp + total_fp
        else 0.0
    )

    recall = (
        total_tp / (total_tp + total_fn)
        if total_tp + total_fn
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "mention_inputs": int(len(frame)),
        "gold_links": int(
            frame["gold_concepts"].map(len).sum()
        ),
        "predicted_links": int(
            frame["predicted_concepts"].map(len).sum()
        ),
        "true_positive_links": int(total_tp),
        "false_positive_links": int(total_fp),
        "false_negative_links": int(total_fn),
        "micro_precision": float(precision),
        "micro_recall": float(recall),
        "micro_f1": float(f1),
        "exact_set_accuracy": float(
            np.mean(exact_matches)
        ) if exact_matches else np.nan,
        "mean_jaccard": float(
            np.mean(jaccards)
        ) if jaccards else np.nan,
        "prediction_coverage": float(
            np.mean(prediction_coverages)
        ) if prediction_coverages else np.nan,
    }


# ------------------------------------------------------------
# 5. Overall, domain, and familiarity results
# ------------------------------------------------------------

result_rows = []

def add_result(scope, value, subset):
    result_rows.append(
        {
            "system":
                "TRAIN_EXACT_NORMALIZED_MENTION_LOOKUP",
            "scope": scope,
            "scope_value": value,
            **evaluate_set_predictions(subset),
        }
    )


add_result(
    "all",
    "development",
    development_mentions,
)

for domain, subset in development_mentions.groupby(
    "domain"
):
    add_result(
        "domain",
        domain,
        subset,
    )

for familiarity in [
    "all_seen",
    "mixed",
    "all_unseen",
]:
    subset = development_mentions[
        development_mentions[
            "concept_familiarity"
        ] == familiarity
    ]

    if len(subset):
        add_result(
            "concept_familiarity",
            familiarity,
            subset,
        )

for mention_seen, subset in development_mentions.groupby(
    "mention_seen_in_train"
):
    add_result(
        "mention_seen_in_train",
        str(bool(mention_seen)),
        subset,
    )

exact_lookup_results = pd.DataFrame(result_rows)


# ------------------------------------------------------------
# 6. Error flags for later analysis
# ------------------------------------------------------------

development_mentions["true_positive_count"] = (
    development_mentions.apply(
        lambda row: len(
            set(row["gold_concepts"])
            & set(row["predicted_concepts"])
        ),
        axis=1,
    )
)

development_mentions["false_positive_count"] = (
    development_mentions.apply(
        lambda row: len(
            set(row["predicted_concepts"])
            - set(row["gold_concepts"])
        ),
        axis=1,
    )
)

development_mentions["false_negative_count"] = (
    development_mentions.apply(
        lambda row: len(
            set(row["gold_concepts"])
            - set(row["predicted_concepts"])
        ),
        axis=1,
    )
)

development_mentions["exact_set_match"] = (
    development_mentions.apply(
        lambda row: (
            set(row["gold_concepts"])
            == set(row["predicted_concepts"])
        ),
        axis=1,
    )
)

development_mentions["gold_concepts_serialized"] = (
    development_mentions[
        "gold_concepts"
    ].map(
        lambda values: " | ".join(
            sorted(values)
        )
    )
)

development_mentions[
    "predicted_concepts_serialized"
] = development_mentions[
    "predicted_concepts"
].map(
    lambda values: " | ".join(
        sorted(values)
    )
)


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("=" * 78)
print("TRAINING-ONLY EXACT MENTION LOOKUP — DEVELOPMENT")
print("=" * 78)

print("Training documents:", (
    taskb_document_split["split"] == "train"
).sum())

print(
    "Training mention dictionary entries:",
    len(training_mention_dictionary),
)

print(
    "Training concept inventory:",
    len(training_concepts),
)

print(
    "Development mention inputs:",
    len(development_mentions),
)

print(
    "Development mentions seen verbatim after normalization:",
    int(
        development_mentions[
            "mention_seen_in_train"
        ].sum()
    ),
)

print(
    "Normalized mention coverage:",
    round(
        development_mentions[
            "mention_seen_in_train"
        ].mean(),
        4,
    ),
)

print("\nEvaluation results:")
display(
    exact_lookup_results.round(4)
)

print("\nConcept familiarity by domain:")
display(
    pd.crosstab(
        development_mentions["domain"],
        development_mentions[
            "concept_familiarity"
        ],
    )
)

print("\nFirst 30 development errors:")
display(
    development_mentions[
        ~development_mentions[
            "exact_set_match"
        ]
    ][
        [
            "domain",
            "num_article",
            "mention",
            "concept_familiarity",
            "mention_seen_in_train",
            "gold_concepts_serialized",
            "predicted_concepts_serialized",
            "true_positive_count",
            "false_positive_count",
            "false_negative_count",
        ]
    ].head(30)
)


# ------------------------------------------------------------
# 8. Save development-only baseline
# ------------------------------------------------------------

EXACT_LOOKUP_RESULTS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_exact_lookup_results.csv"
)

EXACT_LOOKUP_PREDICTIONS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_exact_lookup_predictions.csv"
)

exact_lookup_results.to_csv(
    EXACT_LOOKUP_RESULTS_PATH,
    index=False,
)

development_mentions.drop(
    columns=[
        "gold_concepts",
        "predicted_concepts",
    ]
).to_csv(
    EXACT_LOOKUP_PREDICTIONS_PATH,
    index=False,
)

print("\nSaved exact-lookup results to:")
print(EXACT_LOOKUP_RESULTS_PATH)

print("\nSaved development predictions to:")
print(EXACT_LOOKUP_PREDICTIONS_PATH)

print("\nCell 7 completed successfully.")

In [ ]:
# TASK B — CELL 8
# Morphological normalization and open-vocabulary surface baseline.
# Development only; training evidence only.

import re
import unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Generic orthographic normalization
# ------------------------------------------------------------

BRITISH_TO_AMERICAN = {
    "behaviour": "behavior",
    "behaviours": "behaviors",
    "modelling": "modeling",
    "labelled": "labeled",
    "signalling": "signaling",
    "tumour": "tumor",
    "tumours": "tumors",
    "organisation": "organization",
    "organisations": "organizations",
    "normalisation": "normalization",
    "characterisation": "characterization",
}


def normalize_orthography(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    text = text.casefold().strip()

    # Standardize common English spelling variants.
    tokens = re.findall(
        r"[a-z0-9]+(?:'[a-z0-9]+)?|[-/]",
        text,
    )

    tokens = [
        BRITISH_TO_AMERICAN.get(token, token)
        for token in tokens
    ]

    text = " ".join(tokens)

    # Treat hyphenation and slash variation as spacing variation.
    text = re.sub(r"\s*[-/]\s*", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ------------------------------------------------------------
# 2. Conservative singularization
# ------------------------------------------------------------

IRREGULAR_SINGULARS = {
    "bacteria": "bacterium",
    "children": "child",
    "data": "datum",
    "diseases": "disease",
    "disorders": "disorder",
    "effects": "effect",
    "foods": "food",
    "fungi": "fungus",
    "humans": "human",
    "interactions": "interaction",
    "mechanisms": "mechanism",
    "metabolites": "metabolite",
    "microbes": "microbe",
    "microorganisms": "microorganism",
    "models": "model",
    "patients": "patient",
    "pathways": "pathway",
    "processes": "process",
    "proteins": "protein",
    "receptors": "receptor",
    "responses": "response",
    "strategies": "strategy",
    "studies": "study",
    "symptoms": "symptom",
    "systems": "system",
    "therapies": "therapy",
    "trials": "trial",
}


def singularize_last_token(text):
    tokens = normalize_orthography(text).split()

    if not tokens:
        return ""

    last = tokens[-1]

    if last in IRREGULAR_SINGULARS:
        tokens[-1] = IRREGULAR_SINGULARS[last]

    elif (
        last.endswith("ies")
        and len(last) > 4
    ):
        tokens[-1] = last[:-3] + "y"

    elif (
        last.endswith("sses")
        and len(last) > 5
    ):
        tokens[-1] = last[:-2]

    elif (
        last.endswith(("xes", "zes", "ches", "shes"))
        and len(last) > 4
    ):
        tokens[-1] = last[:-2]

    elif (
        last.endswith("s")
        and not last.endswith(
            (
                "ss",
                "us",
                "is",
            )
        )
        and len(last) > 3
    ):
        tokens[-1] = last[:-1]

    return " ".join(tokens)


def morphological_key(text):
    return singularize_last_token(
        normalize_orthography(text)
    )


# ------------------------------------------------------------
# 3. Training-only morphological mention dictionary
# ------------------------------------------------------------

training_morph_dictionary = defaultdict(set)

for row in training_links.itertuples(index=False):
    key = morphological_key(row.mention)

    training_morph_dictionary[key].add(
        row.concept_normalized_canonical
    )

training_morph_dictionary = {
    key: frozenset(values)
    for key, values in training_morph_dictionary.items()
}


# ------------------------------------------------------------
# 4. Training-only concept-label lookup
#
# This is still restricted to concepts occurring in training.
# It does not use the 315-definition inventory.
# ------------------------------------------------------------

training_concept_label_lookup = defaultdict(set)

for concept in sorted(training_concepts):
    key = morphological_key(concept)

    training_concept_label_lookup[key].add(
        concept
    )

training_concept_label_lookup = {
    key: frozenset(values)
    for key, values
    in training_concept_label_lookup.items()
}


# ------------------------------------------------------------
# 5. Inspect normalization collisions
# ------------------------------------------------------------

mention_key_source_counts = (
    training_links[
        [
            "mention",
            "mention_normalized",
        ]
    ]
    .drop_duplicates()
    .assign(
        morphological_key=lambda frame:
            frame["mention"].map(
                morphological_key
            )
    )
    .groupby("morphological_key")
    .agg(
        normalized_surface_forms=(
            "mention_normalized",
            "nunique",
        ),
        surface_forms=(
            "mention",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
    )
    .reset_index()
)

concepts_per_morph_key = pd.DataFrame(
    [
        {
            "morphological_key": key,
            "mapped_concept_count": len(values),
            "mapped_concepts": " | ".join(
                sorted(values)
            ),
        }
        for key, values
        in training_morph_dictionary.items()
    ]
)

morph_collision_audit = (
    mention_key_source_counts.merge(
        concepts_per_morph_key,
        on="morphological_key",
        how="left",
    )
)

potentially_ambiguous_morph_keys = (
    morph_collision_audit[
        morph_collision_audit[
            "mapped_concept_count"
        ] > 1
    ]
    .copy()
)


# ------------------------------------------------------------
# 6. Three frozen prediction policies
# ------------------------------------------------------------

def predict_exact_lookup(mention):
    return training_mention_dictionary.get(
        normalize_concept(mention),
        frozenset(),
    )


def predict_morphological_lookup(mention):
    key = morphological_key(mention)

    mention_hits = training_morph_dictionary.get(
        key,
        frozenset(),
    )

    if mention_hits:
        return mention_hits

    return training_concept_label_lookup.get(
        key,
        frozenset(),
    )


def predict_morphological_open(mention):
    """
    If no training-supported mapping is available, generate
    a conservative candidate concept directly from the surface
    form. This permits genuinely unseen concepts.
    """
    training_hits = predict_morphological_lookup(
        mention
    )

    if training_hits:
        return training_hits

    generated_label = morphological_key(mention)

    if not generated_label:
        return frozenset()

    return frozenset([generated_label])


# ------------------------------------------------------------
# 7. Apply the policies to development only
# ------------------------------------------------------------

development_morph = (
    development_mentions.copy()
)

development_morph[
    "predicted_exact"
] = development_morph[
    "mention"
].map(predict_exact_lookup)

development_morph[
    "predicted_morph_lookup"
] = development_morph[
    "mention"
].map(predict_morphological_lookup)

development_morph[
    "predicted_morph_open"
] = development_morph[
    "mention"
].map(predict_morphological_open)


# ------------------------------------------------------------
# 8. Evaluate each policy
# ------------------------------------------------------------

system_columns = {
    "EXACT_LOOKUP":
        "predicted_exact",

    "MORPHOLOGICAL_LOOKUP":
        "predicted_morph_lookup",

    "MORPHOLOGICAL_LOOKUP_PLUS_OPEN_SURFACE":
        "predicted_morph_open",
}

morph_result_rows = []

for system_name, prediction_column in (
    system_columns.items()
):
    evaluation_frame = (
        development_morph.copy()
    )

    evaluation_frame[
        "predicted_concepts"
    ] = evaluation_frame[
        prediction_column
    ]

    # Overall
    morph_result_rows.append(
        {
            "system": system_name,
            "scope": "all",
            "scope_value": "development",
            **evaluate_set_predictions(
                evaluation_frame
            ),
        }
    )

    # Domain
    for domain, subset in (
        evaluation_frame.groupby("domain")
    ):
        morph_result_rows.append(
            {
                "system": system_name,
                "scope": "domain",
                "scope_value": domain,
                **evaluate_set_predictions(
                    subset
                ),
            }
        )

    # Concept familiarity
    for familiarity, subset in (
        evaluation_frame.groupby(
            "concept_familiarity"
        )
    ):
        morph_result_rows.append(
            {
                "system": system_name,
                "scope":
                    "concept_familiarity",
                "scope_value": familiarity,
                **evaluate_set_predictions(
                    subset
                ),
            }
        )

morphological_results = pd.DataFrame(
    morph_result_rows
)


# ------------------------------------------------------------
# 9. Per-input comparison
# ------------------------------------------------------------

def exact_match(gold, predicted):
    return set(gold) == set(predicted)


for system_name, prediction_column in (
    system_columns.items()
):
    development_morph[
        f"{system_name}_exact"
    ] = development_morph.apply(
        lambda row: exact_match(
            row["gold_concepts"],
            row[prediction_column],
        ),
        axis=1,
    )


# Cases improved relative to exact lookup
improved_by_morphology = development_morph[
    (
        ~development_morph[
            "EXACT_LOOKUP_exact"
        ]
    )
    &
    (
        development_morph[
            "MORPHOLOGICAL_LOOKUP_exact"
        ]
        |
        development_morph[
            "MORPHOLOGICAL_LOOKUP_PLUS_OPEN_SURFACE_exact"
        ]
    )
].copy()


# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

print("=" * 78)
print("MORPHOLOGICAL AND OPEN-SURFACE BASELINES")
print("=" * 78)

print(
    "Training morphological keys:",
    len(training_morph_dictionary),
)

print(
    "Keys mapping to multiple concepts:",
    len(potentially_ambiguous_morph_keys),
)

if len(potentially_ambiguous_morph_keys):
    print("\nPotentially ambiguous morphological keys:")
    display(
        potentially_ambiguous_morph_keys.head(30)
    )

print("\nOverall development results:")
display(
    morphological_results[
        morphological_results[
            "scope"
        ] == "all"
    ].round(4)
)

print("\nResults by concept familiarity:")
display(
    morphological_results[
        morphological_results[
            "scope"
        ] == "concept_familiarity"
    ].round(4)
)

print("\nCases solved beyond exact lookup:")
display(
    improved_by_morphology[
        [
            "domain",
            "num_article",
            "mention",
            "concept_familiarity",
            "gold_concept_labels",
            "predicted_morph_lookup",
            "predicted_morph_open",
        ]
    ].head(40)
)


# ------------------------------------------------------------
# 11. Save development results
# ------------------------------------------------------------

MORPH_RESULTS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_morphological_baselines.csv"
)

MORPH_PREDICTIONS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_morphological_predictions.csv"
)

MORPH_COLLISION_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_training_morphological_collision_audit.csv"
)

morphological_results.to_csv(
    MORPH_RESULTS_PATH,
    index=False,
)

development_morph_serializable = (
    development_morph.copy()
)

set_columns = [
    "gold_concepts",
    "predicted_concepts",
    "predicted_exact",
    "predicted_morph_lookup",
    "predicted_morph_open",
]

for column in set_columns:
    if column in development_morph_serializable:
        development_morph_serializable[
            column
        ] = development_morph_serializable[
            column
        ].map(
            lambda values: " | ".join(
                sorted(values)
            )
            if isinstance(
                values,
                (set, frozenset),
            )
            else str(values)
        )

development_morph_serializable.to_csv(
    MORPH_PREDICTIONS_PATH,
    index=False,
)

morph_collision_audit.to_csv(
    MORPH_COLLISION_PATH,
    index=False,
)

print("\nSaved morphological results to:")
print(MORPH_RESULTS_PATH)

print("\nSaved morphological predictions to:")
print(MORPH_PREDICTIONS_PATH)

print("\nSaved collision audit to:")
print(MORPH_COLLISION_PATH)

print("\nCell 8 completed successfully.")

In [ ]:
# TASK B — CELL 9
# Training-only compositional concept containment.
# Development tuning only; test remains untouched.

from collections import defaultdict

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Training concept lexicon
# ------------------------------------------------------------

training_concept_lexicon = []

for concept in sorted(training_concepts):
    lexical_form = normalize_orthography(concept)
    tokens = tuple(lexical_form.split())

    if not tokens:
        continue

    training_concept_lexicon.append(
        {
            "concept": concept,
            "lexical_form": lexical_form,
            "tokens": tokens,
            "token_count": len(tokens),
            "character_count": len(
                lexical_form.replace(" ", "")
            ),
        }
    )


# ------------------------------------------------------------
# 2. Find training concepts inside a mention
# ------------------------------------------------------------

def find_concept_containment_matches(
    mention,
    minimum_atomic_characters=4,
):
    mention_tokens = tuple(
        normalize_orthography(mention).split()
    )

    matches = []

    for entry in training_concept_lexicon:
        concept_tokens = entry["tokens"]
        token_count = entry["token_count"]

        if (
            token_count == 1
            and entry["character_count"]
            < minimum_atomic_characters
        ):
            continue

        if token_count > len(mention_tokens):
            continue

        for start in range(
            len(mention_tokens) - token_count + 1
        ):
            end = start + token_count

            if (
                mention_tokens[start:end]
                == concept_tokens
            ):
                matches.append(
                    {
                        "concept": entry["concept"],
                        "start": start,
                        "end": end,
                        "token_count": token_count,
                        "character_count":
                            entry["character_count"],
                    }
                )

    return matches


# ------------------------------------------------------------
# 3. Retain maximal lexical matches
#
# Example:
# "human gut microbiota"
# keeps "human" and "gut microbiota",
# but removes "microbiota" when it is nested inside
# the longer matched label "gut microbiota".
# ------------------------------------------------------------

def retain_maximal_matches(matches):
    retained = []

    for candidate in matches:
        contained_by_longer = False

        for other in matches:
            same_match = (
                candidate["concept"]
                == other["concept"]
                and candidate["start"]
                == other["start"]
                and candidate["end"]
                == other["end"]
            )

            if same_match:
                continue

            candidate_inside_other = (
                other["start"]
                <= candidate["start"]
                and other["end"]
                >= candidate["end"]
            )

            other_is_strictly_longer = (
                other["token_count"]
                > candidate["token_count"]
            )

            if (
                candidate_inside_other
                and other_is_strictly_longer
            ):
                contained_by_longer = True
                break

        if not contained_by_longer:
            retained.append(candidate)

    # Deduplicate concepts while retaining deterministic order.
    unique = {}
    for match in sorted(
        retained,
        key=lambda item: (
            item["start"],
            -item["token_count"],
            item["concept"],
        ),
    ):
        unique.setdefault(
            match["concept"],
            match,
        )

    return list(unique.values())


# ------------------------------------------------------------
# 4. Compositional prediction policy
# ------------------------------------------------------------

def predict_compositional_containment(
    mention,
    minimum_atomic_characters=4,
):
    # Preserve reliable complete mappings first.
    training_supported = (
        predict_morphological_lookup(mention)
    )

    if training_supported:
        return training_supported

    matches = find_concept_containment_matches(
        mention,
        minimum_atomic_characters=
            minimum_atomic_characters,
    )

    maximal_matches = retain_maximal_matches(
        matches
    )

    matched_concepts = frozenset(
        match["concept"]
        for match in maximal_matches
    )

    if matched_concepts:
        return matched_concepts

    # Open-vocabulary fallback for genuinely new labels.
    generated = morphological_key(mention)

    return (
        frozenset([generated])
        if generated
        else frozenset()
    )


# ------------------------------------------------------------
# 5. Development grid
# ------------------------------------------------------------

ATOMIC_CHARACTER_GRID = [
    3,
    4,
    5,
    6,
]

compositional_prediction_columns = {}

for minimum_characters in ATOMIC_CHARACTER_GRID:
    column = (
        f"predicted_compositional_"
        f"minchar{minimum_characters}"
    )

    compositional_prediction_columns[
        minimum_characters
    ] = column

    development_morph[column] = (
        development_morph["mention"].map(
            lambda mention: (
                predict_compositional_containment(
                    mention,
                    minimum_atomic_characters=
                        minimum_characters,
                )
            )
        )
    )


# ------------------------------------------------------------
# 6. Evaluation helper for requested subgroups
# ------------------------------------------------------------

def add_compositional_result(
    output_rows,
    system_name,
    scope,
    scope_value,
    frame,
    prediction_column,
):
    evaluation_frame = frame.copy()

    evaluation_frame[
        "predicted_concepts"
    ] = evaluation_frame[
        prediction_column
    ]

    output_rows.append(
        {
            "system": system_name,
            "scope": scope,
            "scope_value": scope_value,
            **evaluate_set_predictions(
                evaluation_frame
            ),
        }
    )


compositional_result_rows = []

# Include the prior open-surface baseline in the same table.
development_morph[
    "is_multilabel_gold"
] = development_morph[
    "gold_concept_count"
] > 1

for system_name, prediction_column in [
    (
        "MORPHOLOGICAL_LOOKUP_PLUS_OPEN_SURFACE",
        "predicted_morph_open",
    )
] + [
    (
        (
            "COMPOSITIONAL_CONTAINMENT"
            f"_MINCHAR_{minimum_characters}"
        ),
        prediction_column,
    )
    for minimum_characters, prediction_column
    in compositional_prediction_columns.items()
]:

    add_compositional_result(
        compositional_result_rows,
        system_name,
        "all",
        "development",
        development_morph,
        prediction_column,
    )

    for domain, subset in (
        development_morph.groupby("domain")
    ):
        add_compositional_result(
            compositional_result_rows,
            system_name,
            "domain",
            domain,
            subset,
            prediction_column,
        )

    for familiarity, subset in (
        development_morph.groupby(
            "concept_familiarity"
        )
    ):
        add_compositional_result(
            compositional_result_rows,
            system_name,
            "concept_familiarity",
            familiarity,
            subset,
            prediction_column,
        )

    for multilabel, subset in (
        development_morph.groupby(
            "is_multilabel_gold"
        )
    ):
        add_compositional_result(
            compositional_result_rows,
            system_name,
            "gold_cardinality",
            (
                "multilabel"
                if multilabel
                else "single_label"
            ),
            subset,
            prediction_column,
        )


compositional_results = pd.DataFrame(
    compositional_result_rows
)


# ------------------------------------------------------------
# 7. Development selection criterion
#
# Equal weight to link-level micro F1 and exact set accuracy.
# This avoids selecting a policy that obtains high recall by
# indiscriminately adding concepts.
# ------------------------------------------------------------

overall_compositional = (
    compositional_results[
        compositional_results["scope"] == "all"
    ]
    .copy()
)

overall_compositional[
    "selection_score"
] = (
    overall_compositional["micro_f1"]
    + overall_compositional[
        "exact_set_accuracy"
    ]
) / 2

selected_compositional_row = (
    overall_compositional.sort_values(
        [
            "selection_score",
            "micro_f1",
            "exact_set_accuracy",
        ],
        ascending=False,
    )
    .iloc[0]
)

selected_compositional_system = (
    selected_compositional_row["system"]
)

if selected_compositional_system.startswith(
    "COMPOSITIONAL_CONTAINMENT_MINCHAR_"
):
    selected_minimum_characters = int(
        selected_compositional_system.rsplit(
            "_",
            1,
        )[-1]
    )
else:
    selected_minimum_characters = None


# ------------------------------------------------------------
# 8. Compare predictions with the open-surface baseline
# ------------------------------------------------------------

if selected_minimum_characters is not None:
    selected_prediction_column = (
        compositional_prediction_columns[
            selected_minimum_characters
        ]
    )
else:
    selected_prediction_column = (
        "predicted_morph_open"
    )

development_morph[
    "selected_compositional_prediction"
] = development_morph[
    selected_prediction_column
]

development_morph[
    "open_surface_exact"
] = development_morph.apply(
    lambda row: (
        set(row["gold_concepts"])
        == set(row["predicted_morph_open"])
    ),
    axis=1,
)

development_morph[
    "selected_compositional_exact"
] = development_morph.apply(
    lambda row: (
        set(row["gold_concepts"])
        == set(
            row[
                "selected_compositional_prediction"
            ]
        )
    ),
    axis=1,
)

newly_solved = development_morph[
    (
        ~development_morph[
            "open_surface_exact"
        ]
    )
    &
    (
        development_morph[
            "selected_compositional_exact"
        ]
    )
].copy()

newly_broken = development_morph[
    (
        development_morph[
            "open_surface_exact"
        ]
    )
    &
    (
        ~development_morph[
            "selected_compositional_exact"
        ]
    )
].copy()


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("=" * 78)
print("COMPOSITIONAL CONTAINMENT — DEVELOPMENT")
print("=" * 78)

print(
    "Training concept lexicon:",
    len(training_concept_lexicon),
)

print(
    "Selected system:",
    selected_compositional_system,
)

print(
    "Selected minimum atomic characters:",
    selected_minimum_characters,
)

print("\nOverall results:")
display(
    overall_compositional[
        [
            "system",
            "mention_inputs",
            "gold_links",
            "predicted_links",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
            "selection_score",
        ]
    ]
    .sort_values(
        "selection_score",
        ascending=False,
    )
    .round(4)
)

print("\nMulti-label results:")
display(
    compositional_results[
        (
            compositional_results["scope"]
            == "gold_cardinality"
        )
        &
        (
            compositional_results["scope_value"]
            == "multilabel"
        )
    ][
        [
            "system",
            "mention_inputs",
            "gold_links",
            "predicted_links",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
        ]
    ].round(4)
)

print("\nMixed-familiarity results:")
display(
    compositional_results[
        (
            compositional_results["scope"]
            == "concept_familiarity"
        )
        &
        (
            compositional_results["scope_value"]
            == "mixed"
        )
    ][
        [
            "system",
            "mention_inputs",
            "gold_links",
            "predicted_links",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
        ]
    ].round(4)
)

print("\nNewly solved relative to open-surface baseline:")
display(
    newly_solved[
        [
            "domain",
            "num_article",
            "mention",
            "gold_concept_labels",
            "predicted_morph_open",
            "selected_compositional_prediction",
        ]
    ]
)

print("\nNewly broken relative to open-surface baseline:")
display(
    newly_broken[
        [
            "domain",
            "num_article",
            "mention",
            "gold_concept_labels",
            "predicted_morph_open",
            "selected_compositional_prediction",
        ]
    ]
)


# ------------------------------------------------------------
# 10. Save tuning artifacts
# ------------------------------------------------------------

COMPOSITIONAL_RESULTS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_compositional_tuning.csv"
)

COMPOSITIONAL_SELECTION_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_compositional_selection.json"
)

COMPOSITIONAL_PREDICTIONS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_compositional_predictions.csv"
)

compositional_results.to_csv(
    COMPOSITIONAL_RESULTS_PATH,
    index=False,
)

selection_metadata = {
    "selection_split": "development",
    "test_evaluated": False,
    "training_evidence_only": True,
    "selection_metric": (
        "mean_of_micro_f1_and_exact_set_accuracy"
    ),
    "selected_system":
        selected_compositional_system,
    "selected_minimum_atomic_characters":
        selected_minimum_characters,
    "atomic_character_grid":
        ATOMIC_CHARACTER_GRID,
}

COMPOSITIONAL_SELECTION_PATH.write_text(
    json.dumps(
        selection_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

serializable_compositional = (
    development_morph.copy()
)

for column in serializable_compositional.columns:
    if serializable_compositional[
        column
    ].map(
        lambda value: isinstance(
            value,
            (set, frozenset),
        )
    ).any():
        serializable_compositional[
            column
        ] = serializable_compositional[
            column
        ].map(
            lambda values: " | ".join(
                sorted(values)
            )
            if isinstance(
                values,
                (set, frozenset),
            )
            else str(values)
        )

serializable_compositional.to_csv(
    COMPOSITIONAL_PREDICTIONS_PATH,
    index=False,
)

print("\nSaved compositional tuning to:")
print(COMPOSITIONAL_RESULTS_PATH)

print("\nSaved selected configuration to:")
print(COMPOSITIONAL_SELECTION_PATH)

print("\nSaved compositional predictions to:")
print(COMPOSITIONAL_PREDICTIONS_PATH)

print("\nCell 9 completed successfully.")

In [ ]:
# TASK B — CELL 10
# Build PubMedBERT mention-only and contextual mention embeddings
# for TRAIN + DEVELOPMENT only. Test remains untouched.

from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PUBMEDBERT_MODEL = (
    "microsoft/"
    "BiomedNLP-PubMedBERT-base-uncased-abstract"
)

TASKB_TRAIN_ABS_DIR = (
    BASE_DIR
    / "data/taskB/training/abstracts"
)

TASKB_CACHE_DIR = (
    BASE_DIR
    / "journal_experiments/taskB_holdout/cache"
)

TASKB_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TASKB_EMBEDDING_PATH = (
    TASKB_CACHE_DIR
    / "taskB_pubmedbert_train_development_embeddings.npz"
)

TASKB_EMBEDDING_ROWS_PATH = (
    TASKB_CACHE_DIR
    / "taskB_pubmedbert_train_development_embedding_rows.csv"
)

TASKB_EMBEDDING_METADATA_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_pubmedbert_train_development_embedding_metadata.json"
)

CONTEXT_CHARACTERS_PER_SIDE = 240
MAX_OCCURRENCES_PER_MENTION = 20
MAX_SEQUENCE_LENGTH = 256
EMBEDDING_BATCH_SIZE = 64


# ------------------------------------------------------------
# 2. Hardware
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

use_bfloat16 = (
    device.type == "cuda"
    and torch.cuda.get_device_capability(0)[0] >= 8
)

print("=" * 78)
print("PUBMEDBERT EMBEDDING SETUP")
print("=" * 78)

print("Device:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory
            / 1024**3,
            1,
        ),
        "GB",
    )

print("BF16 autocast:", use_bfloat16)
print("Model:", PUBMEDBERT_MODEL)
print(
    "Splits embedded:",
    ["train", "development"],
)


# ------------------------------------------------------------
# 3. Load tokenizer and encoder
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    PUBMEDBERT_MODEL,
    use_fast=True,
)

if not tokenizer.is_fast:
    raise ValueError(
        "A fast tokenizer is required for offsets."
    )

encoder = AutoModel.from_pretrained(
    PUBMEDBERT_MODEL
).to(device)

encoder.eval()

hidden_size = int(
    encoder.config.hidden_size
)

print("Hidden size:", hidden_size)


# ------------------------------------------------------------
# 4. Read abstracts
# ------------------------------------------------------------

def read_taskb_training_abstract(
    domain,
    num_article,
):
    prefix = (
        "pa_"
        if domain == "parkinson"
        else "mh_"
    )

    path = (
        TASKB_TRAIN_ABS_DIR
        / domain
        / f"{prefix}{num_article}.txt"
    )

    if not path.exists():
        raise FileNotFoundError(path)

    return path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).replace("\r\n", "\n")


# ------------------------------------------------------------
# 5. Literal occurrence finder
#
# Offsets remain in the original string. Matching is
# case-insensitive but otherwise literal.
# ------------------------------------------------------------

def find_literal_occurrences(
    text,
    mention,
):
    if not mention:
        return []

    pattern = re.compile(
        re.escape(str(mention)),
        flags=re.IGNORECASE,
    )

    return [
        (match.start(), match.end())
        for match in pattern.finditer(text)
    ]


# ------------------------------------------------------------
# 6. Create one stable row per mention input
# ------------------------------------------------------------

embedding_rows = (
    mention_gold[
        mention_gold["split"].isin(
            ["train", "development"]
        )
    ]
    .copy()
    .sort_values(
        [
            "split",
            "domain",
            "num_article",
            "mention_normalized",
        ]
    )
    .reset_index(drop=True)
)

embedding_rows["embedding_row"] = np.arange(
    len(embedding_rows)
)

embedding_rows["mention_input_id"] = [
    f"taskb_input_{index:04d}"
    for index in range(len(embedding_rows))
]

print(
    "\nMention inputs to embed:",
    len(embedding_rows),
)

print(
    embedding_rows.groupby(
        ["domain", "split"]
    ).size()
)


# ------------------------------------------------------------
# 7. Cache document text
# ------------------------------------------------------------

document_text_cache = {}

for row in (
    embedding_rows[
        [
            "domain",
            "num_article",
        ]
    ]
    .drop_duplicates()
    .itertuples(index=False)
):
    document_text_cache[
        (
            row.domain,
            str(row.num_article),
        )
    ] = read_taskb_training_abstract(
        row.domain,
        str(row.num_article),
    )


# ------------------------------------------------------------
# 8. Prepare mention-only examples
# ------------------------------------------------------------

mention_only_examples = []

for row in embedding_rows.itertuples(
    index=False
):
    mention_only_examples.append(
        {
            "embedding_row":
                row.embedding_row,
            "text":
                str(row.mention),
        }
    )


# ------------------------------------------------------------
# 9. Prepare contextual occurrence examples
# ------------------------------------------------------------

context_examples = []
occurrence_counts = []
literal_alignment_flags = []

for row in embedding_rows.itertuples(
    index=False
):
    text = document_text_cache[
        (
            row.domain,
            str(row.num_article),
        )
    ]

    occurrences = find_literal_occurrences(
        text,
        row.mention,
    )

    occurrences = occurrences[
        :MAX_OCCURRENCES_PER_MENTION
    ]

    occurrence_counts.append(
        len(occurrences)
    )

    literal_alignment_flags.append(
        len(occurrences) > 0
    )

    for occurrence_index, (
        start_char,
        end_char,
    ) in enumerate(occurrences):

        window_start = max(
            0,
            start_char
            - CONTEXT_CHARACTERS_PER_SIDE,
        )

        window_end = min(
            len(text),
            end_char
            + CONTEXT_CHARACTERS_PER_SIDE,
        )

        window_text = text[
            window_start:window_end
        ]

        relative_start = (
            start_char - window_start
        )

        relative_end = (
            end_char - window_start
        )

        context_examples.append(
            {
                "embedding_row":
                    row.embedding_row,
                "occurrence_index":
                    occurrence_index,
                "text":
                    window_text,
                "mention_start":
                    relative_start,
                "mention_end":
                    relative_end,
            }
        )

embedding_rows[
    "literal_occurrence_count"
] = occurrence_counts

embedding_rows[
    "literal_alignment"
] = literal_alignment_flags


# ------------------------------------------------------------
# 10. Mean-pool non-special tokens for mention-only text
# ------------------------------------------------------------

@torch.inference_mode()
def embed_mention_only_examples(
    examples,
):
    output_vectors = np.zeros(
        (
            len(embedding_rows),
            hidden_size,
        ),
        dtype=np.float32,
    )

    for batch_start in tqdm(
        range(
            0,
            len(examples),
            EMBEDDING_BATCH_SIZE,
        ),
        desc="Mention-only embeddings",
    ):
        batch = examples[
            batch_start:
            batch_start
            + EMBEDDING_BATCH_SIZE
        ]

        texts = [
            example["text"]
            for example in batch
        ]

        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_SEQUENCE_LENGTH,
            return_tensors="pt",
            return_special_tokens_mask=True,
        )

        special_tokens_mask = encoded.pop(
            "special_tokens_mask"
        )

        model_inputs = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        autocast_context = (
            torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            )
            if use_bfloat16
            else torch.autocast(
                device_type="cpu",
                enabled=False,
            )
        )

        with autocast_context:
            hidden = encoder(
                **model_inputs
            ).last_hidden_state

        attention_mask = (
            model_inputs["attention_mask"]
            .bool()
        )

        valid_mask = (
            attention_mask
            &
            ~special_tokens_mask.to(
                device
            ).bool()
        )

        denominator = (
            valid_mask.sum(
                dim=1,
                keepdim=True,
            )
            .clamp(min=1)
        )

        pooled = (
            hidden
            * valid_mask.unsqueeze(-1)
        ).sum(dim=1) / denominator

        pooled = (
            pooled.float()
            .cpu()
            .numpy()
        )

        for local_index, example in enumerate(
            batch
        ):
            output_vectors[
                example["embedding_row"]
            ] = pooled[local_index]

    return output_vectors


# ------------------------------------------------------------
# 11. Pool contextualized tokens overlapping the mention
# ------------------------------------------------------------

@torch.inference_mode()
def embed_context_examples(
    examples,
):
    occurrence_vectors = defaultdict(
        list
    )

    token_alignment_failures = 0

    for batch_start in tqdm(
        range(
            0,
            len(examples),
            EMBEDDING_BATCH_SIZE,
        ),
        desc="Contextual occurrence embeddings",
    ):
        batch = examples[
            batch_start:
            batch_start
            + EMBEDDING_BATCH_SIZE
        ]

        texts = [
            example["text"]
            for example in batch
        ]

        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_SEQUENCE_LENGTH,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = encoded.pop(
            "offset_mapping"
        )

        model_inputs = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        autocast_context = (
            torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            )
            if use_bfloat16
            else torch.autocast(
                device_type="cpu",
                enabled=False,
            )
        )

        with autocast_context:
            hidden = encoder(
                **model_inputs
            ).last_hidden_state

        hidden = hidden.float().cpu()
        offsets = offsets.cpu()

        for local_index, example in enumerate(
            batch
        ):
            mention_start = int(
                example["mention_start"]
            )
            mention_end = int(
                example["mention_end"]
            )

            token_ids = []

            for token_index, (
                token_start,
                token_end,
            ) in enumerate(
                offsets[local_index].tolist()
            ):
                if (
                    token_start == 0
                    and token_end == 0
                ):
                    continue

                overlaps_mention = not (
                    token_end <= mention_start
                    or token_start >= mention_end
                )

                if overlaps_mention:
                    token_ids.append(
                        token_index
                    )

            if not token_ids:
                token_alignment_failures += 1
                continue

            vector = (
                hidden[
                    local_index,
                    token_ids,
                ]
                .mean(dim=0)
                .numpy()
                .astype(np.float32)
            )

            occurrence_vectors[
                example["embedding_row"]
            ].append(vector)

    return (
        occurrence_vectors,
        token_alignment_failures,
    )


# ------------------------------------------------------------
# 12. Generate embeddings
# ------------------------------------------------------------

start_time = time.time()

mention_only_embeddings = (
    embed_mention_only_examples(
        mention_only_examples
    )
)

(
    contextual_occurrence_vectors,
    token_alignment_failures,
) = embed_context_examples(
    context_examples
)

contextual_embeddings = np.zeros_like(
    mention_only_embeddings
)

contextual_fallback = []

for row_index in range(
    len(embedding_rows)
):
    vectors = contextual_occurrence_vectors.get(
        row_index,
        [],
    )

    if vectors:
        contextual_embeddings[
            row_index
        ] = np.mean(
            np.vstack(vectors),
            axis=0,
        )
        contextual_fallback.append(False)

    else:
        contextual_embeddings[
            row_index
        ] = mention_only_embeddings[
            row_index
        ]
        contextual_fallback.append(True)

embedding_rows[
    "contextual_fallback_to_mention_only"
] = contextual_fallback

elapsed_minutes = (
    time.time() - start_time
) / 60


# ------------------------------------------------------------
# 13. Numerical validation
# ------------------------------------------------------------

mention_finite = np.isfinite(
    mention_only_embeddings
).all()

context_finite = np.isfinite(
    contextual_embeddings
).all()

mention_norms = np.linalg.norm(
    mention_only_embeddings,
    axis=1,
)

context_norms = np.linalg.norm(
    contextual_embeddings,
    axis=1,
)

assert mention_finite
assert context_finite
assert (mention_norms > 0).all()
assert (context_norms > 0).all()


# ------------------------------------------------------------
# 14. Report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PUBMEDBERT EMBEDDING AUDIT")
print("=" * 78)

print(
    "Mention-only embedding shape:",
    mention_only_embeddings.shape,
)

print(
    "Contextual embedding shape:",
    contextual_embeddings.shape,
)

print(
    "Contextual occurrence examples:",
    len(context_examples),
)

print(
    "Inputs with at least one literal occurrence:",
    int(
        embedding_rows[
            "literal_alignment"
        ].sum()
    ),
)

print(
    "Inputs without a literal occurrence:",
    int(
        (
            ~embedding_rows[
                "literal_alignment"
            ]
        ).sum()
    ),
)

print(
    "Token-alignment failures:",
    token_alignment_failures,
)

print(
    "Contextual fallbacks:",
    int(
        embedding_rows[
            "contextual_fallback_to_mention_only"
        ].sum()
    ),
)

print(
    "Maximum occurrences pooled for one input:",
    int(
        embedding_rows[
            "literal_occurrence_count"
        ].max()
    ),
)

print(
    "Mean mention-only vector norm:",
    float(mention_norms.mean()),
)

print(
    "Mean contextual vector norm:",
    float(context_norms.mean()),
)

print(
    "Elapsed minutes:",
    round(elapsed_minutes, 2),
)

print("\nFallback inputs:")
display(
    embedding_rows[
        embedding_rows[
            "contextual_fallback_to_mention_only"
        ]
    ][
        [
            "split",
            "domain",
            "num_article",
            "mention",
            "literal_occurrence_count",
        ]
    ]
)


# ------------------------------------------------------------
# 15. Save embeddings and provenance
# ------------------------------------------------------------

np.savez_compressed(
    TASKB_EMBEDDING_PATH,
    mention_only=mention_only_embeddings,
    contextual=contextual_embeddings,
)

embedding_rows_to_save = (
    embedding_rows.drop(
        columns=[
            "gold_concepts",
        ],
        errors="ignore",
    )
    .copy()
)

embedding_rows_to_save.to_csv(
    TASKB_EMBEDDING_ROWS_PATH,
    index=False,
)

embedding_metadata = {
    "model": PUBMEDBERT_MODEL,
    "splits": [
        "train",
        "development",
    ],
    "test_embedded": False,
    "rows": int(len(embedding_rows)),
    "hidden_size": hidden_size,
    "context_characters_per_side":
        CONTEXT_CHARACTERS_PER_SIDE,
    "maximum_occurrences_per_mention":
        MAX_OCCURRENCES_PER_MENTION,
    "maximum_sequence_length":
        MAX_SEQUENCE_LENGTH,
    "batch_size":
        EMBEDDING_BATCH_SIZE,
    "contextual_occurrence_examples":
        int(len(context_examples)),
    "inputs_with_literal_alignment":
        int(
            embedding_rows[
                "literal_alignment"
            ].sum()
        ),
    "contextual_fallbacks":
        int(
            embedding_rows[
                "contextual_fallback_to_mention_only"
            ].sum()
        ),
    "token_alignment_failures":
        int(token_alignment_failures),
    "bf16_autocast":
        bool(use_bfloat16),
    "elapsed_minutes":
        float(elapsed_minutes),
}

TASKB_EMBEDDING_METADATA_PATH.write_text(
    json.dumps(
        embedding_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nSaved embeddings to:")
print(TASKB_EMBEDDING_PATH)

print("\nSaved embedding rows to:")
print(TASKB_EMBEDDING_ROWS_PATH)

print("\nSaved embedding metadata to:")
print(TASKB_EMBEDDING_METADATA_PATH)

print("\nCell 10 completed successfully.")

In [ ]:
# TASK B — CELL 11
# Training-only PubMedBERT concept prototypes and
# development ranking diagnostics.
# No prediction threshold is selected in this cell.

import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load embeddings if not already in memory
# ------------------------------------------------------------

if (
    "mention_only_embeddings"
    not in globals()
    or "contextual_embeddings"
    not in globals()
):
    embedding_archive = np.load(
        TASKB_EMBEDDING_PATH
    )

    mention_only_embeddings = (
        embedding_archive["mention_only"]
        .astype(np.float32)
    )

    contextual_embeddings = (
        embedding_archive["contextual"]
        .astype(np.float32)
    )

if "embedding_rows" not in globals():
    embedding_rows = pd.read_csv(
        TASKB_EMBEDDING_ROWS_PATH,
        dtype={"num_article": str},
    )

    # Reattach gold sets from the authoritative links.
    gold_sets_for_restore = (
        taskb_gold_links_split[
            taskb_gold_links_split[
                "split"
            ].isin(["train", "development"])
        ]
        .groupby(
            [
                "domain",
                "document_key",
                "num_article",
                "doi_normalized",
                "mention_normalized",
                "split",
            ],
            as_index=False,
        )
        .agg(
            gold_concepts=(
                "concept_normalized_canonical",
                lambda values: frozenset(values),
            )
        )
    )

    embedding_rows = embedding_rows.merge(
        gold_sets_for_restore,
        on=[
            "domain",
            "document_key",
            "num_article",
            "doi_normalized",
            "mention_normalized",
            "split",
        ],
        how="left",
        validate="one_to_one",
    )

assert (
    len(embedding_rows)
    == len(mention_only_embeddings)
    == len(contextual_embeddings)
)

assert embedding_rows["gold_concepts"].notna().all()


# ------------------------------------------------------------
# 2. L2 normalization
# ------------------------------------------------------------

def l2_normalize(matrix):
    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )

    norms = np.clip(
        norms,
        1e-12,
        None,
    )

    return matrix / norms


mention_only_normalized = l2_normalize(
    mention_only_embeddings
)

contextual_normalized = l2_normalize(
    contextual_embeddings
)


# ------------------------------------------------------------
# 3. Training/development row indices
# ------------------------------------------------------------

train_row_indices = embedding_rows.loc[
    embedding_rows["split"] == "train",
    "embedding_row",
].astype(int).to_numpy()

development_row_indices = embedding_rows.loc[
    embedding_rows["split"] == "development",
    "embedding_row",
].astype(int).to_numpy()

assert len(train_row_indices) == 566
assert len(development_row_indices) == 102


# ------------------------------------------------------------
# 4. Build one prototype per training concept
#
# A multilabel training mention contributes its representation
# to each of its gold concepts.
# ------------------------------------------------------------

prototype_concepts = sorted(
    {
        concept
        for row_index in train_row_indices
        for concept in embedding_rows.loc[
            row_index,
            "gold_concepts",
        ]
    }
)

concept_to_prototype_index = {
    concept: index
    for index, concept
    in enumerate(prototype_concepts)
}


def build_concept_prototypes(
    normalized_embeddings,
):
    vectors_by_concept = {
        concept: []
        for concept in prototype_concepts
    }

    source_rows_by_concept = {
        concept: set()
        for concept in prototype_concepts
    }

    for row_index in train_row_indices:
        concepts = embedding_rows.loc[
            row_index,
            "gold_concepts",
        ]

        vector = normalized_embeddings[
            row_index
        ]

        for concept in concepts:
            vectors_by_concept[
                concept
            ].append(vector)

            source_rows_by_concept[
                concept
            ].add(int(row_index))

    prototypes = np.zeros(
        (
            len(prototype_concepts),
            normalized_embeddings.shape[1],
        ),
        dtype=np.float32,
    )

    support_counts = []

    for concept, prototype_index in (
        concept_to_prototype_index.items()
    ):
        concept_vectors = np.vstack(
            vectors_by_concept[concept]
        )

        mean_vector = concept_vectors.mean(
            axis=0
        )

        mean_vector = mean_vector / max(
            np.linalg.norm(mean_vector),
            1e-12,
        )

        prototypes[
            prototype_index
        ] = mean_vector

        support_counts.append(
            {
                "concept": concept,
                "prototype_index":
                    prototype_index,
                "training_input_support":
                    len(
                        source_rows_by_concept[
                            concept
                        ]
                    ),
            }
        )

    return (
        prototypes,
        pd.DataFrame(support_counts),
    )


(
    mention_only_prototypes,
    prototype_support,
) = build_concept_prototypes(
    mention_only_normalized
)

(
    contextual_prototypes,
    contextual_support_check,
) = build_concept_prototypes(
    contextual_normalized
)

assert prototype_support.equals(
    contextual_support_check
)


# ------------------------------------------------------------
# 5. Development cosine-similarity matrices
# ------------------------------------------------------------

development_mention_vectors = (
    mention_only_normalized[
        development_row_indices
    ]
)

development_context_vectors = (
    contextual_normalized[
        development_row_indices
    ]
)

mention_only_similarity = (
    development_mention_vectors
    @ mention_only_prototypes.T
)

contextual_similarity = (
    development_context_vectors
    @ contextual_prototypes.T
)

assert mention_only_similarity.shape == (
    102,
    len(prototype_concepts),
)

assert contextual_similarity.shape == (
    102,
    len(prototype_concepts),
)


# ------------------------------------------------------------
# 6. Ranking metrics
# ------------------------------------------------------------

RANKING_K_VALUES = [
    1,
    3,
    5,
    10,
    20,
]


def evaluate_rankings(
    similarity_matrix,
    representation_name,
):
    per_input_rows = []
    relevant_link_ranks = []

    for local_index, row_index in enumerate(
        development_row_indices
    ):
        row = embedding_rows.loc[
            row_index
        ]

        gold_all = set(
            row["gold_concepts"]
        )

        gold_seen = (
            gold_all
            & set(prototype_concepts)
        )

        order = np.argsort(
            -similarity_matrix[
                local_index
            ]
        )

        ranked_concepts = [
            prototype_concepts[index]
            for index in order
        ]

        rank_lookup = {
            concept: rank
            for rank, concept
            in enumerate(
                ranked_concepts,
                start=1,
            )
        }

        seen_ranks = [
            rank_lookup[concept]
            for concept in sorted(gold_seen)
        ]

        relevant_link_ranks.extend(
            seen_ranks
        )

        best_relevant_rank = (
            min(seen_ranks)
            if seen_ranks
            else np.nan
        )

        input_result = {
            "representation":
                representation_name,
            "embedding_row":
                int(row_index),
            "mention_input_id":
                row["mention_input_id"],
            "domain":
                row["domain"],
            "num_article":
                row["num_article"],
            "mention":
                row["mention"],
            "gold_seen_count":
                len(gold_seen),
            "gold_unseen_count":
                len(gold_all - gold_seen),
            "best_relevant_rank":
                best_relevant_rank,
            "reciprocal_rank":
                (
                    1.0 / best_relevant_rank
                    if seen_ranks
                    else np.nan
                ),
            "top1_concept":
                ranked_concepts[0],
            "top1_similarity":
                float(
                    similarity_matrix[
                        local_index,
                        order[0],
                    ]
                ),
        }

        for k in RANKING_K_VALUES:
            top_k = set(
                ranked_concepts[:k]
            )

            input_result[
                f"hit_at_{k}"
            ] = (
                int(bool(gold_seen & top_k))
                if gold_seen
                else np.nan
            )

            input_result[
                f"seen_link_recall_at_{k}"
            ] = (
                len(gold_seen & top_k)
                / len(gold_seen)
                if gold_seen
                else np.nan
            )

        input_result[
            "top10_concepts"
        ] = " | ".join(
            ranked_concepts[:10]
        )

        input_result[
            "top10_similarities"
        ] = " | ".join(
            f"{similarity_matrix[local_index, index]:.6f}"
            for index in order[:10]
        )

        per_input_rows.append(
            input_result
        )

    per_input = pd.DataFrame(
        per_input_rows
    )

    eligible = per_input[
        per_input["gold_seen_count"] > 0
    ]

    summary = {
        "representation":
            representation_name,
        "development_inputs":
            len(per_input),
        "inputs_with_seen_gold":
            len(eligible),
        "seen_gold_links":
            len(relevant_link_ranks),
        "mean_reciprocal_rank":
            float(
                eligible[
                    "reciprocal_rank"
                ].mean()
            ),
        "median_best_relevant_rank":
            float(
                eligible[
                    "best_relevant_rank"
                ].median()
            ),
        "mean_relevant_link_rank":
            float(
                np.mean(
                    relevant_link_ranks
                )
            ),
    }

    for k in RANKING_K_VALUES:
        summary[
            f"input_hit_at_{k}"
        ] = float(
            eligible[
                f"hit_at_{k}"
            ].mean()
        )

        summary[
            f"seen_link_recall_at_{k}"
        ] = float(
            eligible[
                f"seen_link_recall_at_{k}"
            ].mean()
        )

    return (
        summary,
        per_input,
    )


(
    mention_ranking_summary,
    mention_ranking_inputs,
) = evaluate_rankings(
    mention_only_similarity,
    "MENTION_ONLY",
)

(
    context_ranking_summary,
    context_ranking_inputs,
) = evaluate_rankings(
    contextual_similarity,
    "CONTEXTUAL_MENTION",
)

ranking_summary = pd.DataFrame(
    [
        mention_ranking_summary,
        context_ranking_summary,
    ]
)

ranking_inputs = pd.concat(
    [
        mention_ranking_inputs,
        context_ranking_inputs,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 7. Direct paired comparison
# ------------------------------------------------------------

paired_ranks = (
    mention_ranking_inputs[
        [
            "mention_input_id",
            "best_relevant_rank",
            "reciprocal_rank",
        ]
    ]
    .rename(
        columns={
            "best_relevant_rank":
                "mention_only_rank",
            "reciprocal_rank":
                "mention_only_rr",
        }
    )
    .merge(
        context_ranking_inputs[
            [
                "mention_input_id",
                "domain",
                "num_article",
                "mention",
                "gold_seen_count",
                "best_relevant_rank",
                "reciprocal_rank",
                "top1_concept",
            ]
        ].rename(
            columns={
                "best_relevant_rank":
                    "contextual_rank",
                "reciprocal_rank":
                    "contextual_rr",
                "top1_concept":
                    "contextual_top1",
            }
        ),
        on="mention_input_id",
        how="inner",
    )
)

paired_ranks["context_rank_improvement"] = (
    paired_ranks["mention_only_rank"]
    - paired_ranks["contextual_rank"]
)

eligible_paired = paired_ranks[
    paired_ranks["gold_seen_count"] > 0
].copy()


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("=" * 78)
print("PUBMEDBERT CONCEPT-PROTOTYPE RANKING")
print("=" * 78)

print(
    "Training concepts represented:",
    len(prototype_concepts),
)

print(
    "Prototype support percentiles:"
)

print(
    prototype_support[
        "training_input_support"
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

print("\nRanking summary:")
display(
    ranking_summary.round(4)
)

print("\nPaired contextual comparison:")
print(
    "Context improves rank:",
    int(
        (
            eligible_paired[
                "context_rank_improvement"
            ] > 0
        ).sum()
    ),
)

print(
    "Context worsens rank:",
    int(
        (
            eligible_paired[
                "context_rank_improvement"
            ] < 0
        ).sum()
    ),
)

print(
    "Rank unchanged:",
    int(
        (
            eligible_paired[
                "context_rank_improvement"
            ] == 0
        ).sum()
    ),
)

print(
    "Median contextual rank change "
    "(positive means improvement):",
    float(
        eligible_paired[
            "context_rank_improvement"
        ].median()
    ),
)

print("\nTwenty largest contextual improvements:")
display(
    eligible_paired.sort_values(
        "context_rank_improvement",
        ascending=False,
    ).head(20)
)

print("\nTwenty largest contextual degradations:")
display(
    eligible_paired.sort_values(
        "context_rank_improvement",
        ascending=True,
    ).head(20)
)


# ------------------------------------------------------------
# 9. Save prototype and ranking artifacts
# ------------------------------------------------------------

TASKB_PROTOTYPE_PATH = (
    TASKB_CACHE_DIR
    / "taskB_pubmedbert_training_concept_prototypes.npz"
)

TASKB_PROTOTYPE_SUPPORT_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_pubmedbert_training_prototype_support.csv"
)

TASKB_RANKING_SUMMARY_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_pubmedbert_ranking_summary.csv"
)

TASKB_RANKING_INPUTS_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_pubmedbert_rankings.csv"
)

TASKB_RANKING_METADATA_PATH = (
    TASKB_RESULTS_DIR
    / "taskB_development_pubmedbert_ranking_metadata.json"
)

np.savez_compressed(
    TASKB_PROTOTYPE_PATH,
    concepts=np.array(
        prototype_concepts,
        dtype=object,
    ),
    mention_only_prototypes=
        mention_only_prototypes,
    contextual_prototypes=
        contextual_prototypes,
)

prototype_support.to_csv(
    TASKB_PROTOTYPE_SUPPORT_PATH,
    index=False,
)

ranking_summary.to_csv(
    TASKB_RANKING_SUMMARY_PATH,
    index=False,
)

ranking_inputs.to_csv(
    TASKB_RANKING_INPUTS_PATH,
    index=False,
)

ranking_metadata = {
    "training_only_prototypes": True,
    "test_evaluated": False,
    "prototype_concepts":
        len(prototype_concepts),
    "ranking_k_values":
        RANKING_K_VALUES,
    "similarity":
        "cosine",
    "prototype_aggregation":
        "mean_of_l2_normalized_training_input_embeddings_then_l2_normalize",
    "multilabel_training_policy":
        "input_representation_contributes_to_each_gold_concept",
}

TASKB_RANKING_METADATA_PATH.write_text(
    json.dumps(
        ranking_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nSaved prototypes to:")
print(TASKB_PROTOTYPE_PATH)

print("\nSaved prototype support to:")
print(TASKB_PROTOTYPE_SUPPORT_PATH)

print("\nSaved ranking summary to:")
print(TASKB_RANKING_SUMMARY_PATH)

print("\nSaved per-input rankings to:")
print(TASKB_RANKING_INPUTS_PATH)

print("\nCell 11 completed successfully.")

In [ ]:
# ============================================================
# CELL 12 — SAVED NEURAL-RANKING SCHEMA AND CONFIDENCE AUDIT
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"
CACHE_DIR = EXPERIMENT_DIR / "cache"

RANKING_PATH = (
    RESULTS_DIR
    / "taskB_development_pubmedbert_rankings.csv"
)

RANKING_SUMMARY_PATH = (
    RESULTS_DIR
    / "taskB_development_pubmedbert_ranking_summary.csv"
)

PROTOTYPE_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_training_concept_prototypes.npz"
)

EMBEDDING_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_train_development_embeddings.npz"
)

EMBEDDING_ROWS_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_train_development_embedding_rows.csv"
)

required_paths = [
    RANKING_PATH,
    RANKING_SUMMARY_PATH,
    PROTOTYPE_PATH,
    EMBEDDING_PATH,
    EMBEDDING_ROWS_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Cell 10/11 artifacts are missing:\n"
        + "\n".join(missing_paths)
    )

rankings = pd.read_csv(RANKING_PATH)
ranking_summary = pd.read_csv(RANKING_SUMMARY_PATH)
embedding_rows = pd.read_csv(EMBEDDING_ROWS_PATH)

with np.load(PROTOTYPE_PATH, allow_pickle=True) as prototype_file:
    prototype_keys = list(prototype_file.files)
    prototype_shapes = {
        key: np.asarray(prototype_file[key]).shape
        for key in prototype_keys
    }

with np.load(EMBEDDING_PATH, allow_pickle=True) as embedding_file:
    embedding_keys = list(embedding_file.files)
    embedding_shapes = {
        key: np.asarray(embedding_file[key]).shape
        for key in embedding_keys
    }

print("=" * 78)
print("SAVED NEURAL-RANKING ARTIFACT AUDIT")
print("=" * 78)

print("\nRanking rows:", len(rankings))
print("Ranking columns:")
for column in rankings.columns:
    print("  ", column)

print("\nRanking summary:")
display(ranking_summary)

print("\nFirst five ranking records:")
display(rankings.head())

print("\nEmbedding-row columns:")
for column in embedding_rows.columns:
    print("  ", column)

print("\nEmbedding rows by split:")
if "split" in embedding_rows.columns:
    print(
        embedding_rows["split"]
        .value_counts(dropna=False)
        .sort_index()
    )
else:
    print("No 'split' column found.")

print("\nPrototype NPZ arrays:")
for key, shape in prototype_shapes.items():
    print(f"  {key}: {shape}")

print("\nEmbedding NPZ arrays:")
for key, shape in embedding_shapes.items():
    print(f"  {key}: {shape}")

# Identify likely similarity/confidence and prediction fields.
similarity_columns = [
    column for column in rankings.columns
    if any(
        token in column.lower()
        for token in [
            "similarity",
            "cosine",
            "score",
            "margin",
            "probability",
            "confidence",
        ]
    )
]

prediction_columns = [
    column for column in rankings.columns
    if any(
        token in column.lower()
        for token in [
            "top1",
            "prediction",
            "predicted",
            "concept",
            "label",
        ]
    )
]

rank_columns = [
    column for column in rankings.columns
    if "rank" in column.lower()
]

print("\nCandidate confidence/similarity columns:")
print(similarity_columns)

print("\nCandidate prediction/concept columns:")
print(prediction_columns)

print("\nCandidate rank columns:")
print(rank_columns)

print("\nNumeric-column distributions:")
numeric_columns = rankings.select_dtypes(
    include=[np.number]
).columns.tolist()

if numeric_columns:
    display(
        rankings[numeric_columns]
        .describe(
            percentiles=[
                0.01,
                0.05,
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
        .T
    )

audit = {
    "ranking_rows": int(len(rankings)),
    "ranking_columns": rankings.columns.tolist(),
    "similarity_columns": similarity_columns,
    "prediction_columns": prediction_columns,
    "rank_columns": rank_columns,
    "prototype_arrays": {
        key: list(shape)
        for key, shape in prototype_shapes.items()
    },
    "embedding_arrays": {
        key: list(shape)
        for key, shape in embedding_shapes.items()
    },
}

AUDIT_PATH = (
    RESULTS_DIR
    / "taskB_development_pubmedbert_ranking_schema_audit.json"
)

with open(AUDIT_PATH, "w", encoding="utf-8") as output_file:
    json.dump(
        audit,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved schema audit to:")
print(AUDIT_PATH)

print("\nCell 12 completed successfully.")

In [ ]:
# ============================================================
# CELL 13 — DEVELOPMENT-ONLY NEURAL RETRIEVAL TUNING
# ============================================================

from pathlib import Path
import ast
import json
import re
import unicodedata

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Paths and resources
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"
PREDICTIONS_DIR = EXPERIMENT_DIR / "predictions"
CACHE_DIR = EXPERIMENT_DIR / "cache"

PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RANKING_PATH = (
    RESULTS_DIR
    / "taskB_development_pubmedbert_rankings.csv"
)

EMBEDDING_ROWS_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_train_development_embedding_rows.csv"
)

rankings = pd.read_csv(RANKING_PATH)
embedding_rows = pd.read_csv(EMBEDDING_ROWS_PATH)


# ------------------------------------------------------------
# 2. Normalization and serialized-list parsing
# ------------------------------------------------------------

def normalize_label(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip()

    # Remove the angle brackets used in some Task B files.
    if value.startswith("<") and value.endswith(">"):
        value = value[1:-1].strip()

    value = value.casefold()
    value = re.sub(r"\s+", " ", value)

    return value


def parse_pipe_values(value):
    if pd.isna(value):
        return []

    if isinstance(value, (list, tuple, np.ndarray)):
        return [
            str(item).strip()
            for item in value
            if str(item).strip()
        ]

    text = str(value).strip()

    if not text:
        return []

    # Accommodate a possible Python-list serialization.
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)

            if isinstance(parsed, (list, tuple)):
                return [
                    str(item).strip()
                    for item in parsed
                    if str(item).strip()
                ]
        except Exception:
            pass

    return [
        item.strip()
        for item in text.split("|")
        if item.strip()
    ]


# ------------------------------------------------------------
# 3. Select mention-only rankings and attach gold labels
# ------------------------------------------------------------

neural = (
    rankings.loc[
        rankings["representation"].eq("MENTION_ONLY")
    ]
    .copy()
    .reset_index(drop=True)
)

development_rows = (
    embedding_rows.loc[
        embedding_rows["split"].eq("development")
    ]
    .copy()
)

if neural["mention_input_id"].duplicated().any():
    raise ValueError(
        "Mention-only ranking table contains duplicate "
        "mention_input_id values."
    )

if development_rows["mention_input_id"].duplicated().any():
    raise ValueError(
        "Development embedding-row table contains duplicate "
        "mention_input_id values."
    )

neural = neural.merge(
    development_rows[
        [
            "mention_input_id",
            "gold_concept_labels",
            "gold_concept_count",
        ]
    ],
    on="mention_input_id",
    how="left",
    validate="one_to_one",
)

if neural["gold_concept_labels"].isna().any():
    missing = neural.loc[
        neural["gold_concept_labels"].isna(),
        "mention_input_id",
    ].tolist()

    raise ValueError(
        "Gold labels could not be attached to: "
        + ", ".join(missing[:10])
    )


# ------------------------------------------------------------
# 4. Recover top-1 and runner-up confidence
# ------------------------------------------------------------

def extract_similarity_list(value):
    values = parse_pipe_values(value)

    output = []

    for item in values:
        try:
            output.append(float(item))
        except (TypeError, ValueError):
            continue

    return output


neural["top10_similarity_values"] = (
    neural["top10_similarities"]
    .apply(extract_similarity_list)
)

neural["runner_up_similarity"] = (
    neural["top10_similarity_values"]
    .apply(
        lambda values:
        values[1] if len(values) >= 2 else np.nan
    )
)

neural["similarity_margin"] = (
    neural["top1_similarity"]
    - neural["runner_up_similarity"]
)

neural["gold_labels"] = (
    neural["gold_concept_labels"]
    .apply(parse_pipe_values)
    .apply(
        lambda labels: {
            normalize_label(label)
            for label in labels
            if normalize_label(label)
        }
    )
)

neural["top1_label"] = (
    neural["top1_concept"]
    .apply(normalize_label)
)

neural["top1_correct"] = neural.apply(
    lambda row:
    row["top1_label"] in row["gold_labels"],
    axis=1,
)

neural["has_seen_gold"] = (
    neural["gold_seen_count"] > 0
)

neural["gold_visibility"] = np.select(
    [
        (neural["gold_seen_count"] > 0)
        & (neural["gold_unseen_count"] == 0),

        (neural["gold_seen_count"] > 0)
        & (neural["gold_unseen_count"] > 0),
    ],
    [
        "all_seen",
        "mixed",
    ],
    default="all_unseen",
)


# ------------------------------------------------------------
# 5. Confidence-discrimination audit
# ------------------------------------------------------------

correctness = (
    neural["top1_correct"]
    .astype(int)
    .to_numpy()
)

confidence_diagnostics = []

for confidence_name in [
    "top1_similarity",
    "similarity_margin",
]:
    scores = (
        neural[confidence_name]
        .fillna(-np.inf)
        .to_numpy()
    )

    if len(np.unique(correctness)) == 2:
        auc = roc_auc_score(
            correctness,
            scores,
        )

        ap = average_precision_score(
            correctness,
            scores,
        )
    else:
        auc = np.nan
        ap = np.nan

    confidence_diagnostics.append(
        {
            "confidence_measure": confidence_name,
            "roc_auc": auc,
            "average_precision": ap,
            "correct_median": neural.loc[
                neural["top1_correct"],
                confidence_name,
            ].median(),
            "incorrect_median": neural.loc[
                ~neural["top1_correct"],
                confidence_name,
            ].median(),
        }
    )

confidence_diagnostics = pd.DataFrame(
    confidence_diagnostics
)


# ------------------------------------------------------------
# 6. Link-level and exact-set evaluation
# ------------------------------------------------------------

def evaluate_predictions(frame, prediction_column):
    gold_links = set()
    predicted_links = set()

    exact_matches = 0
    jaccard_values = []

    for row in frame.itertuples(index=False):
        input_id = row.mention_input_id
        gold_set = row.gold_labels
        predicted_set = getattr(
            row,
            prediction_column,
        )

        gold_links.update(
            (input_id, concept)
            for concept in gold_set
        )

        predicted_links.update(
            (input_id, concept)
            for concept in predicted_set
        )

        exact_matches += int(
            predicted_set == gold_set
        )

        union = gold_set | predicted_set

        if union:
            jaccard_values.append(
                len(gold_set & predicted_set)
                / len(union)
            )
        else:
            jaccard_values.append(1.0)

    true_positive = len(
        gold_links & predicted_links
    )

    false_positive = len(
        predicted_links - gold_links
    )

    false_negative = len(
        gold_links - predicted_links
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive
        else 0.0
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    exact_set_accuracy = (
        exact_matches / len(frame)
        if len(frame)
        else 0.0
    )

    mean_jaccard = (
        float(np.mean(jaccard_values))
        if jaccard_values
        else 0.0
    )

    return {
        "inputs": int(len(frame)),
        "gold_links": int(len(gold_links)),
        "predicted_links": int(len(predicted_links)),
        "true_positive": int(true_positive),
        "false_positive": int(false_positive),
        "false_negative": int(false_negative),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_set_accuracy": exact_set_accuracy,
        "mean_jaccard": mean_jaccard,
    }


# ------------------------------------------------------------
# 7. Development-only threshold grid
# ------------------------------------------------------------

similarity_thresholds = [
    0.00,
    0.85,
    0.88,
    0.90,
    0.92,
    0.94,
    0.95,
    0.96,
    0.97,
    0.98,
    0.99,
    0.995,
]

margin_thresholds = [
    0.000,
    0.002,
    0.005,
    0.010,
    0.015,
    0.020,
    0.030,
    0.050,
]

tuning_records = []
prediction_cache = {}

for similarity_threshold in similarity_thresholds:
    for margin_threshold in margin_thresholds:
        candidate = neural.copy()

        candidate["accepted"] = (
            candidate["top1_similarity"]
            .ge(similarity_threshold)
            &
            candidate["similarity_margin"]
            .fillna(-np.inf)
            .ge(margin_threshold)
        )

        candidate["predicted_labels"] = (
            candidate.apply(
                lambda row:
                {row["top1_label"]}
                if row["accepted"]
                and row["top1_label"]
                else set(),
                axis=1,
            )
        )

        overall = evaluate_predictions(
            candidate,
            "predicted_labels",
        )

        domain_metrics = {}

        for domain, domain_frame in candidate.groupby(
            "domain",
            sort=True,
        ):
            domain_metrics[domain] = evaluate_predictions(
                domain_frame,
                "predicted_labels",
            )

        domain_f1_values = [
            metrics["f1"]
            for metrics in domain_metrics.values()
        ]

        macro_domain_f1 = float(
            np.mean(domain_f1_values)
        )

        # Primary selection emphasizes semantic-link F1;
        # exact-set accuracy distinguishes tied configurations.
        selection_score = (
            0.75 * overall["f1"]
            + 0.25 * overall["exact_set_accuracy"]
        )

        record = {
            "similarity_threshold": similarity_threshold,
            "margin_threshold": margin_threshold,
            "accepted_inputs": int(
                candidate["accepted"].sum()
            ),
            "coverage": float(
                candidate["accepted"].mean()
            ),
            "precision": overall["precision"],
            "recall": overall["recall"],
            "f1": overall["f1"],
            "exact_set_accuracy": (
                overall["exact_set_accuracy"]
            ),
            "mean_jaccard": overall["mean_jaccard"],
            "macro_domain_f1": macro_domain_f1,
            "selection_score": selection_score,
        }

        for domain, metrics in domain_metrics.items():
            prefix = (
                "MH"
                if domain == "mental_health"
                else "PA"
            )

            record[f"{prefix}_precision"] = (
                metrics["precision"]
            )
            record[f"{prefix}_recall"] = (
                metrics["recall"]
            )
            record[f"{prefix}_f1"] = metrics["f1"]
            record[f"{prefix}_exact_set_accuracy"] = (
                metrics["exact_set_accuracy"]
            )

        tuning_records.append(record)

        prediction_cache[
            (
                similarity_threshold,
                margin_threshold,
            )
        ] = candidate

tuning = pd.DataFrame(tuning_records)

tuning = tuning.sort_values(
    [
        "selection_score",
        "f1",
        "exact_set_accuracy",
        "precision",
        "coverage",
        "similarity_threshold",
        "margin_threshold",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        False,
        True,
        True,
    ],
).reset_index(drop=True)

selected = tuning.iloc[0].copy()

selected_key = (
    float(selected["similarity_threshold"]),
    float(selected["margin_threshold"]),
)

selected_predictions = (
    prediction_cache[selected_key]
    .copy()
)

selected_overall = evaluate_predictions(
    selected_predictions,
    "predicted_labels",
)


# ------------------------------------------------------------
# 8. Selected-model subgroup audit
# ------------------------------------------------------------

subgroup_records = []

for subgroup_name, subgroup_frame in [
    ("ALL", selected_predictions),
    *[
        (
            f"VISIBILITY_{visibility}",
            frame,
        )
        for visibility, frame
        in selected_predictions.groupby(
            "gold_visibility",
            sort=True,
        )
    ],
]:
    metrics = evaluate_predictions(
        subgroup_frame,
        "predicted_labels",
    )

    subgroup_records.append(
        {
            "subgroup": subgroup_name,
            **metrics,
        }
    )

subgroup_results = pd.DataFrame(
    subgroup_records
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("=" * 78)
print("MENTION-ONLY PROTOTYPE CONFIDENCE DIAGNOSTIC")
print("=" * 78)

print("\nTop-1 correctness:")
print(
    f"{selected_predictions['top1_correct'].sum()}"
    f"/{len(selected_predictions)} "
    f"({selected_predictions['top1_correct'].mean():.2%})"
)

print("\nConfidence discrimination:")
display(
    confidence_diagnostics.round(4)
)

print("\nConfidence distributions by correctness:")
display(
    neural.groupby("top1_correct")[
        [
            "top1_similarity",
            "similarity_margin",
        ]
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .round(4)
)

print("\n" + "=" * 78)
print("TOP 15 DEVELOPMENT CONFIGURATIONS")
print("=" * 78)

display(
    tuning.head(15).round(4)
)

print("\n" + "=" * 78)
print("SELECTED STANDALONE NEURAL CONFIGURATION")
print("=" * 78)

selected_columns = [
    "similarity_threshold",
    "margin_threshold",
    "accepted_inputs",
    "coverage",
    "precision",
    "recall",
    "f1",
    "exact_set_accuracy",
    "mean_jaccard",
    "macro_domain_f1",
    "selection_score",
]

print(
    selected[selected_columns]
    .to_string()
)

print("\nSelected-configuration subgroup results:")
display(
    subgroup_results.round(4)
)

print("\nRejected high-similarity examples:")
display(
    selected_predictions.loc[
        ~selected_predictions["accepted"]
        & selected_predictions["top1_similarity"].ge(0.95),
        [
            "mention_input_id",
            "domain",
            "mention",
            "top1_concept",
            "top1_similarity",
            "runner_up_similarity",
            "similarity_margin",
            "top1_correct",
            "gold_visibility",
            "gold_concept_labels",
        ],
    ]
    .sort_values(
        [
            "similarity_margin",
            "top1_similarity",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .head(20)
)


# ------------------------------------------------------------
# 10. Save development-only tuning artifacts
# ------------------------------------------------------------

TUNING_PATH = (
    RESULTS_DIR
    / "taskB_development_mention_only_neural_tuning.csv"
)

DIAGNOSTIC_PATH = (
    RESULTS_DIR
    / "taskB_development_mention_only_confidence_diagnostic.csv"
)

SUBGROUP_PATH = (
    RESULTS_DIR
    / "taskB_development_mention_only_neural_subgroups.csv"
)

SELECTED_PREDICTIONS_PATH = (
    PREDICTIONS_DIR
    / "taskB_development_mention_only_neural_selected.csv"
)

SELECTION_PATH = (
    RESULTS_DIR
    / "taskB_development_mention_only_neural_selection.json"
)

tuning.to_csv(
    TUNING_PATH,
    index=False,
)

confidence_diagnostics.to_csv(
    DIAGNOSTIC_PATH,
    index=False,
)

subgroup_results.to_csv(
    SUBGROUP_PATH,
    index=False,
)

selected_output = selected_predictions.copy()

selected_output["gold_labels"] = (
    selected_output["gold_labels"]
    .apply(
        lambda values:
        " | ".join(sorted(values))
    )
)

selected_output["predicted_labels"] = (
    selected_output["predicted_labels"]
    .apply(
        lambda values:
        " | ".join(sorted(values))
    )
)

selected_output = selected_output.drop(
    columns=[
        "top10_similarity_values",
    ],
)

selected_output.to_csv(
    SELECTED_PREDICTIONS_PATH,
    index=False,
)

selection_metadata = {
    "representation": "MENTION_ONLY",
    "selection_split": "development",
    "similarity_threshold": float(
        selected["similarity_threshold"]
    ),
    "margin_threshold": float(
        selected["margin_threshold"]
    ),
    "selection_score_formula": (
        "0.75 * link_micro_f1 + "
        "0.25 * exact_set_accuracy"
    ),
    "selected_metrics": {
        key: (
            int(value)
            if isinstance(
                value,
                (int, np.integer),
            )
            else float(value)
        )
        for key, value in selected_overall.items()
    },
    "contextual_representation_retained_as_ablation_only": True,
}

with open(
    SELECTION_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        selection_metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved complete tuning table to:")
print(TUNING_PATH)

print("\nSaved confidence diagnostic to:")
print(DIAGNOSTIC_PATH)

print("\nSaved subgroup results to:")
print(SUBGROUP_PATH)

print("\nSaved selected predictions to:")
print(SELECTED_PREDICTIONS_PATH)

print("\nSaved selected configuration to:")
print(SELECTION_PATH)

print("\nCell 13 completed successfully.")

In [ ]:
# ============================================================
# CELL 14 — COMPOSITIONAL + NEURAL HYBRID TUNING
# ============================================================

from pathlib import Path
import ast
import json
import re
import unicodedata

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"
PREDICTIONS_DIR = EXPERIMENT_DIR / "predictions"

COMPOSITIONAL_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_predictions.csv"
)

NEURAL_RANKING_PATH = (
    RESULTS_DIR
    / "taskB_development_pubmedbert_rankings.csv"
)

EMBEDDING_ROWS_PATH = (
    EXPERIMENT_DIR
    / "cache"
    / "taskB_pubmedbert_train_development_embedding_rows.csv"
)

for path in [
    COMPOSITIONAL_PATH,
    NEURAL_RANKING_PATH,
    EMBEDDING_ROWS_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)

compositional = pd.read_csv(COMPOSITIONAL_PATH)
rankings = pd.read_csv(NEURAL_RANKING_PATH)
embedding_rows = pd.read_csv(EMBEDDING_ROWS_PATH)


# ------------------------------------------------------------
# 2. Normalization and parsing
# ------------------------------------------------------------

def normalize_label(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip()

    if value.startswith("<") and value.endswith(">"):
        value = value[1:-1].strip()

    value = value.casefold()
    value = re.sub(r"\s+", " ", value)

    return value


def parse_label_set(value):
    if value is None:
        return set()

    if isinstance(value, float) and np.isnan(value):
        return set()

    if isinstance(value, (list, tuple, set, np.ndarray)):
        return {
            normalize_label(item)
            for item in value
            if normalize_label(item)
        }

    text = str(value).strip()

    if not text or text.casefold() in {
        "nan",
        "none",
        "set()",
        "()",
        "[]",
        "{}",
    }:
        return set()

    # DETECH labels written as <Concept>.
    angle_labels = re.findall(
        r"<([^<>]+)>",
        text,
    )

    if angle_labels:
        return {
            normalize_label(item)
            for item in angle_labels
            if normalize_label(item)
        }

    # Python list/tuple/set serializations.
    if (
        text[0] in "[({"
        and text[-1] in "])}"
    ):
        try:
            parsed = ast.literal_eval(text)

            if isinstance(
                parsed,
                (list, tuple, set),
            ):
                return {
                    normalize_label(item)
                    for item in parsed
                    if normalize_label(item)
                }

            if isinstance(parsed, str):
                normalized = normalize_label(parsed)
                return {normalized} if normalized else set()

        except Exception:
            # Some displays use unquoted values:
            # (gut microbiota, human)
            interior = text[1:-1].strip()

            if not interior:
                return set()

            pieces = [
                piece.strip().strip("'\"")
                for piece in interior.split(",")
                if piece.strip()
            ]

            return {
                normalize_label(piece)
                for piece in pieces
                if normalize_label(piece)
            }

    if "|" in text:
        return {
            normalize_label(item)
            for item in text.split("|")
            if normalize_label(item)
        }

    normalized = normalize_label(text)
    return {normalized} if normalized else set()


def parse_similarity_list(value):
    if pd.isna(value):
        return []

    output = []

    for item in str(value).split("|"):
        try:
            output.append(float(item.strip()))
        except (TypeError, ValueError):
            continue

    return output


# ------------------------------------------------------------
# 3. Identify the selected compositional prediction column
# ------------------------------------------------------------

preferred_prediction_columns = [
    "selected_compositional_prediction",
    "compositional_prediction",
    "predicted_compositional",
    "predicted_labels",
    "prediction",
]

compositional_prediction_column = next(
    (
        column
        for column in preferred_prediction_columns
        if column in compositional.columns
    ),
    None,
)

if compositional_prediction_column is None:
    candidates = [
        column
        for column in compositional.columns
        if (
            "compositional" in column.casefold()
            and "predict" in column.casefold()
        )
    ]

    if len(candidates) == 1:
        compositional_prediction_column = candidates[0]
    else:
        raise ValueError(
            "Could not identify the compositional prediction "
            f"column. Available columns: "
            f"{compositional.columns.tolist()}"
        )

print("=" * 78)
print("HYBRID INPUT AUDIT")
print("=" * 78)

print(
    "Compositional prediction column:",
    compositional_prediction_column,
)

print(
    "Compositional rows:",
    len(compositional),
)


# ------------------------------------------------------------
# 4. Build authoritative development table
# ------------------------------------------------------------

development_rows = (
    embedding_rows.loc[
        embedding_rows["split"].eq("development")
    ]
    .copy()
)

mention_only = (
    rankings.loc[
        rankings["representation"].eq("MENTION_ONLY")
    ]
    .copy()
)

mention_only["similarity_values"] = (
    mention_only["top10_similarities"]
    .apply(parse_similarity_list)
)

mention_only["runner_up_similarity"] = (
    mention_only["similarity_values"]
    .apply(
        lambda values:
        values[1] if len(values) >= 2 else np.nan
    )
)

mention_only["similarity_margin"] = (
    mention_only["top1_similarity"]
    - mention_only["runner_up_similarity"]
)

mention_only["neural_label"] = (
    mention_only["top1_concept"]
    .apply(normalize_label)
)


# ------------------------------------------------------------
# 5. Merge using mention_input_id when available
# ------------------------------------------------------------

if (
    "mention_input_id" in compositional.columns
    and "mention_input_id" in mention_only.columns
):
    hybrid = compositional.merge(
        mention_only[
            [
                "mention_input_id",
                "top1_concept",
                "top1_similarity",
                "runner_up_similarity",
                "similarity_margin",
                "neural_label",
            ]
        ],
        on="mention_input_id",
        how="left",
        validate="one_to_one",
    )

    hybrid = hybrid.merge(
        development_rows[
            [
                "mention_input_id",
                "gold_concept_labels",
                "gold_concept_count",
            ]
        ],
        on="mention_input_id",
        how="left",
        validate="one_to_one",
        suffixes=("", "_authoritative"),
    )

else:
    merge_columns = [
        "domain",
        "num_article",
        "mention",
    ]

    missing_merge_columns = [
        column
        for column in merge_columns
        if (
            column not in compositional.columns
            or column not in mention_only.columns
        )
    ]

    if missing_merge_columns:
        raise ValueError(
            "Cannot merge compositional and neural tables. "
            f"Missing columns: {missing_merge_columns}"
        )

    hybrid = compositional.merge(
        mention_only[
            merge_columns
            + [
                "mention_input_id",
                "top1_concept",
                "top1_similarity",
                "runner_up_similarity",
                "similarity_margin",
                "neural_label",
            ]
        ],
        on=merge_columns,
        how="left",
        validate="one_to_one",
    )

    hybrid = hybrid.merge(
        development_rows[
            [
                "mention_input_id",
                "gold_concept_labels",
                "gold_concept_count",
            ]
        ],
        on="mention_input_id",
        how="left",
        validate="one_to_one",
        suffixes=("", "_authoritative"),
    )

if len(hybrid) != 102:
    raise ValueError(
        f"Expected 102 development inputs; found {len(hybrid)}."
    )

if hybrid["top1_similarity"].isna().any():
    raise ValueError(
        "Some development inputs lack neural rankings."
    )


# ------------------------------------------------------------
# 6. Parse gold and compositional prediction sets
# ------------------------------------------------------------

gold_column = (
    "gold_concept_labels_authoritative"
    if "gold_concept_labels_authoritative"
    in hybrid.columns
    else "gold_concept_labels"
)

hybrid["gold_set"] = (
    hybrid[gold_column]
    .apply(parse_label_set)
)

hybrid["compositional_set"] = (
    hybrid[compositional_prediction_column]
    .apply(parse_label_set)
)

if hybrid["gold_set"].apply(len).sum() != 111:
    raise ValueError(
        "Parsed gold-link count is not 111. "
        "Inspect the gold-label serialization."
    )

print(
    "Parsed gold links:",
    hybrid["gold_set"].apply(len).sum(),
)

print(
    "Parsed compositional links:",
    hybrid["compositional_set"].apply(len).sum(),
)

print(
    "Inputs with empty compositional prediction:",
    hybrid["compositional_set"].apply(
        lambda values: len(values) == 0
    ).sum(),
)


# ------------------------------------------------------------
# 7. Evaluation function
# ------------------------------------------------------------

def evaluate_prediction_sets(
    frame,
    prediction_column,
):
    gold_links = set()
    predicted_links = set()

    exact_matches = 0
    jaccard_values = []

    for row in frame.itertuples(index=False):
        input_id = row.mention_input_id
        gold = row.gold_set
        predicted = getattr(
            row,
            prediction_column,
        )

        gold_links.update(
            (input_id, label)
            for label in gold
        )

        predicted_links.update(
            (input_id, label)
            for label in predicted
        )

        exact_matches += int(
            predicted == gold
        )

        union = gold | predicted

        jaccard_values.append(
            (
                len(gold & predicted) / len(union)
                if union
                else 1.0
            )
        )

    true_positive = len(
        gold_links & predicted_links
    )
    false_positive = len(
        predicted_links - gold_links
    )
    false_negative = len(
        gold_links - predicted_links
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive
        else 0.0
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "mention_inputs": int(len(frame)),
        "gold_links": int(len(gold_links)),
        "predicted_links": int(len(predicted_links)),
        "true_positive_links": int(true_positive),
        "false_positive_links": int(false_positive),
        "false_negative_links": int(false_negative),
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": f1,
        "exact_set_accuracy": (
            exact_matches / len(frame)
            if len(frame)
            else 0.0
        ),
        "mean_jaccard": float(
            np.mean(jaccard_values)
        ),
    }


# ------------------------------------------------------------
# 8. Baseline verification
# ------------------------------------------------------------

baseline_metrics = evaluate_prediction_sets(
    hybrid,
    "compositional_set",
)

print("\nCompositional baseline verification:")
for key, value in baseline_metrics.items():
    print(f"{key}: {value}")

if not np.isclose(
    baseline_metrics["micro_f1"],
    0.8372,
    atol=0.001,
):
    raise ValueError(
        "Reconstructed compositional F1 does not match "
        "Cell 9. Stop before hybrid tuning."
    )


# ------------------------------------------------------------
# 9. Hybrid policies
# ------------------------------------------------------------

def apply_hybrid_policy(
    compositional_set,
    neural_label,
    accepted,
    policy,
):
    compositional_set = set(
        compositional_set
    )

    if (
        not accepted
        or not neural_label
    ):
        return compositional_set

    neural_set = {neural_label}

    if policy == "UNION":
        return (
            compositional_set
            | neural_set
        )

    if policy == "REPLACE_SINGLETON":
        if len(compositional_set) <= 1:
            return neural_set

        return compositional_set

    if policy == "REPLACE_ALL":
        return neural_set

    raise ValueError(
        f"Unknown hybrid policy: {policy}"
    )


similarity_thresholds = [
    0.94,
    0.95,
    0.96,
    0.97,
    0.98,
    0.99,
    0.995,
]

margin_thresholds = [
    0.000,
    0.002,
    0.005,
    0.010,
    0.015,
    0.020,
    0.030,
    0.050,
]

policies = [
    "UNION",
    "REPLACE_SINGLETON",
    "REPLACE_ALL",
]

tuning_records = []
prediction_cache = {}

# Include the unchanged baseline explicitly.
baseline_record = {
    "policy": "COMPOSITIONAL_BASELINE",
    "similarity_threshold": np.nan,
    "margin_threshold": np.nan,
    "accepted_neural_inputs": 0,
    "changed_inputs": 0,
    **baseline_metrics,
}

baseline_record["selection_score"] = (
    0.75 * baseline_record["micro_f1"]
    + 0.25 * baseline_record["exact_set_accuracy"]
)

tuning_records.append(
    baseline_record
)

for policy in policies:
    for similarity_threshold in similarity_thresholds:
        for margin_threshold in margin_thresholds:
            candidate = hybrid.copy()

            candidate["neural_accepted"] = (
                candidate["top1_similarity"]
                .ge(similarity_threshold)
                &
                candidate["similarity_margin"]
                .fillna(-np.inf)
                .ge(margin_threshold)
            )

            candidate["hybrid_set"] = (
                candidate.apply(
                    lambda row:
                    apply_hybrid_policy(
                        row["compositional_set"],
                        row["neural_label"],
                        row["neural_accepted"],
                        policy,
                    ),
                    axis=1,
                )
            )

            candidate["prediction_changed"] = (
                candidate["hybrid_set"]
                != candidate["compositional_set"]
            )

            overall = evaluate_prediction_sets(
                candidate,
                "hybrid_set",
            )

            domain_metrics = {}

            for domain, domain_frame in candidate.groupby(
                "domain",
                sort=True,
            ):
                domain_metrics[domain] = (
                    evaluate_prediction_sets(
                        domain_frame,
                        "hybrid_set",
                    )
                )

            selection_score = (
                0.75 * overall["micro_f1"]
                + 0.25
                * overall["exact_set_accuracy"]
            )

            record = {
                "policy": policy,
                "similarity_threshold": (
                    similarity_threshold
                ),
                "margin_threshold": (
                    margin_threshold
                ),
                "accepted_neural_inputs": int(
                    candidate[
                        "neural_accepted"
                    ].sum()
                ),
                "changed_inputs": int(
                    candidate[
                        "prediction_changed"
                    ].sum()
                ),
                **overall,
                "selection_score": selection_score,
            }

            for domain, metrics in domain_metrics.items():
                prefix = (
                    "MH"
                    if domain == "mental_health"
                    else "PA"
                )

                record[
                    f"{prefix}_micro_f1"
                ] = metrics["micro_f1"]

                record[
                    f"{prefix}_exact_set_accuracy"
                ] = metrics[
                    "exact_set_accuracy"
                ]

            tuning_records.append(record)

            prediction_cache[
                (
                    policy,
                    similarity_threshold,
                    margin_threshold,
                )
            ] = candidate


# ------------------------------------------------------------
# 10. Rank configurations
# ------------------------------------------------------------

tuning = pd.DataFrame(
    tuning_records
)

tuning["f1_difference_from_compositional"] = (
    tuning["micro_f1"]
    - baseline_metrics["micro_f1"]
)

tuning[
    "exact_accuracy_difference_from_compositional"
] = (
    tuning["exact_set_accuracy"]
    - baseline_metrics["exact_set_accuracy"]
)

tuning[
    "selection_difference_from_compositional"
] = (
    tuning["selection_score"]
    - baseline_record["selection_score"]
)

tuning = tuning.sort_values(
    [
        "selection_score",
        "micro_f1",
        "exact_set_accuracy",
        "micro_precision",
        "changed_inputs",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        True,
    ],
).reset_index(drop=True)

selected = tuning.iloc[0].copy()

if selected["policy"] == "COMPOSITIONAL_BASELINE":
    selected_predictions = hybrid.copy()
    selected_predictions["neural_accepted"] = False
    selected_predictions["hybrid_set"] = (
        selected_predictions["compositional_set"]
    )
    selected_predictions["prediction_changed"] = False
else:
    selected_key = (
        selected["policy"],
        float(selected["similarity_threshold"]),
        float(selected["margin_threshold"]),
    )

    selected_predictions = (
        prediction_cache[selected_key]
        .copy()
    )


# ------------------------------------------------------------
# 11. Changed-case audit
# ------------------------------------------------------------

changed = selected_predictions.loc[
    selected_predictions["prediction_changed"]
].copy()

changed["compositional_correct_links"] = (
    changed.apply(
        lambda row:
        len(
            row["compositional_set"]
            & row["gold_set"]
        ),
        axis=1,
    )
)

changed["hybrid_correct_links"] = (
    changed.apply(
        lambda row:
        len(
            row["hybrid_set"]
            & row["gold_set"]
        ),
        axis=1,
    )
)

changed["correct_link_change"] = (
    changed["hybrid_correct_links"]
    - changed["compositional_correct_links"]
)

changed["compositional_exact"] = (
    changed["compositional_set"]
    == changed["gold_set"]
)

changed["hybrid_exact"] = (
    changed["hybrid_set"]
    == changed["gold_set"]
)


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("TOP 20 HYBRID CONFIGURATIONS")
print("=" * 78)

display(
    tuning.head(20).round(4)
)

print("\n" + "=" * 78)
print("BEST CONFIGURATION BY POLICY")
print("=" * 78)

best_by_policy = (
    tuning.sort_values(
        [
            "selection_score",
            "micro_f1",
            "exact_set_accuracy",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .groupby(
        "policy",
        as_index=False,
        sort=False,
    )
    .first()
)

display(
    best_by_policy[
        [
            "policy",
            "similarity_threshold",
            "margin_threshold",
            "accepted_neural_inputs",
            "changed_inputs",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
            "selection_score",
            "f1_difference_from_compositional",
            "selection_difference_from_compositional",
        ]
    ].round(4)
)

print("\n" + "=" * 78)
print("SELECTED DEVELOPMENT CONFIGURATION")
print("=" * 78)

selected_fields = [
    "policy",
    "similarity_threshold",
    "margin_threshold",
    "accepted_neural_inputs",
    "changed_inputs",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "exact_set_accuracy",
    "mean_jaccard",
    "selection_score",
    "f1_difference_from_compositional",
    "selection_difference_from_compositional",
]

print(
    selected[selected_fields]
    .to_string()
)

print("\nChanged-case summary:")
if len(changed):
    print(
        changed[
            [
                "correct_link_change",
                "compositional_exact",
                "hybrid_exact",
            ]
        ]
        .agg(
            {
                "correct_link_change": [
                    "count",
                    "sum",
                    "mean",
                ],
                "compositional_exact": "sum",
                "hybrid_exact": "sum",
            }
        )
    )

    changed_display = changed.copy()

    for column in [
        "gold_set",
        "compositional_set",
        "hybrid_set",
    ]:
        changed_display[column] = (
            changed_display[column]
            .apply(
                lambda values:
                " | ".join(sorted(values))
            )
        )

    display(
        changed_display[
            [
                "mention_input_id",
                "domain",
                "num_article",
                "mention",
                "gold_set",
                "compositional_set",
                "top1_concept",
                "top1_similarity",
                "similarity_margin",
                "hybrid_set",
                "correct_link_change",
                "compositional_exact",
                "hybrid_exact",
            ]
        ]
        .sort_values(
            [
                "correct_link_change",
                "top1_similarity",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )
else:
    print("No predictions changed.")


# ------------------------------------------------------------
# 13. Save artifacts
# ------------------------------------------------------------

TUNING_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_neural_hybrid_tuning.csv"
)

SELECTION_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_neural_hybrid_selection.json"
)

SELECTED_PREDICTIONS_PATH = (
    PREDICTIONS_DIR
    / "taskB_development_compositional_neural_hybrid_selected.csv"
)

CHANGED_CASES_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_neural_hybrid_changed_cases.csv"
)

tuning.to_csv(
    TUNING_PATH,
    index=False,
)

selected_output = selected_predictions.copy()

for column in [
    "gold_set",
    "compositional_set",
    "hybrid_set",
]:
    selected_output[column] = (
        selected_output[column]
        .apply(
            lambda values:
            " | ".join(sorted(values))
        )
    )

selected_output = selected_output.drop(
    columns=[
        "similarity_values",
    ],
    errors="ignore",
)

selected_output.to_csv(
    SELECTED_PREDICTIONS_PATH,
    index=False,
)

changed_output = changed.copy()

for column in [
    "gold_set",
    "compositional_set",
    "hybrid_set",
]:
    if column in changed_output.columns:
        changed_output[column] = (
            changed_output[column]
            .apply(
                lambda values:
                " | ".join(sorted(values))
            )
        )

changed_output = changed_output.drop(
    columns=[
        "similarity_values",
    ],
    errors="ignore",
)

changed_output.to_csv(
    CHANGED_CASES_PATH,
    index=False,
)

selection_metadata = {
    "selection_split": "development",
    "compositional_minimum_atomic_characters": 5,
    "selected_policy": selected["policy"],
    "similarity_threshold": (
        None
        if pd.isna(
            selected["similarity_threshold"]
        )
        else float(
            selected["similarity_threshold"]
        )
    ),
    "margin_threshold": (
        None
        if pd.isna(
            selected["margin_threshold"]
        )
        else float(
            selected["margin_threshold"]
        )
    ),
    "selection_score_formula": (
        "0.75 * link_micro_f1 + "
        "0.25 * exact_set_accuracy"
    ),
    "development_metrics": {
        key: (
            int(selected[key])
            if key in {
                "accepted_neural_inputs",
                "changed_inputs",
                "mention_inputs",
                "gold_links",
                "predicted_links",
            }
            else float(selected[key])
        )
        for key in [
            "accepted_neural_inputs",
            "changed_inputs",
            "mention_inputs",
            "gold_links",
            "predicted_links",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
            "selection_score",
            "f1_difference_from_compositional",
            "selection_difference_from_compositional",
        ]
    },
}

with open(
    SELECTION_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        selection_metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved complete hybrid tuning table to:")
print(TUNING_PATH)

print("\nSaved selected hybrid configuration to:")
print(SELECTION_PATH)

print("\nSaved selected predictions to:")
print(SELECTED_PREDICTIONS_PATH)

print("\nSaved changed-case audit to:")
print(CHANGED_CASES_PATH)

print("\nCell 14 completed successfully.")

In [ ]:
# ============================================================
# CELL 14A — REPAIR EMPTY CHANGED-CASE AUDIT AND SAVE RESULTS
# ============================================================

import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Reconstruct changed-case audit safely
# ------------------------------------------------------------

changed = selected_predictions.loc[
    selected_predictions["prediction_changed"]
].copy()

if len(changed) > 0:
    changed["compositional_correct_links"] = [
        len(compositional_set & gold_set)
        for compositional_set, gold_set
        in zip(
            changed["compositional_set"],
            changed["gold_set"],
        )
    ]

    changed["hybrid_correct_links"] = [
        len(hybrid_set & gold_set)
        for hybrid_set, gold_set
        in zip(
            changed["hybrid_set"],
            changed["gold_set"],
        )
    ]

    changed["correct_link_change"] = (
        changed["hybrid_correct_links"]
        - changed["compositional_correct_links"]
    )

    changed["compositional_exact"] = [
        compositional_set == gold_set
        for compositional_set, gold_set
        in zip(
            changed["compositional_set"],
            changed["gold_set"],
        )
    ]

    changed["hybrid_exact"] = [
        hybrid_set == gold_set
        for hybrid_set, gold_set
        in zip(
            changed["hybrid_set"],
            changed["gold_set"],
        )
    ]

else:
    changed["compositional_correct_links"] = pd.Series(
        dtype="int64"
    )
    changed["hybrid_correct_links"] = pd.Series(
        dtype="int64"
    )
    changed["correct_link_change"] = pd.Series(
        dtype="int64"
    )
    changed["compositional_exact"] = pd.Series(
        dtype="bool"
    )
    changed["hybrid_exact"] = pd.Series(
        dtype="bool"
    )


# ------------------------------------------------------------
# 2. Report tuning results
# ------------------------------------------------------------

print("=" * 78)
print("TOP 20 HYBRID CONFIGURATIONS")
print("=" * 78)

display(
    tuning.head(20).round(4)
)

print("\n" + "=" * 78)
print("BEST CONFIGURATION BY POLICY")
print("=" * 78)

best_by_policy = (
    tuning.sort_values(
        [
            "selection_score",
            "micro_f1",
            "exact_set_accuracy",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .groupby(
        "policy",
        as_index=False,
        sort=False,
    )
    .first()
)

display(
    best_by_policy[
        [
            "policy",
            "similarity_threshold",
            "margin_threshold",
            "accepted_neural_inputs",
            "changed_inputs",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
            "selection_score",
            "f1_difference_from_compositional",
            "selection_difference_from_compositional",
        ]
    ].round(4)
)

print("\n" + "=" * 78)
print("SELECTED DEVELOPMENT CONFIGURATION")
print("=" * 78)

selected_fields = [
    "policy",
    "similarity_threshold",
    "margin_threshold",
    "accepted_neural_inputs",
    "changed_inputs",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "exact_set_accuracy",
    "mean_jaccard",
    "selection_score",
    "f1_difference_from_compositional",
    "selection_difference_from_compositional",
]

print(
    selected[selected_fields]
    .to_string()
)

print("\nChanged-case summary:")

if len(changed) > 0:
    print("Changed inputs:", len(changed))
    print(
        "Net change in correct links:",
        int(changed["correct_link_change"].sum()),
    )
    print(
        "Improved inputs:",
        int(
            (
                changed["correct_link_change"] > 0
            ).sum()
        ),
    )
    print(
        "Worsened inputs:",
        int(
            (
                changed["correct_link_change"] < 0
            ).sum()
        ),
    )
    print(
        "Neutral inputs:",
        int(
            (
                changed["correct_link_change"] == 0
            ).sum()
        ),
    )
    print(
        "Exact compositional predictions:",
        int(changed["compositional_exact"].sum()),
    )
    print(
        "Exact hybrid predictions:",
        int(changed["hybrid_exact"].sum()),
    )

    changed_display = changed.copy()

    for column in [
        "gold_set",
        "compositional_set",
        "hybrid_set",
    ]:
        changed_display[column] = (
            changed_display[column]
            .apply(
                lambda values:
                " | ".join(sorted(values))
            )
        )

    display(
        changed_display[
            [
                "mention_input_id",
                "domain",
                "num_article",
                "mention",
                "gold_set",
                "compositional_set",
                "top1_concept",
                "top1_similarity",
                "similarity_margin",
                "hybrid_set",
                "correct_link_change",
                "compositional_exact",
                "hybrid_exact",
            ]
        ]
        .sort_values(
            [
                "correct_link_change",
                "top1_similarity",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )

else:
    print(
        "No predictions changed under the selected "
        "configuration."
    )


# ------------------------------------------------------------
# 3. Save artifacts
# ------------------------------------------------------------

TUNING_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_neural_hybrid_tuning.csv"
)

SELECTION_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_neural_hybrid_selection.json"
)

SELECTED_PREDICTIONS_PATH = (
    PREDICTIONS_DIR
    / "taskB_development_compositional_neural_hybrid_selected.csv"
)

CHANGED_CASES_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_neural_hybrid_changed_cases.csv"
)

tuning.to_csv(
    TUNING_PATH,
    index=False,
)

selected_output = selected_predictions.copy()

for column in [
    "gold_set",
    "compositional_set",
    "hybrid_set",
]:
    selected_output[column] = (
        selected_output[column]
        .apply(
            lambda values:
            " | ".join(sorted(values))
        )
    )

selected_output = selected_output.drop(
    columns=["similarity_values"],
    errors="ignore",
)

selected_output.to_csv(
    SELECTED_PREDICTIONS_PATH,
    index=False,
)

changed_output = changed.copy()

for column in [
    "gold_set",
    "compositional_set",
    "hybrid_set",
]:
    if column in changed_output.columns:
        changed_output[column] = (
            changed_output[column]
            .apply(
                lambda values:
                " | ".join(sorted(values))
            )
        )

changed_output = changed_output.drop(
    columns=["similarity_values"],
    errors="ignore",
)

changed_output.to_csv(
    CHANGED_CASES_PATH,
    index=False,
)


# ------------------------------------------------------------
# 4. Save selected configuration metadata
# ------------------------------------------------------------

def safe_optional_float(value):
    if pd.isna(value):
        return None

    return float(value)


selection_metadata = {
    "selection_split": "development",
    "compositional_minimum_atomic_characters": 5,
    "selected_policy": str(
        selected["policy"]
    ),
    "similarity_threshold": safe_optional_float(
        selected["similarity_threshold"]
    ),
    "margin_threshold": safe_optional_float(
        selected["margin_threshold"]
    ),
    "selection_score_formula": (
        "0.75 * link_micro_f1 + "
        "0.25 * exact_set_accuracy"
    ),
    "development_metrics": {
        "accepted_neural_inputs": int(
            selected["accepted_neural_inputs"]
        ),
        "changed_inputs": int(
            selected["changed_inputs"]
        ),
        "mention_inputs": int(
            selected["mention_inputs"]
        ),
        "gold_links": int(
            selected["gold_links"]
        ),
        "predicted_links": int(
            selected["predicted_links"]
        ),
        "micro_precision": float(
            selected["micro_precision"]
        ),
        "micro_recall": float(
            selected["micro_recall"]
        ),
        "micro_f1": float(
            selected["micro_f1"]
        ),
        "exact_set_accuracy": float(
            selected["exact_set_accuracy"]
        ),
        "mean_jaccard": float(
            selected["mean_jaccard"]
        ),
        "selection_score": float(
            selected["selection_score"]
        ),
        "f1_difference_from_compositional": float(
            selected[
                "f1_difference_from_compositional"
            ]
        ),
        "selection_difference_from_compositional": float(
            selected[
                "selection_difference_from_compositional"
            ]
        ),
    },
}

with open(
    SELECTION_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        selection_metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved complete hybrid tuning table to:")
print(TUNING_PATH)

print("\nSaved selected configuration to:")
print(SELECTION_PATH)

print("\nSaved selected predictions to:")
print(SELECTED_PREDICTIONS_PATH)

print("\nSaved changed-case audit to:")
print(CHANGED_CASES_PATH)

print("\nCell 14A completed successfully.")

In [ ]:
# ============================================================
# CELL 14B — ACTIVE HYBRID COMPLEMENTARITY AUDIT
# ============================================================

import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Find configurations that actually changed predictions
# ------------------------------------------------------------

active_tuning = (
    tuning.loc[
        tuning["changed_inputs"] > 0
    ]
    .copy()
    .sort_values(
        [
            "selection_score",
            "micro_f1",
            "exact_set_accuracy",
            "micro_precision",
            "changed_inputs",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print("=" * 78)
print("ACTIVE HYBRID COMPLEMENTARITY AUDIT")
print("=" * 78)

print(
    "Total hybrid configurations:",
    int(
        tuning["policy"]
        .ne("COMPOSITIONAL_BASELINE")
        .sum()
    ),
)

print(
    "Configurations changing at least one prediction:",
    len(active_tuning),
)

if active_tuning.empty:
    print(
        "\nNo tested hybrid configuration changed any "
        "compositional prediction."
    )

else:
    print("\nTop 20 configurations with actual changes:")

    display(
        active_tuning[
            [
                "policy",
                "similarity_threshold",
                "margin_threshold",
                "accepted_neural_inputs",
                "changed_inputs",
                "micro_precision",
                "micro_recall",
                "micro_f1",
                "exact_set_accuracy",
                "mean_jaccard",
                "selection_score",
                "f1_difference_from_compositional",
                "selection_difference_from_compositional",
            ]
        ]
        .head(20)
        .round(4)
    )

    print("\nBest active configuration by policy:")

    best_active_by_policy = (
        active_tuning
        .groupby(
            "policy",
            as_index=False,
            sort=False,
        )
        .first()
    )

    display(
        best_active_by_policy[
            [
                "policy",
                "similarity_threshold",
                "margin_threshold",
                "accepted_neural_inputs",
                "changed_inputs",
                "micro_precision",
                "micro_recall",
                "micro_f1",
                "exact_set_accuracy",
                "selection_score",
                "f1_difference_from_compositional",
                "selection_difference_from_compositional",
            ]
        ].round(4)
    )


# ------------------------------------------------------------
# 2. Inspect the best active configuration
# ------------------------------------------------------------

if not active_tuning.empty:
    best_active = active_tuning.iloc[0].copy()

    best_active_key = (
        str(best_active["policy"]),
        float(best_active["similarity_threshold"]),
        float(best_active["margin_threshold"]),
    )

    best_active_predictions = (
        prediction_cache[best_active_key]
        .copy()
    )

    active_changes = (
        best_active_predictions.loc[
            best_active_predictions[
                "prediction_changed"
            ]
        ]
        .copy()
    )

    active_changes["compositional_tp"] = [
        len(compositional_set & gold_set)
        for compositional_set, gold_set
        in zip(
            active_changes["compositional_set"],
            active_changes["gold_set"],
        )
    ]

    active_changes["hybrid_tp"] = [
        len(hybrid_set & gold_set)
        for hybrid_set, gold_set
        in zip(
            active_changes["hybrid_set"],
            active_changes["gold_set"],
        )
    ]

    active_changes["compositional_fp"] = [
        len(compositional_set - gold_set)
        for compositional_set, gold_set
        in zip(
            active_changes["compositional_set"],
            active_changes["gold_set"],
        )
    ]

    active_changes["hybrid_fp"] = [
        len(hybrid_set - gold_set)
        for hybrid_set, gold_set
        in zip(
            active_changes["hybrid_set"],
            active_changes["gold_set"],
        )
    ]

    active_changes["tp_change"] = (
        active_changes["hybrid_tp"]
        - active_changes["compositional_tp"]
    )

    active_changes["fp_change"] = (
        active_changes["hybrid_fp"]
        - active_changes["compositional_fp"]
    )

    active_changes["compositional_exact"] = [
        compositional_set == gold_set
        for compositional_set, gold_set
        in zip(
            active_changes["compositional_set"],
            active_changes["gold_set"],
        )
    ]

    active_changes["hybrid_exact"] = [
        hybrid_set == gold_set
        for hybrid_set, gold_set
        in zip(
            active_changes["hybrid_set"],
            active_changes["gold_set"],
        )
    ]

    print("\n" + "=" * 78)
    print("BEST ACTIVE CONFIGURATION")
    print("=" * 78)

    print(
        best_active[
            [
                "policy",
                "similarity_threshold",
                "margin_threshold",
                "accepted_neural_inputs",
                "changed_inputs",
                "micro_precision",
                "micro_recall",
                "micro_f1",
                "exact_set_accuracy",
                "selection_score",
                "f1_difference_from_compositional",
                "selection_difference_from_compositional",
            ]
        ].to_string()
    )

    print("\nChanged-case effect:")
    print(
        "Added correct links:",
        int(
            active_changes["tp_change"]
            .clip(lower=0)
            .sum()
        ),
    )
    print(
        "Removed correct links:",
        int(
            -active_changes["tp_change"]
            .clip(upper=0)
            .sum()
        ),
    )
    print(
        "Added false-positive links:",
        int(
            active_changes["fp_change"]
            .clip(lower=0)
            .sum()
        ),
    )
    print(
        "Removed false-positive links:",
        int(
            -active_changes["fp_change"]
            .clip(upper=0)
            .sum()
        ),
    )

    display_frame = active_changes.copy()

    for column in [
        "gold_set",
        "compositional_set",
        "hybrid_set",
    ]:
        display_frame[column] = (
            display_frame[column]
            .apply(
                lambda labels:
                " | ".join(sorted(labels))
            )
        )

    print("\nAll changed cases:")

    display(
        display_frame[
            [
                "mention_input_id",
                "domain",
                "num_article",
                "mention",
                "gold_set",
                "compositional_set",
                "top1_concept",
                "top1_similarity",
                "similarity_margin",
                "hybrid_set",
                "tp_change",
                "fp_change",
                "compositional_exact",
                "hybrid_exact",
            ]
        ]
        .sort_values(
            [
                "tp_change",
                "fp_change",
                "top1_similarity",
            ],
            ascending=[
                False,
                True,
                False,
            ],
        )
    )


# ------------------------------------------------------------
# 3. Save audit
# ------------------------------------------------------------

ACTIVE_TUNING_PATH = (
    RESULTS_DIR
    / "taskB_development_active_hybrid_complementarity.csv"
)

ACTIVE_CASES_PATH = (
    RESULTS_DIR
    / "taskB_development_best_active_hybrid_changed_cases.csv"
)

ACTIVE_SUMMARY_PATH = (
    RESULTS_DIR
    / "taskB_development_active_hybrid_complementarity.json"
)

active_tuning.to_csv(
    ACTIVE_TUNING_PATH,
    index=False,
)

if not active_tuning.empty:
    active_cases_output = active_changes.copy()

    for column in [
        "gold_set",
        "compositional_set",
        "hybrid_set",
    ]:
        active_cases_output[column] = (
            active_cases_output[column]
            .apply(
                lambda labels:
                " | ".join(sorted(labels))
            )
        )

    active_cases_output = (
        active_cases_output.drop(
            columns=["similarity_values"],
            errors="ignore",
        )
    )

    active_cases_output.to_csv(
        ACTIVE_CASES_PATH,
        index=False,
    )

    active_summary = {
        "active_configurations": int(
            len(active_tuning)
        ),
        "best_active_policy": str(
            best_active["policy"]
        ),
        "similarity_threshold": float(
            best_active["similarity_threshold"]
        ),
        "margin_threshold": float(
            best_active["margin_threshold"]
        ),
        "changed_inputs": int(
            best_active["changed_inputs"]
        ),
        "micro_f1": float(
            best_active["micro_f1"]
        ),
        "f1_difference_from_compositional": float(
            best_active[
                "f1_difference_from_compositional"
            ]
        ),
        "selection_difference_from_compositional": float(
            best_active[
                "selection_difference_from_compositional"
            ]
        ),
    }

else:
    pd.DataFrame().to_csv(
        ACTIVE_CASES_PATH,
        index=False,
    )

    active_summary = {
        "active_configurations": 0,
        "best_active_policy": None,
    }

with open(
    ACTIVE_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        active_summary,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved active tuning audit to:")
print(ACTIVE_TUNING_PATH)

print("\nSaved best active changed cases to:")
print(ACTIVE_CASES_PATH)

print("\nSaved active audit summary to:")
print(ACTIVE_SUMMARY_PATH)

print("\nCell 14B completed successfully.")

In [ ]:
# ============================================================
# CELL 15 — LEAKAGE-SAFE DEFINITION EVALUATION PROTOCOL
# ============================================================

from pathlib import Path
import ast
import json
import re
import unicodedata

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLITS_DIR = EXPERIMENT_DIR / "splits"

GOLD_LINKS_PATH = (
    RESULTS_DIR
    / "taskB_authoritative_unique_gold_links.csv"
)

REFERENCE_DEFINITIONS_PATH = (
    RESULTS_DIR
    / "taskB_authoritative_reference_definitions.csv"
)

SPLIT_PATH = (
    SPLITS_DIR
    / "taskB_document_holdout_grouped_70_15_15.csv"
)

COMPOSITIONAL_PREDICTIONS_PATH = (
    RESULTS_DIR
    / "taskB_development_compositional_predictions.csv"
)

for path in [
    GOLD_LINKS_PATH,
    REFERENCE_DEFINITIONS_PATH,
    SPLIT_PATH,
    COMPOSITIONAL_PREDICTIONS_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)

gold_links = pd.read_csv(GOLD_LINKS_PATH)
reference_definitions = pd.read_csv(
    REFERENCE_DEFINITIONS_PATH
)
document_split = pd.read_csv(SPLIT_PATH)
compositional_predictions = pd.read_csv(
    COMPOSITIONAL_PREDICTIONS_PATH
)


# ------------------------------------------------------------
# 2. Schema helpers
# ------------------------------------------------------------

def find_column(
    frame,
    preferred_names,
    contains_all=None,
):
    for name in preferred_names:
        if name in frame.columns:
            return name

    if contains_all:
        candidates = [
            column
            for column in frame.columns
            if all(
                token.casefold()
                in column.casefold()
                for token in contains_all
            )
        ]

        if len(candidates) == 1:
            return candidates[0]

    raise ValueError(
        "Could not identify required column. "
        f"Available columns: {frame.columns.tolist()}"
    )


gold_concept_column = find_column(
    gold_links,
    [
        "concept_canonical",
        "canonical_concept",
        "concept",
        "concept_label",
    ],
    contains_all=["concept"],
)

definition_concept_column = find_column(
    reference_definitions,
    [
        "concept_canonical",
        "canonical_concept",
        "concept",
        "concept_label",
    ],
    contains_all=["concept"],
)

definition_text_column = find_column(
    reference_definitions,
    [
        "reference_definition",
        "definition",
        "definition_gold",
        "intensional_definition",
        "Intensional definition",
    ],
    contains_all=["definition"],
)

prediction_column = find_column(
    compositional_predictions,
    [
        "selected_compositional_prediction",
        "compositional_prediction",
        "predicted_compositional",
        "predicted_labels",
    ],
    contains_all=[
        "compositional",
        "prediction",
    ],
)

print("=" * 78)
print("DEFINITION RESOURCE SCHEMA")
print("=" * 78)

print("Gold concept column:", gold_concept_column)
print(
    "Definition concept column:",
    definition_concept_column,
)
print(
    "Definition text column:",
    definition_text_column,
)
print(
    "Compositional prediction column:",
    prediction_column,
)


# ------------------------------------------------------------
# 3. Normalization
# ------------------------------------------------------------

def normalize_concept(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip()

    if text.startswith("<") and text.endswith(">"):
        text = text[1:-1].strip()

    text = text.casefold()
    text = re.sub(r"\s+", " ", text)

    return text


def parse_concept_set(value):
    if value is None:
        return set()

    if isinstance(value, float) and np.isnan(value):
        return set()

    if isinstance(value, (list, tuple, set, np.ndarray)):
        return {
            normalize_concept(item)
            for item in value
            if normalize_concept(item)
        }

    text = str(value).strip()

    if not text or text.casefold() in {
        "nan",
        "none",
        "set()",
        "()",
        "[]",
        "{}",
    }:
        return set()

    angle_labels = re.findall(
        r"<([^<>]+)>",
        text,
    )

    if angle_labels:
        return {
            normalize_concept(item)
            for item in angle_labels
            if normalize_concept(item)
        }

    if (
        text[0] in "[({"
        and text[-1] in "])}"
    ):
        try:
            parsed = ast.literal_eval(text)

            if isinstance(
                parsed,
                (list, tuple, set),
            ):
                return {
                    normalize_concept(item)
                    for item in parsed
                    if normalize_concept(item)
                }

        except Exception:
            interior = text[1:-1].strip()

            return {
                normalize_concept(item)
                for item in interior.split(",")
                if normalize_concept(item)
            }

    if "|" in text:
        return {
            normalize_concept(item)
            for item in text.split("|")
            if normalize_concept(item)
        }

    normalized = normalize_concept(text)

    return {normalized} if normalized else set()


# ------------------------------------------------------------
# 4. Attach authoritative split to semantic links
# ------------------------------------------------------------

gold_links["concept_key"] = (
    gold_links[gold_concept_column]
    .apply(normalize_concept)
)

reference_definitions["concept_key"] = (
    reference_definitions[
        definition_concept_column
    ]
    .apply(normalize_concept)
)

# Keep one authoritative definition per concept.
definition_duplicates = (
    reference_definitions.groupby(
        "concept_key"
    )[definition_text_column]
    .nunique()
)

conflicting_definitions = (
    definition_duplicates[
        definition_duplicates > 1
    ]
)

if len(conflicting_definitions):
    raise ValueError(
        "Some canonical concepts have multiple different "
        "reference definitions:\n"
        + conflicting_definitions.to_string()
    )

definition_inventory = (
    reference_definitions[
        [
            "concept_key",
            definition_concept_column,
            definition_text_column,
        ]
    ]
    .drop_duplicates("concept_key")
    .rename(
        columns={
            definition_concept_column:
                "concept_canonical",
            definition_text_column:
                "reference_definition",
        }
    )
)

merge_keys = [
    "domain",
    "num_article",
]

for key in merge_keys:
    if key not in gold_links.columns:
        raise ValueError(
            f"Gold links lack merge column: {key}"
        )

    if key not in document_split.columns:
        raise ValueError(
            f"Split table lacks merge column: {key}"
        )

gold_links["num_article"] = (
    gold_links["num_article"]
    .astype(str)
)

document_split["num_article"] = (
    document_split["num_article"]
    .astype(str)
)

split_lookup = (
    document_split[
        [
            "domain",
            "num_article",
            "split",
        ]
    ]
    .drop_duplicates()
)

links_with_split = gold_links.merge(
    split_lookup,
    on=[
        "domain",
        "num_article",
    ],
    how="left",
    validate="many_to_one",
)

if links_with_split["split"].isna().any():
    missing = links_with_split.loc[
        links_with_split["split"].isna(),
        [
            "domain",
            "num_article",
        ],
    ].drop_duplicates()

    raise ValueError(
        "Some semantic links could not be assigned to a split:\n"
        + missing.head(20).to_string(index=False)
    )


# ------------------------------------------------------------
# 5. Define training-known and development-target concepts
# ------------------------------------------------------------

training_concepts = set(
    links_with_split.loc[
        links_with_split["split"].eq("train"),
        "concept_key",
    ]
)

development_concepts = set(
    links_with_split.loc[
        links_with_split["split"].eq("development"),
        "concept_key",
    ]
)

development_seen_concepts = (
    development_concepts
    & training_concepts
)

development_unseen_concepts = (
    development_concepts
    - training_concepts
)

all_definition_concepts = set(
    definition_inventory["concept_key"]
)

missing_development_definitions = (
    development_concepts
    - all_definition_concepts
)

if missing_development_definitions:
    raise ValueError(
        "Development concepts without reference definitions: "
        + ", ".join(
            sorted(missing_development_definitions)
        )
    )


# ------------------------------------------------------------
# 6. Build leakage-safe training definition dictionary
# ------------------------------------------------------------

training_definition_inventory = (
    definition_inventory.loc[
        definition_inventory[
            "concept_key"
        ].isin(training_concepts)
    ]
    .copy()
)

development_definition_manifest = (
    definition_inventory.loc[
        definition_inventory[
            "concept_key"
        ].isin(development_concepts)
    ]
    .copy()
)

development_definition_manifest[
    "definition_visibility"
] = np.where(
    development_definition_manifest[
        "concept_key"
    ].isin(training_concepts),
    "seen_in_training",
    "unseen_in_training",
)


# ------------------------------------------------------------
# 7. Concept frequency and domain membership
# ------------------------------------------------------------

training_frequency = (
    links_with_split.loc[
        links_with_split["split"].eq("train")
    ]
    .groupby("concept_key")
    .size()
    .rename("training_link_count")
)

development_frequency = (
    links_with_split.loc[
        links_with_split["split"].eq(
            "development"
        )
    ]
    .groupby("concept_key")
    .size()
    .rename("development_link_count")
)

development_domains = (
    links_with_split.loc[
        links_with_split["split"].eq(
            "development"
        )
    ]
    .groupby("concept_key")["domain"]
    .agg(
        lambda values:
        " | ".join(
            sorted(set(values))
        )
    )
    .rename("development_domains")
)

development_definition_manifest = (
    development_definition_manifest
    .merge(
        training_frequency,
        on="concept_key",
        how="left",
    )
    .merge(
        development_frequency,
        on="concept_key",
        how="left",
    )
    .merge(
        development_domains,
        on="concept_key",
        how="left",
    )
)

development_definition_manifest[
    "training_link_count"
] = (
    development_definition_manifest[
        "training_link_count"
    ]
    .fillna(0)
    .astype(int)
)

development_definition_manifest[
    "development_link_count"
] = (
    development_definition_manifest[
        "development_link_count"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 8. Audit predicted concepts from the locked linker
# ------------------------------------------------------------

compositional_predictions[
    "predicted_concept_set"
] = (
    compositional_predictions[
        prediction_column
    ]
    .apply(parse_concept_set)
)

predicted_development_concepts = set().union(
    *compositional_predictions[
        "predicted_concept_set"
    ].tolist()
)

predicted_seen_concepts = (
    predicted_development_concepts
    & training_concepts
)

predicted_unseen_concepts = (
    predicted_development_concepts
    - training_concepts
)

predicted_with_reference = (
    predicted_development_concepts
    & all_definition_concepts
)

predicted_without_reference = (
    predicted_development_concepts
    - all_definition_concepts
)


# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

summary = {
    "training_concepts": len(
        training_concepts
    ),
    "training_definitions_available": len(
        training_definition_inventory
    ),
    "development_target_concepts": len(
        development_concepts
    ),
    "development_seen_concepts": len(
        development_seen_concepts
    ),
    "development_unseen_concepts": len(
        development_unseen_concepts
    ),
    "development_seen_ratio": (
        len(development_seen_concepts)
        / len(development_concepts)
        if development_concepts
        else 0.0
    ),
    "locked_compositional_predicted_concepts": len(
        predicted_development_concepts
    ),
    "predicted_seen_concepts": len(
        predicted_seen_concepts
    ),
    "predicted_unseen_concepts": len(
        predicted_unseen_concepts
    ),
    "predicted_concepts_with_reference": len(
        predicted_with_reference
    ),
    "predicted_concepts_without_reference": len(
        predicted_without_reference
    ),
}

print("\n" + "=" * 78)
print("LEAKAGE-SAFE DEFINITION PROTOCOL")
print("=" * 78)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nUnseen development target concepts:")

display(
    development_definition_manifest.loc[
        development_definition_manifest[
            "definition_visibility"
        ].eq("unseen_in_training"),
        [
            "concept_key",
            "concept_canonical",
            "reference_definition",
            "development_domains",
            "development_link_count",
        ],
    ]
    .sort_values(
        [
            "development_domains",
            "concept_key",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\nPredicted development concepts without an "
    "authoritative reference definition:"
)

if predicted_without_reference:
    print(
        "\n".join(
            sorted(predicted_without_reference)
        )
    )
else:
    print("None")


# ------------------------------------------------------------
# 10. Save protocol artifacts
# ------------------------------------------------------------

TRAINING_DEFINITION_PATH = (
    RESULTS_DIR
    / "taskB_training_visible_definition_inventory.csv"
)

DEVELOPMENT_MANIFEST_PATH = (
    RESULTS_DIR
    / "taskB_development_definition_evaluation_manifest.csv"
)

PROTOCOL_PATH = (
    RESULTS_DIR
    / "taskB_definition_evaluation_protocol.json"
)

training_definition_inventory.to_csv(
    TRAINING_DEFINITION_PATH,
    index=False,
)

development_definition_manifest.to_csv(
    DEVELOPMENT_MANIFEST_PATH,
    index=False,
)

protocol = {
    "evaluation_split": "development",
    "definition_training_rule": (
        "A reference definition is available to the "
        "system only when its concept occurs in a "
        "training document."
    ),
    "seen_concept_policy": (
        "Retrieve the training-visible canonical "
        "definition exactly."
    ),
    "unseen_concept_policy": (
        "Generate a definition without consulting the "
        "held-out reference definition."
    ),
    "primary_definition_evaluation_unit": (
        "unique canonical concept"
    ),
    "reporting_strata": [
        "all development concepts",
        "seen-in-training concepts",
        "unseen-in-training concepts",
    ],
    "summary": summary,
}

with open(
    PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        protocol,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved training-visible definitions to:")
print(TRAINING_DEFINITION_PATH)

print("\nSaved development definition manifest to:")
print(DEVELOPMENT_MANIFEST_PATH)

print("\nSaved definition protocol to:")
print(PROTOCOL_PATH)

print("\nCell 15 completed successfully.")

In [ ]:
# ============================================================
# CELL 16 — CONCEPT-LEVEL DEFINITION HOLDOUT FEASIBILITY AUDIT
# ============================================================

from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths and authoritative resources
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"

GOLD_LINKS_PATH = (
    RESULTS_DIR
    / "taskB_authoritative_unique_gold_links.csv"
)

REFERENCE_DEFINITIONS_PATH = (
    RESULTS_DIR
    / "taskB_authoritative_reference_definitions.csv"
)

gold_links = pd.read_csv(
    GOLD_LINKS_PATH
)

definitions = pd.read_csv(
    REFERENCE_DEFINITIONS_PATH
)


# ------------------------------------------------------------
# 2. Normalization
# ------------------------------------------------------------

def normalize_concept(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip()

    if text.startswith("<") and text.endswith(">"):
        text = text[1:-1].strip()

    text = text.casefold()
    text = re.sub(r"\s+", " ", text)

    return text


gold_links["concept_key"] = (
    gold_links["concept_canonical"]
    .apply(normalize_concept)
)

definitions["concept_key"] = (
    definitions["concept_canonical"]
    .apply(normalize_concept)
)


# ------------------------------------------------------------
# 3. Validate one definition per concept
# ------------------------------------------------------------

definition_conflicts = (
    definitions.groupby("concept_key")[
        "definition"
    ]
    .nunique()
)

definition_conflicts = definition_conflicts.loc[
    definition_conflicts > 1
]

if len(definition_conflicts):
    raise ValueError(
        "Some canonical concepts have conflicting "
        "reference definitions:\n"
        + definition_conflicts.to_string()
    )

definition_inventory = (
    definitions[
        [
            "concept_key",
            "concept_canonical",
            "definition",
        ]
    ]
    .drop_duplicates("concept_key")
    .copy()
)

if len(definition_inventory) != 315:
    raise ValueError(
        "Expected 315 canonical definitions; "
        f"found {len(definition_inventory)}."
    )


# ------------------------------------------------------------
# 4. Build concept-level occurrence metadata
# ------------------------------------------------------------

gold_links["document_key"] = (
    gold_links["domain"].astype(str)
    + "::"
    + gold_links["num_article"].astype(str)
)

concept_support = (
    gold_links.groupby("concept_key")
    .agg(
        semantic_link_count=(
            "concept_key",
            "size",
        ),
        document_count=(
            "document_key",
            "nunique",
        ),
        unique_mention_count=(
            "mention",
            "nunique",
        ),
        domain_count=(
            "domain",
            "nunique",
        ),
        domains=(
            "domain",
            lambda values:
            " | ".join(
                sorted(set(values))
            ),
        ),
        character_length_mean=(
            "mention",
            lambda values:
            float(
                np.mean(
                    [
                        len(str(value))
                        for value in values
                    ]
                )
            ),
        ),
        mention_word_count_mean=(
            "mention",
            lambda values:
            float(
                np.mean(
                    [
                        len(str(value).split())
                        for value in values
                    ]
                )
            ),
        ),
    )
    .reset_index()
)

concept_metadata = (
    definition_inventory.merge(
        concept_support,
        on="concept_key",
        how="left",
        validate="one_to_one",
    )
)

support_columns = [
    "semantic_link_count",
    "document_count",
    "unique_mention_count",
    "domain_count",
]

if concept_metadata[
    support_columns
].isna().any().any():
    missing = concept_metadata.loc[
        concept_metadata[
            "semantic_link_count"
        ].isna(),
        [
            "concept_key",
            "concept_canonical",
        ],
    ]

    raise ValueError(
        "Definitions without semantic-link evidence:\n"
        + missing.to_string(index=False)
    )


# ------------------------------------------------------------
# 5. Define support strata
# ------------------------------------------------------------

def support_bin(value):
    value = int(value)

    if value == 1:
        return "1_link"

    if value == 2:
        return "2_links"

    if value <= 4:
        return "3_to_4_links"

    if value <= 9:
        return "5_to_9_links"

    return "10_plus_links"


concept_metadata["support_bin"] = (
    concept_metadata[
        "semantic_link_count"
    ]
    .apply(support_bin)
)

concept_metadata["domain_stratum"] = (
    concept_metadata["domains"]
    .replace(
        {
            "mental_health":
                "mental_health_only",
            "parkinson":
                "parkinson_only",
            "mental_health | parkinson":
                "both_domains",
        }
    )
)

concept_metadata["split_stratum"] = (
    concept_metadata["domain_stratum"]
    + "::"
    + concept_metadata["support_bin"]
)


# ------------------------------------------------------------
# 6. Definition-length metadata
# ------------------------------------------------------------

concept_metadata["definition_word_count"] = (
    concept_metadata["definition"]
    .astype(str)
    .str.split()
    .str.len()
)

concept_metadata[
    "definition_character_count"
] = (
    concept_metadata["definition"]
    .astype(str)
    .str.len()
)


# ------------------------------------------------------------
# 7. Feasibility summaries
# ------------------------------------------------------------

print("=" * 78)
print("CONCEPT-LEVEL DEFINITION HOLDOUT FEASIBILITY")
print("=" * 78)

print(
    "Canonical concepts:",
    len(concept_metadata),
)

print(
    "Semantic links represented:",
    int(
        concept_metadata[
            "semantic_link_count"
        ].sum()
    ),
)

print("\nConcepts by domain membership:")
print(
    concept_metadata[
        "domain_stratum"
    ]
    .value_counts()
)

print("\nConcepts by occurrence support:")
print(
    concept_metadata[
        "support_bin"
    ]
    .value_counts()
    .reindex(
        [
            "1_link",
            "2_links",
            "3_to_4_links",
            "5_to_9_links",
            "10_plus_links",
        ],
        fill_value=0,
    )
)

print("\nJoint split strata:")
joint_strata = (
    concept_metadata.groupby(
        [
            "domain_stratum",
            "support_bin",
        ],
        observed=True,
    )
    .size()
    .rename("concepts")
    .reset_index()
)

display(
    joint_strata.sort_values(
        [
            "domain_stratum",
            "support_bin",
        ]
    )
)

print("\nOccurrence-support percentiles:")
display(
    concept_metadata[
        [
            "semantic_link_count",
            "document_count",
            "unique_mention_count",
        ]
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .round(3)
)

print("\nDefinition-length percentiles:")
display(
    concept_metadata[
        [
            "definition_word_count",
            "definition_character_count",
        ]
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .round(3)
)

print("\nTwenty highest-support concepts:")
display(
    concept_metadata[
        [
            "concept_canonical",
            "domains",
            "semantic_link_count",
            "document_count",
            "unique_mention_count",
            "definition_word_count",
        ]
    ]
    .sort_values(
        [
            "semantic_link_count",
            "document_count",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(20)
)


# ------------------------------------------------------------
# 8. Estimate expected split sizes
# ------------------------------------------------------------

expected_split_sizes = {
    "train": round(
        len(concept_metadata) * 0.70
    ),
    "development": round(
        len(concept_metadata) * 0.15
    ),
    "test": (
        len(concept_metadata)
        - round(
            len(concept_metadata) * 0.70
        )
        - round(
            len(concept_metadata) * 0.15
        )
    ),
}

print("\nExpected 70/15/15 concept split:")
print(expected_split_sizes)


# ------------------------------------------------------------
# 9. Save feasibility audit
# ------------------------------------------------------------

METADATA_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_metadata_before_split.csv"
)

FEASIBILITY_PATH = (
    RESULTS_DIR
    / "taskB_definition_holdout_feasibility.json"
)

concept_metadata.to_csv(
    METADATA_PATH,
    index=False,
)

feasibility = {
    "canonical_concepts": int(
        len(concept_metadata)
    ),
    "semantic_links": int(
        concept_metadata[
            "semantic_link_count"
        ].sum()
    ),
    "expected_split_sizes": (
        expected_split_sizes
    ),
    "domain_membership_counts": {
        str(key): int(value)
        for key, value
        in concept_metadata[
            "domain_stratum"
        ]
        .value_counts()
        .items()
    },
    "support_bin_counts": {
        str(key): int(value)
        for key, value
        in concept_metadata[
            "support_bin"
        ]
        .value_counts()
        .items()
    },
    "proposed_evaluation_design": {
        "unit": "unique canonical concept",
        "split": "70/15/15",
        "stratification": [
            "domain membership",
            "semantic-link support",
        ],
        "reference_definition_access": (
            "Only definitions assigned to the "
            "training partition may be used as "
            "prompt examples or retrieval resources."
        ),
        "target_occurrence_evidence": (
            "Mention forms and document contexts "
            "remain available because they constitute "
            "the inference evidence, not the withheld "
            "definition target."
        ),
    },
}

with open(
    FEASIBILITY_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        feasibility,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved concept metadata to:")
print(METADATA_PATH)

print("\nSaved feasibility audit to:")
print(FEASIBILITY_PATH)

print("\nCell 16 completed successfully.")

In [ ]:
# ============================================================
# CELL 17 — BALANCED CONCEPT-LEVEL DEFINITION HOLDOUT SPLIT
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load concept metadata
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLITS_DIR = EXPERIMENT_DIR / "splits"

SPLITS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_metadata_before_split.csv"
)

concept_metadata = pd.read_csv(
    METADATA_PATH
)

if len(concept_metadata) != 315:
    raise ValueError(
        f"Expected 315 concepts; found {len(concept_metadata)}."
    )


# ------------------------------------------------------------
# 2. Exact target sizes
# ------------------------------------------------------------

N_CONCEPTS = len(concept_metadata)

N_TRAIN = 220
N_DEVELOPMENT = 47
N_TEST = N_CONCEPTS - N_TRAIN - N_DEVELOPMENT

assert N_TEST == 48


# ------------------------------------------------------------
# 3. Balance features
# ------------------------------------------------------------

balance_frame = pd.DataFrame(
    index=concept_metadata.index
)

# Domain-membership indicators.
for domain_value in [
    "mental_health_only",
    "parkinson_only",
    "both_domains",
]:
    balance_frame[
        f"domain_{domain_value}"
    ] = (
        concept_metadata[
            "domain_stratum"
        ].eq(domain_value)
        .astype(float)
    )

# Support-bin indicators.
for support_value in [
    "1_link",
    "2_links",
    "3_to_4_links",
    "5_to_9_links",
    "10_plus_links",
]:
    balance_frame[
        f"support_{support_value}"
    ] = (
        concept_metadata[
            "support_bin"
        ].eq(support_value)
        .astype(float)
    )

# Continuous metadata. Log transforms limit domination
# by a few very frequent concepts.
balance_frame[
    "log_semantic_link_count"
] = np.log1p(
    concept_metadata[
        "semantic_link_count"
    ].astype(float)
)

balance_frame[
    "log_document_count"
] = np.log1p(
    concept_metadata[
        "document_count"
    ].astype(float)
)

balance_frame[
    "log_unique_mention_count"
] = np.log1p(
    concept_metadata[
        "unique_mention_count"
    ].astype(float)
)

balance_frame[
    "definition_word_count"
] = concept_metadata[
    "definition_word_count"
].astype(float)

balance_frame[
    "mention_word_count_mean"
] = concept_metadata[
    "mention_word_count_mean"
].astype(float)


# ------------------------------------------------------------
# 4. Standardize features globally
# ------------------------------------------------------------

feature_names = (
    balance_frame.columns.tolist()
)

feature_matrix = (
    balance_frame
    .to_numpy(dtype=np.float64)
)

global_mean = feature_matrix.mean(axis=0)

global_standard_deviation = (
    feature_matrix.std(
        axis=0,
        ddof=0,
    )
)

global_standard_deviation[
    global_standard_deviation == 0
] = 1.0

standardized_features = (
    feature_matrix - global_mean
) / global_standard_deviation


# ------------------------------------------------------------
# 5. Candidate validity constraints
# ------------------------------------------------------------

domain_values = (
    concept_metadata[
        "domain_stratum"
    ]
    .to_numpy()
)

support_values = (
    concept_metadata[
        "support_bin"
    ]
    .to_numpy()
)


def split_is_valid(
    development_indices,
    test_indices,
):
    # Both held-out partitions must represent every
    # domain-membership category.
    for indices in [
        development_indices,
        test_indices,
    ]:
        present_domains = set(
            domain_values[indices]
        )

        if present_domains != {
            "mental_health_only",
            "parkinson_only",
            "both_domains",
        }:
            return False

        # Require usable evidence-rich subsets for the
        # all-context versus collapse-selected comparison.
        high_support_count = np.isin(
            support_values[indices],
            [
                "5_to_9_links",
                "10_plus_links",
            ],
        ).sum()

        multi_occurrence_count = (
            support_values[indices]
            != "1_link"
        ).sum()

        if high_support_count < 5:
            return False

        if multi_occurrence_count < 15:
            return False

    return True


# ------------------------------------------------------------
# 6. Metadata-only random search
# ------------------------------------------------------------

SEARCH_SEEDS = 50000

best_candidate = None
best_objective = np.inf
valid_candidate_count = 0

all_indices = np.arange(
    N_CONCEPTS
)

for seed in range(SEARCH_SEEDS):
    rng = np.random.default_rng(seed)

    permutation = rng.permutation(
        all_indices
    )

    train_indices = permutation[
        :N_TRAIN
    ]

    development_indices = permutation[
        N_TRAIN:
        N_TRAIN + N_DEVELOPMENT
    ]

    test_indices = permutation[
        N_TRAIN + N_DEVELOPMENT:
    ]

    if not split_is_valid(
        development_indices,
        test_indices,
    ):
        continue

    valid_candidate_count += 1

    split_indices = {
        "train": train_indices,
        "development": development_indices,
        "test": test_indices,
    }

    standardized_differences = []

    for split_name, indices in (
        split_indices.items()
    ):
        split_mean = (
            standardized_features[
                indices
            ].mean(axis=0)
        )

        standardized_differences.extend(
            np.abs(split_mean).tolist()
        )

    standardized_differences = np.array(
        standardized_differences
    )

    # Prioritize the largest imbalance, then the
    # overall average imbalance.
    maximum_difference = float(
        standardized_differences.max()
    )

    mean_difference = float(
        standardized_differences.mean()
    )

    objective = (
        maximum_difference
        + 0.25 * mean_difference
    )

    if objective < best_objective:
        best_objective = objective

        best_candidate = {
            "seed": seed,
            "train_indices":
                train_indices.copy(),
            "development_indices":
                development_indices.copy(),
            "test_indices":
                test_indices.copy(),
            "maximum_standardized_difference":
                maximum_difference,
            "mean_absolute_standardized_difference":
                mean_difference,
            "objective": objective,
        }

if best_candidate is None:
    raise RuntimeError(
        "No valid concept split was found."
    )


# ------------------------------------------------------------
# 7. Assign selected split
# ------------------------------------------------------------

concept_split = concept_metadata.copy()

concept_split["split"] = ""

concept_split.loc[
    best_candidate["train_indices"],
    "split",
] = "train"

concept_split.loc[
    best_candidate[
        "development_indices"
    ],
    "split",
] = "development"

concept_split.loc[
    best_candidate["test_indices"],
    "split",
] = "test"

if concept_split["split"].eq("").any():
    raise RuntimeError(
        "Some concepts were not assigned."
    )

if concept_split[
    "concept_key"
].duplicated().any():
    raise RuntimeError(
        "A concept appears more than once in "
        "the split table."
    )


# ------------------------------------------------------------
# 8. Balance summary
# ------------------------------------------------------------

count_summary = (
    concept_split.groupby("split")
    .size()
    .rename("concepts")
)

domain_summary = (
    concept_split.groupby(
        [
            "split",
            "domain_stratum",
        ],
        observed=True,
    )
    .size()
    .rename("concepts")
    .reset_index()
)

support_summary = (
    concept_split.groupby(
        [
            "split",
            "support_bin",
        ],
        observed=True,
    )
    .size()
    .rename("concepts")
    .reset_index()
)

numeric_features = [
    "semantic_link_count",
    "document_count",
    "unique_mention_count",
    "definition_word_count",
    "mention_word_count_mean",
]

numeric_summary = (
    concept_split.groupby("split")[
        numeric_features
    ]
    .mean()
    .round(3)
)

balance_records = []

for split_name in [
    "train",
    "development",
    "test",
]:
    indices = concept_split.index[
        concept_split["split"].eq(
            split_name
        )
    ].to_numpy()

    split_mean = (
        standardized_features[
            indices
        ].mean(axis=0)
    )

    for feature_name, difference in zip(
        feature_names,
        split_mean,
    ):
        balance_records.append(
            {
                "split": split_name,
                "feature": feature_name,
                "standardized_difference":
                    float(difference),
                "absolute_difference":
                    float(abs(difference)),
            }
        )

balance_audit = pd.DataFrame(
    balance_records
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("=" * 78)
print("SELECTED CONCEPT-LEVEL DEFINITION SPLIT")
print("=" * 78)

print(
    "Selected metadata-balance seed:",
    best_candidate["seed"],
)

print(
    "Valid candidates evaluated:",
    valid_candidate_count,
)

print(
    "Balance objective:",
    best_candidate["objective"],
)

print(
    "Maximum absolute standardized difference:",
    best_candidate[
        "maximum_standardized_difference"
    ],
)

print(
    "Mean absolute standardized difference:",
    best_candidate[
        "mean_absolute_standardized_difference"
    ],
)

print("\nConcept counts:")
print(count_summary)

print("\nDomain-membership counts:")
display(
    domain_summary.pivot(
        index="domain_stratum",
        columns="split",
        values="concepts",
    )
    .fillna(0)
    .astype(int)
)

print("\nOccurrence-support counts:")
display(
    support_summary.pivot(
        index="support_bin",
        columns="split",
        values="concepts",
    )
    .fillna(0)
    .astype(int)
)

print("\nMean numeric characteristics:")
display(numeric_summary)

print("\nTwenty largest balance differences:")
display(
    balance_audit.sort_values(
        "absolute_difference",
        ascending=False,
    )
    .head(20)
    .round(4)
)


# ------------------------------------------------------------
# 10. Reference-definition access audit
# ------------------------------------------------------------

concept_sets = {
    split_name: set(
        concept_split.loc[
            concept_split[
                "split"
            ].eq(split_name),
            "concept_key",
        ]
    )
    for split_name in [
        "train",
        "development",
        "test",
    ]
}

train_development_overlap = (
    concept_sets["train"]
    & concept_sets["development"]
)

train_test_overlap = (
    concept_sets["train"]
    & concept_sets["test"]
)

development_test_overlap = (
    concept_sets["development"]
    & concept_sets["test"]
)

print("\nReference-target overlap audit:")
print(
    "Train ∩ development:",
    len(train_development_overlap),
)
print(
    "Train ∩ test:",
    len(train_test_overlap),
)
print(
    "Development ∩ test:",
    len(development_test_overlap),
)

assert not train_development_overlap
assert not train_test_overlap
assert not development_test_overlap


# ------------------------------------------------------------
# 11. Save authoritative concept split
# ------------------------------------------------------------

CONCEPT_SPLIT_PATH = (
    SPLITS_DIR
    / "taskB_definition_concept_holdout_70_15_15.csv"
)

BALANCE_AUDIT_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_holdout_balance_audit.csv"
)

SPLIT_METADATA_PATH = (
    SPLITS_DIR
    / "taskB_definition_concept_holdout_metadata.json"
)

concept_split.to_csv(
    CONCEPT_SPLIT_PATH,
    index=False,
)

balance_audit.to_csv(
    BALANCE_AUDIT_PATH,
    index=False,
)

split_metadata = {
    "selected_seed": int(
        best_candidate["seed"]
    ),
    "search_seeds": SEARCH_SEEDS,
    "valid_candidates_evaluated": int(
        valid_candidate_count
    ),
    "target_sizes": {
        "train": N_TRAIN,
        "development": N_DEVELOPMENT,
        "test": N_TEST,
    },
    "balance_features": feature_names,
    "constraints": {
        "all_domain_memberships_in_each_heldout_split":
            True,
        "minimum_high_support_concepts_per_heldout_split":
            5,
        "minimum_multi_occurrence_concepts_per_heldout_split":
            15,
    },
    "objective": float(
        best_candidate["objective"]
    ),
    "maximum_absolute_standardized_difference":
        float(
            best_candidate[
                "maximum_standardized_difference"
            ]
        ),
    "mean_absolute_standardized_difference":
        float(
            best_candidate[
                "mean_absolute_standardized_difference"
            ]
        ),
    "reference_definition_overlap": {
        "train_development": 0,
        "train_test": 0,
        "development_test": 0,
    },
}

with open(
    SPLIT_METADATA_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        split_metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nAuthoritative concept split saved to:")
print(CONCEPT_SPLIT_PATH)

print("\nBalance audit saved to:")
print(BALANCE_AUDIT_PATH)

print("\nSplit metadata saved to:")
print(SPLIT_METADATA_PATH)

print("\nCell 17 completed successfully.")

In [ ]:
# ============================================================
# CELL 18 — FULL-CORPUS TASK B OCCURRENCE EVIDENCE EMBEDDINGS
# ============================================================

from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"
CACHE_DIR = EXPERIMENT_DIR / "cache"

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MENTION_INPUTS_PATH = (
    RESULTS_DIR
    / "taskB_authoritative_mention_inputs.csv"
)

MODEL_NAME = (
    "microsoft/"
    "BiomedNLP-PubMedBERT-base-uncased-abstract"
)

MAX_LENGTH = 512
STRIDE = 128
MODEL_BATCH_SIZE = 8
CONTEXT_CHARACTERS = 180

ABSTRACT_DIRECTORIES = {
    "mental_health": (
        BASE_DIR
        / "data"
        / "taskA"
        / "training"
        / "abstracts"
        / "mental_health"
    ),
    "parkinson": (
        BASE_DIR
        / "data"
        / "taskA"
        / "training"
        / "abstracts"
        / "parkinson"
    ),
}

FILE_PREFIXES = {
    "mental_health": "mh_",
    "parkinson": "pa_",
}

mention_inputs = pd.read_csv(
    MENTION_INPUTS_PATH
)

# ------------------------------------------------------------
# Add stable mention-input identifiers when absent
# ------------------------------------------------------------

mention_inputs = (
    mention_inputs
    .reset_index(drop=True)
)

if "mention_input_id" not in mention_inputs.columns:
    mention_inputs.insert(
        0,
        "mention_input_id",
        [
            f"taskb_input_{index:04d}"
            for index in range(
                len(mention_inputs)
            )
        ],
    )

if mention_inputs[
    "mention_input_id"
].duplicated().any():
    raise ValueError(
        "Generated mention_input_id values are not unique."
    )

print(
    "Mention-input identifier source:",
    (
        "authoritative file"
        if "mention_input_id"
        in pd.read_csv(
            MENTION_INPUTS_PATH,
            nrows=1,
        ).columns
        else "stable row-order reconstruction"
    ),
)

print(
    "First mention-input IDs:",
    mention_inputs[
        "mention_input_id"
    ].head().tolist(),
)

if len(mention_inputs) != 790:
    raise ValueError(
        "Expected 790 authoritative mention inputs; "
        f"found {len(mention_inputs)}."
    )

mention_inputs["num_article"] = (
    mention_inputs["num_article"]
    .astype(str)
)


# ------------------------------------------------------------
# 2. Resolve and load all abstract texts
# ------------------------------------------------------------

def resolve_abstract_path(
    domain,
    num_article,
):
    directory = ABSTRACT_DIRECTORIES[
        domain
    ]

    prefix = FILE_PREFIXES[
        domain
    ]

    expected_path = (
        directory
        / f"{prefix}{num_article}.txt"
    )

    if expected_path.exists():
        return expected_path

    # Defensive fallback for numeric IDs stored as "32.0".
    try:
        normalized_id = str(
            int(float(num_article))
        )
    except Exception:
        normalized_id = str(
            num_article
        )

    fallback_path = (
        directory
        / f"{prefix}{normalized_id}.txt"
    )

    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        f"No abstract found for "
        f"{domain} article {num_article}"
    )


document_rows = (
    mention_inputs[
        [
            "domain",
            "num_article",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "domain",
            "num_article",
        ]
    )
    .reset_index(drop=True)
)

document_texts = {}

for row in document_rows.itertuples(
    index=False
):
    path = resolve_abstract_path(
        row.domain,
        row.num_article,
    )

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    document_texts[
        (
            row.domain,
            row.num_article,
        )
    ] = text

print("=" * 78)
print("FULL-CORPUS TASK B EMBEDDING SETUP")
print("=" * 78)

print("Mention inputs:", len(mention_inputs))
print("Documents:", len(document_rows))
print(
    "Documents by domain:\n",
    document_rows["domain"]
    .value_counts(),
)


# ------------------------------------------------------------
# 3. Find literal mention occurrences
# ------------------------------------------------------------

def find_literal_occurrences(
    text,
    mention,
):
    mention = str(mention)

    if not mention:
        return []

    # Case-insensitive literal matching. No word-boundary
    # restriction is added because Task B mentions may contain
    # punctuation, hyphens, Greek symbols, or parentheses.
    pattern = re.compile(
        re.escape(mention),
        flags=re.IGNORECASE,
    )

    return [
        (
            match.start(),
            match.end(),
        )
        for match in pattern.finditer(text)
    ]


mention_occurrences = {}

for row in mention_inputs.itertuples(
    index=False
):
    text = document_texts[
        (
            row.domain,
            row.num_article,
        )
    ]

    mention_occurrences[
        row.mention_input_id
    ] = find_literal_occurrences(
        text,
        row.mention,
    )


# ------------------------------------------------------------
# 4. Load PubMedBERT
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

use_fp16 = (
    device.type == "cuda"
)

print("\nDevice:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory
            / 1024**3,
            1,
        ),
        "GB",
    )

print("FP16 autocast:", use_fp16)
print("Model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
)

model.to(device)
model.eval()

hidden_size = int(
    model.config.hidden_size
)

print("Hidden size:", hidden_size)


# ------------------------------------------------------------
# 5. Tokenize every document with overflow windows
# ------------------------------------------------------------

window_records = []

for row in document_rows.itertuples(
    index=False
):
    text = document_texts[
        (
            row.domain,
            row.num_article,
        )
    ]

    encoding = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        padding=False,
    )

    for window_index in range(
        len(encoding["input_ids"])
    ):
        window_records.append(
            {
                "domain": row.domain,
                "num_article":
                    row.num_article,
                "window_index":
                    window_index,
                "input_ids":
                    encoding[
                        "input_ids"
                    ][window_index],
                "attention_mask":
                    encoding[
                        "attention_mask"
                    ][window_index],
                "offset_mapping":
                    encoding[
                        "offset_mapping"
                    ][window_index],
                "special_tokens_mask":
                    encoding[
                        "special_tokens_mask"
                    ][window_index],
            }
        )

print(
    "\nToken windows created:",
    len(window_records),
)


# ------------------------------------------------------------
# 6. Encode document windows
# ------------------------------------------------------------

window_hidden_states = []

encoding_start_time = time.time()

for batch_start in tqdm(
    range(
        0,
        len(window_records),
        MODEL_BATCH_SIZE,
    ),
    desc="Encoding document windows",
):
    batch_records = window_records[
        batch_start:
        batch_start + MODEL_BATCH_SIZE
    ]

    batch_features = [
        {
            "input_ids":
                record["input_ids"],
            "attention_mask":
                record["attention_mask"],
        }
        for record in batch_records
    ]

    padded = tokenizer.pad(
        batch_features,
        padding=True,
        return_tensors="pt",
    )

    padded = {
        key: value.to(device)
        for key, value in padded.items()
    }

    with torch.no_grad():
        if use_fp16:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                outputs = model(**padded)
        else:
            outputs = model(**padded)

    hidden = (
        outputs.last_hidden_state
        .float()
        .cpu()
        .numpy()
    )

    for batch_index, record in enumerate(
        batch_records
    ):
        original_length = len(
            record["input_ids"]
        )

        window_hidden_states.append(
            hidden[
                batch_index,
                :original_length,
                :,
            ]
        )

elapsed_minutes = (
    time.time() - encoding_start_time
) / 60.0


# ------------------------------------------------------------
# 7. Index windows by document
# ------------------------------------------------------------

windows_by_document = {}

for window_row_index, record in enumerate(
    window_records
):
    key = (
        record["domain"],
        record["num_article"],
    )

    windows_by_document.setdefault(
        key,
        [],
    ).append(window_row_index)


# ------------------------------------------------------------
# 8. Pool every literal occurrence
# ------------------------------------------------------------

def pool_occurrence_from_windows(
    document_key,
    start_char,
    end_char,
):
    candidate_pools = []

    for window_row_index in (
        windows_by_document[
            document_key
        ]
    ):
        record = window_records[
            window_row_index
        ]

        offsets = record[
            "offset_mapping"
        ]

        special_mask = record[
            "special_tokens_mask"
        ]

        token_indices = []

        for token_index, (
            token_start,
            token_end,
        ) in enumerate(offsets):
            if special_mask[token_index] == 1:
                continue

            if token_end <= token_start:
                continue

            # Token overlaps the mention character span.
            if (
                token_start < end_char
                and token_end > start_char
            ):
                token_indices.append(
                    token_index
                )

        if not token_indices:
            continue

        covered_start = min(
            offsets[token_index][0]
            for token_index in token_indices
        )

        covered_end = max(
            offsets[token_index][1]
            for token_index in token_indices
        )

        # Require the window to cover the complete mention.
        if (
            covered_start > start_char
            or covered_end < end_char
        ):
            continue

        hidden = window_hidden_states[
            window_row_index
        ]

        pooled = hidden[
            token_indices,
            :,
        ].mean(axis=0)

        # Prefer the most compact window representation.
        excess_characters = (
            (start_char - covered_start)
            + (covered_end - end_char)
        )

        candidate_pools.append(
            (
                excess_characters,
                window_row_index,
                pooled,
            )
        )

    if not candidate_pools:
        return None, None

    candidate_pools.sort(
        key=lambda item:
        (
            item[0],
            item[1],
        )
    )

    _, selected_window, vector = (
        candidate_pools[0]
    )

    return (
        vector.astype(np.float32),
        selected_window,
    )


occurrence_rows = []
occurrence_vectors = []

mention_level_vectors = np.zeros(
    (
        len(mention_inputs),
        hidden_size,
    ),
    dtype=np.float32,
)

mention_level_matched = np.zeros(
    len(mention_inputs),
    dtype=bool,
)

mention_input_row_lookup = {
    mention_input_id: row_index
    for row_index, mention_input_id
    in enumerate(
        mention_inputs[
            "mention_input_id"
        ]
    )
}

for row in tqdm(
    mention_inputs.itertuples(index=False),
    total=len(mention_inputs),
    desc="Pooling mention occurrences",
):
    document_key = (
        row.domain,
        row.num_article,
    )

    text = document_texts[
        document_key
    ]

    spans = mention_occurrences[
        row.mention_input_id
    ]

    pooled_for_input = []

    for occurrence_index, (
        start_char,
        end_char,
    ) in enumerate(spans):
        vector, selected_window = (
            pool_occurrence_from_windows(
                document_key,
                start_char,
                end_char,
            )
        )

        if vector is None:
            continue

        left = max(
            0,
            start_char - CONTEXT_CHARACTERS,
        )

        right = min(
            len(text),
            end_char + CONTEXT_CHARACTERS,
        )

        context = re.sub(
            r"\s+",
            " ",
            text[left:right],
        ).strip()

        occurrence_vector_index = len(
            occurrence_vectors
        )

        occurrence_vectors.append(
            vector
        )

        pooled_for_input.append(
            vector
        )

        occurrence_rows.append(
            {
                "occurrence_vector_index":
                    occurrence_vector_index,
                "mention_input_id":
                    row.mention_input_id,
                "domain":
                    row.domain,
                "num_article":
                    row.num_article,
                "doi_normalized":
                    row.doi_normalized,
                "mention":
                    row.mention,
                "mention_normalized":
                    row.mention_normalized,
                "occurrence_index":
                    occurrence_index,
                "start_char":
                    start_char,
                "end_char":
                    end_char,
                "selected_window_row":
                    selected_window,
                "context":
                    context,
            }
        )

    mention_row_index = (
        mention_input_row_lookup[
            row.mention_input_id
        ]
    )

    if pooled_for_input:
        mention_level_vectors[
            mention_row_index
        ] = np.mean(
            np.vstack(
                pooled_for_input
            ),
            axis=0,
        )

        mention_level_matched[
            mention_row_index
        ] = True


# ------------------------------------------------------------
# 9. Mention-only fallback for unmatched inputs
# ------------------------------------------------------------

unmatched_indices = np.where(
    ~mention_level_matched
)[0]

if len(unmatched_indices):
    unmatched_mentions = (
        mention_inputs.iloc[
            unmatched_indices
        ]["mention"]
        .astype(str)
        .tolist()
    )

    for batch_start in tqdm(
        range(
            0,
            len(unmatched_mentions),
            64,
        ),
        desc="Encoding unmatched mentions",
    ):
        batch_mentions = unmatched_mentions[
            batch_start:
            batch_start + 64
        ]

        encoded = tokenizer(
            batch_mentions,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(device)
            for key, value
            in encoded.items()
        }

        with torch.no_grad():
            if use_fp16:
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    outputs = model(
                        **encoded
                    )
            else:
                outputs = model(
                    **encoded
                )

        hidden = (
            outputs.last_hidden_state
            .float()
        )

        attention = encoded[
            "attention_mask"
        ].bool()

        # Remove CLS and SEP from pooling.
        token_mask = attention.clone()
        token_mask[:, 0] = False

        sequence_lengths = (
            attention.sum(dim=1)
        )

        for batch_index, length in enumerate(
            sequence_lengths.tolist()
        ):
            if length > 1:
                token_mask[
                    batch_index,
                    length - 1,
                ] = False

        denominator = (
            token_mask.sum(
                dim=1,
                keepdim=True,
            )
            .clamp(min=1)
        )

        pooled = (
            hidden
            * token_mask.unsqueeze(-1)
        ).sum(dim=1) / denominator

        pooled = (
            pooled.cpu()
            .numpy()
            .astype(np.float32)
        )

        target_indices = unmatched_indices[
            batch_start:
            batch_start + len(
                batch_mentions
            )
        ]

        mention_level_vectors[
            target_indices
        ] = pooled


# ------------------------------------------------------------
# 10. Final audit
# ------------------------------------------------------------

occurrence_table = pd.DataFrame(
    occurrence_rows
)

mention_evidence_table = (
    mention_inputs.copy()
)

mention_evidence_table[
    "embedding_row"
] = np.arange(
    len(mention_evidence_table)
)

mention_evidence_table[
    "literal_occurrence_count"
] = (
    mention_evidence_table[
        "mention_input_id"
    ]
    .map(
        occurrence_table.groupby(
            "mention_input_id"
        )
        .size()
    )
    .fillna(0)
    .astype(int)
)

mention_evidence_table[
    "literal_alignment"
] = (
    mention_level_matched
)

mention_evidence_table[
    "fallback_to_mention_only"
] = (
    ~mention_level_matched
)

print("\n" + "=" * 78)
print("FULL-CORPUS OCCURRENCE-EVIDENCE AUDIT")
print("=" * 78)

print(
    "Mention-level embedding shape:",
    mention_level_vectors.shape,
)

print(
    "Occurrence-level embedding shape:",
    (
        len(occurrence_vectors),
        hidden_size,
    ),
)

print(
    "Literal occurrence records:",
    len(occurrence_table),
)

print(
    "Inputs with literal alignment:",
    int(
        mention_level_matched.sum()
    ),
)

print(
    "Inputs using mention-only fallback:",
    int(
        (~mention_level_matched).sum()
    ),
)

print(
    "All mention vectors finite:",
    bool(
        np.isfinite(
            mention_level_vectors
        ).all()
    ),
)

if occurrence_vectors:
    occurrence_matrix = np.vstack(
        occurrence_vectors
    ).astype(np.float32)
else:
    occurrence_matrix = np.empty(
        (
            0,
            hidden_size,
        ),
        dtype=np.float32,
    )

print(
    "All occurrence vectors finite:",
    bool(
        np.isfinite(
            occurrence_matrix
        ).all()
    ),
)

print(
    "Maximum literal occurrences for one input:",
    int(
        mention_evidence_table[
            "literal_occurrence_count"
        ].max()
    ),
)

print(
    "Elapsed encoding minutes:",
    round(
        elapsed_minutes,
        2,
    ),
)

if len(unmatched_indices):
    print("\nMention-only fallback inputs:")

    display(
        mention_evidence_table.loc[
            mention_evidence_table[
                "fallback_to_mention_only"
            ],
            [
                "mention_input_id",
                "domain",
                "num_article",
                "mention",
            ],
        ]
    )


# ------------------------------------------------------------
# 11. Save evidence artifacts
# ------------------------------------------------------------

MENTION_EMBEDDING_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_mentions_contextual_embeddings.npy"
)

OCCURRENCE_EMBEDDING_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_literal_occurrence_embeddings.npy"
)

MENTION_EVIDENCE_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_mentions_evidence_rows.csv"
)

OCCURRENCE_EVIDENCE_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_literal_occurrence_rows.csv"
)

METADATA_OUTPUT_PATH = (
    RESULTS_DIR
    / "taskB_pubmedbert_full_corpus_evidence_metadata.json"
)

np.save(
    MENTION_EMBEDDING_PATH,
    mention_level_vectors,
)

np.save(
    OCCURRENCE_EMBEDDING_PATH,
    occurrence_matrix,
)

mention_evidence_table.to_csv(
    MENTION_EVIDENCE_PATH,
    index=False,
)

occurrence_table.to_csv(
    OCCURRENCE_EVIDENCE_PATH,
    index=False,
)

metadata = {
    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "stride": STRIDE,
    "context_characters_each_side":
        CONTEXT_CHARACTERS,
    "documents": int(
        len(document_rows)
    ),
    "mention_inputs": int(
        len(mention_evidence_table)
    ),
    "literal_occurrence_records": int(
        len(occurrence_table)
    ),
    "literal_aligned_inputs": int(
        mention_level_matched.sum()
    ),
    "mention_only_fallback_inputs": int(
        (~mention_level_matched).sum()
    ),
    "hidden_size": hidden_size,
    "elapsed_minutes": elapsed_minutes,
    "reference_definitions_accessed": False,
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved mention embeddings to:")
print(MENTION_EMBEDDING_PATH)

print("\nSaved occurrence embeddings to:")
print(OCCURRENCE_EMBEDDING_PATH)

print("\nSaved mention evidence rows to:")
print(MENTION_EVIDENCE_PATH)

print("\nSaved occurrence evidence rows to:")
print(OCCURRENCE_EVIDENCE_PATH)

print("\nSaved embedding metadata to:")
print(METADATA_OUTPUT_PATH)

print("\nCell 18 completed successfully.")

In [ ]:
# ============================================================
# CELL 19 — TRAINING-ONLY COLLAPSE FIT FOR DEFINITION EVIDENCE
# ============================================================

from pathlib import Path
import json
import re
import unicodedata

import joblib
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = (
    BASE_DIR
    / "journal_experiments"
    / "taskB_holdout"
)

RESULTS_DIR = EXPERIMENT_DIR / "results"
CACHE_DIR = EXPERIMENT_DIR / "cache"
SPLITS_DIR = EXPERIMENT_DIR / "splits"
MODELS_DIR = EXPERIMENT_DIR / "models"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

GOLD_LINKS_PATH = (
    RESULTS_DIR
    / "taskB_authoritative_unique_gold_links.csv"
)

MENTION_EVIDENCE_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_mentions_evidence_rows.csv"
)

OCCURRENCE_ROWS_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_literal_occurrence_rows.csv"
)

OCCURRENCE_EMBEDDINGS_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_literal_occurrence_embeddings.npy"
)

CONCEPT_SPLIT_PATH = (
    SPLITS_DIR
    / "taskB_definition_concept_holdout_70_15_15.csv"
)

N_CLUSTERS = 100
RANDOM_SEED = 42

gold_links = pd.read_csv(
    GOLD_LINKS_PATH
)

mention_evidence = pd.read_csv(
    MENTION_EVIDENCE_PATH
)

occurrence_rows = pd.read_csv(
    OCCURRENCE_ROWS_PATH
)

occurrence_embeddings = np.load(
    OCCURRENCE_EMBEDDINGS_PATH
)

concept_split = pd.read_csv(
    CONCEPT_SPLIT_PATH
)


# ------------------------------------------------------------
# 2. Basic provenance checks
# ------------------------------------------------------------

if len(occurrence_rows) != len(
    occurrence_embeddings
):
    raise ValueError(
        "Occurrence-row count does not match "
        "occurrence-embedding count."
    )

if not np.array_equal(
    occurrence_rows[
        "occurrence_vector_index"
    ].to_numpy(),
    np.arange(len(occurrence_rows)),
):
    raise ValueError(
        "Occurrence-vector indices are not aligned "
        "with the embedding matrix."
    )

if not np.isfinite(
    occurrence_embeddings
).all():
    raise ValueError(
        "Occurrence embeddings contain nonfinite values."
    )


# ------------------------------------------------------------
# 3. Normalization
# ------------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip()

    text = text.casefold()
    text = re.sub(r"\s+", " ", text)

    return text


def normalize_concept(value):
    text = normalize_text(value)

    if text.startswith("<") and text.endswith(">"):
        text = text[1:-1].strip()

    return text


gold_links["num_article"] = (
    gold_links["num_article"]
    .astype(str)
)

mention_evidence["num_article"] = (
    mention_evidence["num_article"]
    .astype(str)
)

occurrence_rows["num_article"] = (
    occurrence_rows["num_article"]
    .astype(str)
)

gold_links["mention_key"] = (
    gold_links["mention"]
    .apply(normalize_text)
)

mention_evidence["mention_key"] = (
    mention_evidence["mention"]
    .apply(normalize_text)
)

gold_links["concept_key"] = (
    gold_links["concept_canonical"]
    .apply(normalize_concept)
)

concept_split["concept_key"] = (
    concept_split["concept_key"]
    .apply(normalize_concept)
)


# ------------------------------------------------------------
# 4. Recover mention_input_id for every semantic link
# ------------------------------------------------------------

mention_lookup = (
    mention_evidence[
        [
            "mention_input_id",
            "domain",
            "num_article",
            "mention_key",
        ]
    ]
    .drop_duplicates()
)

if mention_lookup.duplicated(
    [
        "domain",
        "num_article",
        "mention_key",
    ]
).any():
    duplicates = mention_lookup.loc[
        mention_lookup.duplicated(
            [
                "domain",
                "num_article",
                "mention_key",
            ],
            keep=False,
        )
    ]

    raise ValueError(
        "Mention lookup is not unique:\n"
        + duplicates.head(20).to_string(
            index=False
        )
    )

links_with_ids = gold_links.merge(
    mention_lookup,
    on=[
        "domain",
        "num_article",
        "mention_key",
    ],
    how="left",
    validate="many_to_one",
)

if links_with_ids[
    "mention_input_id"
].isna().any():
    missing = links_with_ids.loc[
        links_with_ids[
            "mention_input_id"
        ].isna(),
        [
            "domain",
            "num_article",
            "mention",
            "concept_canonical",
        ],
    ]

    raise ValueError(
        "Some semantic links could not be mapped "
        "to mention_input_id:\n"
        + missing.head(20).to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# 5. Attach concept-level definition split
# ------------------------------------------------------------

split_lookup = (
    concept_split[
        [
            "concept_key",
            "split",
        ]
    ]
    .drop_duplicates("concept_key")
)

links_with_ids = links_with_ids.merge(
    split_lookup,
    on="concept_key",
    how="left",
    validate="many_to_one",
)

if links_with_ids["split"].isna().any():
    missing_concepts = (
        links_with_ids.loc[
            links_with_ids["split"].isna(),
            "concept_key",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Concepts missing from definition split: "
        + ", ".join(missing_concepts[:20])
    )


# ------------------------------------------------------------
# 6. Determine strict training-fit occurrences
# ------------------------------------------------------------

# Every literal occurrence inherits all concept partitions
# attached to its mention input.
mention_partition_sets = (
    links_with_ids.groupby(
        "mention_input_id"
    )["split"]
    .agg(
        lambda values:
        frozenset(values)
    )
)

occurrence_rows[
    "linked_concept_splits"
] = (
    occurrence_rows[
        "mention_input_id"
    ]
    .map(
        mention_partition_sets
    )
)

if occurrence_rows[
    "linked_concept_splits"
].isna().any():
    raise ValueError(
        "Some occurrences lack linked concept partitions."
    )

occurrence_rows[
    "strict_training_fit_eligible"
] = (
    occurrence_rows[
        "linked_concept_splits"
    ]
    .apply(
        lambda values:
        values == frozenset({"train"})
    )
)

training_fit_indices = (
    occurrence_rows.loc[
        occurrence_rows[
            "strict_training_fit_eligible"
        ],
        "occurrence_vector_index",
    ]
    .astype(int)
    .to_numpy()
)

if len(training_fit_indices) < N_CLUSTERS:
    raise ValueError(
        "Too few strict training occurrences for "
        f"K={N_CLUSTERS}: {len(training_fit_indices)}"
    )

training_fit_embeddings = (
    occurrence_embeddings[
        training_fit_indices
    ]
)


# ------------------------------------------------------------
# 7. Fit frozen scaler and K-means state
# ------------------------------------------------------------

print("=" * 78)
print("TRAINING-ONLY DEFINITION-EVIDENCE COLLAPSE FIT")
print("=" * 78)

print(
    "All literal occurrences:",
    len(occurrence_rows),
)

print(
    "Strict training-fit occurrences:",
    len(training_fit_indices),
)

print(
    "Occurrences excluded from fitting:",
    len(occurrence_rows)
    - len(training_fit_indices),
)

print("Clusters:", N_CLUSTERS)
print("Random seed:", RANDOM_SEED)

scaler = StandardScaler()

training_scaled = scaler.fit_transform(
    training_fit_embeddings
)

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=20,
    max_iter=500,
    algorithm="lloyd",
)

kmeans.fit(training_scaled)

all_scaled = scaler.transform(
    occurrence_embeddings
)

all_distances = kmeans.transform(
    all_scaled
)

nearest_cluster = all_distances.argmin(
    axis=1
)

nearest_distance = all_distances.min(
    axis=1
)


# ------------------------------------------------------------
# 8. Training-derived distance temperature
# ------------------------------------------------------------

training_nearest_distance = (
    nearest_distance[
        training_fit_indices
    ]
)

training_q25 = float(
    np.quantile(
        training_nearest_distance,
        0.25,
    )
)

training_q75 = float(
    np.quantile(
        training_nearest_distance,
        0.75,
    )
)

distance_temperature = (
    training_q75 - training_q25
)

if distance_temperature <= 0:
    distance_temperature = float(
        np.std(
            training_nearest_distance
        )
    )

if distance_temperature <= 0:
    raise ValueError(
        "Could not derive a positive collapse "
        "temperature."
    )

# Higher value = closer to a learned state centroid.
collapse_score = np.exp(
    -nearest_distance
    / distance_temperature
)


# ------------------------------------------------------------
# 9. Attach scores to occurrence evidence
# ------------------------------------------------------------

scored_occurrences = occurrence_rows.copy()

scored_occurrences[
    "collapse_cluster"
] = nearest_cluster

scored_occurrences[
    "collapse_distance"
] = nearest_distance

scored_occurrences[
    "collapse_score"
] = collapse_score

scored_occurrences[
    "linked_concept_splits"
] = (
    scored_occurrences[
        "linked_concept_splits"
    ]
    .apply(
        lambda values:
        " | ".join(sorted(values))
    )
)


# ------------------------------------------------------------
# 10. Expand occurrences to concept-specific evidence
# ------------------------------------------------------------

concept_evidence = (
    scored_occurrences.merge(
        links_with_ids[
            [
                "mention_input_id",
                "concept_key",
                "concept_canonical",
                "split",
            ]
        ].drop_duplicates(),
        on="mention_input_id",
        how="inner",
        validate="many_to_many",
    )
)

# The same literal context can occur under more than one
# concept for legitimate multi-label mention inputs.
concept_evidence = (
    concept_evidence.drop_duplicates(
        [
            "occurrence_vector_index",
            "concept_key",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Concept-level evidence summary
# ------------------------------------------------------------

concept_evidence_summary = (
    concept_evidence.groupby(
        [
            "split",
            "concept_key",
            "concept_canonical",
        ],
        as_index=False,
    )
    .agg(
        occurrence_evidence_count=(
            "occurrence_vector_index",
            "size",
        ),
        unique_context_count=(
            "context",
            "nunique",
        ),
        unique_mention_count=(
            "mention",
            "nunique",
        ),
        occupied_cluster_count=(
            "collapse_cluster",
            "nunique",
        ),
        collapse_min=(
            "collapse_score",
            "min",
        ),
        collapse_median=(
            "collapse_score",
            "median",
        ),
        collapse_max=(
            "collapse_score",
            "max",
        ),
        distance_median=(
            "collapse_distance",
            "median",
        ),
    )
)


# ------------------------------------------------------------
# 12. Audit
# ------------------------------------------------------------

cluster_sizes = np.bincount(
    kmeans.labels_,
    minlength=N_CLUSTERS,
)

print("\nK-means iterations:", kmeans.n_iter_)

print(
    "Training distance temperature:",
    round(
        distance_temperature,
        6,
    ),
)

print(
    "Training cluster-size minimum:",
    int(cluster_sizes.min()),
)

print(
    "Training cluster-size median:",
    float(
        np.median(cluster_sizes)
    ),
)

print(
    "Training cluster-size maximum:",
    int(cluster_sizes.max()),
)

print("\nEvidence rows by concept split:")
print(
    concept_evidence.groupby("split")
    .size()
)

print("\nConcepts represented by split:")
print(
    concept_evidence.groupby("split")[
        "concept_key"
    ]
    .nunique()
)

print("\nConcept evidence-count percentiles:")
display(
    concept_evidence_summary.groupby(
        "split"
    )[
        "occurrence_evidence_count"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .round(3)
)

print("\nCollapse-score percentiles by split:")
display(
    concept_evidence.groupby("split")[
        "collapse_score"
    ]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
        ]
    )
    .round(4)
)

print("\nDevelopment concepts with highest evidence:")
display(
    concept_evidence_summary.loc[
        concept_evidence_summary[
            "split"
        ].eq("development")
    ]
    .sort_values(
        "occurrence_evidence_count",
        ascending=False,
    )
    .head(20)
)


# ------------------------------------------------------------
# 13. Save frozen state and evidence
# ------------------------------------------------------------

SCALER_PATH = (
    MODELS_DIR
    / "taskB_definition_collapse_scaler_K100.joblib"
)

KMEANS_PATH = (
    MODELS_DIR
    / "taskB_definition_collapse_kmeans_K100_seed42.joblib"
)

SCORED_OCCURRENCE_PATH = (
    RESULTS_DIR
    / "taskB_definition_occurrences_with_collapse_K100.csv"
)

CONCEPT_EVIDENCE_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_evidence_with_collapse_K100.csv"
)

CONCEPT_SUMMARY_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_evidence_summary_K100.csv"
)

METADATA_PATH = (
    RESULTS_DIR
    / "taskB_definition_collapse_K100_metadata.json"
)

joblib.dump(
    scaler,
    SCALER_PATH,
)

joblib.dump(
    kmeans,
    KMEANS_PATH,
)

scored_occurrences.to_csv(
    SCORED_OCCURRENCE_PATH,
    index=False,
)

concept_evidence.to_csv(
    CONCEPT_EVIDENCE_PATH,
    index=False,
)

concept_evidence_summary.to_csv(
    CONCEPT_SUMMARY_PATH,
    index=False,
)

metadata = {
    "clusters": N_CLUSTERS,
    "random_seed": RANDOM_SEED,
    "embedding_dimensions": int(
        occurrence_embeddings.shape[1]
    ),
    "all_literal_occurrences": int(
        len(occurrence_rows)
    ),
    "strict_training_fit_occurrences": int(
        len(training_fit_indices)
    ),
    "excluded_from_fit": int(
        len(occurrence_rows)
        - len(training_fit_indices)
    ),
    "fit_rule": (
        "Occurrence is used for scaler and K-means "
        "fitting only when every linked concept belongs "
        "to the definition-training partition."
    ),
    "temperature_source": (
        "Training-only nearest-centroid distance IQR"
    ),
    "distance_temperature": float(
        distance_temperature
    ),
    "kmeans_iterations": int(
        kmeans.n_iter_
    ),
    "cluster_size_minimum": int(
        cluster_sizes.min()
    ),
    "cluster_size_median": float(
        np.median(cluster_sizes)
    ),
    "cluster_size_maximum": int(
        cluster_sizes.max()
    ),
    "reference_definitions_used_for_fit": False,
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved scaler to:")
print(SCALER_PATH)

print("\nSaved K-means state to:")
print(KMEANS_PATH)

print("\nSaved scored occurrences to:")
print(SCORED_OCCURRENCE_PATH)

print("\nSaved concept evidence to:")
print(CONCEPT_EVIDENCE_PATH)

print("\nSaved concept evidence summary to:")
print(CONCEPT_SUMMARY_PATH)

print("\nSaved collapse metadata to:")
print(METADATA_PATH)

print("\nCell 19 completed successfully.")

In [ ]:
# ============================================================
# CELL 19A — ADD MENTION-ONLY FALLBACK EVIDENCE
# ============================================================

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load frozen resources
# ------------------------------------------------------------

MENTION_EMBEDDINGS_PATH = (
    CACHE_DIR
    / "taskB_pubmedbert_all_mentions_contextual_embeddings.npy"
)

SCALER_PATH = (
    MODELS_DIR
    / "taskB_definition_collapse_scaler_K100.joblib"
)

KMEANS_PATH = (
    MODELS_DIR
    / "taskB_definition_collapse_kmeans_K100_seed42.joblib"
)

COLLAPSE_METADATA_PATH = (
    RESULTS_DIR
    / "taskB_definition_collapse_K100_metadata.json"
)

mention_embeddings = np.load(
    MENTION_EMBEDDINGS_PATH
)

frozen_scaler = joblib.load(
    SCALER_PATH
)

frozen_kmeans = joblib.load(
    KMEANS_PATH
)

with open(
    COLLAPSE_METADATA_PATH,
    "r",
    encoding="utf-8",
) as input_file:
    collapse_metadata = json.load(
        input_file
    )

frozen_temperature = float(
    collapse_metadata[
        "distance_temperature"
    ]
)

if len(mention_embeddings) != len(
    mention_evidence
):
    raise ValueError(
        "Mention embedding count does not match "
        "mention evidence rows."
    )


# ------------------------------------------------------------
# 2. Identify mention-only fallback inputs
# ------------------------------------------------------------

fallback_mentions = (
    mention_evidence.loc[
        mention_evidence[
            "fallback_to_mention_only"
        ].astype(bool)
    ]
    .copy()
)

if fallback_mentions.empty:
    print(
        "No mention-only fallback evidence is required."
    )

else:
    fallback_embedding_indices = (
        fallback_mentions[
            "embedding_row"
        ]
        .astype(int)
        .to_numpy()
    )

    fallback_vectors = (
        mention_embeddings[
            fallback_embedding_indices
        ]
    )

    fallback_scaled = (
        frozen_scaler.transform(
            fallback_vectors
        )
    )

    fallback_distances = (
        frozen_kmeans.transform(
            fallback_scaled
        )
    )

    fallback_clusters = (
        fallback_distances.argmin(
            axis=1
        )
    )

    fallback_nearest_distances = (
        fallback_distances.min(
            axis=1
        )
    )

    fallback_collapse_scores = np.exp(
        -fallback_nearest_distances
        / frozen_temperature
    )


# ------------------------------------------------------------
# 3. Create synthetic evidence rows
# ------------------------------------------------------------

fallback_rows = []

if not fallback_mentions.empty:
    for fallback_position, (
        row_index,
        row,
    ) in enumerate(
        fallback_mentions.iterrows()
    ):
        partition_set = (
            mention_partition_sets.get(
                row["mention_input_id"],
                frozenset(),
            )
        )

        fallback_rows.append(
            {
                # Negative indices cannot collide with
                # literal-occurrence vector indices.
                "occurrence_vector_index":
                    -(fallback_position + 1),
                "mention_input_id":
                    row["mention_input_id"],
                "domain":
                    row["domain"],
                "num_article":
                    str(row["num_article"]),
                "doi_normalized":
                    row["doi_normalized"],
                "mention":
                    row["mention"],
                "mention_normalized":
                    row["mention_normalized"],
                "occurrence_index":
                    -1,
                "start_char":
                    np.nan,
                "end_char":
                    np.nan,
                "selected_window_row":
                    np.nan,
                # No document passage is presented as
                # evidence when the mention was not found.
                "context":
                    "",
                "linked_concept_splits":
                    " | ".join(
                        sorted(partition_set)
                    ),
                "strict_training_fit_eligible":
                    False,
                "collapse_cluster":
                    int(
                        fallback_clusters[
                            fallback_position
                        ]
                    ),
                "collapse_distance":
                    float(
                        fallback_nearest_distances[
                            fallback_position
                        ]
                    ),
                "collapse_score":
                    float(
                        fallback_collapse_scores[
                            fallback_position
                        ]
                    ),
                "evidence_source":
                    "mention_only_fallback",
            }
        )

fallback_scored = pd.DataFrame(
    fallback_rows
)


# ------------------------------------------------------------
# 4. Append without changing literal evidence
# ------------------------------------------------------------

scored_occurrences_complete = (
    scored_occurrences.copy()
)

scored_occurrences_complete[
    "evidence_source"
] = "literal_occurrence"

if len(fallback_scored):
    scored_occurrences_complete = pd.concat(
        [
            scored_occurrences_complete,
            fallback_scored,
        ],
        ignore_index=True,
        sort=False,
    )


# ------------------------------------------------------------
# 5. Expand complete evidence to concepts
# ------------------------------------------------------------

concept_evidence_complete = (
    scored_occurrences_complete.merge(
        links_with_ids[
            [
                "mention_input_id",
                "concept_key",
                "concept_canonical",
                "split",
            ]
        ]
        .drop_duplicates(),
        on="mention_input_id",
        how="inner",
        validate="many_to_many",
    )
)

concept_evidence_complete = (
    concept_evidence_complete
    .drop_duplicates(
        [
            "occurrence_vector_index",
            "concept_key",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Rebuild complete concept summary
# ------------------------------------------------------------

concept_evidence_summary_complete = (
    concept_evidence_complete.groupby(
        [
            "split",
            "concept_key",
            "concept_canonical",
        ],
        as_index=False,
    )
    .agg(
        total_evidence_count=(
            "occurrence_vector_index",
            "size",
        ),
        literal_occurrence_count=(
            "evidence_source",
            lambda values:
            int(
                (
                    values
                    == "literal_occurrence"
                ).sum()
            ),
        ),
        fallback_evidence_count=(
            "evidence_source",
            lambda values:
            int(
                (
                    values
                    == "mention_only_fallback"
                ).sum()
            ),
        ),
        unique_context_count=(
            "context",
            lambda values:
            int(
                pd.Series(values)
                .replace("", np.nan)
                .nunique()
            ),
        ),
        unique_mention_count=(
            "mention",
            "nunique",
        ),
        occupied_cluster_count=(
            "collapse_cluster",
            "nunique",
        ),
        collapse_min=(
            "collapse_score",
            "min",
        ),
        collapse_median=(
            "collapse_score",
            "median",
        ),
        collapse_max=(
            "collapse_score",
            "max",
        ),
        distance_median=(
            "collapse_distance",
            "median",
        ),
    )
)


# ------------------------------------------------------------
# 7. Coverage audit against authoritative concept split
# ------------------------------------------------------------

expected_concepts = {
    split_name: set(
        concept_split.loc[
            concept_split[
                "split"
            ].eq(split_name),
            "concept_key",
        ]
    )
    for split_name in [
        "train",
        "development",
        "test",
    ]
}

represented_concepts = {
    split_name: set(
        concept_evidence_complete.loc[
            concept_evidence_complete[
                "split"
            ].eq(split_name),
            "concept_key",
        ]
    )
    for split_name in [
        "train",
        "development",
        "test",
    ]
}

coverage_records = []

for split_name in [
    "train",
    "development",
    "test",
]:
    missing = (
        expected_concepts[split_name]
        - represented_concepts[split_name]
    )

    coverage_records.append(
        {
            "split": split_name,
            "expected_concepts": len(
                expected_concepts[
                    split_name
                ]
            ),
            "represented_concepts": len(
                represented_concepts[
                    split_name
                ]
            ),
            "missing_concepts": len(
                missing
            ),
            "missing_concept_labels":
                " | ".join(sorted(missing)),
        }
    )

coverage_audit = pd.DataFrame(
    coverage_records
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("=" * 78)
print("COMPLETE DEFINITION-EVIDENCE COVERAGE")
print("=" * 78)

print(
    "Literal occurrence evidence rows:",
    int(
        (
            scored_occurrences_complete[
                "evidence_source"
            ]
            == "literal_occurrence"
        ).sum()
    ),
)

print(
    "Mention-only fallback evidence rows:",
    int(
        (
            scored_occurrences_complete[
                "evidence_source"
            ]
            == "mention_only_fallback"
        ).sum()
    ),
)

print(
    "Concept-expanded evidence rows:",
    len(concept_evidence_complete),
)

print("\nConcept coverage:")
display(coverage_audit)

print("\nFallback evidence by concept split:")
print(
    concept_evidence_complete.loc[
        concept_evidence_complete[
            "evidence_source"
        ].eq("mention_only_fallback")
    ]
    .groupby("split")
    .size()
)

print("\nConcepts relying on fallback evidence:")
display(
    concept_evidence_summary_complete.loc[
        concept_evidence_summary_complete[
            "fallback_evidence_count"
        ] > 0,
        [
            "split",
            "concept_key",
            "concept_canonical",
            "total_evidence_count",
            "literal_occurrence_count",
            "fallback_evidence_count",
            "collapse_median",
        ],
    ]
    .sort_values(
        [
            "split",
            "concept_key",
        ]
    )
)

if coverage_audit[
    "missing_concepts"
].sum() != 0:
    raise ValueError(
        "Some split concepts still lack evidence."
    )


# ------------------------------------------------------------
# 9. Overwrite evidence artifacts with complete versions
# ------------------------------------------------------------

COMPLETE_OCCURRENCE_PATH = (
    RESULTS_DIR
    / "taskB_definition_occurrences_with_collapse_K100_complete.csv"
)

COMPLETE_CONCEPT_EVIDENCE_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_evidence_with_collapse_K100_complete.csv"
)

COMPLETE_SUMMARY_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_evidence_summary_K100_complete.csv"
)

COVERAGE_AUDIT_PATH = (
    RESULTS_DIR
    / "taskB_definition_evidence_coverage_audit.csv"
)

COMPLETE_METADATA_PATH = (
    RESULTS_DIR
    / "taskB_definition_collapse_K100_complete_metadata.json"
)

scored_occurrences_complete.to_csv(
    COMPLETE_OCCURRENCE_PATH,
    index=False,
)

concept_evidence_complete.to_csv(
    COMPLETE_CONCEPT_EVIDENCE_PATH,
    index=False,
)

concept_evidence_summary_complete.to_csv(
    COMPLETE_SUMMARY_PATH,
    index=False,
)

coverage_audit.to_csv(
    COVERAGE_AUDIT_PATH,
    index=False,
)

complete_metadata = {
    **collapse_metadata,
    "literal_occurrence_evidence_rows": int(
        (
            scored_occurrences_complete[
                "evidence_source"
            ]
            == "literal_occurrence"
        ).sum()
    ),
    "mention_only_fallback_evidence_rows": int(
        (
            scored_occurrences_complete[
                "evidence_source"
            ]
            == "mention_only_fallback"
        ).sum()
    ),
    "all_split_concepts_represented": True,
    "fallback_evidence_used_for_model_fit": False,
}

with open(
    COMPLETE_METADATA_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        complete_metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved complete occurrence evidence to:")
print(COMPLETE_OCCURRENCE_PATH)

print("\nSaved complete concept evidence to:")
print(COMPLETE_CONCEPT_EVIDENCE_PATH)

print("\nSaved complete concept summary to:")
print(COMPLETE_SUMMARY_PATH)

print("\nSaved coverage audit to:")
print(COVERAGE_AUDIT_PATH)

print("\nSaved complete metadata to:")
print(COMPLETE_METADATA_PATH)

print("\nCell 19A completed successfully.")

In [ ]:
# ============================================================
# CELL 20 — COLLAPSE K-SENSITIVITY FOR DEFINITION EVIDENCE
# ============================================================

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

K_VALUES = [
    25,
    50,
    100,
]

RANDOM_SEED = 42


# ------------------------------------------------------------
# 2. Reconstruct vectors for every complete evidence row
# ------------------------------------------------------------

mention_vector_lookup = {
    mention_input_id:
        mention_embeddings[
            int(embedding_row)
        ]
    for mention_input_id, embedding_row
    in zip(
        mention_evidence[
            "mention_input_id"
        ],
        mention_evidence[
            "embedding_row"
        ],
    )
}

complete_evidence_vectors = []

for row in (
    scored_occurrences_complete
    .itertuples(index=False)
):
    vector_index = int(
        row.occurrence_vector_index
    )

    if vector_index >= 0:
        vector = occurrence_embeddings[
            vector_index
        ]
    else:
        vector = mention_vector_lookup[
            row.mention_input_id
        ]

    complete_evidence_vectors.append(
        vector
    )

complete_evidence_vectors = np.vstack(
    complete_evidence_vectors
).astype(np.float32)

if complete_evidence_vectors.shape[0] != len(
    scored_occurrences_complete
):
    raise ValueError(
        "Complete evidence vectors are misaligned."
    )

if not np.isfinite(
    complete_evidence_vectors
).all():
    raise ValueError(
        "Complete evidence vectors contain "
        "nonfinite values."
    )


# ------------------------------------------------------------
# 3. Strict training-fit vector indices
# ------------------------------------------------------------

strict_fit_mask = (
    scored_occurrences_complete[
        "strict_training_fit_eligible"
    ]
    .fillna(False)
    .astype(bool)
    .to_numpy()
)

literal_mask = (
    scored_occurrences_complete[
        "evidence_source"
    ]
    .eq("literal_occurrence")
    .to_numpy()
)

fit_mask = (
    strict_fit_mask
    & literal_mask
)

fit_row_indices = np.where(
    fit_mask
)[0]

fit_vectors = complete_evidence_vectors[
    fit_row_indices
]

print("=" * 78)
print("COLLAPSE K-SENSITIVITY SETUP")
print("=" * 78)

print(
    "Complete evidence rows:",
    len(
        scored_occurrences_complete
    ),
)

print(
    "Strict literal training-fit rows:",
    len(fit_row_indices),
)

print(
    "Embedding dimensions:",
    complete_evidence_vectors.shape[1],
)

print(
    "K values:",
    K_VALUES,
)


# ------------------------------------------------------------
# 4. Fit a common training-only scaler
# ------------------------------------------------------------

sensitivity_scaler = StandardScaler()

fit_scaled = (
    sensitivity_scaler.fit_transform(
        fit_vectors
    )
)

complete_scaled = (
    sensitivity_scaler.transform(
        complete_evidence_vectors
    )
)

SENSITIVITY_SCALER_PATH = (
    MODELS_DIR
    / "taskB_definition_collapse_scaler_Ksensitivity.joblib"
)

joblib.dump(
    sensitivity_scaler,
    SENSITIVITY_SCALER_PATH,
)


# ------------------------------------------------------------
# 5. Fit and score each K
# ------------------------------------------------------------

scored_k = (
    scored_occurrences_complete
    .copy()
)

sensitivity_records = []

model_metadata = {}

for k_value in K_VALUES:
    print(
        "\n" + "-" * 78
    )
    print(f"FITTING K={k_value}")
    print("-" * 78)

    kmeans_model = KMeans(
        n_clusters=k_value,
        random_state=RANDOM_SEED,
        n_init=20,
        max_iter=500,
        algorithm="lloyd",
    )

    kmeans_model.fit(
        fit_scaled
    )

    all_distances = (
        kmeans_model.transform(
            complete_scaled
        )
    )

    nearest_cluster = (
        all_distances.argmin(
            axis=1
        )
    )

    nearest_distance = (
        all_distances.min(
            axis=1
        )
    )

    training_distance = (
        nearest_distance[
            fit_row_indices
        ]
    )

    q25 = float(
        np.quantile(
            training_distance,
            0.25,
        )
    )

    q75 = float(
        np.quantile(
            training_distance,
            0.75,
        )
    )

    temperature = q75 - q25

    if temperature <= 0:
        temperature = float(
            np.std(
                training_distance
            )
        )

    if temperature <= 0:
        raise ValueError(
            f"Invalid temperature for K={k_value}"
        )

    collapse = np.exp(
        -nearest_distance
        / temperature
    )

    scored_k[
        f"cluster_K{k_value}"
    ] = nearest_cluster

    scored_k[
        f"distance_K{k_value}"
    ] = nearest_distance

    scored_k[
        f"collapse_K{k_value}"
    ] = collapse

    cluster_sizes = np.bincount(
        kmeans_model.labels_,
        minlength=k_value,
    )

    model_path = (
        MODELS_DIR
        / (
            "taskB_definition_collapse_"
            f"kmeans_K{k_value}_seed42.joblib"
        )
    )

    joblib.dump(
        kmeans_model,
        model_path,
    )

    split_labels = (
        scored_k[
            "mention_input_id"
        ]
        .map(
            links_with_ids.groupby(
                "mention_input_id"
            )["split"]
            .agg(
                lambda values:
                (
                    "mixed"
                    if len(set(values)) > 1
                    else list(
                        set(values)
                    )[0]
                )
            )
        )
    )

    # Distribution summaries are calculated on evidence
    # attached exclusively to one partition. Mixed inputs
    # remain scored but are omitted from this comparison.
    for split_name in [
        "train",
        "development",
        "test",
    ]:
        split_mask = (
            split_labels
            .eq(split_name)
            .to_numpy()
        )

        split_values = collapse[
            split_mask
        ]

        train_values = collapse[
            split_labels
            .eq("train")
            .to_numpy()
        ]

        if len(split_values) and len(
            train_values
        ):
            ks_statistic = float(
                ks_2samp(
                    train_values,
                    split_values,
                ).statistic
            )
        else:
            ks_statistic = np.nan

        sensitivity_records.append(
            {
                "K": k_value,
                "split": split_name,
                "evidence_rows": int(
                    split_mask.sum()
                ),
                "collapse_q05": float(
                    np.quantile(
                        split_values,
                        0.05,
                    )
                ),
                "collapse_q25": float(
                    np.quantile(
                        split_values,
                        0.25,
                    )
                ),
                "collapse_median": float(
                    np.quantile(
                        split_values,
                        0.50,
                    )
                ),
                "collapse_q75": float(
                    np.quantile(
                        split_values,
                        0.75,
                    )
                ),
                "collapse_q95": float(
                    np.quantile(
                        split_values,
                        0.95,
                    )
                ),
                "train_split_ks_statistic":
                    ks_statistic,
                "temperature": temperature,
                "kmeans_iterations": int(
                    kmeans_model.n_iter_
                ),
                "cluster_size_minimum": int(
                    cluster_sizes.min()
                ),
                "cluster_size_median": float(
                    np.median(
                        cluster_sizes
                    )
                ),
                "cluster_size_maximum": int(
                    cluster_sizes.max()
                ),
                "singleton_clusters": int(
                    (
                        cluster_sizes == 1
                    ).sum()
                ),
            }
        )

    model_metadata[
        f"K{k_value}"
    ] = {
        "model_path": str(
            model_path
        ),
        "temperature": float(
            temperature
        ),
        "kmeans_iterations": int(
            kmeans_model.n_iter_
        ),
        "cluster_size_minimum": int(
            cluster_sizes.min()
        ),
        "cluster_size_median": float(
            np.median(
                cluster_sizes
            )
        ),
        "cluster_size_maximum": int(
            cluster_sizes.max()
        ),
        "singleton_clusters": int(
            (
                cluster_sizes == 1
            ).sum()
        ),
    }

    print(
        "Iterations:",
        kmeans_model.n_iter_,
    )
    print(
        "Temperature:",
        round(
            temperature,
            6,
        ),
    )
    print(
        "Cluster sizes — min:",
        int(cluster_sizes.min()),
        "median:",
        float(
            np.median(
                cluster_sizes
            )
        ),
        "max:",
        int(cluster_sizes.max()),
    )
    print(
        "Singleton clusters:",
        int(
            (
                cluster_sizes == 1
            ).sum()
        ),
    )


sensitivity_summary = pd.DataFrame(
    sensitivity_records
)


# ------------------------------------------------------------
# 6. Expand K-specific scores to concepts
# ------------------------------------------------------------

score_columns = [
    "mention_input_id",
    "occurrence_vector_index",
]

for k_value in K_VALUES:
    score_columns.extend(
        [
            f"cluster_K{k_value}",
            f"distance_K{k_value}",
            f"collapse_K{k_value}",
        ]
    )

concept_evidence_k = (
    concept_evidence_complete.drop(
        columns=[
            column
            for column
            in concept_evidence_complete.columns
            if column.startswith(
                (
                    "collapse_cluster",
                    "collapse_distance",
                    "collapse_score",
                )
            )
        ],
        errors="ignore",
    )
    .merge(
        scored_k[
            score_columns
        ],
        on=[
            "mention_input_id",
            "occurrence_vector_index",
        ],
        how="left",
        validate="many_to_one",
    )
)

for k_value in K_VALUES:
    if concept_evidence_k[
        f"collapse_K{k_value}"
    ].isna().any():
        raise ValueError(
            f"Missing concept-evidence scores for K={k_value}"
        )


# ------------------------------------------------------------
# 7. Top-context stability across K
# ------------------------------------------------------------

top_context_records = []

for (
    split_name,
    concept_key,
), group in concept_evidence_k.groupby(
    [
        "split",
        "concept_key",
    ],
    sort=True,
):
    record = {
        "split": split_name,
        "concept_key": concept_key,
        "evidence_count": len(group),
    }

    for k_value in K_VALUES:
        top_row = group.sort_values(
            [
                f"collapse_K{k_value}",
                "occurrence_vector_index",
            ],
            ascending=[
                False,
                True,
            ],
        ).iloc[0]

        record[
            f"top_evidence_K{k_value}"
        ] = (
            f"{top_row['mention_input_id']}"
            f"::{int(top_row['occurrence_vector_index'])}"
        )

    record[
        "same_top_context_all_K"
    ] = (
        len(
            {
                record[
                    f"top_evidence_K{k_value}"
                ]
                for k_value in K_VALUES
            }
        )
        == 1
    )

    top_context_records.append(
        record
    )

top_context_stability = pd.DataFrame(
    top_context_records
)


# ------------------------------------------------------------
# 8. Pairwise top-context agreement
# ------------------------------------------------------------

agreement_records = []

multi_evidence_stability = (
    top_context_stability.loc[
        top_context_stability[
            "evidence_count"
        ] >= 2
    ]
)

for first_k_index in range(
    len(K_VALUES)
):
    for second_k_index in range(
        first_k_index + 1,
        len(K_VALUES),
    ):
        first_k = K_VALUES[
            first_k_index
        ]

        second_k = K_VALUES[
            second_k_index
        ]

        for split_name, group in (
            multi_evidence_stability
            .groupby("split")
        ):
            agreement = (
                group[
                    f"top_evidence_K{first_k}"
                ]
                == group[
                    f"top_evidence_K{second_k}"
                ]
            ).mean()

            agreement_records.append(
                {
                    "split": split_name,
                    "K_first": first_k,
                    "K_second": second_k,
                    "multi_evidence_concepts":
                        len(group),
                    "top_context_agreement":
                        float(agreement),
                }
            )

top_context_agreement = pd.DataFrame(
    agreement_records
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("COLLAPSE K-SENSITIVITY SUMMARY")
print("=" * 78)

display(
    sensitivity_summary[
        [
            "K",
            "split",
            "evidence_rows",
            "collapse_q25",
            "collapse_median",
            "collapse_q75",
            "train_split_ks_statistic",
            "temperature",
            "cluster_size_minimum",
            "cluster_size_median",
            "cluster_size_maximum",
            "singleton_clusters",
        ]
    ].round(4)
)

print("\nTop-context agreement for multi-evidence concepts:")
display(
    top_context_agreement.round(4)
)

print("\nAll-K top-context stability:")
display(
    top_context_stability.groupby(
        "split"
    )[
        "same_top_context_all_K"
    ]
    .agg(
        [
            "count",
            "sum",
            "mean",
        ]
    )
    .round(4)
)


# ------------------------------------------------------------
# 10. Save K-sensitivity artifacts
# ------------------------------------------------------------

SCORED_K_PATH = (
    RESULTS_DIR
    / "taskB_definition_occurrences_collapse_K25_K50_K100.csv"
)

CONCEPT_EVIDENCE_K_PATH = (
    RESULTS_DIR
    / "taskB_definition_concept_evidence_collapse_K25_K50_K100.csv"
)

SENSITIVITY_SUMMARY_PATH = (
    RESULTS_DIR
    / "taskB_definition_collapse_K_sensitivity.csv"
)

TOP_STABILITY_PATH = (
    RESULTS_DIR
    / "taskB_definition_collapse_top_context_stability.csv"
)

TOP_AGREEMENT_PATH = (
    RESULTS_DIR
    / "taskB_definition_collapse_top_context_agreement.csv"
)

SENSITIVITY_METADATA_PATH = (
    RESULTS_DIR
    / "taskB_definition_collapse_K_sensitivity_metadata.json"
)

scored_k.to_csv(
    SCORED_K_PATH,
    index=False,
)

concept_evidence_k.to_csv(
    CONCEPT_EVIDENCE_K_PATH,
    index=False,
)

sensitivity_summary.to_csv(
    SENSITIVITY_SUMMARY_PATH,
    index=False,
)

top_context_stability.to_csv(
    TOP_STABILITY_PATH,
    index=False,
)

top_context_agreement.to_csv(
    TOP_AGREEMENT_PATH,
    index=False,
)

sensitivity_metadata = {
    "K_values": K_VALUES,
    "random_seed": RANDOM_SEED,
    "scaler_fit_rows": int(
        len(fit_row_indices)
    ),
    "scaler_fit_rule": (
        "Strict training-partition literal occurrences only"
    ),
    "fallback_rows_used_for_fit": False,
    "reference_definitions_used": False,
    "models": model_metadata,
    "K_selection_rule": (
        "No K selected from geometry alone; K will be "
        "selected using development-only definition metrics."
    ),
}

with open(
    SENSITIVITY_METADATA_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        sensitivity_metadata,
        output_file,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved K-scored occurrences to:")
print(SCORED_K_PATH)

print("\nSaved K-scored concept evidence to:")
print(CONCEPT_EVIDENCE_K_PATH)

print("\nSaved sensitivity summary to:")
print(SENSITIVITY_SUMMARY_PATH)

print("\nSaved top-context stability to:")
print(TOP_STABILITY_PATH)

print("\nSaved top-context agreement to:")
print(TOP_AGREEMENT_PATH)

print("\nSaved sensitivity metadata to:")
print(SENSITIVITY_METADATA_PATH)

print("\nCell 20 completed successfully.")

In [ ]:
# ================================================================
# CELL 21A — REPAIR AND LOCK DEVELOPMENT PROMPT MANIFEST
# ================================================================
#
# Repairs:
#   1. ALL_CONTEXTS -> UNRANKED_CONTEXTS
#   2. Cleans context snippets that begin/end mid-word.
#   3. Keeps an equal maximum of three contexts in every
#      evidence-bearing condition.
#   4. Performs a row-wise reference-leakage check.
#
# This cell makes no API calls and does not access test references.
# ================================================================

from pathlib import Path
import json
import re
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

INPUT_MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_prompt_manifest.csv"
)

LOCKED_MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_prompt_manifest_locked.csv"
)

LOCKED_METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_prompt_manifest_locked_metadata.json"
)

MAX_CONTEXTS = 3
MAX_CONTEXT_CHARACTERS = 800


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def clean_text(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def clean_context_edges(value):
    """
    Clean whitespace and remove visibly clipped partial words at
    the beginning or end of a stored character window.

    This is deliberately conservative: an edge is trimmed only
    when it looks like an incomplete lowercase word.
    """

    value = clean_text(value)

    if not value:
        return ""

    # A lowercase fragment at the beginning usually indicates that
    # the context window began in the middle of a word.
    if re.match(r"^[a-z]{1,12}\s", value):
        value = re.sub(
            r"^[^\s]+\s+",
            "",
            value,
            count=1,
        )

    # A lowercase fragment at the end, without sentence punctuation,
    # usually indicates that the window ended in the middle of a word.
    if (
        value
        and value[-1].islower()
        and not re.search(r"[.!?]['\")\]]?$", value)
        and " " in value
    ):
        value = value.rsplit(" ", 1)[0].rstrip()

    if len(value) > MAX_CONTEXT_CHARACTERS:
        value = value[:MAX_CONTEXT_CHARACTERS]

        if " " in value:
            value = value.rsplit(" ", 1)[0].rstrip()

        value += " …"

    return value


def unique_preserving_order(values):
    output = []
    seen = set()

    for value in values:
        value = clean_context_edges(value)

        if not value:
            continue

        key = value.casefold()

        if key not in seen:
            seen.add(key)
            output.append(value)

    return output


def parse_contexts(value):
    if pd.isna(value) or not str(value).strip():
        return []

    parsed = json.loads(value)

    if not isinstance(parsed, list):
        raise ValueError(
            "selected_contexts_json must contain a JSON list."
        )

    return unique_preserving_order(parsed)[:MAX_CONTEXTS]


def format_context_block(contexts):
    if not contexts:
        return "(No contextual occurrence evidence was available.)"

    return "\n".join(
        f"{index}. {context}"
        for index, context in enumerate(contexts, start=1)
    )


def make_prompt(concept_label, contexts, condition):
    concept_label = clean_text(concept_label)

    if condition == "CONCEPT_ONLY":
        evidence_section = (
            "No occurrence context is supplied for this condition."
        )
    elif contexts:
        evidence_section = (
            "Occurrence evidence:\n"
            f"{format_context_block(contexts)}"
        )
    else:
        evidence_section = (
            "No literal occurrence context was available for this "
            "concept. Base the definition on the concept label alone."
        )

    return (
        "Write one concise definitional sentence for the biomedical "
        "or scientific concept below.\n\n"
        f"Concept: {concept_label}\n\n"
        f"{evidence_section}\n\n"
        "Requirements:\n"
        "- Define the concept itself, not the surrounding article.\n"
        "- Use the occurrence evidence only when it is informative.\n"
        "- Do not mention that evidence, a passage, an article, or a dataset.\n"
        "- Do not add citations, examples, headings, or bullet points.\n"
        "- Return only one self-contained sentence."
    )


# ----------------------------------------------------------------
# LOAD AND REPAIR
# ----------------------------------------------------------------

if not INPUT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Input manifest not found:\n{INPUT_MANIFEST_PATH}"
    )

manifest = pd.read_csv(INPUT_MANIFEST_PATH)

required_columns = {
    "concept_key",
    "concept_canonical",
    "definition_split",
    "condition",
    "available_unique_contexts",
    "selected_contexts_json",
    "reference_definition",
}

missing_columns = required_columns.difference(
    manifest.columns
)

if missing_columns:
    raise ValueError(
        "Manifest is missing required columns:\n"
        f"{sorted(missing_columns)}"
    )

manifest = manifest.copy()

manifest["condition"] = manifest["condition"].replace(
    {
        "ALL_CONTEXTS": "UNRANKED_CONTEXTS",
    }
)

manifest["_parsed_contexts"] = (
    manifest["selected_contexts_json"]
    .map(parse_contexts)
)

manifest["selected_context_count"] = (
    manifest["_parsed_contexts"]
    .map(len)
)

manifest["selected_contexts_json"] = (
    manifest["_parsed_contexts"]
    .map(
        lambda values: json.dumps(
            values,
            ensure_ascii=False,
        )
    )
)

manifest["prompt"] = manifest.apply(
    lambda row: make_prompt(
        concept_label=row["concept_canonical"],
        contexts=row["_parsed_contexts"],
        condition=row["condition"],
    ),
    axis=1,
)

manifest = manifest.drop(
    columns=["_parsed_contexts"]
)


# ----------------------------------------------------------------
# LOCKED AUDIT
# ----------------------------------------------------------------

expected_conditions = {
    "CONCEPT_ONLY",
    "UNRANKED_CONTEXTS",
    "COLLAPSE_K25",
    "COLLAPSE_K50",
    "COLLAPSE_K100",
}

development_concepts = manifest[
    "concept_key"
].nunique()

condition_counts = (
    manifest.groupby("condition")[
        "concept_key"
    ]
    .nunique()
)

if set(condition_counts.index) != expected_conditions:
    raise AssertionError(
        "Unexpected conditions after repair:\n"
        f"{condition_counts.index.tolist()}"
    )

if not condition_counts.eq(development_concepts).all():
    raise AssertionError(
        "Every condition must contain every development concept."
    )

duplicate_rows = manifest.duplicated(
    subset=["concept_key", "condition"]
).sum()

if duplicate_rows:
    raise AssertionError(
        f"Duplicate concept-condition rows: {duplicate_rows}"
    )

if manifest["definition_split"].astype(
    str
).str.casefold().ne("development").any():
    raise AssertionError(
        "The manifest contains a non-development concept."
    )

if manifest["selected_context_count"].gt(
    MAX_CONTEXTS
).any():
    raise AssertionError(
        "A prompt contains more than the locked number of contexts."
    )

# Row-wise leakage audit.
reference_leakage = manifest.apply(
    lambda row: (
        bool(clean_text(row["reference_definition"]))
        and clean_text(
            row["reference_definition"]
        ).casefold()
        in clean_text(row["prompt"]).casefold()
    ),
    axis=1,
)

if reference_leakage.any():
    leaking_rows = manifest.loc[
        reference_leakage,
        ["concept_key", "condition"],
    ]

    raise AssertionError(
        "Reference-definition leakage detected:\n"
        f"{leaking_rows.to_string(index=False)}"
    )

condition_audit = (
    manifest.groupby("condition")
    .agg(
        concepts=("concept_key", "nunique"),
        prompts=("prompt", "size"),
        mean_selected_contexts=(
            "selected_context_count",
            "mean",
        ),
        concepts_without_context=(
            "selected_context_count",
            lambda values: int((values == 0).sum()),
        ),
        mean_prompt_characters=(
            "prompt",
            lambda values: float(
                values.str.len().mean()
            ),
        ),
        maximum_prompt_characters=(
            "prompt",
            lambda values: int(
                values.str.len().max()
            ),
        ),
    )
    .reset_index()
)


# ----------------------------------------------------------------
# DISPLAY
# ----------------------------------------------------------------

print("=" * 78)
print("LOCKED DEVELOPMENT PROMPT MANIFEST")
print("=" * 78)

print("Development concepts:", development_concepts)
print("Conditions:", sorted(expected_conditions))
print("Manifest rows:", len(manifest))
print("Reference leakage rows:", int(reference_leakage.sum()))
print("Maximum contexts per evidence prompt:", MAX_CONTEXTS)

print("\nCondition audit:")
print(
    condition_audit.to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
    )
)

no_context_concepts = (
    manifest.loc[
        manifest["condition"].eq("UNRANKED_CONTEXTS")
        & manifest["selected_context_count"].eq(0),
        ["concept_key", "concept_canonical"],
    ]
    .drop_duplicates()
)

print("\nConcepts without literal contextual evidence:")
if no_context_concepts.empty:
    print("None")
else:
    print(no_context_concepts.to_string(index=False))

print("\nRepaired example prompts:")

for condition in [
    "UNRANKED_CONTEXTS",
    "COLLAPSE_K25",
    "COLLAPSE_K50",
    "COLLAPSE_K100",
]:
    example = manifest.loc[
        manifest["condition"].eq(condition)
    ].iloc[0]

    print("\n" + "-" * 78)
    print(condition)
    print("-" * 78)
    print(example["prompt"])


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

manifest.to_csv(
    LOCKED_MANIFEST_PATH,
    index=False,
)

metadata = {
    "status": "locked_before_generation",
    "development_concepts": int(
        development_concepts
    ),
    "conditions": sorted(expected_conditions),
    "manifest_rows": int(len(manifest)),
    "maximum_contexts_per_evidence_prompt": int(
        MAX_CONTEXTS
    ),
    "maximum_context_characters": int(
        MAX_CONTEXT_CHARACTERS
    ),
    "matched_context_budget": True,
    "unranked_condition": (
        "First three unique contexts in stable corpus order."
    ),
    "collapse_conditions": (
        "Top three unique contexts ranked by the corresponding "
        "training-fitted collapse score."
    ),
    "test_concepts_included": False,
    "reference_definitions_inserted_into_prompts": False,
    "reference_leakage_rows": int(
        reference_leakage.sum()
    ),
    "concepts_without_literal_context": (
        no_context_concepts[
            "concept_key"
        ].tolist()
    ),
    "condition_audit": condition_audit.to_dict(
        orient="records"
    ),
    "input_manifest": str(
        INPUT_MANIFEST_PATH
    ),
    "locked_manifest": str(
        LOCKED_MANIFEST_PATH
    ),
}

with open(
    LOCKED_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved locked manifest to:")
print(LOCKED_MANIFEST_PATH)

print("\nSaved locked metadata to:")
print(LOCKED_METADATA_PATH)

print("\nCell 21A completed successfully.")

In [ ]:
# ================================================================
# CELL 21B — LOCK ISO-STYLE DEFINITION PROMPTS
# ================================================================
#
# Replaces the generic generation instruction with the original
# DETECH ISO-style intensional-definition prompt.
#
# No API calls are made.
# Development references remain excluded from prompts.
# ================================================================

from pathlib import Path
import hashlib
import json
import re

import pandas as pd


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

INPUT_MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_prompt_manifest_locked.csv"
)

ISO_MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_prompt_manifest_ISO_locked.csv"
)

ISO_METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_prompt_manifest_ISO_locked_metadata.json"
)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def clean_text(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def parse_contexts(value):
    if pd.isna(value) or not str(value).strip():
        return []

    parsed = json.loads(value)

    if not isinstance(parsed, list):
        raise ValueError(
            "selected_contexts_json must contain a JSON list."
        )

    return [
        clean_text(context)
        for context in parsed
        if clean_text(context)
    ]


def format_context_block(contexts):
    return "\n".join(
        f"{index}. {context}"
        for index, context in enumerate(
            contexts,
            start=1,
        )
    )


def make_iso_prompt(
    concept_label,
    contexts,
    condition,
):
    concept_label = clean_text(concept_label)

    prompt_parts = [
        (
            "You are a terminology expert working on "
            "biomedical concept definitions."
        ),
        (
            f"Write exactly one intensional definition "
            f"for the concept {concept_label}."
        ),
    ]

    if condition != "CONCEPT_ONLY" and contexts:
        prompt_parts.append(
            "Use the following occurrence evidence to identify "
            "the genus and distinguishing characteristics of the "
            "concept. Ignore article-specific details that are not "
            "part of the concept's meaning.\n\n"
            "Occurrence evidence:\n"
            f"{format_context_block(contexts)}"
        )

    prompt_parts.append(
        "Requirements:\n"
        "- Use an ISO 1087 (2019) and ISO 704 (2022)-based "
        "ontological intensional style\n"
        "- Start with the genus term, not with the concept label\n"
        "- Do NOT begin with \"A\", \"An\", or \"The\" unless "
        "absolutely necessary\n"
        "- Prefer the pattern: genus + that/which + "
        "distinguishing characteristics\n"
        "- Be concise and precise\n"
        "- Use specialized biomedical language when appropriate\n"
        "- Do not give examples\n"
        "- Do not mention the evidence, passage, article, "
        "context, dataset, or these instructions\n"
        "- Do not explain your reasoning\n"
        "- Output only the definition text"
    )

    return "\n\n".join(prompt_parts)


def prompt_hash(value):
    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()


# ----------------------------------------------------------------
# LOAD MANIFEST
# ----------------------------------------------------------------

if not INPUT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Locked manifest not found:\n{INPUT_MANIFEST_PATH}"
    )

manifest = pd.read_csv(
    INPUT_MANIFEST_PATH
)

required_columns = {
    "concept_key",
    "concept_canonical",
    "definition_split",
    "condition",
    "selected_context_count",
    "selected_contexts_json",
    "reference_definition",
}

missing_columns = required_columns.difference(
    manifest.columns
)

if missing_columns:
    raise ValueError(
        "Manifest is missing required columns:\n"
        f"{sorted(missing_columns)}"
    )

manifest = manifest.copy()

manifest["_parsed_contexts"] = (
    manifest["selected_contexts_json"]
    .map(parse_contexts)
)

manifest["prompt"] = manifest.apply(
    lambda row: make_iso_prompt(
        concept_label=row["concept_canonical"],
        contexts=row["_parsed_contexts"],
        condition=row["condition"],
    ),
    axis=1,
)

manifest["prompt_style"] = (
    "DETECH_ISO_INTENSIONAL_V1"
)

manifest["prompt_hash"] = (
    manifest["prompt"].map(prompt_hash)
)

manifest = manifest.drop(
    columns=["_parsed_contexts"]
)


# ----------------------------------------------------------------
# AUDIT
# ----------------------------------------------------------------

expected_conditions = {
    "CONCEPT_ONLY",
    "UNRANKED_CONTEXTS",
    "COLLAPSE_K25",
    "COLLAPSE_K50",
    "COLLAPSE_K100",
}

if len(manifest) != 235:
    raise AssertionError(
        f"Expected 235 prompt rows; found {len(manifest)}."
    )

if manifest["concept_key"].nunique() != 47:
    raise AssertionError(
        "Expected 47 development concepts."
    )

if set(manifest["condition"]) != expected_conditions:
    raise AssertionError(
        "Unexpected condition set:\n"
        f"{sorted(manifest['condition'].unique())}"
    )

if manifest.duplicated(
    subset=["concept_key", "condition"]
).any():
    raise AssertionError(
        "Duplicate concept-condition rows detected."
    )

if manifest["definition_split"].astype(
    str
).str.casefold().ne("development").any():
    raise AssertionError(
        "A non-development concept appears in the manifest."
    )

# Reference definitions must not occur inside their corresponding
# generation prompts.
reference_leakage = manifest.apply(
    lambda row: (
        bool(clean_text(row["reference_definition"]))
        and clean_text(
            row["reference_definition"]
        ).casefold()
        in clean_text(row["prompt"]).casefold()
    ),
    axis=1,
)

if reference_leakage.any():
    raise AssertionError(
        "Reference-definition leakage detected."
    )

required_instruction_checks = {
    "mentions_ISO_1087": (
        manifest["prompt"]
        .str.contains(
            "ISO 1087",
            regex=False,
        )
        .all()
    ),
    "mentions_ISO_704": (
        manifest["prompt"]
        .str.contains(
            "ISO 704",
            regex=False,
        )
        .all()
    ),
    "requires_genus_first": (
        manifest["prompt"]
        .str.contains(
            "Start with the genus term",
            regex=False,
        )
        .all()
    ),
    "prohibits_reasoning": (
        manifest["prompt"]
        .str.contains(
            "Do not explain your reasoning",
            regex=False,
        )
        .all()
    ),
    "reference_leakage_absent": (
        not reference_leakage.any()
    ),
}

if not all(required_instruction_checks.values()):
    raise AssertionError(
        "One or more prompt-instruction checks failed:\n"
        f"{required_instruction_checks}"
    )

condition_audit = (
    manifest.groupby("condition")
    .agg(
        concepts=("concept_key", "nunique"),
        prompts=("prompt", "size"),
        mean_selected_contexts=(
            "selected_context_count",
            "mean",
        ),
        mean_prompt_characters=(
            "prompt",
            lambda values: float(
                values.str.len().mean()
            ),
        ),
        maximum_prompt_characters=(
            "prompt",
            lambda values: int(
                values.str.len().max()
            ),
        ),
    )
    .reset_index()
)


# ----------------------------------------------------------------
# DISPLAY
# ----------------------------------------------------------------

print("=" * 78)
print("ISO-STYLE DEVELOPMENT PROMPT MANIFEST")
print("=" * 78)

print("Prompt style: DETECH_ISO_INTENSIONAL_V1")
print("Development concepts:", manifest["concept_key"].nunique())
print("Conditions:", sorted(expected_conditions))
print("Manifest rows:", len(manifest))
print("Reference leakage rows:", int(reference_leakage.sum()))

print("\nInstruction checks:")
for check_name, passed in required_instruction_checks.items():
    print(f"{check_name}: {passed}")

print("\nCondition audit:")
print(
    condition_audit.to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
    )
)

print("\nConcept-only example:")
print("-" * 78)

print(
    manifest.loc[
        manifest["condition"].eq("CONCEPT_ONLY")
    ].iloc[0]["prompt"]
)

print("\nCollapse-evidence example:")
print("-" * 78)

print(
    manifest.loc[
        manifest["condition"].eq("COLLAPSE_K25")
    ].iloc[0]["prompt"]
)


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

manifest.to_csv(
    ISO_MANIFEST_PATH,
    index=False,
)

metadata = {
    "status": "locked_before_generation",
    "prompt_style": "DETECH_ISO_INTENSIONAL_V1",
    "prompt_basis": (
        "Original DETECH baseline prompt with matched "
        "occurrence-evidence instructions added."
    ),
    "development_concepts": int(
        manifest["concept_key"].nunique()
    ),
    "conditions": sorted(expected_conditions),
    "manifest_rows": int(len(manifest)),
    "reference_definitions_inserted_into_prompts": False,
    "reference_leakage_rows": int(
        reference_leakage.sum()
    ),
    "test_concepts_included": False,
    "instruction_checks": required_instruction_checks,
    "input_manifest": str(
        INPUT_MANIFEST_PATH
    ),
    "output_manifest": str(
        ISO_MANIFEST_PATH
    ),
    "condition_audit": (
        condition_audit.to_dict(
            orient="records"
        )
    ),
}

with open(
    ISO_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved ISO-style locked manifest to:")
print(ISO_MANIFEST_PATH)

print("\nSaved ISO-style metadata to:")
print(ISO_METADATA_PATH)

print("\nCell 21B completed successfully.")

In [ ]:
# ================================================================
# CELL 21C — REPAIR ISO METADATA JSON
# ================================================================

import json
import numpy as np

def json_converter(value):
    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    raise TypeError(
        f"Object of type {type(value).__name__} "
        "is not JSON serializable"
    )

# Overwrite the incomplete JSON created by Cell 21B.
with open(
    ISO_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
        default=json_converter,
    )

# Confirm that the saved file is valid JSON.
with open(
    ISO_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    validated_metadata = json.load(file_handle)

print("=" * 78)
print("ISO METADATA REPAIR")
print("=" * 78)
print("ISO manifest exists:", ISO_MANIFEST_PATH.exists())
print("ISO manifest rows:", len(pd.read_csv(ISO_MANIFEST_PATH)))
print("Metadata JSON validated:", True)
print("Prompt style:", validated_metadata["prompt_style"])

print("\nSaved repaired metadata to:")
print(ISO_METADATA_PATH)

print("\nCell 21C completed successfully.")

In [ ]:
# ================================================================
# CELL 22B — ISO-PROMPT TWO-CALL SMOKE TEST
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import re

import pandas as pd
from google.colab import userdata
from openai import OpenAI


# ----------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

ISO_MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_prompt_manifest_ISO_locked.csv"
)

ISO_SMOKE_OUTPUT_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_ISO_api_smoke_test.csv"
)

ISO_SMOKE_METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_api_smoke_test_metadata.json"
)

MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.0
MAX_COMPLETION_TOKENS = 120

SYSTEM_MESSAGE = (
    "You are a terminology expert generating ontological "
    "intensional definitions. Follow the requested ISO-style, "
    "genus-first format exactly and output only the definition."
)


# ----------------------------------------------------------------
# SECURE CLIENT
# ----------------------------------------------------------------

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "The Colab secret OPENAI_API_KEY is missing or empty."
    )

client = OpenAI(api_key=api_key)


# ----------------------------------------------------------------
# LOAD ISO MANIFEST
# ----------------------------------------------------------------

if not ISO_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"ISO manifest not found:\n{ISO_MANIFEST_PATH}"
    )

iso_manifest = pd.read_csv(
    ISO_MANIFEST_PATH
)

smoke_rows = (
    iso_manifest.loc[
        iso_manifest["concept_key"]
        .astype(str)
        .str.casefold()
        .eq("gut microbiota")
        & iso_manifest["condition"].isin(
            ["CONCEPT_ONLY", "COLLAPSE_K25"]
        )
    ]
    .copy()
)

condition_order = {
    "CONCEPT_ONLY": 0,
    "COLLAPSE_K25": 1,
}

smoke_rows["_order"] = (
    smoke_rows["condition"].map(condition_order)
)

smoke_rows = (
    smoke_rows
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

if len(smoke_rows) != 2:
    raise AssertionError(
        f"Expected two ISO smoke rows; found {len(smoke_rows)}."
    )

print("=" * 78)
print("ISO-PROMPT SMOKE TEST")
print("=" * 78)
print("Manifest:", ISO_MANIFEST_PATH.name)
print("Model:", MODEL_NAME)
print("Concept:", smoke_rows.iloc[0]["concept_canonical"])
print("Conditions:", smoke_rows["condition"].tolist())

print("\nPrompt character counts:")
print(
    smoke_rows[
        ["condition", "prompt"]
    ].assign(
        prompt_characters=lambda frame:
        frame["prompt"].str.len()
    )[
        ["condition", "prompt_characters"]
    ].to_string(index=False)
)


# ----------------------------------------------------------------
# GENERATE
# ----------------------------------------------------------------

generation_rows = []

for row in smoke_rows.itertuples(index=False):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_MESSAGE,
            },
            {
                "role": "user",
                "content": row.prompt,
            },
        ],
        temperature=TEMPERATURE,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )

    output = (
        response.choices[0].message.content or ""
    ).strip()

    output = re.sub(
        r"^\s*definition\s*:\s*",
        "",
        output,
        flags=re.IGNORECASE,
    ).strip().strip('"').strip()

    usage = response.usage

    generation_rows.append(
        {
            "concept_key": row.concept_key,
            "concept_canonical": row.concept_canonical,
            "condition": row.condition,
            "prompt_style": row.prompt_style,
            "generated_definition": output,
            "reference_definition": row.reference_definition,
            "requested_model": MODEL_NAME,
            "returned_model": response.model,
            "response_id": response.id,
            "system_fingerprint": getattr(
                response,
                "system_fingerprint",
                None,
            ),
            "finish_reason": (
                response.choices[0].finish_reason
            ),
            "prompt_tokens": (
                usage.prompt_tokens
                if usage is not None
                else None
            ),
            "completion_tokens": (
                usage.completion_tokens
                if usage is not None
                else None
            ),
            "request_time_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }
    )

    print("\n" + "-" * 78)
    print("Condition:", row.condition)
    print("Generated definition:")
    print(output)

iso_smoke_results = pd.DataFrame(
    generation_rows
)


# ----------------------------------------------------------------
# ISO-STYLE AUDIT
# ----------------------------------------------------------------

concept_label_plain = (
    iso_smoke_results["concept_canonical"]
    .str.replace("<", "", regex=False)
    .str.replace(">", "", regex=False)
    .str.strip()
)

iso_smoke_results["starts_with_concept_label"] = [
    definition.casefold().startswith(
        concept.casefold()
    )
    for definition, concept in zip(
        iso_smoke_results["generated_definition"],
        concept_label_plain,
    )
]

iso_smoke_results["starts_with_article"] = (
    iso_smoke_results["generated_definition"]
    .str.contains(
        r"^(?:A|An|The)\s+",
        case=False,
        regex=True,
    )
)

iso_smoke_results["uses_refers_to"] = (
    iso_smoke_results["generated_definition"]
    .str.contains(
        r"\brefers\s+to\b",
        case=False,
        regex=True,
    )
)

iso_smoke_results["mentions_evidence"] = (
    iso_smoke_results["generated_definition"]
    .str.contains(
        r"\b(?:evidence|passage|article|dataset|context)\b",
        case=False,
        regex=True,
    )
)

iso_smoke_results["word_count"] = (
    iso_smoke_results["generated_definition"]
    .str.findall(r"\b[\w'-]+\b")
    .str.len()
)

iso_smoke_results["provisional_iso_format_pass"] = ~(
    iso_smoke_results[
        [
            "starts_with_concept_label",
            "starts_with_article",
            "uses_refers_to",
            "mentions_evidence",
        ]
    ].any(axis=1)
)

print("\n" + "=" * 78)
print("ISO-STYLE SMOKE AUDIT")
print("=" * 78)

print(
    iso_smoke_results[
        [
            "condition",
            "word_count",
            "starts_with_concept_label",
            "starts_with_article",
            "uses_refers_to",
            "mentions_evidence",
            "provisional_iso_format_pass",
            "prompt_tokens",
            "completion_tokens",
        ]
    ].to_string(index=False)
)


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

iso_smoke_results.to_csv(
    ISO_SMOKE_OUTPUT_PATH,
    index=False,
)

metadata = {
    "purpose": "ISO-style two-prompt smoke test",
    "manifest": str(ISO_MANIFEST_PATH),
    "prompt_style": "DETECH_ISO_INTENSIONAL_V1",
    "model": MODEL_NAME,
    "temperature": TEMPERATURE,
    "maximum_completion_tokens": MAX_COMPLETION_TOKENS,
    "system_message": SYSTEM_MESSAGE,
    "api_calls": 2,
    "test_concepts_included": False,
    "reference_definitions_sent_to_api": False,
    "all_provisional_iso_checks_passed": bool(
        iso_smoke_results[
            "provisional_iso_format_pass"
        ].all()
    ),
    "output": str(ISO_SMOKE_OUTPUT_PATH),
}

with open(
    ISO_SMOKE_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved ISO smoke-test outputs to:")
print(ISO_SMOKE_OUTPUT_PATH)

print("\nSaved ISO smoke-test metadata to:")
print(ISO_SMOKE_METADATA_PATH)

print("\nCell 22B completed successfully.")

In [ ]:
# ================================================================
# CELL 23 — FULL RESUMABLE DEVELOPMENT GENERATION
# ================================================================
#
# Generates definitions for:
#   47 development concepts × 5 conditions = 235 calls
#
# Features:
#   - Randomized call order with fixed seed 42
#   - Incremental saving after every successful call
#   - Safe resumption following interruption
#   - Exponential retry for transient API errors
#   - Reference definitions are never sent to the API
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import random
import re
import time

import pandas as pd
from google.colab import userdata
from openai import OpenAI
from tqdm.auto import tqdm


# ----------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_prompt_manifest_ISO_locked.csv"
)

OUTPUT_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_ISO_generations_all_conditions.csv"
)

METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_generation_metadata.json"
)

MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.0
MAX_COMPLETION_TOKENS = 120

RANDOMIZATION_SEED = 42
MAX_RETRIES = 6
INITIAL_RETRY_SECONDS = 2.0

SYSTEM_MESSAGE = (
    "You are a terminology expert generating ontological "
    "intensional definitions. Follow the requested ISO-style, "
    "genus-first format exactly and output only the definition."
)

# ----------------------------------------------------------------
# SECURE CLIENT
# ----------------------------------------------------------------

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "The Colab secret OPENAI_API_KEY is missing or empty."
    )

client = OpenAI(api_key=api_key)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def clean_generated_definition(value):
    value = str(value).strip()

    value = re.sub(
        r"^\s*definition\s*:\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )

    return value.strip().strip('"').strip()


def make_generation_id(concept_key, condition):
    raw_value = f"{concept_key}|||{condition}"

    return hashlib.sha256(
        raw_value.encode("utf-8")
    ).hexdigest()[:20]


def make_prompt_hash(prompt):
    return hashlib.sha256(
        str(prompt).encode("utf-8")
    ).hexdigest()


def atomic_save_csv(frame, path):
    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    frame.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def call_definition_api(prompt):
    delay = INITIAL_RETRY_SECONDS

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_MESSAGE,
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                temperature=TEMPERATURE,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )

            return response

        except Exception as error:
            if attempt == MAX_RETRIES:
                raise

            print(
                f"\nAPI attempt {attempt} failed: "
                f"{type(error).__name__}: {error}"
            )
            print(
                f"Retrying in {delay:.1f} seconds..."
            )

            time.sleep(delay)
            delay *= 2


# ----------------------------------------------------------------
# LOAD AND VALIDATE LOCKED MANIFEST
# ----------------------------------------------------------------

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Locked manifest not found:\n{MANIFEST_PATH}"
    )

manifest = pd.read_csv(MANIFEST_PATH)

required_columns = {
    "concept_key",
    "concept_canonical",
    "definition_split",
    "condition",
    "selected_context_count",
    "selected_contexts_json",
    "prompt",
    "reference_definition",
}

missing_columns = required_columns.difference(
    manifest.columns
)

if missing_columns:
    raise ValueError(
        "Locked manifest is missing columns:\n"
        f"{sorted(missing_columns)}"
    )

if len(manifest) != 235:
    raise AssertionError(
        f"Expected 235 manifest rows; found {len(manifest)}."
    )

if manifest["concept_key"].nunique() != 47:
    raise AssertionError(
        "Expected 47 unique development concepts."
    )

if manifest["definition_split"].astype(
    str
).str.casefold().ne("development").any():
    raise AssertionError(
        "Manifest contains non-development concepts."
    )

manifest = manifest.copy()

manifest["generation_id"] = manifest.apply(
    lambda row: make_generation_id(
        row["concept_key"],
        row["condition"],
    ),
    axis=1,
)

manifest["prompt_hash"] = (
    manifest["prompt"].map(make_prompt_hash)
)

if manifest["generation_id"].duplicated().any():
    raise AssertionError(
        "Duplicate generation identifiers detected."
    )


# ----------------------------------------------------------------
# FIXED RANDOMIZED CALL ORDER
# ----------------------------------------------------------------

generation_ids = manifest[
    "generation_id"
].tolist()

randomizer = random.Random(
    RANDOMIZATION_SEED
)

randomizer.shuffle(generation_ids)

order_lookup = {
    generation_id: order
    for order, generation_id in enumerate(
        generation_ids,
        start=1,
    )
}

manifest["randomized_call_order"] = (
    manifest["generation_id"].map(order_lookup)
)

manifest = manifest.sort_values(
    "randomized_call_order"
).reset_index(drop=True)


# ----------------------------------------------------------------
# LOAD EXISTING PROGRESS
# ----------------------------------------------------------------

if OUTPUT_PATH.exists():
    completed = pd.read_csv(OUTPUT_PATH)

    if "generation_id" not in completed.columns:
        raise ValueError(
            "Existing output file has no generation_id column."
        )

    if completed["generation_id"].duplicated().any():
        raise ValueError(
            "Existing output contains duplicate generation IDs."
        )

    # Ensure resumed rows came from the current locked prompts.
    existing_validation = completed.merge(
        manifest[
            ["generation_id", "prompt_hash"]
        ],
        on="generation_id",
        how="left",
        suffixes=("_saved", "_current"),
        validate="one_to_one",
    )

    unknown_ids = existing_validation[
        "prompt_hash_current"
    ].isna()

    if unknown_ids.any():
        raise ValueError(
            "Existing output contains generation IDs not found "
            "in the current manifest."
        )

    changed_prompts = (
        existing_validation["prompt_hash_saved"]
        != existing_validation["prompt_hash_current"]
    )

    if changed_prompts.any():
        raise ValueError(
            "Existing generations were produced from different "
            "prompt content. Move or rename the existing output "
            "before beginning a new run."
        )

    completed_rows = completed.to_dict(
        orient="records"
    )

else:
    completed = pd.DataFrame()
    completed_rows = []

completed_ids = {
    row["generation_id"]
    for row in completed_rows
}

pending = manifest.loc[
    ~manifest["generation_id"].isin(
        completed_ids
    )
].copy()


# ----------------------------------------------------------------
# RUN SUMMARY
# ----------------------------------------------------------------

print("=" * 78)
print("FULL DEVELOPMENT GENERATION")
print("=" * 78)

print("Model:", MODEL_NAME)
print("Temperature:", TEMPERATURE)
print("Maximum completion tokens:", MAX_COMPLETION_TOKENS)
print("Randomization seed:", RANDOMIZATION_SEED)
print("Total expected calls:", len(manifest))
print("Previously completed:", len(completed_ids))
print("Pending calls:", len(pending))
print("Reference definitions sent to API: False")

print("\nPending calls by condition:")
print(
    pending["condition"]
    .value_counts()
    .sort_index()
)


# ----------------------------------------------------------------
# GENERATE
# ----------------------------------------------------------------

run_start = time.time()

for row in tqdm(
    pending.itertuples(index=False),
    total=len(pending),
    desc="Generating development definitions",
):
    request_time = datetime.now(
        timezone.utc
    ).isoformat()

    # Only the locked prompt is sent. The reference definition
    # remains local and is attached after generation for scoring.
    response = call_definition_api(
        prompt=row.prompt
    )

    raw_output = (
        response.choices[0].message.content or ""
    ).strip()

    generated_definition = (
        clean_generated_definition(raw_output)
    )

    usage = response.usage

    result_row = {
        "generation_id": row.generation_id,
        "prompt_style": row.prompt_style,
        "randomized_call_order": (
            row.randomized_call_order
        ),
        "concept_key": row.concept_key,
        "concept_canonical": (
            row.concept_canonical
        ),
        "definition_split": (
            row.definition_split
        ),
        "condition": row.condition,
        "selected_context_count": (
            row.selected_context_count
        ),
        "selected_contexts_json": (
            row.selected_contexts_json
        ),
        "prompt_hash": row.prompt_hash,

        "requested_model": MODEL_NAME,
        "returned_model": response.model,
        "temperature": TEMPERATURE,
        "max_completion_tokens": (
            MAX_COMPLETION_TOKENS
        ),

        "generated_definition": (
            generated_definition
        ),
        "raw_model_output": raw_output,

        # Local evaluation field; never transmitted.
        "reference_definition": (
            row.reference_definition
        ),

        "response_id": response.id,
        "system_fingerprint": getattr(
            response,
            "system_fingerprint",
            None,
        ),
        "finish_reason": (
            response.choices[0].finish_reason
        ),
        "prompt_tokens": (
            usage.prompt_tokens
            if usage is not None
            else None
        ),
        "completion_tokens": (
            usage.completion_tokens
            if usage is not None
            else None
        ),
        "total_tokens": (
            usage.total_tokens
            if usage is not None
            else None
        ),
        "request_time_utc": request_time,
    }

    completed_rows.append(result_row)

    current_results = pd.DataFrame(
        completed_rows
    ).sort_values(
        "randomized_call_order"
    )

    # Save immediately after every successful response.
    atomic_save_csv(
        current_results,
        OUTPUT_PATH,
    )


elapsed_minutes = (
    time.time() - run_start
) / 60


# ----------------------------------------------------------------
# FINAL AUDIT
# ----------------------------------------------------------------

results = pd.read_csv(OUTPUT_PATH)

results["is_empty"] = (
    results["generated_definition"]
    .fillna("")
    .str.strip()
    .eq("")
)

results["word_count"] = (
    results["generated_definition"]
    .fillna("")
    .str.findall(r"\b[\w'-]+\b")
    .str.len()
)

results["sentence_terminal"] = (
    results["generated_definition"]
    .fillna("")
    .str.endswith((".", "!", "?"))
)

# Noncapturing group avoids the earlier pandas warning.
results["mentions_evidence"] = (
    results["generated_definition"]
    .fillna("")
    .str.contains(
        r"\b(?:evidence|passage|article|dataset|context)\b",
        case=False,
        regex=True,
    )
)

results["finish_reason_stop"] = (
    results["finish_reason"].eq("stop")
)

if len(results) != len(manifest):
    raise AssertionError(
        f"Expected {len(manifest)} completed rows; "
        f"found {len(results)}."
    )

if results["generation_id"].duplicated().any():
    raise AssertionError(
        "Duplicate completed generations detected."
    )

if results["is_empty"].any():
    raise AssertionError(
        "At least one generated definition is empty."
    )

result_keys = set(
    zip(
        results["concept_key"],
        results["condition"],
    )
)

manifest_keys = set(
    zip(
        manifest["concept_key"],
        manifest["condition"],
    )
)

if result_keys != manifest_keys:
    raise AssertionError(
        "Completed concept-condition keys do not match "
        "the locked manifest."
    )

condition_audit = (
    results.groupby("condition")
    .agg(
        definitions=(
            "generation_id",
            "size",
        ),
        mean_word_count=(
            "word_count",
            "mean",
        ),
        minimum_word_count=(
            "word_count",
            "min",
        ),
        maximum_word_count=(
            "word_count",
            "max",
        ),
        terminal_sentence_rate=(
            "sentence_terminal",
            "mean",
        ),
        evidence_mention_rate=(
            "mentions_evidence",
            "mean",
        ),
        stop_finish_rate=(
            "finish_reason_stop",
            "mean",
        ),
        mean_prompt_tokens=(
            "prompt_tokens",
            "mean",
        ),
        mean_completion_tokens=(
            "completion_tokens",
            "mean",
        ),
    )
    .reset_index()
)

print("\n" + "=" * 78)
print("FULL DEVELOPMENT GENERATION AUDIT")
print("=" * 78)

print("Completed generations:", len(results))
print("Unique concepts:", results["concept_key"].nunique())
print("Conditions:", results["condition"].nunique())
print("Empty generations:", int(results["is_empty"].sum()))
print(
    "Non-stop finish reasons:",
    int((~results["finish_reason_stop"]).sum()),
)
print(
    "Definitions mentioning evidence/context:",
    int(results["mentions_evidence"].sum()),
)
print(
    "Elapsed minutes for pending calls:",
    round(elapsed_minutes, 2),
)

print("\nCondition audit:")
print(
    condition_audit.to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)

print("\nNon-stop responses:")
non_stop = results.loc[
    ~results["finish_reason_stop"],
    [
        "concept_canonical",
        "condition",
        "finish_reason",
        "generated_definition",
    ],
]

if non_stop.empty:
    print("None")
else:
    print(non_stop.to_string(index=False))

print("\nDefinitions mentioning evidence-related language:")
evidence_mentions = results.loc[
    results["mentions_evidence"],
    [
        "concept_canonical",
        "condition",
        "generated_definition",
    ],
]

if evidence_mentions.empty:
    print("None")
else:
    print(evidence_mentions.to_string(index=False))


# ----------------------------------------------------------------
# SAVE FINAL AUDIT COLUMNS AND METADATA
# ----------------------------------------------------------------

atomic_save_csv(
    results.sort_values(
        ["concept_key", "condition"]
    ),
    OUTPUT_PATH,
)

metadata = {
    "status": "complete",
    "model_requested": MODEL_NAME,
    "temperature": TEMPERATURE,
    "prompt_style": "DETECH_ISO_INTENSIONAL_V1",
    "maximum_completion_tokens": (
        MAX_COMPLETION_TOKENS
    ),
    "system_message": SYSTEM_MESSAGE,
    "randomization_seed": RANDOMIZATION_SEED,
    "development_concepts": int(
        results["concept_key"].nunique()
    ),
    "conditions": sorted(
        results["condition"].unique().tolist()
    ),
    "completed_generations": int(
        len(results)
    ),
    "empty_generations": int(
        results["is_empty"].sum()
    ),
    "non_stop_finish_reasons": int(
        (~results["finish_reason_stop"]).sum()
    ),
    "reference_definitions_sent_to_api": False,
    "test_concepts_included": False,
    "api_key_source": (
        "Colab secret OPENAI_API_KEY"
    ),
    "api_key_recorded": False,
    "manifest_path": str(MANIFEST_PATH),
    "output_path": str(OUTPUT_PATH),
    "condition_audit": (
        condition_audit.to_dict(
            orient="records"
        )
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved complete development generations to:")
print(OUTPUT_PATH)

print("\nSaved generation metadata to:")
print(METADATA_PATH)

print("\nCell 23 completed successfully.")

In [ ]:
# ================================================================
# CELL 24 — DEVELOPMENT DEFINITION EVALUATION
# ================================================================
#
# Metrics:
#   - Corpus BLEU
#   - Sentence BLEU distribution and quartiles
#   - BiomedBERT BERTScore precision, recall, and F1
#   - Per-concept paired contrasts
#   - ISO-style output diagnostics
#
# Development concepts only.
# No test data are accessed.
# ================================================================

!pip -q install --upgrade sacrebleu bert-score

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import torch
from sacrebleu import corpus_bleu, sentence_bleu
from bert_score import BERTScorer


# ----------------------------------------------------------------
# PATHS AND CONFIGURATION
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

GENERATION_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_ISO_generations_all_conditions.csv"
)

DETAILED_RESULTS_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_detailed_metrics.csv"
)

CONDITION_RESULTS_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_condition_metrics.csv"
)

PAIRED_RESULTS_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_paired_contrasts.csv"
)

METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_evaluation_metadata.json"
)

BIOMEDBERT_MODEL = (
    "microsoft/"
    "BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

BIOMEDBERT_LAYERS = 12
BERTSCORE_BATCH_SIZE = 16

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def clean_text(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def plain_concept_label(value):
    value = clean_text(value)

    if value.startswith("<") and value.endswith(">"):
        value = value[1:-1]

    return value.strip()


def begins_with_label(definition, concept):
    definition = clean_text(definition).casefold()
    concept = plain_concept_label(concept).casefold()

    return definition.startswith(concept)


# ----------------------------------------------------------------
# LOAD AND VALIDATE
# ----------------------------------------------------------------

if not GENERATION_PATH.exists():
    raise FileNotFoundError(
        f"Generation file not found:\n{GENERATION_PATH}"
    )

results = pd.read_csv(GENERATION_PATH)

required_columns = {
    "concept_key",
    "concept_canonical",
    "definition_split",
    "condition",
    "generated_definition",
    "reference_definition",
    "finish_reason",
}

missing_columns = required_columns.difference(
    results.columns
)

if missing_columns:
    raise ValueError(
        "Generation file is missing columns:\n"
        f"{sorted(missing_columns)}"
    )

results = results.copy()

results["generated_definition"] = (
    results["generated_definition"]
    .map(clean_text)
)

results["reference_definition"] = (
    results["reference_definition"]
    .map(clean_text)
)

if len(results) != 235:
    raise AssertionError(
        f"Expected 235 generations; found {len(results)}."
    )

if results["concept_key"].nunique() != 47:
    raise AssertionError(
        "Expected 47 development concepts."
    )

if results.duplicated(
    subset=["concept_key", "condition"]
).any():
    raise AssertionError(
        "Duplicate concept-condition rows detected."
    )

if results["generated_definition"].eq("").any():
    raise AssertionError(
        "At least one generated definition is empty."
    )

if results["reference_definition"].eq("").any():
    raise AssertionError(
        "At least one reference definition is empty."
    )

if results["definition_split"].astype(
    str
).str.casefold().ne("development").any():
    raise AssertionError(
        "Non-development rows detected."
    )

print("=" * 78)
print("DEVELOPMENT EVALUATION SETUP")
print("=" * 78)

print("Rows:", len(results))
print("Concepts:", results["concept_key"].nunique())
print("Conditions:", sorted(results["condition"].unique()))
print("Device:", DEVICE)
print("BiomedBERT model:", BIOMEDBERT_MODEL)


# ----------------------------------------------------------------
# ISO-STYLE DIAGNOSTICS
# ----------------------------------------------------------------

results["word_count"] = (
    results["generated_definition"]
    .str.findall(r"\b[\w'-]+\b")
    .str.len()
)

results["starts_with_concept_label"] = results.apply(
    lambda row: begins_with_label(
        row["generated_definition"],
        row["concept_canonical"],
    ),
    axis=1,
)

results["starts_with_article"] = (
    results["generated_definition"]
    .str.contains(
        r"^(?:A|An|The)\s+",
        case=False,
        regex=True,
    )
)

results["uses_refers_to"] = (
    results["generated_definition"]
    .str.contains(
        r"\brefers\s+to\b",
        case=False,
        regex=True,
    )
)

results["contains_multiple_sentences"] = (
    results["generated_definition"]
    .str.count(r"[.!?]+")
    .gt(1)
)

results["provisional_iso_format_pass"] = ~(
    results[
        [
            "starts_with_concept_label",
            "starts_with_article",
            "uses_refers_to",
            "contains_multiple_sentences",
        ]
    ].any(axis=1)
)


# ----------------------------------------------------------------
# SENTENCE BLEU
# ----------------------------------------------------------------

results["sentence_bleu"] = [
    float(
        sentence_bleu(
            hypothesis=prediction,
            references=[reference],
            smooth_method="exp",
            use_effective_order=True,
        ).score
    )
    for prediction, reference in zip(
        results["generated_definition"],
        results["reference_definition"],
    )
]


# ----------------------------------------------------------------
# BIOMEDBERT BERTSCORE
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("COMPUTING BIOMEDBERT BERTSCORE")
print("=" * 78)

predictions = results[
    "generated_definition"
].tolist()

references = results[
    "reference_definition"
].tolist()

# Initialize explicitly so we can repair BiomedBERT's
# impractically large tokenizer maximum length.
biomedbert_scorer = BERTScorer(
    model_type=BIOMEDBERT_MODEL,
    num_layers=BIOMEDBERT_LAYERS,
    batch_size=BERTSCORE_BATCH_SIZE,
    nthreads=4,
    idf=False,
    device=DEVICE,
    lang=None,
    rescale_with_baseline=False,
    use_fast_tokenizer=True,
)

biomedbert_scorer._tokenizer.model_max_length = 512

with torch.inference_mode():
    bert_precision, bert_recall, bert_f1 = (
        biomedbert_scorer.score(
            cands=predictions,
            refs=references,
            batch_size=BERTSCORE_BATCH_SIZE,
        )
    )

results["biomedbert_precision"] = (
    bert_precision.cpu().numpy()
)

results["biomedbert_recall"] = (
    bert_recall.cpu().numpy()
)

results["biomedbert_f1"] = (
    bert_f1.cpu().numpy()
)

if not np.isfinite(
    results[
        [
            "sentence_bleu",
            "biomedbert_precision",
            "biomedbert_recall",
            "biomedbert_f1",
        ]
    ].to_numpy()
).all():
    raise AssertionError(
        "Non-finite evaluation values detected."
    )


# ----------------------------------------------------------------
# CONDITION-LEVEL METRICS
# ----------------------------------------------------------------

condition_rows = []

for condition, condition_frame in results.groupby(
    "condition",
    sort=True,
):
    condition_frame = condition_frame.sort_values(
        "concept_key"
    )

    condition_predictions = condition_frame[
        "generated_definition"
    ].tolist()

    condition_references = condition_frame[
        "reference_definition"
    ].tolist()

    # SacreBLEU expects a list of reference streams.
    condition_corpus_bleu = corpus_bleu(
        condition_predictions,
        [condition_references],
    ).score

    condition_rows.append(
        {
            "condition": condition,
            "n_concepts": int(
                len(condition_frame)
            ),
            "corpus_bleu": float(
                condition_corpus_bleu
            ),
            "sentence_bleu_mean": float(
                condition_frame[
                    "sentence_bleu"
                ].mean()
            ),
            "sentence_bleu_q25": float(
                condition_frame[
                    "sentence_bleu"
                ].quantile(0.25)
            ),
            "sentence_bleu_median": float(
                condition_frame[
                    "sentence_bleu"
                ].median()
            ),
            "sentence_bleu_q75": float(
                condition_frame[
                    "sentence_bleu"
                ].quantile(0.75)
            ),
            "biomedbert_precision_mean": float(
                condition_frame[
                    "biomedbert_precision"
                ].mean()
            ),
            "biomedbert_recall_mean": float(
                condition_frame[
                    "biomedbert_recall"
                ].mean()
            ),
            "biomedbert_f1_mean": float(
                condition_frame[
                    "biomedbert_f1"
                ].mean()
            ),
            "biomedbert_f1_q25": float(
                condition_frame[
                    "biomedbert_f1"
                ].quantile(0.25)
            ),
            "biomedbert_f1_median": float(
                condition_frame[
                    "biomedbert_f1"
                ].median()
            ),
            "biomedbert_f1_q75": float(
                condition_frame[
                    "biomedbert_f1"
                ].quantile(0.75)
            ),
            "mean_word_count": float(
                condition_frame[
                    "word_count"
                ].mean()
            ),
            "iso_format_pass_rate": float(
                condition_frame[
                    "provisional_iso_format_pass"
                ].mean()
            ),
            "starts_with_label_count": int(
                condition_frame[
                    "starts_with_concept_label"
                ].sum()
            ),
            "starts_with_article_count": int(
                condition_frame[
                    "starts_with_article"
                ].sum()
            ),
            "uses_refers_to_count": int(
                condition_frame[
                    "uses_refers_to"
                ].sum()
            ),
            "multiple_sentence_count": int(
                condition_frame[
                    "contains_multiple_sentences"
                ].sum()
            ),
        }
    )

condition_metrics = pd.DataFrame(
    condition_rows
)

# Transparent combined descriptive score:
# BLEU is placed on [0,1], then averaged with BiomedBERT F1.
# This is reported for comparison but is not yet used to lock K.
condition_metrics[
    "descriptive_combined_score"
] = (
    (
        condition_metrics["corpus_bleu"] / 100.0
    )
    + condition_metrics["biomedbert_f1_mean"]
) / 2.0

condition_metrics = condition_metrics.sort_values(
    [
        "descriptive_combined_score",
        "biomedbert_f1_mean",
        "corpus_bleu",
    ],
    ascending=False,
).reset_index(drop=True)


# ----------------------------------------------------------------
# PAIRED PER-CONCEPT CONTRASTS
# ----------------------------------------------------------------

paired_rows = []

comparison_baselines = [
    "CONCEPT_ONLY",
    "UNRANKED_CONTEXTS",
]

target_conditions = [
    "COLLAPSE_K25",
    "COLLAPSE_K50",
    "COLLAPSE_K100",
]

paired_metrics = [
    "sentence_bleu",
    "biomedbert_precision",
    "biomedbert_recall",
    "biomedbert_f1",
]

for baseline_condition in comparison_baselines:
    baseline = (
        results.loc[
            results["condition"].eq(
                baseline_condition
            ),
            ["concept_key"] + paired_metrics,
        ]
        .set_index("concept_key")
        .sort_index()
    )

    for target_condition in target_conditions:
        target = (
            results.loc[
                results["condition"].eq(
                    target_condition
                ),
                ["concept_key"] + paired_metrics,
            ]
            .set_index("concept_key")
            .sort_index()
        )

        if not baseline.index.equals(target.index):
            raise AssertionError(
                "Paired concept indices do not match."
            )

        for metric in paired_metrics:
            differences = (
                target[metric] - baseline[metric]
            )

            paired_rows.append(
                {
                    "target_condition": target_condition,
                    "baseline_condition": baseline_condition,
                    "metric": metric,
                    "n_concepts": int(
                        len(differences)
                    ),
                    "mean_difference": float(
                        differences.mean()
                    ),
                    "median_difference": float(
                        differences.median()
                    ),
                    "difference_q25": float(
                        differences.quantile(0.25)
                    ),
                    "difference_q75": float(
                        differences.quantile(0.75)
                    ),
                    "positive_concepts": int(
                        differences.gt(0).sum()
                    ),
                    "negative_concepts": int(
                        differences.lt(0).sum()
                    ),
                    "zero_concepts": int(
                        differences.eq(0).sum()
                    ),
                }
            )

paired_contrasts = pd.DataFrame(
    paired_rows
)


# ----------------------------------------------------------------
# DISPLAY
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("DEVELOPMENT CONDITION RESULTS")
print("=" * 78)

display_columns = [
    "condition",
    "corpus_bleu",
    "sentence_bleu_median",
    "biomedbert_precision_mean",
    "biomedbert_recall_mean",
    "biomedbert_f1_mean",
    "biomedbert_f1_q25",
    "biomedbert_f1_median",
    "biomedbert_f1_q75",
    "iso_format_pass_rate",
    "descriptive_combined_score",
]

print(
    condition_metrics[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\n" + "=" * 78)
print("COLLAPSE MINUS BASELINE — PAIRED CONCEPT RESULTS")
print("=" * 78)

print(
    paired_contrasts.loc[
        paired_contrasts["metric"].isin(
            ["sentence_bleu", "biomedbert_f1"]
        )
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\n" + "=" * 78)
print("ISO-FORMAT FAILURES")
print("=" * 78)

iso_failures = results.loc[
    ~results["provisional_iso_format_pass"],
    [
        "concept_canonical",
        "condition",
        "generated_definition",
        "starts_with_concept_label",
        "starts_with_article",
        "uses_refers_to",
        "contains_multiple_sentences",
    ],
].sort_values(
    ["condition", "concept_canonical"]
)

if iso_failures.empty:
    print("None")
else:
    print(
        iso_failures.to_string(index=False)
    )


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

results.to_csv(
    DETAILED_RESULTS_PATH,
    index=False,
)

condition_metrics.to_csv(
    CONDITION_RESULTS_PATH,
    index=False,
)

paired_contrasts.to_csv(
    PAIRED_RESULTS_PATH,
    index=False,
)

metadata = {
    "evaluation_split": "development",
    "development_concepts": int(
        results["concept_key"].nunique()
    ),
    "conditions": sorted(
        results["condition"].unique().tolist()
    ),
    "bleu_library": "sacrebleu",
    "corpus_bleu_reference_structure": (
        "one reference stream containing 47 aligned references"
    ),
    "sentence_bleu_smoothing": "exp",
    "sentence_bleu_use_effective_order": True,
    "semantic_metric": "BERTScore",
    "semantic_encoder": BIOMEDBERT_MODEL,
    "semantic_encoder_layers": BIOMEDBERT_LAYERS,
    "bertscore_idf": False,
    "bertscore_rescaled_with_baseline": False,
    "device": DEVICE,
    "test_concepts_accessed": False,
    "k_selected": False,
    "descriptive_combined_score": (
        "mean of corpus BLEU divided by 100 and "
        "mean BiomedBERT BERTScore F1"
    ),
    "generation_file": str(GENERATION_PATH),
    "detailed_results": str(
        DETAILED_RESULTS_PATH
    ),
    "condition_results": str(
        CONDITION_RESULTS_PATH
    ),
    "paired_results": str(
        PAIRED_RESULTS_PATH
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved detailed metrics to:")
print(DETAILED_RESULTS_PATH)

print("\nSaved condition metrics to:")
print(CONDITION_RESULTS_PATH)

print("\nSaved paired contrasts to:")
print(PAIRED_RESULTS_PATH)

print("\nSaved evaluation metadata to:")
print(METADATA_PATH)

print("\nCell 24 completed successfully.")

In [ ]:
# ================================================================
# CELL 25 — PAIRED DEVELOPMENT BOOTSTRAP AND SELECTION LOCK
# ================================================================
#
# Selection rule:
#   Among collapse conditions only, select the condition with the
#   highest development descriptive combined score:
#
#       0.5 × (corpus BLEU / 100)
#     + 0.5 × mean BiomedBERT F1
#
# Bootstrap:
#   Paired resampling of the 47 development concepts.
#
# Test concepts remain untouched.
# ================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sacrebleu import corpus_bleu
from tqdm.auto import tqdm


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"

DETAILED_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_metrics_with_ROUGE.csv"
)

CONDITION_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_summary_with_ROUGE.csv"
)

BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_paired_bootstrap.csv"
)

BOOTSTRAP_SUMMARY_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_paired_bootstrap_summary.csv"
)

LOCKED_PROTOCOL_PATH = (
    RESULTS_DIR /
    "taskB_definition_locked_test_protocol.json"
)


# ----------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 42

COLLAPSE_CONDITIONS = [
    "COLLAPSE_K25",
    "COLLAPSE_K50",
    "COLLAPSE_K100",
]

CONTRASTS = [
    ("COLLAPSE_K50", "CONCEPT_ONLY"),
    ("UNRANKED_CONTEXTS", "CONCEPT_ONLY"),
    ("COLLAPSE_K50", "UNRANKED_CONTEXTS"),
    ("COLLAPSE_K50", "COLLAPSE_K25"),
    ("COLLAPSE_K50", "COLLAPSE_K100"),
]


# ----------------------------------------------------------------
# LOAD RESULTS
# ----------------------------------------------------------------

detailed = pd.read_csv(DETAILED_PATH)
condition_metrics = pd.read_csv(CONDITION_PATH)

required_detailed_columns = {
    "concept_key",
    "condition",
    "generated_definition",
    "reference_definition",
    "sentence_bleu",
    "biomedbert_f1",
}

missing = required_detailed_columns.difference(
    detailed.columns
)

if missing:
    raise ValueError(
        f"Detailed results missing columns: {sorted(missing)}"
    )

if detailed["concept_key"].nunique() != 47:
    raise AssertionError(
        "Expected 47 development concepts."
    )


# ----------------------------------------------------------------
# APPLY THE PREDEFINED K-SELECTION RULE
# ----------------------------------------------------------------

collapse_ranking = (
    condition_metrics.loc[
        condition_metrics["condition"].isin(
            COLLAPSE_CONDITIONS
        )
    ]
    .sort_values(
        [
            "descriptive_combined_score",
            "biomedbert_f1_mean",
            "corpus_bleu",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

selected_collapse_condition = collapse_ranking.iloc[
    0
]["condition"]

selected_k = int(
    selected_collapse_condition.replace(
        "COLLAPSE_K",
        "",
    )
)

print("=" * 78)
print("DEVELOPMENT COLLAPSE SELECTION")
print("=" * 78)

print(
    collapse_ranking[
        [
            "condition",
            "corpus_bleu",
            "biomedbert_f1_mean",
            "descriptive_combined_score",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nSelected collapse condition:", selected_collapse_condition)
print("Selected K:", selected_k)


# ----------------------------------------------------------------
# ALIGN CONDITIONS BY CONCEPT
# ----------------------------------------------------------------

condition_frames = {}

for condition, frame in detailed.groupby(
    "condition"
):
    aligned = (
        frame.sort_values("concept_key")
        .reset_index(drop=True)
    )

    condition_frames[condition] = aligned

reference_keys = condition_frames[
    "CONCEPT_ONLY"
]["concept_key"].tolist()

for condition, frame in condition_frames.items():
    if frame["concept_key"].tolist() != reference_keys:
        raise AssertionError(
            f"Concept alignment failure for {condition}."
        )

    if (
        frame["reference_definition"].tolist()
        != condition_frames[
            "CONCEPT_ONLY"
        ]["reference_definition"].tolist()
    ):
        raise AssertionError(
            f"Reference mismatch for {condition}."
        )

n_concepts = len(reference_keys)


# ----------------------------------------------------------------
# METRIC FUNCTION
# ----------------------------------------------------------------

def calculate_sample_metrics(
    frame,
    sampled_indices,
):
    sampled = frame.iloc[
        sampled_indices
    ]

    predictions = sampled[
        "generated_definition"
    ].tolist()

    references = sampled[
        "reference_definition"
    ].tolist()

    sample_corpus_bleu = float(
        corpus_bleu(
            predictions,
            [references],
        ).score
    )

    sample_sentence_bleu = float(
        sampled["sentence_bleu"].mean()
    )

    sample_biomedbert_f1 = float(
        sampled["biomedbert_f1"].mean()
    )

    sample_combined = (
        (sample_corpus_bleu / 100.0)
        + sample_biomedbert_f1
    ) / 2.0

    return {
        "corpus_bleu": sample_corpus_bleu,
        "mean_sentence_bleu": sample_sentence_bleu,
        "mean_biomedbert_f1": sample_biomedbert_f1,
        "combined_score": sample_combined,
    }


# ----------------------------------------------------------------
# OBSERVED CONTRASTS
# ----------------------------------------------------------------

all_indices = np.arange(n_concepts)

observed_contrasts = {}

for target, baseline in CONTRASTS:
    target_metrics = calculate_sample_metrics(
        condition_frames[target],
        all_indices,
    )

    baseline_metrics = calculate_sample_metrics(
        condition_frames[baseline],
        all_indices,
    )

    contrast_name = f"{target}_MINUS_{baseline}"

    observed_contrasts[contrast_name] = {
        metric: (
            target_metrics[metric]
            - baseline_metrics[metric]
        )
        for metric in target_metrics
    }


# ----------------------------------------------------------------
# PAIRED BOOTSTRAP
# ----------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

for replicate in tqdm(
    range(BOOTSTRAP_REPLICATES),
    desc="Paired concept bootstrap",
):
    sampled_indices = rng.integers(
        low=0,
        high=n_concepts,
        size=n_concepts,
    )

    sampled_condition_metrics = {}

    needed_conditions = sorted(
        set(
            condition
            for contrast in CONTRASTS
            for condition in contrast
        )
    )

    for condition in needed_conditions:
        sampled_condition_metrics[
            condition
        ] = calculate_sample_metrics(
            condition_frames[condition],
            sampled_indices,
        )

    for target, baseline in CONTRASTS:
        contrast_name = (
            f"{target}_MINUS_{baseline}"
        )

        for metric in [
            "corpus_bleu",
            "mean_sentence_bleu",
            "mean_biomedbert_f1",
            "combined_score",
        ]:
            bootstrap_rows.append(
                {
                    "replicate": replicate,
                    "contrast": contrast_name,
                    "target_condition": target,
                    "baseline_condition": baseline,
                    "metric": metric,
                    "difference": (
                        sampled_condition_metrics[
                            target
                        ][metric]
                        - sampled_condition_metrics[
                            baseline
                        ][metric]
                    ),
                }
            )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# ----------------------------------------------------------------
# SUMMARIZE BOOTSTRAP DISTRIBUTIONS
# ----------------------------------------------------------------

summary_rows = []

for (
    contrast,
    target,
    baseline,
    metric,
), frame in bootstrap_results.groupby(
    [
        "contrast",
        "target_condition",
        "baseline_condition",
        "metric",
    ],
    sort=True,
):
    differences = frame[
        "difference"
    ].to_numpy()

    observed_difference = (
        observed_contrasts[contrast][metric]
    )

    probability_greater = float(
        np.mean(differences > 0)
    )

    probability_less = float(
        np.mean(differences < 0)
    )

    summary_rows.append(
        {
            "contrast": contrast,
            "target_condition": target,
            "baseline_condition": baseline,
            "metric": metric,
            "observed_difference": float(
                observed_difference
            ),
            "bootstrap_mean_difference": float(
                np.mean(differences)
            ),
            "bootstrap_median_difference": float(
                np.median(differences)
            ),
            "ci_lower_95": float(
                np.quantile(
                    differences,
                    0.025,
                )
            ),
            "ci_upper_95": float(
                np.quantile(
                    differences,
                    0.975,
                )
            ),
            "probability_greater_than_zero": (
                probability_greater
            ),
            "probability_less_than_zero": (
                probability_less
            ),
        }
    )

bootstrap_summary = pd.DataFrame(
    summary_rows
)


# ----------------------------------------------------------------
# DISPLAY
# ----------------------------------------------------------------

print("\n" + "=" * 78)
print("PAIRED DEVELOPMENT BOOTSTRAP")
print("=" * 78)

display_summary = bootstrap_summary.loc[
    bootstrap_summary["metric"].isin(
        [
            "corpus_bleu",
            "mean_biomedbert_f1",
            "combined_score",
        ]
    )
].copy()

print(
    display_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


# ----------------------------------------------------------------
# LOCK TEST PROTOCOL
# ----------------------------------------------------------------

selected_row = collapse_ranking.iloc[0]

locked_protocol = {
    "status": "locked_before_test_generation",
    "task": "Task B definition generation",
    "prompt_style": "DETECH_ISO_INTENSIONAL_V1",
    "generation_model": "gpt-4o-mini",
    "temperature": 0.0,
    "maximum_completion_tokens": 120,
    "context_budget": 3,
    "development_concepts": 47,
    "selection_rule": (
        "Among K=25, K=50, and K=100, maximize the "
        "development descriptive combined score: "
        "0.5*(corpus BLEU/100) + "
        "0.5*(mean BiomedBERT BERTScore F1)."
    ),
    "selected_collapse_condition": (
        selected_collapse_condition
    ),
    "selected_k": selected_k,
    "selected_development_corpus_bleu": float(
        selected_row["corpus_bleu"]
    ),
    "selected_development_biomedbert_f1": float(
        selected_row["biomedbert_f1_mean"]
    ),
    "selected_development_combined_score": float(
        selected_row[
            "descriptive_combined_score"
        ]
    ),
    "locked_test_conditions": [
        "CONCEPT_ONLY",
        "UNRANKED_CONTEXTS",
        selected_collapse_condition,
    ],
    "test_references_available_to_generator": False,
    "test_generation_completed": False,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "semantic_metric": (
        "BERTScore using "
        "microsoft/BiomedNLP-BiomedBERT-base-uncased-"
        "abstract-fulltext"
    ),
    "development_detailed_results": str(
        DETAILED_PATH
    ),
    "development_condition_results": str(
        CONDITION_PATH
    ),
    "bootstrap_summary": str(
        BOOTSTRAP_SUMMARY_PATH
    ),
}

bootstrap_results.to_csv(
    BOOTSTRAP_REPLICATES_PATH,
    index=False,
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)

with open(
    LOCKED_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        locked_protocol,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\n" + "=" * 78)
print("LOCKED TEST PROTOCOL")
print("=" * 78)

print("Selected collapse condition:", selected_collapse_condition)
print("Selected K:", selected_k)
print(
    "Locked test conditions:",
    locked_protocol["locked_test_conditions"],
)
print("Test generation completed: False")

print("\nSaved bootstrap replicates to:")
print(BOOTSTRAP_REPLICATES_PATH)

print("\nSaved bootstrap summary to:")
print(BOOTSTRAP_SUMMARY_PATH)

print("\nSaved locked test protocol to:")
print(LOCKED_PROTOCOL_PATH)

print("\nCell 25 completed successfully.")

In [ ]:
# ================================================================
# CELL 25A — RESTORE TEST-GENERATION STATUS AFTER RESELECTION
# ================================================================

from datetime import datetime, timezone
import json

CANONICAL_TEST_OUTPUT = (
    PREDICTIONS_DIR /
    "taskB_definition_test_ISO_generations_prompt_deduplicated.csv"
)

if not CANONICAL_TEST_OUTPUT.exists():
    raise FileNotFoundError(
        f"Canonical test output not found:\n{CANONICAL_TEST_OUTPUT}"
    )

canonical_test = pd.read_csv(
    CANONICAL_TEST_OUTPUT
)

if len(canonical_test) != 144:
    raise AssertionError(
        f"Expected 144 canonical test rows; found {len(canonical_test)}."
    )

with open(
    LOCKED_PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    protocol = json.load(file_handle)

protocol["test_generation_completed"] = True
protocol["test_generation_output"] = str(
    CANONICAL_TEST_OUTPUT
)
protocol["identical_prompt_outputs_canonicalized"] = True
protocol["canonicalization_rule"] = (
    "Earliest randomized API response for each exact prompt hash."
)
protocol["test_evaluation_completed"] = True
protocol["test_metrics_output"] = str(
    RESULTS_DIR /
    "taskB_definition_test_ISO_condition_metrics.csv"
)
protocol["protocol_repaired_utc"] = datetime.now(
    timezone.utc
).isoformat()

with open(
    LOCKED_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        protocol,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("=" * 78)
print("LOCKED PROTOCOL REPAIRED")
print("=" * 78)
print("Selected K:", protocol["selected_k"])
print(
    "Test generation completed:",
    protocol["test_generation_completed"],
)
print(
    "Canonicalized outputs:",
    protocol["identical_prompt_outputs_canonicalized"],
)
print(
    "Test evaluation completed:",
    protocol["test_evaluation_completed"],
)

print("\nUpdated protocol:")
print(LOCKED_PROTOCOL_PATH)

print("\nCell 25A completed successfully.")

In [ ]:
# ================================================================
# CELL 26 — BUILD REFERENCE-BLIND TEST PROMPT MANIFEST
# ================================================================
#
# Locked conditions:
#   1. CONCEPT_ONLY
#   2. UNRANKED_CONTEXTS
#   3. COLLAPSE_K50
#
# This cell:
#   - Uses only concept labels and occurrence evidence.
#   - Does not load authoritative definitions.
#   - Does not include a reference-definition column.
#   - Makes no API calls.
# ================================================================

from pathlib import Path
import hashlib
import json
import re

import pandas as pd


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

EVIDENCE_PATH = (
    RESULTS_DIR /
    "taskB_definition_concept_evidence_collapse_K25_K50_K100.csv"
)

LOCKED_PROTOCOL_PATH = (
    RESULTS_DIR /
    "taskB_definition_locked_test_protocol.json"
)

TEST_MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_test_prompt_manifest_ISO_locked.csv"
)

TEST_METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_prompt_manifest_ISO_locked_metadata.json"
)

MAX_CONTEXTS = 3
MAX_CONTEXT_CHARACTERS = 800


# ----------------------------------------------------------------
# LOAD LOCKED PROTOCOL
# ----------------------------------------------------------------

with open(
    LOCKED_PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    locked_protocol = json.load(file_handle)

if locked_protocol["selected_k"] != 50:
    raise AssertionError(
        "The locked protocol does not specify K=50."
    )

expected_conditions = {
    "CONCEPT_ONLY",
    "UNRANKED_CONTEXTS",
    "COLLAPSE_K50",
}

if set(
    locked_protocol["locked_test_conditions"]
) != expected_conditions:
    raise AssertionError(
        "Unexpected locked test conditions."
    )


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def clean_text(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def clean_context_edges(value):
    value = clean_text(value)

    if not value:
        return ""

    if re.match(r"^[a-z]{1,12}\s", value):
        value = re.sub(
            r"^[^\s]+\s+",
            "",
            value,
            count=1,
        )

    if (
        value
        and value[-1].islower()
        and not re.search(r"[.!?]['\")\]]?$", value)
        and " " in value
    ):
        value = value.rsplit(
            " ",
            1,
        )[0].rstrip()

    if len(value) > MAX_CONTEXT_CHARACTERS:
        value = value[
            :MAX_CONTEXT_CHARACTERS
        ]

        if " " in value:
            value = value.rsplit(
                " ",
                1,
            )[0].rstrip()

        value += " …"

    return value


def unique_preserving_order(values):
    output = []
    seen = set()

    for value in values:
        value = clean_context_edges(value)

        if not value:
            continue

        key = value.casefold()

        if key not in seen:
            seen.add(key)
            output.append(value)

    return output


def format_context_block(contexts):
    return "\n".join(
        f"{index}. {context}"
        for index, context in enumerate(
            contexts,
            start=1,
        )
    )


def make_iso_prompt(
    concept_label,
    contexts,
    condition,
):
    prompt_parts = [
        (
            "You are a terminology expert working on "
            "biomedical concept definitions."
        ),
        (
            f"Write exactly one intensional definition "
            f"for the concept {clean_text(concept_label)}."
        ),
    ]

    if condition != "CONCEPT_ONLY" and contexts:
        prompt_parts.append(
            "Use the following occurrence evidence to identify "
            "the genus and distinguishing characteristics of the "
            "concept. Ignore article-specific details that are not "
            "part of the concept's meaning.\n\n"
            "Occurrence evidence:\n"
            f"{format_context_block(contexts)}"
        )

    prompt_parts.append(
        "Requirements:\n"
        "- Use an ISO 1087 (2019) and ISO 704 (2022)-based "
        "ontological intensional style\n"
        "- Start with the genus term, not with the concept label\n"
        "- Do NOT begin with \"A\", \"An\", or \"The\" unless "
        "absolutely necessary\n"
        "- Prefer the pattern: genus + that/which + "
        "distinguishing characteristics\n"
        "- Be concise and precise\n"
        "- Use specialized biomedical language when appropriate\n"
        "- Do not give examples\n"
        "- Do not mention the evidence, passage, article, "
        "context, dataset, or these instructions\n"
        "- Do not explain your reasoning\n"
        "- Output only the definition text"
    )

    return "\n\n".join(prompt_parts)


def make_prompt_hash(prompt):
    return hashlib.sha256(
        prompt.encode("utf-8")
    ).hexdigest()


# ----------------------------------------------------------------
# LOAD ONLY EVIDENCE — NOT REFERENCE DEFINITIONS
# ----------------------------------------------------------------

if not EVIDENCE_PATH.exists():
    raise FileNotFoundError(
        f"Evidence file not found:\n{EVIDENCE_PATH}"
    )

evidence = pd.read_csv(EVIDENCE_PATH)

required_columns = {
    "concept_key",
    "concept_canonical",
    "split",
    "context",
    "collapse_K50",
}

missing_columns = required_columns.difference(
    evidence.columns
)

if missing_columns:
    raise ValueError(
        f"Evidence file missing columns: {sorted(missing_columns)}"
    )

evidence["split_standardized"] = (
    evidence["split"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

test_evidence = evidence.loc[
    evidence["split_standardized"].eq("test")
].copy()

test_concepts = (
    test_evidence[
        [
            "concept_key",
            "concept_canonical",
        ]
    ]
    .drop_duplicates(
        subset=["concept_key"]
    )
    .sort_values("concept_key")
    .reset_index(drop=True)
)

if len(test_concepts) != 48:
    raise AssertionError(
        f"Expected 48 test concepts; found {len(test_concepts)}."
    )

# Confirm that no definition-like column has been loaded.
definition_columns = [
    column
    for column in test_evidence.columns
    if "definition" in column.casefold()
]

if definition_columns:
    raise AssertionError(
        "Definition-bearing columns unexpectedly present:\n"
        f"{definition_columns}"
    )


# ----------------------------------------------------------------
# BUILD LOCKED TEST PROMPTS
# ----------------------------------------------------------------

tie_columns = [
    column
    for column in [
        "domain",
        "num_article",
        "start_char",
        "end_char",
        "context",
    ]
    if column in test_evidence.columns
]

manifest_rows = []

for concept_row in test_concepts.itertuples(
    index=False
):
    concept_key = concept_row.concept_key
    concept_label = concept_row.concept_canonical

    concept_evidence = test_evidence.loc[
        test_evidence["concept_key"].eq(
            concept_key
        )
    ].copy()

    concept_evidence["_context_clean"] = (
        concept_evidence["context"]
        .map(clean_context_edges)
    )

    concept_evidence = concept_evidence.loc[
        concept_evidence[
            "_context_clean"
        ].ne("")
    ].copy()

    if tie_columns:
        concept_evidence = (
            concept_evidence.sort_values(
                tie_columns,
                kind="mergesort",
            )
        )

    concept_evidence = (
        concept_evidence.drop_duplicates(
            subset=["_context_clean"],
            keep="first",
        )
    )

    unranked_contexts = (
        unique_preserving_order(
            concept_evidence[
                "_context_clean"
            ].tolist()
        )[:MAX_CONTEXTS]
    )

    collapse_ranked = (
        concept_evidence.sort_values(
            ["collapse_K50"] + tie_columns,
            ascending=(
                [False]
                + [True] * len(tie_columns)
            ),
            kind="mergesort",
        )
    )

    collapse_contexts = (
        unique_preserving_order(
            collapse_ranked[
                "_context_clean"
            ].tolist()
        )[:MAX_CONTEXTS]
    )

    condition_contexts = {
        "CONCEPT_ONLY": [],
        "UNRANKED_CONTEXTS": (
            unranked_contexts
        ),
        "COLLAPSE_K50": (
            collapse_contexts
        ),
    }

    for condition, contexts in (
        condition_contexts.items()
    ):
        prompt = make_iso_prompt(
            concept_label=concept_label,
            contexts=contexts,
            condition=condition,
        )

        manifest_rows.append(
            {
                "concept_key": concept_key,
                "concept_canonical": (
                    concept_label
                ),
                "definition_split": "test",
                "condition": condition,
                "selected_k": (
                    50
                    if condition == "COLLAPSE_K50"
                    else None
                ),
                "available_unique_contexts": int(
                    concept_evidence[
                        "_context_clean"
                    ].nunique()
                ),
                "selected_context_count": int(
                    len(contexts)
                ),
                "selected_contexts_json": (
                    json.dumps(
                        contexts,
                        ensure_ascii=False,
                    )
                ),
                "prompt_style": (
                    "DETECH_ISO_INTENSIONAL_V1"
                ),
                "prompt": prompt,
                "prompt_hash": (
                    make_prompt_hash(prompt)
                ),
            }
        )

test_manifest = pd.DataFrame(
    manifest_rows
)


# ----------------------------------------------------------------
# AUDIT
# ----------------------------------------------------------------

if len(test_manifest) != 144:
    raise AssertionError(
        f"Expected 144 rows; found {len(test_manifest)}."
    )

if test_manifest["concept_key"].nunique() != 48:
    raise AssertionError(
        "Expected 48 unique test concepts."
    )

if set(test_manifest["condition"]) != (
    expected_conditions
):
    raise AssertionError(
        "Test conditions do not match the locked protocol."
    )

if test_manifest.duplicated(
    subset=["concept_key", "condition"]
).any():
    raise AssertionError(
        "Duplicate concept-condition rows detected."
    )

if any(
    "definition" in column.casefold()
    for column in test_manifest.columns
):
    # definition_split is permitted metadata, but no definition text.
    unexpected = [
        column
        for column in test_manifest.columns
        if (
            "definition" in column.casefold()
            and column != "definition_split"
        )
    ]

    if unexpected:
        raise AssertionError(
            "Reference-bearing manifest columns detected:\n"
            f"{unexpected}"
        )

condition_audit = (
    test_manifest.groupby("condition")
    .agg(
        concepts=("concept_key", "nunique"),
        prompts=("prompt", "size"),
        mean_selected_contexts=(
            "selected_context_count",
            "mean",
        ),
        concepts_without_context=(
            "selected_context_count",
            lambda values: int(
                (values == 0).sum()
            ),
        ),
        mean_prompt_characters=(
            "prompt",
            lambda values: float(
                values.str.len().mean()
            ),
        ),
        maximum_prompt_characters=(
            "prompt",
            lambda values: int(
                values.str.len().max()
            ),
        ),
    )
    .reset_index()
)

print("=" * 78)
print("REFERENCE-BLIND TEST PROMPT MANIFEST")
print("=" * 78)

print("Test concepts:", test_manifest["concept_key"].nunique())
print("Conditions:", sorted(expected_conditions))
print("Manifest rows:", len(test_manifest))
print("Reference definitions loaded:", False)
print("Reference-definition columns:", [])

print("\nCondition audit:")
print(
    condition_audit.to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
    )
)

print("\nConcepts without contextual evidence:")
no_context = (
    test_manifest.loc[
        test_manifest["condition"].eq(
            "UNRANKED_CONTEXTS"
        )
        & test_manifest[
            "selected_context_count"
        ].eq(0),
        [
            "concept_key",
            "concept_canonical",
        ],
    ]
    .drop_duplicates()
)

if no_context.empty:
    print("None")
else:
    print(no_context.to_string(index=False))


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

test_manifest.to_csv(
    TEST_MANIFEST_PATH,
    index=False,
)

metadata = {
    "status": "locked_before_test_generation",
    "test_concepts": 48,
    "conditions": sorted(expected_conditions),
    "manifest_rows": int(len(test_manifest)),
    "selected_k": 50,
    "context_budget": MAX_CONTEXTS,
    "prompt_style": "DETECH_ISO_INTENSIONAL_V1",
    "reference_definitions_loaded": False,
    "reference_definitions_in_manifest": False,
    "reference_definitions_in_prompts": False,
    "generation_completed": False,
    "locked_protocol": str(
        LOCKED_PROTOCOL_PATH
    ),
    "evidence_source": str(
        EVIDENCE_PATH
    ),
    "output_manifest": str(
        TEST_MANIFEST_PATH
    ),
    "condition_audit": (
        condition_audit.to_dict(
            orient="records"
        )
    ),
}

with open(
    TEST_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved reference-blind test manifest to:")
print(TEST_MANIFEST_PATH)

print("\nSaved manifest metadata to:")
print(TEST_METADATA_PATH)

print("\nCell 26 completed successfully.")

In [ ]:
# ================================================================
# CELL 27 — REFERENCE-BLIND LOCKED TEST GENERATION
# ================================================================
#
# Generates:
#   48 concepts × 3 locked conditions = 144 definitions
#
# Reference definitions are not loaded or attached.
# The run is randomized, incrementally saved, and resumable.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import random
import re
import time

import pandas as pd
from google.colab import userdata
from openai import OpenAI
from tqdm.auto import tqdm


# ----------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

MANIFEST_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_test_prompt_manifest_ISO_locked.csv"
)

OUTPUT_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_test_ISO_generations_locked.csv"
)

METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_ISO_generation_metadata.json"
)

LOCKED_PROTOCOL_PATH = (
    RESULTS_DIR /
    "taskB_definition_locked_test_protocol.json"
)

MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.0
MAX_COMPLETION_TOKENS = 120

RANDOMIZATION_SEED = 42
MAX_RETRIES = 6
INITIAL_RETRY_SECONDS = 2.0

SYSTEM_MESSAGE = (
    "You are a terminology expert generating ontological "
    "intensional definitions. Follow the requested ISO-style, "
    "genus-first format exactly and output only the definition."
)


# ----------------------------------------------------------------
# SECURE CLIENT
# ----------------------------------------------------------------

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "The Colab secret OPENAI_API_KEY is missing or empty."
    )

client = OpenAI(api_key=api_key)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def clean_generated_definition(value):
    value = str(value).strip()

    value = re.sub(
        r"^\s*definition\s*:\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )

    return value.strip().strip('"').strip()


def make_generation_id(
    concept_key,
    condition,
):
    raw_value = (
        f"test|||{concept_key}|||{condition}"
    )

    return hashlib.sha256(
        raw_value.encode("utf-8")
    ).hexdigest()[:20]


def atomic_save_csv(frame, path):
    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    frame.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(path)


def call_definition_api(prompt):
    delay = INITIAL_RETRY_SECONDS

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):
        try:
            return client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_MESSAGE,
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                temperature=TEMPERATURE,
                max_completion_tokens=(
                    MAX_COMPLETION_TOKENS
                ),
            )

        except Exception as error:
            if attempt == MAX_RETRIES:
                raise

            print(
                f"\nAPI attempt {attempt} failed: "
                f"{type(error).__name__}: {error}"
            )
            print(
                f"Retrying in {delay:.1f} seconds..."
            )

            time.sleep(delay)
            delay *= 2


# ----------------------------------------------------------------
# LOAD AND VALIDATE MANIFEST
# ----------------------------------------------------------------

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Test manifest not found:\n{MANIFEST_PATH}"
    )

manifest = pd.read_csv(MANIFEST_PATH)

required_columns = {
    "concept_key",
    "concept_canonical",
    "definition_split",
    "condition",
    "selected_context_count",
    "selected_contexts_json",
    "prompt_style",
    "prompt",
    "prompt_hash",
}

missing_columns = required_columns.difference(
    manifest.columns
)

if missing_columns:
    raise ValueError(
        f"Manifest missing columns: {sorted(missing_columns)}"
    )

unexpected_reference_columns = [
    column
    for column in manifest.columns
    if (
        "definition" in column.casefold()
        and column != "definition_split"
    )
]

if unexpected_reference_columns:
    raise AssertionError(
        "Reference-bearing columns found:\n"
        f"{unexpected_reference_columns}"
    )

if len(manifest) != 144:
    raise AssertionError(
        f"Expected 144 rows; found {len(manifest)}."
    )

if manifest["concept_key"].nunique() != 48:
    raise AssertionError(
        "Expected 48 test concepts."
    )

if manifest["definition_split"].astype(
    str
).str.casefold().ne("test").any():
    raise AssertionError(
        "Non-test rows found in the locked manifest."
    )

expected_conditions = {
    "CONCEPT_ONLY",
    "UNRANKED_CONTEXTS",
    "COLLAPSE_K50",
}

if set(manifest["condition"]) != expected_conditions:
    raise AssertionError(
        "Manifest conditions do not match the locked protocol."
    )

# Independently verify every saved prompt hash.
reconstructed_hashes = manifest["prompt"].map(
    lambda value: hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()
)

if not reconstructed_hashes.equals(
    manifest["prompt_hash"]
):
    raise AssertionError(
        "At least one test prompt hash is invalid."
    )

manifest = manifest.copy()

manifest["generation_id"] = manifest.apply(
    lambda row: make_generation_id(
        row["concept_key"],
        row["condition"],
    ),
    axis=1,
)

if manifest["generation_id"].duplicated().any():
    raise AssertionError(
        "Duplicate test generation IDs detected."
    )


# ----------------------------------------------------------------
# FIXED RANDOMIZED ORDER
# ----------------------------------------------------------------

generation_ids = manifest[
    "generation_id"
].tolist()

randomizer = random.Random(
    RANDOMIZATION_SEED
)

randomizer.shuffle(generation_ids)

order_lookup = {
    generation_id: order
    for order, generation_id in enumerate(
        generation_ids,
        start=1,
    )
}

manifest["randomized_call_order"] = (
    manifest["generation_id"].map(
        order_lookup
    )
)

manifest = manifest.sort_values(
    "randomized_call_order"
).reset_index(drop=True)


# ----------------------------------------------------------------
# LOAD EXISTING PROGRESS
# ----------------------------------------------------------------

if OUTPUT_PATH.exists():
    completed = pd.read_csv(
        OUTPUT_PATH
    )

    required_saved_columns = {
        "generation_id",
        "prompt_hash",
    }

    if not required_saved_columns.issubset(
        completed.columns
    ):
        raise ValueError(
            "Existing output lacks generation_id "
            "or prompt_hash."
        )

    if completed[
        "generation_id"
    ].duplicated().any():
        raise ValueError(
            "Existing output contains duplicate IDs."
        )

    validation = completed.merge(
        manifest[
            [
                "generation_id",
                "prompt_hash",
            ]
        ],
        on="generation_id",
        how="left",
        suffixes=("_saved", "_current"),
        validate="one_to_one",
    )

    if validation[
        "prompt_hash_current"
    ].isna().any():
        raise ValueError(
            "Existing output contains unknown IDs."
        )

    if (
        validation["prompt_hash_saved"]
        != validation["prompt_hash_current"]
    ).any():
        raise ValueError(
            "Saved outputs were generated from different prompts."
        )

    completed_rows = completed.to_dict(
        orient="records"
    )

else:
    completed_rows = []

completed_ids = {
    row["generation_id"]
    for row in completed_rows
}

pending = manifest.loc[
    ~manifest["generation_id"].isin(
        completed_ids
    )
].copy()


# ----------------------------------------------------------------
# RUN SUMMARY
# ----------------------------------------------------------------

print("=" * 78)
print("REFERENCE-BLIND LOCKED TEST GENERATION")
print("=" * 78)

print("Model:", MODEL_NAME)
print("Prompt style: DETECH_ISO_INTENSIONAL_V1")
print("Temperature:", TEMPERATURE)
print("Maximum completion tokens:", MAX_COMPLETION_TOKENS)
print("Randomization seed:", RANDOMIZATION_SEED)
print("Total expected calls:", len(manifest))
print("Previously completed:", len(completed_ids))
print("Pending calls:", len(pending))
print("Reference definitions loaded:", False)
print("Reference definitions sent to API:", False)

print("\nPending calls by condition:")
print(
    pending["condition"]
    .value_counts()
    .sort_index()
)


# ----------------------------------------------------------------
# GENERATE
# ----------------------------------------------------------------

run_start = time.time()

for row in tqdm(
    pending.itertuples(index=False),
    total=len(pending),
    desc="Generating locked test definitions",
):
    request_time = datetime.now(
        timezone.utc
    ).isoformat()

    response = call_definition_api(
        prompt=row.prompt
    )

    raw_output = (
        response.choices[0].message.content
        or ""
    ).strip()

    generated_definition = (
        clean_generated_definition(
            raw_output
        )
    )

    usage = response.usage

    result_row = {
        "generation_id": row.generation_id,
        "randomized_call_order": (
            row.randomized_call_order
        ),
        "concept_key": row.concept_key,
        "concept_canonical": (
            row.concept_canonical
        ),
        "definition_split": (
            row.definition_split
        ),
        "condition": row.condition,
        "selected_context_count": (
            row.selected_context_count
        ),
        "selected_contexts_json": (
            row.selected_contexts_json
        ),
        "prompt_style": row.prompt_style,
        "prompt_hash": row.prompt_hash,

        "requested_model": MODEL_NAME,
        "returned_model": response.model,
        "temperature": TEMPERATURE,
        "max_completion_tokens": (
            MAX_COMPLETION_TOKENS
        ),

        "generated_definition": (
            generated_definition
        ),
        "raw_model_output": raw_output,

        "response_id": response.id,
        "system_fingerprint": getattr(
            response,
            "system_fingerprint",
            None,
        ),
        "finish_reason": (
            response.choices[0].finish_reason
        ),
        "prompt_tokens": (
            usage.prompt_tokens
            if usage is not None
            else None
        ),
        "completion_tokens": (
            usage.completion_tokens
            if usage is not None
            else None
        ),
        "total_tokens": (
            usage.total_tokens
            if usage is not None
            else None
        ),
        "request_time_utc": request_time,
    }

    completed_rows.append(
        result_row
    )

    current_results = (
        pd.DataFrame(completed_rows)
        .sort_values(
            "randomized_call_order"
        )
    )

    atomic_save_csv(
        current_results,
        OUTPUT_PATH,
    )


elapsed_minutes = (
    time.time() - run_start
) / 60


# ----------------------------------------------------------------
# FINAL STRUCTURAL AUDIT
# ----------------------------------------------------------------

results = pd.read_csv(
    OUTPUT_PATH
)

results["is_empty"] = (
    results["generated_definition"]
    .fillna("")
    .str.strip()
    .eq("")
)

results["word_count"] = (
    results["generated_definition"]
    .fillna("")
    .str.findall(r"\b[\w'-]+\b")
    .str.len()
)

results["sentence_terminal"] = (
    results["generated_definition"]
    .fillna("")
    .str.endswith((".", "!", "?"))
)

results["finish_reason_stop"] = (
    results["finish_reason"].eq("stop")
)

if len(results) != 144:
    raise AssertionError(
        f"Expected 144 completed rows; found {len(results)}."
    )

if results["generation_id"].duplicated().any():
    raise AssertionError(
        "Duplicate completed generations detected."
    )

if results["is_empty"].any():
    raise AssertionError(
        "At least one test generation is empty."
    )

if (
    ~results["finish_reason_stop"]
).any():
    raise AssertionError(
        "At least one test generation did not finish normally."
    )

if any(
    "reference" in column.casefold()
    for column in results.columns
):
    raise AssertionError(
        "A reference-bearing column entered the generated output."
    )

condition_audit = (
    results.groupby("condition")
    .agg(
        definitions=(
            "generation_id",
            "size",
        ),
        mean_word_count=(
            "word_count",
            "mean",
        ),
        minimum_word_count=(
            "word_count",
            "min",
        ),
        maximum_word_count=(
            "word_count",
            "max",
        ),
        terminal_sentence_rate=(
            "sentence_terminal",
            "mean",
        ),
        mean_prompt_tokens=(
            "prompt_tokens",
            "mean",
        ),
        mean_completion_tokens=(
            "completion_tokens",
            "mean",
        ),
    )
    .reset_index()
)

print("\n" + "=" * 78)
print("LOCKED TEST GENERATION AUDIT")
print("=" * 78)

print("Completed generations:", len(results))
print("Unique test concepts:", results["concept_key"].nunique())
print("Conditions:", results["condition"].nunique())
print("Empty generations:", int(results["is_empty"].sum()))
print(
    "Non-stop finish reasons:",
    int((~results["finish_reason_stop"]).sum()),
)
print(
    "Elapsed minutes for pending calls:",
    round(elapsed_minutes, 2),
)

print("\nCondition audit:")
print(
    condition_audit.to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}",
    )
)


# ----------------------------------------------------------------
# SAVE AND UPDATE PROTOCOL
# ----------------------------------------------------------------

atomic_save_csv(
    results.sort_values(
        ["concept_key", "condition"]
    ),
    OUTPUT_PATH,
)

metadata = {
    "status": "complete",
    "evaluation_split": "test",
    "test_concepts": 48,
    "conditions": sorted(
        results["condition"].unique().tolist()
    ),
    "completed_generations": int(
        len(results)
    ),
    "model_requested": MODEL_NAME,
    "temperature": TEMPERATURE,
    "maximum_completion_tokens": (
        MAX_COMPLETION_TOKENS
    ),
    "system_message": SYSTEM_MESSAGE,
    "prompt_style": "DETECH_ISO_INTENSIONAL_V1",
    "randomization_seed": RANDOMIZATION_SEED,
    "reference_definitions_loaded": False,
    "reference_definitions_sent_to_api": False,
    "reference_definitions_in_output": False,
    "manifest": str(MANIFEST_PATH),
    "output": str(OUTPUT_PATH),
    "condition_audit": (
        condition_audit.to_dict(
            orient="records"
        )
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    LOCKED_PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    protocol = json.load(file_handle)

protocol[
    "test_generation_completed"
] = True

protocol[
    "test_generation_output"
] = str(OUTPUT_PATH)

protocol[
    "test_generation_completed_utc"
] = datetime.now(
    timezone.utc
).isoformat()

with open(
    LOCKED_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        protocol,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved locked test generations to:")
print(OUTPUT_PATH)

print("\nSaved generation metadata to:")
print(METADATA_PATH)

print("\nUpdated locked protocol:")
print(LOCKED_PROTOCOL_PATH)

print("\nCell 27 completed successfully.")

In [ ]:
# ================================================================
# CELL 28 — FINAL DEVELOPMENT AND TEST METRICS WITH ROUGE
# ================================================================
#
# Adds:
#   - Corpus and sentence BLEU
#   - ROUGE-1 precision/recall/F1
#   - ROUGE-2 precision/recall/F1
#   - ROUGE-L precision/recall/F1
#   - BiomedBERT BERTScore precision/recall/F1
#   - Means, medians, and quartiles
#
# Test references are loaded only after locked generation.
# ================================================================

!pip -q install --upgrade sacrebleu rouge-score bert-score

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import torch
from sacrebleu import corpus_bleu, sentence_bleu
from rouge_score import rouge_scorer
from bert_score import BERTScorer


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"
SPLITS_DIR = BASE_DIR / "splits"

DEVELOPMENT_GENERATION_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_development_ISO_generations_prompt_deduplicated.csv"
)

TEST_GENERATION_PATH = (
    PREDICTIONS_DIR /
    "taskB_definition_test_ISO_generations_prompt_deduplicated.csv"
)

CONCEPT_SPLIT_PATH = (
    SPLITS_DIR /
    "taskB_definition_concept_holdout_70_15_15.csv"
)

DEVELOPMENT_DETAILED_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_metrics_with_ROUGE.csv"
)

DEVELOPMENT_SUMMARY_PATH = (
    RESULTS_DIR /
    "taskB_definition_development_ISO_summary_with_ROUGE.csv"
)

TEST_DETAILED_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_ISO_detailed_metrics.csv"
)

TEST_SUMMARY_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_ISO_condition_metrics.csv"
)

METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_final_metric_metadata.json"
)

BIOMEDBERT_MODEL = (
    "microsoft/"
    "BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

BIOMEDBERT_LAYERS = 12
BERTSCORE_BATCH_SIZE = 16

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def clean_text(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def add_lexical_metrics(frame):
    frame = frame.copy()

    frame["sentence_bleu"] = [
        float(
            sentence_bleu(
                hypothesis=prediction,
                references=[reference],
                smooth_method="exp",
                use_effective_order=True,
            ).score
        )
        for prediction, reference in zip(
            frame["generated_definition"],
            frame["reference_definition"],
        )
    ]

    scorer = rouge_scorer.RougeScorer(
        [
            "rouge1",
            "rouge2",
            "rougeL",
        ],
        use_stemmer=True,
    )

    rouge_rows = []

    for prediction, reference in zip(
        frame["generated_definition"],
        frame["reference_definition"],
    ):
        scores = scorer.score(
            target=reference,
            prediction=prediction,
        )

        rouge_rows.append(
            {
                "rouge1_precision": (
                    scores["rouge1"].precision
                ),
                "rouge1_recall": (
                    scores["rouge1"].recall
                ),
                "rouge1_f1": (
                    scores["rouge1"].fmeasure
                ),
                "rouge2_precision": (
                    scores["rouge2"].precision
                ),
                "rouge2_recall": (
                    scores["rouge2"].recall
                ),
                "rouge2_f1": (
                    scores["rouge2"].fmeasure
                ),
                "rougeL_precision": (
                    scores["rougeL"].precision
                ),
                "rougeL_recall": (
                    scores["rougeL"].recall
                ),
                "rougeL_f1": (
                    scores["rougeL"].fmeasure
                ),
            }
        )

    rouge_frame = pd.DataFrame(
        rouge_rows,
        index=frame.index,
    )

    return pd.concat(
        [frame, rouge_frame],
        axis=1,
    )


def summarize_conditions(frame):
    summary_rows = []

    for condition, subset in frame.groupby(
        "condition",
        sort=True,
    ):
        subset = subset.sort_values(
            "concept_key"
        )

        predictions = subset[
            "generated_definition"
        ].tolist()

        references = subset[
            "reference_definition"
        ].tolist()

        row = {
            "condition": condition,
            "n_concepts": int(len(subset)),
            "corpus_bleu": float(
                corpus_bleu(
                    predictions,
                    [references],
                ).score
            ),
        }

        distribution_metrics = [
            "sentence_bleu",
            "rouge1_precision",
            "rouge1_recall",
            "rouge1_f1",
            "rouge2_precision",
            "rouge2_recall",
            "rouge2_f1",
            "rougeL_precision",
            "rougeL_recall",
            "rougeL_f1",
            "biomedbert_precision",
            "biomedbert_recall",
            "biomedbert_f1",
        ]

        for metric in distribution_metrics:
            row[f"{metric}_mean"] = float(
                subset[metric].mean()
            )
            row[f"{metric}_q25"] = float(
                subset[metric].quantile(0.25)
            )
            row[f"{metric}_median"] = float(
                subset[metric].median()
            )
            row[f"{metric}_q75"] = float(
                subset[metric].quantile(0.75)
            )

        row["descriptive_combined_score"] = (
            (row["corpus_bleu"] / 100.0)
            + row["biomedbert_f1_mean"]
        ) / 2.0

        summary_rows.append(row)

    return (
        pd.DataFrame(summary_rows)
        .sort_values(
            [
                "descriptive_combined_score",
                "biomedbert_f1_mean",
                "corpus_bleu",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )


# ----------------------------------------------------------------
# LOAD DEVELOPMENT OUTPUTS
# ----------------------------------------------------------------

development = pd.read_csv(
    DEVELOPMENT_GENERATION_PATH
)

development["generated_definition"] = (
    development["generated_definition"]
    .map(clean_text)
)

development["reference_definition"] = (
    development["reference_definition"]
    .map(clean_text)
)

if len(development) != 235:
    raise AssertionError(
        f"Expected 235 development rows; found {len(development)}."
    )


# ----------------------------------------------------------------
# LOAD TEST OUTPUTS AND EXPOSE REFERENCES FOR FIRST TIME
# ----------------------------------------------------------------

test = pd.read_csv(
    TEST_GENERATION_PATH
)

concept_split = pd.read_csv(
    CONCEPT_SPLIT_PATH,
    usecols=[
        "concept_key",
        "definition",
        "split",
    ],
)

test_references = (
    concept_split.loc[
        concept_split["split"]
        .astype(str)
        .str.casefold()
        .eq("test"),
        [
            "concept_key",
            "definition",
        ],
    ]
    .drop_duplicates(
        subset=["concept_key"]
    )
    .rename(
        columns={
            "definition": (
                "reference_definition"
            )
        }
    )
)

if len(test_references) != 48:
    raise AssertionError(
        f"Expected 48 test references; found {len(test_references)}."
    )

test = test.merge(
    test_references,
    on="concept_key",
    how="left",
    validate="many_to_one",
)

test["generated_definition"] = (
    test["generated_definition"]
    .map(clean_text)
)

test["reference_definition"] = (
    test["reference_definition"]
    .map(clean_text)
)

if len(test) != 144:
    raise AssertionError(
        f"Expected 144 test rows; found {len(test)}."
    )

if test["reference_definition"].eq("").any():
    raise AssertionError(
        "At least one test reference failed to merge."
    )


# ----------------------------------------------------------------
# LEXICAL METRICS
# ----------------------------------------------------------------

development = add_lexical_metrics(
    development
)

test = add_lexical_metrics(
    test
)


# ----------------------------------------------------------------
# BIOMEDBERT BERTSCORE
# ----------------------------------------------------------------

print("=" * 78)
print("FINAL METRIC EVALUATION")
print("=" * 78)

print("Development rows:", len(development))
print("Test rows:", len(test))
print("Device:", DEVICE)
print("BiomedBERT:", BIOMEDBERT_MODEL)
print("Test references exposed after generation:", True)

biomedbert_scorer = BERTScorer(
    model_type=BIOMEDBERT_MODEL,
    num_layers=BIOMEDBERT_LAYERS,
    batch_size=BERTSCORE_BATCH_SIZE,
    nthreads=4,
    idf=False,
    device=DEVICE,
    lang=None,
    rescale_with_baseline=False,
    use_fast_tokenizer=True,
)

# Prevent tokenizer overflow with current Transformers versions.
biomedbert_scorer._tokenizer.model_max_length = 512


def add_biomedbert_metrics(frame):
    with torch.inference_mode():
        precision, recall, f1 = (
            biomedbert_scorer.score(
                cands=frame[
                    "generated_definition"
                ].tolist(),
                refs=frame[
                    "reference_definition"
                ].tolist(),
                batch_size=(
                    BERTSCORE_BATCH_SIZE
                ),
            )
        )

    frame = frame.copy()

    frame["biomedbert_precision"] = (
        precision.cpu().numpy()
    )
    frame["biomedbert_recall"] = (
        recall.cpu().numpy()
    )
    frame["biomedbert_f1"] = (
        f1.cpu().numpy()
    )

    return frame


development = add_biomedbert_metrics(
    development
)

test = add_biomedbert_metrics(
    test
)


# ----------------------------------------------------------------
# SUMMARIZE
# ----------------------------------------------------------------

development_summary = summarize_conditions(
    development
)

test_summary = summarize_conditions(
    test
)

display_columns = [
    "condition",
    "corpus_bleu",
    "rouge1_f1_mean",
    "rouge2_f1_mean",
    "rougeL_f1_mean",
    "biomedbert_precision_mean",
    "biomedbert_recall_mean",
    "biomedbert_f1_mean",
    "descriptive_combined_score",
]

print("\n" + "=" * 78)
print("DEVELOPMENT RESULTS WITH ROUGE")
print("=" * 78)

print(
    development_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\n" + "=" * 78)
print("LOCKED TEST RESULTS")
print("=" * 78)

print(
    test_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\nTest quartiles:")

quartile_columns = [
    "condition",
    "sentence_bleu_q25",
    "sentence_bleu_median",
    "sentence_bleu_q75",
    "rouge1_f1_q25",
    "rouge1_f1_median",
    "rouge1_f1_q75",
    "rouge2_f1_q25",
    "rouge2_f1_median",
    "rouge2_f1_q75",
    "rougeL_f1_q25",
    "rougeL_f1_median",
    "rougeL_f1_q75",
    "biomedbert_f1_q25",
    "biomedbert_f1_median",
    "biomedbert_f1_q75",
]

print(
    test_summary[
        quartile_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

development.to_csv(
    DEVELOPMENT_DETAILED_PATH,
    index=False,
)

development_summary.to_csv(
    DEVELOPMENT_SUMMARY_PATH,
    index=False,
)

test.to_csv(
    TEST_DETAILED_PATH,
    index=False,
)

test_summary.to_csv(
    TEST_SUMMARY_PATH,
    index=False,
)

metadata = {
    "identical_prompt_outputs_canonicalized": True,
    "canonicalization_rule": (
    "Earliest randomized API response for each exact prompt hash."
    ),
    "development_generation_input": str(
    DEVELOPMENT_GENERATION_PATH
    ),
    "test_generation_input": str(
    TEST_GENERATION_PATH
    ),
    "development_concepts": 47,
    "test_concepts": 48,
    "test_generation_completed_before_reference_loading": True,
    "bleu_library": "sacrebleu",
    "rouge_library": "rouge-score",
    "rouge_types": [
        "rouge1",
        "rouge2",
        "rougeL",
    ],
    "rouge_use_stemmer": True,
    "semantic_metric": "BERTScore",
    "semantic_encoder": BIOMEDBERT_MODEL,
    "semantic_encoder_layers": BIOMEDBERT_LAYERS,
    "bertscore_idf": False,
    "bertscore_rescaled_with_baseline": False,
    "development_detailed_output": str(
        DEVELOPMENT_DETAILED_PATH
    ),
    "development_summary_output": str(
        DEVELOPMENT_SUMMARY_PATH
    ),
    "test_detailed_output": str(
        TEST_DETAILED_PATH
    ),
    "test_summary_output": str(
        TEST_SUMMARY_PATH
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved development detailed metrics to:")
print(DEVELOPMENT_DETAILED_PATH)

print("\nSaved development summary to:")
print(DEVELOPMENT_SUMMARY_PATH)

print("\nSaved locked test detailed metrics to:")
print(TEST_DETAILED_PATH)

print("\nSaved locked test summary to:")
print(TEST_SUMMARY_PATH)

print("\nSaved metric metadata to:")
print(METADATA_PATH)

print("\nCell 28 completed successfully.")

In [ ]:
# ================================================================
# CELL 29 — FINAL PAIRED TEST BOOTSTRAP
# ================================================================
#
# Contrasts:
#   1. COLLAPSE_K50 − UNRANKED_CONTEXTS
#   2. COLLAPSE_K50 − CONCEPT_ONLY
#   3. UNRANKED_CONTEXTS − CONCEPT_ONLY
#
# Metrics:
#   - Corpus BLEU
#   - Mean sentence BLEU
#   - Mean ROUGE-1/2/L F1
#   - Mean BiomedBERT F1
#   - BLEU/BiomedBERT descriptive combined score
#
# Uncertainty is estimated across test concepts.
# ================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sacrebleu import corpus_bleu
from tqdm.auto import tqdm


# ----------------------------------------------------------------
# PATHS AND CONFIGURATION
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"

TEST_DETAILED_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_ISO_detailed_metrics.csv"
)

BOOTSTRAP_REPLICATES_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_ISO_paired_bootstrap.csv"
)

BOOTSTRAP_SUMMARY_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_ISO_paired_bootstrap_summary.csv"
)

STATISTICAL_METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_test_ISO_statistical_metadata.json"
)

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 42

CONTRASTS = [
    (
        "COLLAPSE_K50",
        "UNRANKED_CONTEXTS",
    ),
    (
        "COLLAPSE_K50",
        "CONCEPT_ONLY",
    ),
    (
        "UNRANKED_CONTEXTS",
        "CONCEPT_ONLY",
    ),
]

METRICS = [
    "corpus_bleu",
    "mean_sentence_bleu",
    "mean_rouge1_f1",
    "mean_rouge2_f1",
    "mean_rougeL_f1",
    "mean_biomedbert_f1",
    "combined_score",
]


# ----------------------------------------------------------------
# LOAD AND ALIGN TEST RESULTS
# ----------------------------------------------------------------

test = pd.read_csv(
    TEST_DETAILED_PATH
)

required_columns = {
    "concept_key",
    "condition",
    "generated_definition",
    "reference_definition",
    "sentence_bleu",
    "rouge1_f1",
    "rouge2_f1",
    "rougeL_f1",
    "biomedbert_f1",
}

missing_columns = required_columns.difference(
    test.columns
)

if missing_columns:
    raise ValueError(
        f"Test results missing columns: {sorted(missing_columns)}"
    )

if test["concept_key"].nunique() != 48:
    raise AssertionError(
        "Expected 48 unique test concepts."
    )

condition_frames = {}

for condition, frame in test.groupby(
    "condition"
):
    condition_frames[condition] = (
        frame.sort_values("concept_key")
        .reset_index(drop=True)
    )

reference_keys = condition_frames[
    "CONCEPT_ONLY"
]["concept_key"].tolist()

reference_definitions = condition_frames[
    "CONCEPT_ONLY"
]["reference_definition"].tolist()

for condition, frame in (
    condition_frames.items()
):
    if frame["concept_key"].tolist() != (
        reference_keys
    ):
        raise AssertionError(
            f"Concept misalignment for {condition}."
        )

    if frame[
        "reference_definition"
    ].tolist() != reference_definitions:
        raise AssertionError(
            f"Reference mismatch for {condition}."
        )

n_concepts = len(reference_keys)


# ----------------------------------------------------------------
# METRIC FUNCTION
# ----------------------------------------------------------------

def calculate_metrics(
    frame,
    sampled_indices,
):
    sampled = frame.iloc[
        sampled_indices
    ]

    predictions = sampled[
        "generated_definition"
    ].tolist()

    references = sampled[
        "reference_definition"
    ].tolist()

    sample_corpus_bleu = float(
        corpus_bleu(
            predictions,
            [references],
        ).score
    )

    sample_biomedbert = float(
        sampled["biomedbert_f1"].mean()
    )

    return {
        "corpus_bleu": (
            sample_corpus_bleu
        ),
        "mean_sentence_bleu": float(
            sampled["sentence_bleu"].mean()
        ),
        "mean_rouge1_f1": float(
            sampled["rouge1_f1"].mean()
        ),
        "mean_rouge2_f1": float(
            sampled["rouge2_f1"].mean()
        ),
        "mean_rougeL_f1": float(
            sampled["rougeL_f1"].mean()
        ),
        "mean_biomedbert_f1": (
            sample_biomedbert
        ),
        "combined_score": (
            (sample_corpus_bleu / 100.0)
            + sample_biomedbert
        ) / 2.0,
    }


# ----------------------------------------------------------------
# OBSERVED DIFFERENCES
# ----------------------------------------------------------------

all_indices = np.arange(
    n_concepts
)

observed_differences = {}

for target, baseline in CONTRASTS:
    target_metrics = calculate_metrics(
        condition_frames[target],
        all_indices,
    )

    baseline_metrics = calculate_metrics(
        condition_frames[baseline],
        all_indices,
    )

    contrast_name = (
        f"{target}_MINUS_{baseline}"
    )

    observed_differences[
        contrast_name
    ] = {
        metric: (
            target_metrics[metric]
            - baseline_metrics[metric]
        )
        for metric in METRICS
    }


# ----------------------------------------------------------------
# PAIRED BOOTSTRAP
# ----------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

needed_conditions = sorted(
    set(
        condition
        for contrast in CONTRASTS
        for condition in contrast
    )
)

for replicate in tqdm(
    range(BOOTSTRAP_REPLICATES),
    desc="Paired test bootstrap",
):
    sampled_indices = rng.integers(
        low=0,
        high=n_concepts,
        size=n_concepts,
    )

    sampled_metrics = {
        condition: calculate_metrics(
            condition_frames[condition],
            sampled_indices,
        )
        for condition in needed_conditions
    }

    for target, baseline in CONTRASTS:
        contrast_name = (
            f"{target}_MINUS_{baseline}"
        )

        for metric in METRICS:
            bootstrap_rows.append(
                {
                    "replicate": replicate,
                    "contrast": contrast_name,
                    "target_condition": target,
                    "baseline_condition": baseline,
                    "metric": metric,
                    "difference": (
                        sampled_metrics[
                            target
                        ][metric]
                        - sampled_metrics[
                            baseline
                        ][metric]
                    ),
                }
            )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# ----------------------------------------------------------------
# SUMMARIZE
# ----------------------------------------------------------------

summary_rows = []

for (
    contrast,
    target,
    baseline,
    metric,
), frame in bootstrap_results.groupby(
    [
        "contrast",
        "target_condition",
        "baseline_condition",
        "metric",
    ],
    sort=True,
):
    values = frame[
        "difference"
    ].to_numpy()

    probability_positive = float(
        np.mean(values > 0)
    )

    probability_negative = float(
        np.mean(values < 0)
    )

    # Descriptive two-sided bootstrap tail probability.
    two_sided_tail_probability = min(
        1.0,
        2.0 * min(
            probability_positive,
            probability_negative,
        ),
    )

    summary_rows.append(
        {
            "contrast": contrast,
            "target_condition": target,
            "baseline_condition": baseline,
            "metric": metric,
            "observed_difference": float(
                observed_differences[
                    contrast
                ][metric]
            ),
            "bootstrap_mean_difference": float(
                np.mean(values)
            ),
            "bootstrap_median_difference": float(
                np.median(values)
            ),
            "ci_lower_95": float(
                np.quantile(
                    values,
                    0.025,
                )
            ),
            "ci_upper_95": float(
                np.quantile(
                    values,
                    0.975,
                )
            ),
            "probability_greater_than_zero": (
                probability_positive
            ),
            "probability_less_than_zero": (
                probability_negative
            ),
            "two_sided_bootstrap_tail_probability": (
                two_sided_tail_probability
            ),
        }
    )

bootstrap_summary = pd.DataFrame(
    summary_rows
)


# ----------------------------------------------------------------
# DISPLAY
# ----------------------------------------------------------------

print("=" * 78)
print("FINAL PAIRED TEST BOOTSTRAP")
print("=" * 78)

print("Test concepts:", n_concepts)
print("Bootstrap replicates:", BOOTSTRAP_REPLICATES)
print("Bootstrap seed:", BOOTSTRAP_SEED)

for target, baseline in CONTRASTS:
    contrast_name = (
        f"{target}_MINUS_{baseline}"
    )

    print("\n" + "-" * 78)
    print(contrast_name)
    print("-" * 78)

    contrast_table = bootstrap_summary.loc[
        bootstrap_summary[
            "contrast"
        ].eq(contrast_name),
        [
            "metric",
            "observed_difference",
            "ci_lower_95",
            "ci_upper_95",
            "probability_greater_than_zero",
            "two_sided_bootstrap_tail_probability",
        ],
    ]

    print(
        contrast_table.to_string(
            index=False,
            float_format=lambda value: f"{value:.6f}",
        )
    )


# ----------------------------------------------------------------
# AUTOMATED CLAIM CALIBRATION
# ----------------------------------------------------------------

def interval_interpretation(
    lower,
    upper,
):
    if lower > 0:
        return "reliable_positive"

    if upper < 0:
        return "reliable_negative"

    return "inconclusive"


claim_table = bootstrap_summary.copy()

claim_table["interval_interpretation"] = (
    claim_table.apply(
        lambda row: interval_interpretation(
            row["ci_lower_95"],
            row["ci_upper_95"],
        ),
        axis=1,
    )
)

print("\n" + "=" * 78)
print("INTERVAL INTERPRETATION")
print("=" * 78)

print(
    claim_table[
        [
            "contrast",
            "metric",
            "observed_difference",
            "ci_lower_95",
            "ci_upper_95",
            "interval_interpretation",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

bootstrap_results.to_csv(
    BOOTSTRAP_REPLICATES_PATH,
    index=False,
)

claim_table.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)

metadata = {
    "evaluation_split": "test",
    "test_concepts": n_concepts,
    "paired_resampling_unit": "concept",
    "bootstrap_replicates": (
        BOOTSTRAP_REPLICATES
    ),
    "bootstrap_seed": BOOTSTRAP_SEED,
    "confidence_interval": (
        "percentile 95%"
    ),
    "metrics": METRICS,
    "contrasts": [
        {
            "target": target,
            "baseline": baseline,
        }
        for target, baseline in CONTRASTS
    ],
    "uncertainty_scope": (
        "Sampling variability across concepts; "
        "does not estimate repeated-LLM-generation variability."
    ),
    "test_detailed_metrics": str(
        TEST_DETAILED_PATH
    ),
    "bootstrap_replicates_output": str(
        BOOTSTRAP_REPLICATES_PATH
    ),
    "bootstrap_summary_output": str(
        BOOTSTRAP_SUMMARY_PATH
    ),
}

with open(
    STATISTICAL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved bootstrap replicates to:")
print(BOOTSTRAP_REPLICATES_PATH)

print("\nSaved bootstrap summary to:")
print(BOOTSTRAP_SUMMARY_PATH)

print("\nSaved statistical metadata to:")
print(STATISTICAL_METADATA_PATH)

print("\nCell 29 completed successfully.")

In [ ]:
# ================================================================
# CELL 30 — ACTIVE COLLAPSE-INTERVENTION AUDIT
# ================================================================
#
# Determines:
#   - How often K=50 selected different contexts from the
#     unranked baseline.
#   - How often different evidence changed the generated output.
#   - Metric differences specifically among affected concepts.
#
# No API calls are made.
# ================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

FILES = {
    "development": {
        "manifest": (
            PREDICTIONS_DIR /
            "taskB_definition_development_prompt_manifest_ISO_locked.csv"
        ),
        "metrics": (
            RESULTS_DIR /
            "taskB_definition_development_ISO_metrics_with_ROUGE.csv"
        ),
    },
    "test": {
        "manifest": (
            PREDICTIONS_DIR /
            "taskB_definition_test_prompt_manifest_ISO_locked.csv"
        ),
        "metrics": (
            RESULTS_DIR /
            "taskB_definition_test_ISO_detailed_metrics.csv"
        ),
    },
}

AUDIT_PATH = (
    RESULTS_DIR /
    "taskB_definition_collapse_active_intervention_audit.csv"
)

SUMMARY_PATH = (
    RESULTS_DIR /
    "taskB_definition_collapse_active_intervention_summary.csv"
)

METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_collapse_active_intervention_metadata.json"
)


# ----------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------

def parse_contexts(value):
    if pd.isna(value) or not str(value).strip():
        return []

    parsed = json.loads(value)

    return [
        str(context).strip()
        for context in parsed
        if str(context).strip()
    ]


def context_jaccard(first, second):
    first_set = set(first)
    second_set = set(second)

    if not first_set and not second_set:
        return 1.0

    union = first_set | second_set

    return len(
        first_set & second_set
    ) / len(union)


# ----------------------------------------------------------------
# BUILD PER-CONCEPT AUDIT
# ----------------------------------------------------------------

all_audits = []

metric_columns = [
    "sentence_bleu",
    "rouge1_f1",
    "rouge2_f1",
    "rougeL_f1",
    "biomedbert_f1",
]

for split_name, paths in FILES.items():
    manifest = pd.read_csv(
        paths["manifest"]
    )

    metrics = pd.read_csv(
        paths["metrics"]
    )

    unranked_manifest = manifest.loc[
        manifest["condition"].eq(
            "UNRANKED_CONTEXTS"
        ),
        [
            "concept_key",
            "concept_canonical",
            "available_unique_contexts",
            "selected_contexts_json",
        ],
    ].rename(
        columns={
            "selected_contexts_json": (
                "unranked_contexts_json"
            )
        }
    )

    collapse_manifest = manifest.loc[
        manifest["condition"].eq(
            "COLLAPSE_K50"
        ),
        [
            "concept_key",
            "selected_contexts_json",
        ],
    ].rename(
        columns={
            "selected_contexts_json": (
                "collapse_contexts_json"
            )
        }
    )

    audit = unranked_manifest.merge(
        collapse_manifest,
        on="concept_key",
        how="inner",
        validate="one_to_one",
    )

    audit["unranked_contexts"] = (
        audit["unranked_contexts_json"]
        .map(parse_contexts)
    )

    audit["collapse_contexts"] = (
        audit["collapse_contexts_json"]
        .map(parse_contexts)
    )

    audit["same_context_order"] = [
        first == second
        for first, second in zip(
            audit["unranked_contexts"],
            audit["collapse_contexts"],
        )
    ]

    audit["same_context_set"] = [
        set(first) == set(second)
        for first, second in zip(
            audit["unranked_contexts"],
            audit["collapse_contexts"],
        )
    ]

    audit["context_jaccard"] = [
        context_jaccard(
            first,
            second,
        )
        for first, second in zip(
            audit["unranked_contexts"],
            audit["collapse_contexts"],
        )
    ]

    audit["collapse_changed_evidence"] = (
        ~audit["same_context_order"]
    )

    unranked_metrics = metrics.loc[
        metrics["condition"].eq(
            "UNRANKED_CONTEXTS"
        ),
        [
            "concept_key",
            "generated_definition",
        ] + metric_columns,
    ].rename(
        columns={
            "generated_definition": (
                "unranked_definition"
            ),
            **{
                metric: f"unranked_{metric}"
                for metric in metric_columns
            },
        }
    )

    collapse_metrics = metrics.loc[
        metrics["condition"].eq(
            "COLLAPSE_K50"
        ),
        [
            "concept_key",
            "generated_definition",
        ] + metric_columns,
    ].rename(
        columns={
            "generated_definition": (
                "collapse_definition"
            ),
            **{
                metric: f"collapse_{metric}"
                for metric in metric_columns
            },
        }
    )

    audit = audit.merge(
        unranked_metrics,
        on="concept_key",
        how="left",
        validate="one_to_one",
    ).merge(
        collapse_metrics,
        on="concept_key",
        how="left",
        validate="one_to_one",
    )

    audit["generated_output_changed"] = (
        audit["collapse_definition"]
        .str.strip()
        .str.casefold()
        != audit["unranked_definition"]
        .str.strip()
        .str.casefold()
    )

    for metric in metric_columns:
        audit[f"difference_{metric}"] = (
            audit[f"collapse_{metric}"]
            - audit[f"unranked_{metric}"]
        )

    audit.insert(
        0,
        "split",
        split_name,
    )

    audit = audit.drop(
        columns=[
            "unranked_contexts",
            "collapse_contexts",
        ]
    )

    all_audits.append(audit)

active_audit = pd.concat(
    all_audits,
    ignore_index=True,
)


# ----------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------

summary_rows = []

for split_name, split_frame in (
    active_audit.groupby("split")
):
    for subgroup_name, subgroup in [
        ("ALL_CONCEPTS", split_frame),
        (
            "EVIDENCE_CHANGED",
            split_frame.loc[
                split_frame[
                    "collapse_changed_evidence"
                ]
            ],
        ),
        (
            "OUTPUT_CHANGED",
            split_frame.loc[
                split_frame[
                    "generated_output_changed"
                ]
            ],
        ),
    ]:
        row = {
            "split": split_name,
            "subgroup": subgroup_name,
            "concepts": int(
                len(subgroup)
            ),
            "evidence_changed": int(
                subgroup[
                    "collapse_changed_evidence"
                ].sum()
            ),
            "output_changed": int(
                subgroup[
                    "generated_output_changed"
                ].sum()
            ),
            "mean_context_jaccard": (
                float(
                    subgroup[
                        "context_jaccard"
                    ].mean()
                )
                if len(subgroup)
                else np.nan
            ),
        }

        for metric in metric_columns:
            difference_column = (
                f"difference_{metric}"
            )

            row[
                f"mean_difference_{metric}"
            ] = (
                float(
                    subgroup[
                        difference_column
                    ].mean()
                )
                if len(subgroup)
                else np.nan
            )

            row[
                f"positive_concepts_{metric}"
            ] = int(
                subgroup[
                    difference_column
                ].gt(0).sum()
            )

            row[
                f"negative_concepts_{metric}"
            ] = int(
                subgroup[
                    difference_column
                ].lt(0).sum()
            )

        summary_rows.append(row)

summary = pd.DataFrame(
    summary_rows
)


# ----------------------------------------------------------------
# DISPLAY
# ----------------------------------------------------------------

print("=" * 78)
print("ACTIVE COLLAPSE-INTERVENTION AUDIT")
print("=" * 78)

for split_name, split_frame in (
    active_audit.groupby("split")
):
    print("\n" + "-" * 78)
    print(split_name.upper())
    print("-" * 78)

    print("Concepts:", len(split_frame))
    print(
        "Concepts where selected evidence changed:",
        int(
            split_frame[
                "collapse_changed_evidence"
            ].sum()
        ),
    )
    print(
        "Concepts where generated output changed:",
        int(
            split_frame[
                "generated_output_changed"
            ].sum()
        ),
    )
    print(
        "Changed evidence but identical output:",
        int(
            (
                split_frame[
                    "collapse_changed_evidence"
                ]
                & ~split_frame[
                    "generated_output_changed"
                ]
            ).sum()
        ),
    )

print("\nMetric differences: K50 minus unranked")
print(
    summary[
        [
            "split",
            "subgroup",
            "concepts",
            "evidence_changed",
            "output_changed",
            "mean_context_jaccard",
            "mean_difference_sentence_bleu",
            "mean_difference_rouge1_f1",
            "mean_difference_rouge2_f1",
            "mean_difference_rougeL_f1",
            "mean_difference_biomedbert_f1",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nTest concepts with changed evidence:")
test_changed = active_audit.loc[
    active_audit["split"].eq("test")
    & active_audit[
        "collapse_changed_evidence"
    ],
    [
        "concept_canonical",
        "available_unique_contexts",
        "context_jaccard",
        "generated_output_changed",
        "difference_sentence_bleu",
        "difference_rouge1_f1",
        "difference_rouge2_f1",
        "difference_rougeL_f1",
        "difference_biomedbert_f1",
    ],
].sort_values(
    "difference_biomedbert_f1",
    ascending=False,
)

if test_changed.empty:
    print("None")
else:
    print(
        test_changed.to_string(
            index=False,
            float_format=lambda value: f"{value:.6f}",
        )
    )


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

active_audit.to_csv(
    AUDIT_PATH,
    index=False,
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

metadata = {
    "purpose": (
        "Determine when collapse ranking materially changed "
        "the evidence supplied to the definition generator."
    ),
    "collapse_condition": "COLLAPSE_K50",
    "baseline_condition": "UNRANKED_CONTEXTS",
    "context_budget": 3,
    "splits": ["development", "test"],
    "audit_output": str(AUDIT_PATH),
    "summary_output": str(SUMMARY_PATH),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved active-intervention audit to:")
print(AUDIT_PATH)

print("\nSaved active-intervention summary to:")
print(SUMMARY_PATH)

print("\nSaved audit metadata to:")
print(METADATA_PATH)

print("\nCell 30 completed successfully.")

In [ ]:
# ================================================================
# CELL 31 — CANONICALIZE IDENTICAL-PROMPT OUTPUTS
# ================================================================
#
# If multiple conditions used the exact same prompt, assign them
# the response from the earliest randomized API call.
#
# This removes API nondeterminism from comparisons in which the
# experimental input was identical.
#
# No new API calls are made.
# ================================================================

from pathlib import Path
import json

import pandas as pd


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

FILES = {
    "development": {
        "input": (
            PREDICTIONS_DIR /
            "taskB_definition_development_ISO_generations_all_conditions.csv"
        ),
        "output": (
            PREDICTIONS_DIR /
            "taskB_definition_development_ISO_generations_prompt_deduplicated.csv"
        ),
    },
    "test": {
        "input": (
            PREDICTIONS_DIR /
            "taskB_definition_test_ISO_generations_locked.csv"
        ),
        "output": (
            PREDICTIONS_DIR /
            "taskB_definition_test_ISO_generations_prompt_deduplicated.csv"
        ),
    },
}

AUDIT_PATH = (
    RESULTS_DIR /
    "taskB_definition_identical_prompt_canonicalization_audit.csv"
)

METADATA_PATH = (
    RESULTS_DIR /
    "taskB_definition_identical_prompt_canonicalization_metadata.json"
)


# ----------------------------------------------------------------
# CANONICALIZE EACH SPLIT
# ----------------------------------------------------------------

audit_rows = []

response_columns = [
    "generated_definition",
    "raw_model_output",
    "response_id",
    "system_fingerprint",
    "finish_reason",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "request_time_utc",
]

for split_name, paths in FILES.items():
    frame = pd.read_csv(
        paths["input"]
    )

    required_columns = {
        "generation_id",
        "randomized_call_order",
        "concept_key",
        "condition",
        "prompt_hash",
        "generated_definition",
    }

    missing = required_columns.difference(
        frame.columns
    )

    if missing:
        raise ValueError(
            f"{split_name} file missing columns: {sorted(missing)}"
        )

    frame = frame.copy()

    # Preserve every originally generated response for audit.
    for column in response_columns:
        if column in frame.columns:
            frame[
                f"original_{column}"
            ] = frame[column]

    # Earliest call for each exact prompt hash.
    canonical_sources = (
        frame.sort_values(
            "randomized_call_order"
        )
        .drop_duplicates(
            subset=["prompt_hash"],
            keep="first",
        )
    )

    canonical_columns = [
        "prompt_hash",
        "generation_id",
        "condition",
        "randomized_call_order",
    ] + [
        column
        for column in response_columns
        if column in frame.columns
    ]

    canonical_sources = (
        canonical_sources[
            canonical_columns
        ]
        .rename(
            columns={
                "generation_id": (
                    "canonical_source_generation_id"
                ),
                "condition": (
                    "canonical_source_condition"
                ),
                "randomized_call_order": (
                    "canonical_source_call_order"
                ),
                **{
                    column: f"canonical_{column}"
                    for column in response_columns
                    if column in frame.columns
                },
            }
        )
    )

    frame = frame.merge(
        canonical_sources,
        on="prompt_hash",
        how="left",
        validate="many_to_one",
    )

    frame["prompt_multiplicity"] = (
        frame.groupby(
            "prompt_hash"
        )["prompt_hash"].transform("size")
    )

    frame["canonical_response_reused"] = (
        frame["generation_id"]
        != frame[
            "canonical_source_generation_id"
        ]
    )

    frame["output_changed_by_canonicalization"] = (
        frame["original_generated_definition"]
        .fillna("")
        .str.strip()
        != frame[
            "canonical_generated_definition"
        ]
        .fillna("")
        .str.strip()
    )

    # Replace response fields with the canonical response.
    for column in response_columns:
        canonical_column = (
            f"canonical_{column}"
        )

        if canonical_column in frame.columns:
            frame[column] = frame[
                canonical_column
            ]

    canonical_drop_columns = [
        f"canonical_{column}"
        for column in response_columns
        if f"canonical_{column}" in frame.columns
    ]

    frame = frame.drop(
        columns=canonical_drop_columns
    )

    # Exact prompt identity must now imply output identity.
    output_counts = (
        frame.groupby(
            "prompt_hash"
        )["generated_definition"]
        .nunique(dropna=False)
    )

    if output_counts.gt(1).any():
        raise AssertionError(
            f"{split_name}: identical prompts still have "
            "different canonical outputs."
        )

    frame.to_csv(
        paths["output"],
        index=False,
    )

    shared_prompt_rows = frame.loc[
        frame["prompt_multiplicity"].gt(1)
    ]

    audit_rows.append(
        {
            "split": split_name,
            "rows": int(len(frame)),
            "unique_prompt_hashes": int(
                frame["prompt_hash"].nunique()
            ),
            "shared_prompt_hashes": int(
                shared_prompt_rows[
                    "prompt_hash"
                ].nunique()
            ),
            "rows_sharing_a_prompt": int(
                len(shared_prompt_rows)
            ),
            "canonical_responses_reused": int(
                frame[
                    "canonical_response_reused"
                ].sum()
            ),
            "outputs_changed_by_canonicalization": int(
                frame[
                    "output_changed_by_canonicalization"
                ].sum()
            ),
            "output_path": str(
                paths["output"]
            ),
        }
    )

audit = pd.DataFrame(
    audit_rows
)


# ----------------------------------------------------------------
# DISPLAY AND SAVE
# ----------------------------------------------------------------

print("=" * 78)
print("IDENTICAL-PROMPT CANONICALIZATION")
print("=" * 78)

print(
    audit.to_string(
        index=False
    )
)

print(
    "\nCanonical rule: response from the earliest randomized "
    "API call for each exact prompt hash."
)

print(
    "Identical prompt hashes now imply identical generated outputs: True"
)

audit.to_csv(
    AUDIT_PATH,
    index=False,
)

metadata = {
    "purpose": (
        "Remove API-generation nondeterminism from comparisons "
        "where experimental prompts were exactly identical."
    ),
    "canonical_selection_rule": (
        "For each exact prompt hash, use the response from the "
        "smallest randomized_call_order."
    ),
    "selection_rule_uses_metric_results": False,
    "new_api_calls_made": False,
    "splits": audit.to_dict(
        orient="records"
    ),
    "audit_output": str(AUDIT_PATH),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved canonicalization audit to:")
print(AUDIT_PATH)

print("\nSaved canonicalization metadata to:")
print(METADATA_PATH)

print("\nCell 31 completed successfully.")

In [ ]:
# ================================================================
# CELL 32 — RELOAD LOCKED CONCEPT-LINKING ARTIFACTS
# ================================================================

from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"
PREDICTIONS_DIR = BASE_DIR / "predictions"

SELECTION_PATH = (
    RESULTS_DIR /
    "taskB_development_compositional_selection.json"
)

GOLD_PATH = (
    RESULTS_DIR /
    "taskB_authoritative_gold_links_with_split.csv"
)

MENTIONS_PATH = (
    RESULTS_DIR /
    "taskB_authoritative_mention_inputs_with_split.csv"
)

DEVELOPMENT_PREDICTIONS_PATH = (
    RESULTS_DIR /
    "taskB_development_compositional_predictions.csv"
)

MORPHOLOGICAL_PREDICTIONS_PATH = (
    RESULTS_DIR /
    "taskB_development_morphological_predictions.csv"
)

for path in [
    SELECTION_PATH,
    GOLD_PATH,
    MENTIONS_PATH,
    DEVELOPMENT_PREDICTIONS_PATH,
    MORPHOLOGICAL_PREDICTIONS_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required artifact not found:\n{path}"
        )

with open(
    SELECTION_PATH,
    "r",
    encoding="utf-8",
) as file_handle:
    compositional_selection = json.load(
        file_handle
    )

gold_links = pd.read_csv(GOLD_PATH)
mention_inputs = pd.read_csv(MENTIONS_PATH)
development_predictions = pd.read_csv(
    DEVELOPMENT_PREDICTIONS_PATH
)
morphological_predictions = pd.read_csv(
    MORPHOLOGICAL_PREDICTIONS_PATH
)

print("=" * 78)
print("LOCKED CONCEPT-LINKING ARTIFACTS")
print("=" * 78)

print("\nLocked compositional selection:")
print(
    json.dumps(
        compositional_selection,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nGold-link columns:")
print(list(gold_links.columns))

print("\nMention-input columns:")
print(list(mention_inputs.columns))

print("\nDevelopment compositional-prediction columns:")
print(list(development_predictions.columns))

print("\nMorphological-prediction columns:")
print(list(morphological_predictions.columns))

print("\nMention inputs by split:")
print(
    mention_inputs.groupby(
        ["domain", "split"]
    ).size()
)

print("\nGold links by split:")
print(
    gold_links.groupby(
        ["domain", "split"]
    ).size()
)

print("\nFirst five compositional development predictions:")
print(
    development_predictions.head().to_string(
        index=False
    )
)

print("\nFirst five morphological development predictions:")
print(
    morphological_predictions.head().to_string(
        index=False
    )
)

print("\nCell 32 completed successfully.")

In [ ]:
# ================================================================
# CELL 33 — RECONSTRUCT LOCKED COMPOSITIONAL LINKER
# ================================================================
#
# Reconstructs:
#   - Exact training lookup
#   - Morphological lookup
#   - Open-surface fallback
#   - Non-overlapping compositional containment, minchar=5
#
# Uses training and development only.
# Test predictions are not generated or evaluated.
# ================================================================

!pip -q install inflect

from pathlib import Path
import ast
import json
import re
import unicodedata

import inflect
import pandas as pd


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026/"
    "journal_experiments/taskB_holdout"
)

RESULTS_DIR = BASE_DIR / "results"

GOLD_PATH = (
    RESULTS_DIR /
    "taskB_authoritative_gold_links_with_split.csv"
)

MENTIONS_PATH = (
    RESULTS_DIR /
    "taskB_authoritative_mention_inputs_with_split.csv"
)

SAVED_DEVELOPMENT_PATH = (
    RESULTS_DIR /
    "taskB_development_compositional_predictions.csv"
)

AUDIT_PATH = (
    RESULTS_DIR /
    "taskB_compositional_linker_reconstruction_audit.csv"
)

MINIMUM_ATOMIC_CHARACTERS = 5

inflection_engine = inflect.engine()


# ----------------------------------------------------------------
# NORMALIZATION
# ----------------------------------------------------------------

BRITISH_TO_AMERICAN = {
    "behaviour": "behavior",
    "behaviours": "behaviors",
    "tumour": "tumor",
    "tumours": "tumors",
    "signalling": "signaling",
    "modelling": "modeling",
    "organisation": "organization",
    "organisations": "organizations",
}

IRREGULAR_SINGULARS = {
    "bacteria": "bacterium",
    "fungi": "fungus",
    "criteria": "criterion",
    "phenomena": "phenomenon",
    "analyses": "analysis",
    "diagnoses": "diagnosis",
    "indices": "index",
    "matrices": "matrix",
    "media": "medium",
    "mice": "mouse",
    "people": "person",
}


def normalize_basic(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize(
        "NFKC",
        str(value),
    ).casefold()

    value = value.replace("α", "alpha")
    value = value.replace("β", "beta")
    value = value.replace("γ", "gamma")

    value = re.sub(
        r"[‐‑‒–—−]",
        "-",
        value,
    )

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value,
    )

    return re.sub(
        r"\s+",
        " ",
        value,
    ).strip()


def singularize_token(token):
    token = BRITISH_TO_AMERICAN.get(
        token,
        token,
    )

    if token in IRREGULAR_SINGULARS:
        return IRREGULAR_SINGULARS[token]

    # Preserve common biomedical endings that are not plurals.
    if token.endswith(
        (
            "ss",
            "us",
            "is",
            "ics",
        )
    ):
        return token

    singular = inflection_engine.singular_noun(
        token
    )

    if singular:
        return str(singular)

    return token


def morphological_key(value):
    basic = normalize_basic(value)

    return " ".join(
        singularize_token(token)
        for token in basic.split()
    )


def parse_concept_set(value):
    if pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    # Handle serialized Python/JSON collections if present.
    if (
        value.startswith("[")
        or value.startswith("(")
        or value.startswith("{")
    ):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(
                parsed,
                (list, tuple, set),
            ):
                return {
                    normalize_basic(item)
                    for item in parsed
                    if normalize_basic(item)
                }
        except Exception:
            pass

    return {
        normalize_basic(item)
        for item in value.split("|")
        if normalize_basic(item)
    }


def serialize_concept_set(values):
    return " | ".join(
        sorted(values)
    )


# ----------------------------------------------------------------
# LOAD TRAINING AND DEVELOPMENT DATA
# ----------------------------------------------------------------

gold = pd.read_csv(GOLD_PATH)
mentions = pd.read_csv(MENTIONS_PATH)
saved_development = pd.read_csv(
    SAVED_DEVELOPMENT_PATH
)

training_gold = gold.loc[
    gold["split"].eq("train")
].copy()

development_mentions = mentions.loc[
    mentions["split"].eq("development")
].copy()

print("=" * 78)
print("RECONSTRUCTION SETUP")
print("=" * 78)

print("Training gold links:", len(training_gold))
print(
    "Training concepts:",
    training_gold[
        "concept_normalized_canonical"
    ].nunique(),
)
print(
    "Development mentions:",
    len(development_mentions),
)


# ----------------------------------------------------------------
# TRAINING-ONLY LOOKUP TABLES
# ----------------------------------------------------------------

exact_lookup = {}

for mention_key, frame in training_gold.groupby(
    "mention_normalized"
):
    exact_lookup[
        normalize_basic(mention_key)
    ] = {
        normalize_basic(concept)
        for concept in frame[
            "concept_normalized_canonical"
        ]
        if normalize_basic(concept)
    }

morphological_lookup = {}

for mention_key, concepts in exact_lookup.items():
    key = morphological_key(
        mention_key
    )

    morphological_lookup.setdefault(
        key,
        set(),
    ).update(concepts)

training_concepts = sorted(
    {
        normalize_basic(concept)
        for concept in training_gold[
            "concept_normalized_canonical"
        ]
        if normalize_basic(concept)
    }
)

# Candidate concepts are matched morphologically against mention tokens.
atomic_candidates = []

for concept in training_concepts:
    concept_key = normalize_basic(
        concept
    )
    concept_tokens = tuple(
        concept_key.split()
    )

    character_count = len(
        re.sub(
            r"[^a-z0-9]",
            "",
            concept_key,
        )
    )

    if (
        concept_tokens
        and character_count
        >= MINIMUM_ATOMIC_CHARACTERS
    ):
        atomic_candidates.append(
            {
                "concept": concept,
                "tokens": concept_tokens,
                "token_count": len(
                    concept_tokens
                ),
                "character_count": (
                    character_count
                ),
            }
        )

print("Exact lookup keys:", len(exact_lookup))
print(
    "Morphological lookup keys:",
    len(morphological_lookup),
)
print(
    "Eligible atomic concepts:",
    len(atomic_candidates),
)


# ----------------------------------------------------------------
# LOCKED LINKER
# ----------------------------------------------------------------

def find_contiguous_matches(
    mention_tokens,
):
    matches = []

    for candidate in atomic_candidates:
        candidate_tokens = candidate[
            "tokens"
        ]

        width = len(candidate_tokens)

        if width > len(mention_tokens):
            continue

        for start in range(
            len(mention_tokens) - width + 1
        ):
            end = start + width

            if tuple(
                mention_tokens[start:end]
            ) == candidate_tokens:
                matches.append(
                    {
                        "concept": candidate[
                            "concept"
                        ],
                        "start": start,
                        "end": end,
                        "token_count": width,
                        "character_count": (
                            candidate[
                                "character_count"
                            ]
                        ),
                    }
                )

    return matches


def select_nonoverlapping_matches(
    matches,
):
    # Longest and most specific spans first.
    ordered = sorted(
        matches,
        key=lambda item: (
            -item["token_count"],
            -item["character_count"],
            item["start"],
            item["concept"],
        ),
    )

    selected = []
    occupied_tokens = set()

    for match in ordered:
        span_tokens = set(
            range(
                match["start"],
                match["end"],
            )
        )

        if occupied_tokens.isdisjoint(
            span_tokens
        ):
            selected.append(match)
            occupied_tokens.update(
                span_tokens
            )

    selected = sorted(
        selected,
        key=lambda item: (
            item["start"],
            item["end"],
            item["concept"],
        ),
    )

    return {
        match["concept"]
        for match in selected
    }


training_concept_keys = {
    normalize_basic(concept)
    for concept in training_gold[
        "concept_canonical"
    ].dropna()
}

def predict_linker(mention):
    exact_key = normalize_basic(
        mention
    )

    morph_key = morphological_key(
        mention
    )

    exact_prediction = exact_lookup.get(
        exact_key,
        set(),
    )

    morph_prediction = morphological_lookup.get(
        morph_key,
        set(),
    )

    if morph_prediction:
        open_prediction = set(
            morph_prediction
        )
    elif morph_key:
        open_prediction = {
            morph_key
        }
    else:
        open_prediction = set()

    # Preserve a complete morphological prediction when it is
    # an authoritative concept observed in the training data.
    if (
        len(open_prediction) == 1
        and next(iter(open_prediction))
        in training_concept_keys
    ):
        compositional_prediction = set(
            open_prediction
        )

    else:
        mention_tokens = normalize_basic(
            mention
        ).split()

        matches = find_contiguous_matches(
            mention_tokens
        )

        compositional_prediction = (
            select_nonoverlapping_matches(
                matches
            )
        )

        # If no training-visible component is found,
        # retain the open-surface prediction.
        if not compositional_prediction:
            compositional_prediction = set(
                open_prediction
            )

    return {
        "predicted_exact_reconstructed": (
            exact_prediction
        ),
        "predicted_morph_lookup_reconstructed": (
            morph_prediction
        ),
        "predicted_morph_open_reconstructed": (
            open_prediction
        ),
        "predicted_compositional_reconstructed": (
            compositional_prediction
        ),
    }

# ----------------------------------------------------------------
# RECONSTRUCT DEVELOPMENT PREDICTIONS
# ----------------------------------------------------------------

reconstructed_rows = []

for row in development_mentions.itertuples(
    index=False
):
    prediction = predict_linker(
        row.mention_normalized
    )

    reconstructed_rows.append(
        {
            "domain": row.domain,
            "document_key": row.document_key,
            "mention_normalized": (
                row.mention_normalized
            ),
            **{
                key: serialize_concept_set(
                    value
                )
                for key, value in (
                    prediction.items()
                )
            },
        }
    )

reconstructed = pd.DataFrame(
    reconstructed_rows
)

saved_columns = (
    saved_development[
        [
            "domain",
            "document_key",
            "mention_normalized",
            "predicted_exact",
            "predicted_morph_lookup",
            "predicted_morph_open",
            "selected_compositional_prediction",
        ]
    ]
    .copy()
)

for column in [
    "predicted_exact",
    "predicted_morph_lookup",
    "predicted_morph_open",
    "selected_compositional_prediction",
]:
    saved_columns[
        f"{column}_set"
    ] = saved_columns[column].map(
        parse_concept_set
    )

audit = saved_columns.merge(
    reconstructed,
    on=[
        "domain",
        "document_key",
        "mention_normalized",
    ],
    how="outer",
    validate="one_to_one",
)

comparison_pairs = {
    "exact_agreement": (
        "predicted_exact_set",
        "predicted_exact_reconstructed",
    ),
    "morph_lookup_agreement": (
        "predicted_morph_lookup_set",
        "predicted_morph_lookup_reconstructed",
    ),
    "morph_open_agreement": (
        "predicted_morph_open_set",
        "predicted_morph_open_reconstructed",
    ),
    "compositional_agreement": (
        "selected_compositional_prediction_set",
        "predicted_compositional_reconstructed",
    ),
}

for agreement_column, (
    saved_column,
    reconstructed_column,
) in comparison_pairs.items():
    audit[reconstructed_column] = (
        audit[reconstructed_column]
        .map(parse_concept_set)
    )

    audit[agreement_column] = [
        saved == reconstructed_value
        for saved, reconstructed_value in zip(
            audit[saved_column],
            audit[reconstructed_column],
        )
    ]

print("\n" + "=" * 78)
print("DEVELOPMENT REPRODUCTION AGREEMENT")
print("=" * 78)

for agreement_column in comparison_pairs:
    print(
        f"{agreement_column}: "
        f"{audit[agreement_column].mean():.4f} "
        f"({audit[agreement_column].sum()}/{len(audit)})"
    )

disagreements = audit.loc[
    ~audit["compositional_agreement"]
].copy()

print("\nFirst 30 compositional disagreements:")
if disagreements.empty:
    print("None")
else:
    print(
        disagreements[
            [
                "domain",
                "document_key",
                "mention_normalized",
                "selected_compositional_prediction",
                "predicted_compositional_reconstructed",
                "predicted_morph_open",
                "predicted_morph_open_reconstructed",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

audit.to_csv(
    AUDIT_PATH,
    index=False,
)

print("\nSaved reconstruction audit to:")
print(AUDIT_PATH)

if not audit[
    "compositional_agreement"
].all():
    print(
        "\nACTION REQUIRED: Do not evaluate the test split yet. "
        "The reconstruction must first match the saved "
        "development predictions exactly."
    )
else:
    print(
        "\nRECONSTRUCTION PASSED: The locked linker can now "
        "be applied to the untouched test split."
    )

print("\nCell 33 completed successfully.")

In [ ]:
# ================================================================
# CELL 34 — LOCKED COMPOSITIONAL LINKER TEST EVALUATION
# ================================================================
#
# Applies the development-verified linker to the untouched internal
# test split. No parameters are selected or modified here.
# ================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

TEST_PREDICTIONS_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_predictions_locked.csv"
)

TEST_RESULTS_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_results_locked.csv"
)

TEST_METADATA_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_results_locked.json"
)


# ----------------------------------------------------------------
# LOCKED TEST INPUTS
# ----------------------------------------------------------------

test_mentions = mentions.loc[
    mentions["split"].eq("test")
].copy()

print("=" * 78)
print("LOCKED COMPOSITIONAL TEST INFERENCE")
print("=" * 78)

print("Test mention inputs:", len(test_mentions))
print("Minimum atomic characters:", MINIMUM_ATOMIC_CHARACTERS)
print("Training-only lookup concepts:", len(training_concepts))
print("Development reconstruction passed: True")


# ----------------------------------------------------------------
# GENERATE TEST PREDICTIONS BEFORE EVALUATION
# ----------------------------------------------------------------

prediction_rows = []

for row in test_mentions.itertuples(
    index=False
):
    prediction = predict_linker(
        row.mention_normalized
    )

    prediction_rows.append(
        {
            "domain": row.domain,
            "document_key": row.document_key,
            "num_article": getattr(
                row,
                "num_article",
                None,
            ),
            "mention": getattr(
                row,
                "mention",
                row.mention_normalized,
            ),
            "mention_normalized": (
                row.mention_normalized
            ),
            **{
                key: serialize_concept_set(
                    value
                )
                for key, value
                in prediction.items()
            },
        }
    )

test_predictions = pd.DataFrame(
    prediction_rows
)

test_predictions.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)

print("\nTest predictions generated:", len(test_predictions))
print("Saved predictions before evaluation to:")
print(TEST_PREDICTIONS_PATH)


# ----------------------------------------------------------------
# LOAD TEST GOLD ONLY AFTER PREDICTIONS ARE FROZEN
# ----------------------------------------------------------------

test_gold = gold.loc[
    gold["split"].eq("test")
].copy()

test_gold_sets = (
    test_gold
    .groupby(
        [
            "domain",
            "document_key",
            "mention_normalized",
        ],
        dropna=False,
    )["concept_normalized_canonical"]
    .apply(
        lambda values: {
            normalize_basic(value)
            for value in values
            if normalize_basic(value)
        }
    )
    .rename("gold_set")
    .reset_index()
)

evaluation = test_predictions.merge(
    test_gold_sets,
    on=[
        "domain",
        "document_key",
        "mention_normalized",
    ],
    how="left",
    validate="one_to_one",
)

evaluation["gold_set"] = evaluation[
    "gold_set"
].map(
    lambda value: (
        value
        if isinstance(value, set)
        else set()
    )
)

prediction_columns = {
    "EXACT_LOOKUP": (
        "predicted_exact_reconstructed"
    ),
    "MORPHOLOGICAL_LOOKUP": (
        "predicted_morph_lookup_reconstructed"
    ),
    "MORPHOLOGICAL_OPEN": (
        "predicted_morph_open_reconstructed"
    ),
    "COMPOSITIONAL_MINCHAR_5": (
        "predicted_compositional_reconstructed"
    ),
}

for column in prediction_columns.values():
    evaluation[f"{column}_set"] = (
        evaluation[column].map(
            parse_concept_set
        )
    )


# ----------------------------------------------------------------
# EVALUATION FUNCTIONS
# ----------------------------------------------------------------

def evaluate_prediction_sets(
    frame,
    prediction_column,
):
    gold_sets = frame["gold_set"].tolist()
    predicted_sets = frame[
        prediction_column
    ].tolist()

    true_positives = sum(
        len(gold_set & predicted_set)
        for gold_set, predicted_set
        in zip(gold_sets, predicted_sets)
    )

    false_positives = sum(
        len(predicted_set - gold_set)
        for gold_set, predicted_set
        in zip(gold_sets, predicted_sets)
    )

    false_negatives = sum(
        len(gold_set - predicted_set)
        for gold_set, predicted_set
        in zip(gold_sets, predicted_sets)
    )

    precision_denominator = (
        true_positives + false_positives
    )

    recall_denominator = (
        true_positives + false_negatives
    )

    precision = (
        true_positives / precision_denominator
        if precision_denominator
        else 0.0
    )

    recall = (
        true_positives / recall_denominator
        if recall_denominator
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    exact_set_accuracy = np.mean(
        [
            gold_set == predicted_set
            for gold_set, predicted_set
            in zip(gold_sets, predicted_sets)
        ]
    )

    jaccard_values = []

    for gold_set, predicted_set in zip(
        gold_sets,
        predicted_sets,
    ):
        union = gold_set | predicted_set

        jaccard_values.append(
            (
                len(gold_set & predicted_set)
                / len(union)
            )
            if union
            else 1.0
        )

    return {
        "mention_inputs": len(frame),
        "gold_links": sum(
            len(value)
            for value in gold_sets
        ),
        "predicted_links": sum(
            len(value)
            for value in predicted_sets
        ),
        "true_positive_links": true_positives,
        "false_positive_links": false_positives,
        "false_negative_links": false_negatives,
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": f1,
        "exact_set_accuracy": (
            exact_set_accuracy
        ),
        "mean_jaccard": float(
            np.mean(jaccard_values)
        ),
    }


# ----------------------------------------------------------------
# OVERALL AND DOMAIN-SPECIFIC RESULTS
# ----------------------------------------------------------------

result_rows = []

evaluation_groups = [
    ("ALL", evaluation)
]

evaluation_groups.extend(
    (
        domain,
        domain_frame,
    )
    for domain, domain_frame
    in evaluation.groupby(
        "domain",
        sort=True,
    )
)

for system_name, column in (
    prediction_columns.items()
):
    set_column = f"{column}_set"

    for domain, frame in evaluation_groups:
        metrics = evaluate_prediction_sets(
            frame,
            set_column,
        )

        result_rows.append(
            {
                "system": system_name,
                "domain": domain,
                **metrics,
            }
        )

test_results = pd.DataFrame(
    result_rows
)

print("\n" + "=" * 78)
print("LOCKED TEST RESULTS")
print("=" * 78)

display_columns = [
    "system",
    "domain",
    "mention_inputs",
    "predicted_links",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "exact_set_accuracy",
    "mean_jaccard",
]

print(
    test_results[
        display_columns
    ].round(4).to_string(
        index=False
    )
)


# ----------------------------------------------------------------
# SAVE DETAILED AND SUMMARY RESULTS
# ----------------------------------------------------------------

evaluation_to_save = evaluation.copy()

evaluation_to_save["gold_set"] = (
    evaluation_to_save["gold_set"].map(
        serialize_concept_set
    )
)

for column in prediction_columns.values():
    evaluation_to_save.drop(
        columns=[f"{column}_set"],
        inplace=True,
    )

evaluation_to_save.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)

test_results.to_csv(
    TEST_RESULTS_PATH,
    index=False,
)

selected_result = (
    test_results.loc[
        (
            test_results["system"]
            .eq("COMPOSITIONAL_MINCHAR_5")
        )
        & test_results["domain"].eq("ALL")
    ]
    .iloc[0]
    .to_dict()
)

metadata = {
    "evaluation_split": "test",
    "official_hidden_test": False,
    "internal_document_holdout": True,
    "test_tuning_performed": False,
    "training_evidence_only": True,
    "development_reconstruction_agreement": 1.0,
    "selected_system": (
        "COMPOSITIONAL_MINCHAR_5"
    ),
    "minimum_atomic_characters": (
        MINIMUM_ATOMIC_CHARACTERS
    ),
    "test_mention_inputs": int(
        len(test_mentions)
    ),
    "test_gold_links": int(
        len(test_gold)
    ),
    "selected_test_result": {
        key: (
            value.item()
            if hasattr(value, "item")
            else value
        )
        for key, value
        in selected_result.items()
    },
}

with open(
    TEST_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved evaluated test predictions to:")
print(TEST_PREDICTIONS_PATH)

print("\nSaved locked test results to:")
print(TEST_RESULTS_PATH)

print("\nSaved evaluation metadata to:")
print(TEST_METADATA_PATH)

print("\nCell 34 completed successfully.")

In [ ]:
# ================================================================
# CELL 35 — PAIRED DOCUMENT BOOTSTRAP FOR CONCEPT LINKING
# ================================================================

import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ----------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 42

MORPH_COLUMN = (
    "predicted_morph_open_reconstructed_set"
)

COMPOSITIONAL_COLUMN = (
    "predicted_compositional_reconstructed_set"
)

BOOTSTRAP_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_document_bootstrap.csv"
)

BOOTSTRAP_SUMMARY_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_document_bootstrap_summary.csv"
)

CHANGE_AUDIT_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_change_audit.csv"
)

BOOTSTRAP_METADATA_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_document_bootstrap_metadata.json"
)


# ----------------------------------------------------------------
# ENSURE SET COLUMNS EXIST
# ----------------------------------------------------------------

for raw_column in [
    "predicted_morph_open_reconstructed",
    "predicted_compositional_reconstructed",
]:
    set_column = f"{raw_column}_set"

    if set_column not in evaluation.columns:
        evaluation[set_column] = (
            evaluation[raw_column].map(
                parse_concept_set
            )
        )


# ----------------------------------------------------------------
# METRIC FUNCTION
# ----------------------------------------------------------------

def compact_metrics(
    frame,
    prediction_column,
):
    gold_sets = frame["gold_set"].tolist()
    predicted_sets = frame[
        prediction_column
    ].tolist()

    tp = sum(
        len(gold & predicted)
        for gold, predicted
        in zip(gold_sets, predicted_sets)
    )

    fp = sum(
        len(predicted - gold)
        for gold, predicted
        in zip(gold_sets, predicted_sets)
    )

    fn = sum(
        len(gold - predicted)
        for gold, predicted
        in zip(gold_sets, predicted_sets)
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    micro_f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    exact_accuracy = np.mean(
        [
            gold == predicted
            for gold, predicted
            in zip(gold_sets, predicted_sets)
        ]
    )

    jaccard = np.mean(
        [
            (
                len(gold & predicted)
                / len(gold | predicted)
                if gold | predicted
                else 1.0
            )
            for gold, predicted
            in zip(gold_sets, predicted_sets)
        ]
    )

    return {
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": micro_f1,
        "exact_set_accuracy": exact_accuracy,
        "mean_jaccard": jaccard,
    }


# ----------------------------------------------------------------
# OBSERVED CONTRAST
# ----------------------------------------------------------------

morph_observed = compact_metrics(
    evaluation,
    MORPH_COLUMN,
)

compositional_observed = compact_metrics(
    evaluation,
    COMPOSITIONAL_COLUMN,
)

metric_names = list(
    morph_observed.keys()
)

observed_differences = {
    metric: (
        compositional_observed[metric]
        - morph_observed[metric]
    )
    for metric in metric_names
}


# ----------------------------------------------------------------
# PAIRED DOCUMENT BOOTSTRAP
# ----------------------------------------------------------------

document_keys = (
    evaluation["document_key"]
    .drop_duplicates()
    .tolist()
)

document_frames = {
    document_key: evaluation.loc[
        evaluation["document_key"].eq(
            document_key
        )
    ].copy()
    for document_key in document_keys
}

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

for replicate in tqdm(
    range(BOOTSTRAP_REPLICATES),
    desc="Paired document bootstrap",
):
    sampled_documents = rng.choice(
        document_keys,
        size=len(document_keys),
        replace=True,
    )

    sampled_frame = pd.concat(
        [
            document_frames[document_key]
            for document_key
            in sampled_documents
        ],
        ignore_index=True,
    )

    morph_metrics = compact_metrics(
        sampled_frame,
        MORPH_COLUMN,
    )

    compositional_metrics = compact_metrics(
        sampled_frame,
        COMPOSITIONAL_COLUMN,
    )

    row = {
        "replicate": replicate,
    }

    for metric in metric_names:
        row[f"morphological_open_{metric}"] = (
            morph_metrics[metric]
        )

        row[f"compositional_{metric}"] = (
            compositional_metrics[metric]
        )

        row[f"difference_{metric}"] = (
            compositional_metrics[metric]
            - morph_metrics[metric]
        )

    bootstrap_rows.append(row)

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# ----------------------------------------------------------------
# SUMMARIZE INTERVALS
# ----------------------------------------------------------------

summary_rows = []

for metric in metric_names:
    differences = bootstrap_results[
        f"difference_{metric}"
    ].to_numpy()

    lower, upper = np.quantile(
        differences,
        [0.025, 0.975],
    )

    probability_positive = float(
        np.mean(differences > 0)
    )

    probability_negative = float(
        np.mean(differences < 0)
    )

    summary_rows.append(
        {
            "contrast": (
                "COMPOSITIONAL_MINUS_"
                "MORPHOLOGICAL_OPEN"
            ),
            "metric": metric,
            "morphological_open_observed": (
                morph_observed[metric]
            ),
            "compositional_observed": (
                compositional_observed[metric]
            ),
            "observed_difference": (
                observed_differences[metric]
            ),
            "bootstrap_mean_difference": (
                differences.mean()
            ),
            "bootstrap_median_difference": (
                np.median(differences)
            ),
            "ci_lower_95": lower,
            "ci_upper_95": upper,
            "probability_greater_than_zero": (
                probability_positive
            ),
            "probability_less_than_zero": (
                probability_negative
            ),
            "interval_interpretation": (
                "compositional_higher"
                if lower > 0
                else (
                    "compositional_lower"
                    if upper < 0
                    else "inconclusive"
                )
            ),
        }
    )

bootstrap_summary = pd.DataFrame(
    summary_rows
)


# ----------------------------------------------------------------
# PREDICTION-CHANGE AUDIT
# ----------------------------------------------------------------

change_audit = evaluation[
    [
        "domain",
        "document_key",
        "num_article",
        "mention",
        "mention_normalized",
        "gold_set",
        MORPH_COLUMN,
        COMPOSITIONAL_COLUMN,
    ]
].copy()

change_audit["prediction_changed"] = [
    morph != compositional
    for morph, compositional
    in zip(
        change_audit[MORPH_COLUMN],
        change_audit[COMPOSITIONAL_COLUMN],
    )
]

change_audit["morph_correct_links"] = [
    len(gold & predicted)
    for gold, predicted
    in zip(
        change_audit["gold_set"],
        change_audit[MORPH_COLUMN],
    )
]

change_audit["compositional_correct_links"] = [
    len(gold & predicted)
    for gold, predicted
    in zip(
        change_audit["gold_set"],
        change_audit[COMPOSITIONAL_COLUMN],
    )
]

change_audit["correct_link_difference"] = (
    change_audit[
        "compositional_correct_links"
    ]
    - change_audit[
        "morph_correct_links"
    ]
)

change_audit["morph_exact"] = [
    gold == predicted
    for gold, predicted
    in zip(
        change_audit["gold_set"],
        change_audit[MORPH_COLUMN],
    )
]

change_audit["compositional_exact"] = [
    gold == predicted
    for gold, predicted
    in zip(
        change_audit["gold_set"],
        change_audit[COMPOSITIONAL_COLUMN],
    )
]

change_audit["exact_outcome"] = np.select(
    [
        (
            ~change_audit["morph_exact"]
            & change_audit[
                "compositional_exact"
            ]
        ),
        (
            change_audit["morph_exact"]
            & ~change_audit[
                "compositional_exact"
            ]
        ),
        (
            change_audit["morph_exact"]
            & change_audit[
                "compositional_exact"
            ]
        ),
    ],
    [
        "IMPROVED",
        "REGRESSED",
        "BOTH_CORRECT",
    ],
    default="BOTH_INCORRECT",
)


# ----------------------------------------------------------------
# OUTPUT
# ----------------------------------------------------------------

print("=" * 78)
print("PAIRED DOCUMENT-BOOTSTRAP CONTRAST")
print("=" * 78)

print(
    bootstrap_summary.round(6).to_string(
        index=False
    )
)

changed = change_audit.loc[
    change_audit["prediction_changed"]
].copy()

print("\n" + "=" * 78)
print("COMPOSITIONAL INTERVENTION AUDIT")
print("=" * 78)

print("Test inputs:", len(change_audit))
print(
    "Inputs with changed predictions:",
    int(change_audit["prediction_changed"].sum()),
)

print("\nExact-set outcomes among changed inputs:")
print(
    changed["exact_outcome"]
    .value_counts()
    .to_string()
)

print("\nChanged predictions:")
print(
    changed[
        [
            "domain",
            "num_article",
            "mention",
            "gold_set",
            MORPH_COLUMN,
            COMPOSITIONAL_COLUMN,
            "exact_outcome",
        ]
    ].to_string(
        index=False
    )
)


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

bootstrap_results.to_csv(
    BOOTSTRAP_PATH,
    index=False,
)

bootstrap_summary.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
)

change_audit_to_save = (
    change_audit.copy()
)

for column in [
    "gold_set",
    MORPH_COLUMN,
    COMPOSITIONAL_COLUMN,
]:
    change_audit_to_save[column] = (
        change_audit_to_save[column].map(
            serialize_concept_set
        )
    )

change_audit_to_save.to_csv(
    CHANGE_AUDIT_PATH,
    index=False,
)

metadata = {
    "split": "test",
    "official_hidden_test": False,
    "internal_document_holdout": True,
    "contrast": (
        "COMPOSITIONAL_MINCHAR_5_MINUS_"
        "MORPHOLOGICAL_OPEN"
    ),
    "bootstrap_unit": "document",
    "documents": len(document_keys),
    "mention_inputs": len(evaluation),
    "replicates": BOOTSTRAP_REPLICATES,
    "seed": BOOTSTRAP_SEED,
    "parameters_retuned_on_test": False,
}

with open(
    BOOTSTRAP_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        metadata,
        file_handle,
        indent=2,
    )

print("\nSaved bootstrap replicates to:")
print(BOOTSTRAP_PATH)

print("\nSaved bootstrap summary to:")
print(BOOTSTRAP_SUMMARY_PATH)

print("\nSaved prediction-change audit to:")
print(CHANGE_AUDIT_PATH)

print("\nSaved bootstrap metadata to:")
print(BOOTSTRAP_METADATA_PATH)

print("\nCell 35 completed successfully.")

In [ ]:
# ================================================================
# CELL 36 — TEST SUBGROUP AND FAILURE-MODE ANALYSIS
# ================================================================

import json

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------

SUBGROUP_RESULTS_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_subgroup_results.csv"
)

SUBGROUP_SUMMARY_PATH = (
    RESULTS_DIR /
    "taskB_test_compositional_subgroup_summary.json"
)


# ----------------------------------------------------------------
# DEFINE TRAINING-FAMILIARITY GROUPS
# ----------------------------------------------------------------

analysis = evaluation.copy()

analysis["exact_training_match"] = (
    analysis[
        "predicted_exact_reconstructed_set"
    ].map(bool)
)

analysis["morphological_training_match"] = (
    analysis[
        "predicted_morph_lookup_reconstructed_set"
    ].map(bool)
)

analysis["familiarity_group"] = np.select(
    [
        analysis["exact_training_match"],
        (
            ~analysis["exact_training_match"]
            & analysis[
                "morphological_training_match"
            ]
        ),
    ],
    [
        "EXACT_TRAINING_MATCH",
        "MORPHOLOGICAL_TRAINING_MATCH",
    ],
    default="OPEN_UNSEEN_SURFACE",
)

analysis["gold_cardinality"] = (
    analysis["gold_set"].map(len)
)

analysis["cardinality_group"] = np.where(
    analysis["gold_cardinality"].eq(1),
    "SINGLE_LABEL",
    "MULTI_LABEL",
)

analysis["mention_word_count"] = (
    analysis["mention_normalized"]
    .fillna("")
    .str.split()
    .str.len()
)

analysis["length_group"] = np.where(
    analysis["mention_word_count"].eq(1),
    "SINGLE_WORD",
    "MULTIWORD",
)


# ----------------------------------------------------------------
# SUBGROUP EVALUATION
# ----------------------------------------------------------------

systems = {
    "MORPHOLOGICAL_OPEN": (
        "predicted_morph_open_reconstructed_set"
    ),
    "COMPOSITIONAL_MINCHAR_5": (
        "predicted_compositional_reconstructed_set"
    ),
}

subgroup_rows = []

grouping_variables = {
    "familiarity": "familiarity_group",
    "gold_cardinality": "cardinality_group",
    "mention_length": "length_group",
}

for analysis_name, group_column in (
    grouping_variables.items()
):
    for subgroup, frame in analysis.groupby(
        group_column,
        sort=True,
    ):
        for system_name, prediction_column in (
            systems.items()
        ):
            metrics = compact_metrics(
                frame,
                prediction_column,
            )

            subgroup_rows.append(
                {
                    "analysis": analysis_name,
                    "subgroup": subgroup,
                    "system": system_name,
                    "mention_inputs": len(frame),
                    **metrics,
                }
            )

subgroup_results = pd.DataFrame(
    subgroup_rows
)


# ----------------------------------------------------------------
# PAIRED DIFFERENCES WITHIN EACH SUBGROUP
# ----------------------------------------------------------------

difference_rows = []

for analysis_name, group_column in (
    grouping_variables.items()
):
    for subgroup, frame in analysis.groupby(
        group_column,
        sort=True,
    ):
        morph_metrics = compact_metrics(
            frame,
            (
                "predicted_morph_open_"
                "reconstructed_set"
            ),
        )

        compositional_metrics = compact_metrics(
            frame,
            (
                "predicted_compositional_"
                "reconstructed_set"
            ),
        )

        difference_rows.append(
            {
                "analysis": analysis_name,
                "subgroup": subgroup,
                "mention_inputs": len(frame),
                "micro_precision_difference": (
                    compositional_metrics[
                        "micro_precision"
                    ]
                    - morph_metrics[
                        "micro_precision"
                    ]
                ),
                "micro_recall_difference": (
                    compositional_metrics[
                        "micro_recall"
                    ]
                    - morph_metrics[
                        "micro_recall"
                    ]
                ),
                "micro_f1_difference": (
                    compositional_metrics[
                        "micro_f1"
                    ]
                    - morph_metrics[
                        "micro_f1"
                    ]
                ),
                "exact_accuracy_difference": (
                    compositional_metrics[
                        "exact_set_accuracy"
                    ]
                    - morph_metrics[
                        "exact_set_accuracy"
                    ]
                ),
                "mean_jaccard_difference": (
                    compositional_metrics[
                        "mean_jaccard"
                    ]
                    - morph_metrics[
                        "mean_jaccard"
                    ]
                ),
            }
        )

difference_summary = pd.DataFrame(
    difference_rows
)


# ----------------------------------------------------------------
# INTERVENTION OUTCOMES BY SUBGROUP
# ----------------------------------------------------------------

analysis["prediction_changed"] = [
    morph != compositional
    for morph, compositional
    in zip(
        analysis[
            "predicted_morph_open_"
            "reconstructed_set"
        ],
        analysis[
            "predicted_compositional_"
            "reconstructed_set"
        ],
    )
]

analysis["morph_exact"] = [
    gold == predicted
    for gold, predicted
    in zip(
        analysis["gold_set"],
        analysis[
            "predicted_morph_open_"
            "reconstructed_set"
        ],
    )
]

analysis["compositional_exact"] = [
    gold == predicted
    for gold, predicted
    in zip(
        analysis["gold_set"],
        analysis[
            "predicted_compositional_"
            "reconstructed_set"
        ],
    )
]

analysis["intervention_outcome"] = np.select(
    [
        (
            analysis["prediction_changed"]
            & ~analysis["morph_exact"]
            & analysis["compositional_exact"]
        ),
        (
            analysis["prediction_changed"]
            & analysis["morph_exact"]
            & ~analysis["compositional_exact"]
        ),
        analysis["prediction_changed"],
    ],
    [
        "IMPROVED",
        "REGRESSED",
        "CHANGED_WITHOUT_EXACT_GAIN",
    ],
    default="UNCHANGED",
)

intervention_rows = []

for analysis_name, group_column in (
    grouping_variables.items()
):
    for subgroup, frame in analysis.groupby(
        group_column,
        sort=True,
    ):
        changed_frame = frame.loc[
            frame["prediction_changed"]
        ]

        intervention_rows.append(
            {
                "analysis": analysis_name,
                "subgroup": subgroup,
                "mention_inputs": len(frame),
                "changed_predictions": len(
                    changed_frame
                ),
                "improved": int(
                    changed_frame[
                        "intervention_outcome"
                    ].eq("IMPROVED").sum()
                ),
                "regressed": int(
                    changed_frame[
                        "intervention_outcome"
                    ].eq("REGRESSED").sum()
                ),
                "changed_without_exact_gain": int(
                    changed_frame[
                        "intervention_outcome"
                    ]
                    .eq(
                        "CHANGED_WITHOUT_EXACT_GAIN"
                    )
                    .sum()
                ),
            }
        )

intervention_summary = pd.DataFrame(
    intervention_rows
)


# ----------------------------------------------------------------
# PRINT RESULTS
# ----------------------------------------------------------------

print("=" * 78)
print("TEST SUBGROUP PERFORMANCE")
print("=" * 78)

print(
    subgroup_results[
        [
            "analysis",
            "subgroup",
            "system",
            "mention_inputs",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 78)
print("COMPOSITIONAL − MORPHOLOGICAL-OPEN")
print("=" * 78)

print(
    difference_summary
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 78)
print("INTERVENTION OUTCOMES BY SUBGROUP")
print("=" * 78)

print(
    intervention_summary.to_string(
        index=False
    )
)


# ----------------------------------------------------------------
# SAVE
# ----------------------------------------------------------------

subgroup_results.to_csv(
    SUBGROUP_RESULTS_PATH,
    index=False,
)

summary_payload = {
    "split": "test",
    "systems": list(systems.keys()),
    "subgroup_differences": (
        difference_summary.to_dict(
            orient="records"
        )
    ),
    "intervention_outcomes": (
        intervention_summary.to_dict(
            orient="records"
        )
    ),
}

with open(
    SUBGROUP_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        summary_payload,
        file_handle,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved subgroup results to:")
print(SUBGROUP_RESULTS_PATH)

print("\nSaved subgroup summary to:")
print(SUBGROUP_SUMMARY_PATH)

print("\nCell 36 completed successfully.")

In [ ]:
# ================================================================
# CELL 37 — CONSOLIDATE TASK B PUBLICATION RESULTS
# ================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from openpyxl.styles import (
    Alignment,
    Font,
    PatternFill,
)
from openpyxl.utils import get_column_letter


# ----------------------------------------------------------------
# OUTPUT PATHS
# ----------------------------------------------------------------

PUBLICATION_WORKBOOK_PATH = (
    RESULTS_DIR /
    "DETECH_TaskB_holdout_publication_results.xlsx"
)

PUBLICATION_SUMMARY_PATH = (
    RESULTS_DIR /
    "DETECH_TaskB_holdout_publication_summary.json"
)


# ----------------------------------------------------------------
# SUPPORT FUNCTIONS
# ----------------------------------------------------------------

def load_required_csv(filename):
    path = RESULTS_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Required result file not found: {path}"
        )

    return pd.read_csv(path)


def make_json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return [
            make_json_safe(item)
            for item in value.tolist()
        ]

    if isinstance(value, set):
        return sorted(
            make_json_safe(item)
            for item in value
        )

    if isinstance(value, np.generic):
        return value.item()

    if pd.isna(value):
        return None

    return value


# ----------------------------------------------------------------
# RECONSTRUCT DEVELOPMENT LINKING SUMMARY
# ----------------------------------------------------------------

development_gold = gold.loc[
    gold["split"].eq("development")
].copy()

development_gold_sets = (
    development_gold
    .groupby(
        [
            "domain",
            "document_key",
            "mention_normalized",
        ],
        dropna=False,
    )["concept_normalized_canonical"]
    .apply(
        lambda values: {
            normalize_basic(value)
            for value in values
            if normalize_basic(value)
        }
    )
    .rename("gold_set")
    .reset_index()
)

development_evaluation = (
    saved_development.merge(
        development_gold_sets,
        on=[
            "domain",
            "document_key",
            "mention_normalized",
        ],
        how="left",
        validate="one_to_one",
    )
)

development_linking_columns = {
    "EXACT_LOOKUP": "predicted_exact",
    "MORPHOLOGICAL_LOOKUP": (
        "predicted_morph_lookup"
    ),
    "MORPHOLOGICAL_OPEN": (
        "predicted_morph_open"
    ),
    "COMPOSITIONAL_MINCHAR_5": (
        "selected_compositional_prediction"
    ),
}

development_linking_rows = []

for system_name, raw_column in (
    development_linking_columns.items()
):
    temporary_column = (
        f"{raw_column}_publication_set"
    )

    development_evaluation[
        temporary_column
    ] = development_evaluation[
        raw_column
    ].map(parse_concept_set)

    groups = [
        ("ALL", development_evaluation)
    ]

    groups.extend(
        (
            domain,
            domain_frame,
        )
        for domain, domain_frame
        in development_evaluation.groupby(
            "domain",
            sort=True,
        )
    )

    for domain, frame in groups:
        metrics = evaluate_prediction_sets(
            frame,
            temporary_column,
        )

        development_linking_rows.append(
            {
                "split": "development",
                "system": system_name,
                "domain": domain,
                **metrics,
            }
        )

development_linking_summary = pd.DataFrame(
    development_linking_rows
)


# ----------------------------------------------------------------
# LOAD FINAL TEST LINKING RESULTS
# ----------------------------------------------------------------

test_linking_summary = load_required_csv(
    "taskB_test_compositional_results_locked.csv"
)

test_linking_summary.insert(
    0,
    "split",
    "test",
)

linking_main_results = pd.concat(
    [
        development_linking_summary,
        test_linking_summary,
    ],
    ignore_index=True,
    sort=False,
)

linking_bootstrap = load_required_csv(
    "taskB_test_compositional_"
    "document_bootstrap_summary.csv"
)

linking_subgroups = load_required_csv(
    "taskB_test_compositional_"
    "subgroup_results.csv"
)

linking_changes = load_required_csv(
    "taskB_test_compositional_"
    "change_audit.csv"
)


# ----------------------------------------------------------------
# LOAD FINAL DEFINITION RESULTS
# ----------------------------------------------------------------

definition_development = load_required_csv(
    "taskB_definition_development_"
    "ISO_summary_with_ROUGE.csv"
)

definition_test = load_required_csv(
    "taskB_definition_test_"
    "ISO_condition_metrics.csv"
)

definition_development_bootstrap = (
    load_required_csv(
        "taskB_definition_development_"
        "ISO_paired_bootstrap_summary.csv"
    )
)

definition_test_bootstrap = (
    load_required_csv(
        "taskB_definition_test_"
        "ISO_paired_bootstrap_summary.csv"
    )
)

definition_intervention = (
    load_required_csv(
        "taskB_definition_collapse_"
        "active_intervention_summary.csv"
    )
)


# ----------------------------------------------------------------
# CREATE PUBLICATION OVERVIEW
# ----------------------------------------------------------------

def select_result(
    frame,
    split,
    system,
    domain="ALL",
):
    selected = frame.loc[
        frame["split"].eq(split)
        & frame["system"].eq(system)
        & frame["domain"].eq(domain)
    ]

    if len(selected) != 1:
        raise ValueError(
            "Could not identify one linking result "
            f"for {split}, {system}, {domain}."
        )

    return selected.iloc[0]


development_compositional = select_result(
    linking_main_results,
    "development",
    "COMPOSITIONAL_MINCHAR_5",
)

test_compositional = select_result(
    linking_main_results,
    "test",
    "COMPOSITIONAL_MINCHAR_5",
)

test_morphological = select_result(
    linking_main_results,
    "test",
    "MORPHOLOGICAL_OPEN",
)

definition_test_indexed = (
    definition_test.set_index("condition")
)

overview_rows = [
    {
        "result_area": "Concept linking",
        "comparison": (
            "Selected compositional system — "
            "development"
        ),
        "primary_metric": "micro_f1",
        "value": development_compositional[
            "micro_f1"
        ],
        "interpretation": (
            "Development-selected system."
        ),
    },
    {
        "result_area": "Concept linking",
        "comparison": (
            "Selected compositional system — test"
        ),
        "primary_metric": "micro_f1",
        "value": test_compositional["micro_f1"],
        "interpretation": (
            "Locked internal document-holdout result."
        ),
    },
    {
        "result_area": "Concept linking",
        "comparison": (
            "Morphological-open baseline — test"
        ),
        "primary_metric": "micro_f1",
        "value": test_morphological["micro_f1"],
        "interpretation": (
            "Descriptively exceeded the selected "
            "compositional system."
        ),
    },
    {
        "result_area": "Concept linking",
        "comparison": (
            "Compositional minus morphological-open"
        ),
        "primary_metric": "micro_f1_difference",
        "value": (
            test_compositional["micro_f1"]
            - test_morphological["micro_f1"]
        ),
        "interpretation": (
            "Document-bootstrap interval included zero."
        ),
    },
    {
        "result_area": "Definition generation",
        "comparison": "Concept-only — test",
        "primary_metric": (
            "descriptive_combined_score"
        ),
        "value": definition_test_indexed.loc[
            "CONCEPT_ONLY",
            "descriptive_combined_score",
        ],
        "interpretation": (
            "Descriptively strongest locked test "
            "condition."
        ),
    },
    {
        "result_area": "Definition generation",
        "comparison": "Unranked contexts — test",
        "primary_metric": (
            "descriptive_combined_score"
        ),
        "value": definition_test_indexed.loc[
            "UNRANKED_CONTEXTS",
            "descriptive_combined_score",
        ],
        "interpretation": (
            "No reliable difference from concept-only."
        ),
    },
    {
        "result_area": "Definition generation",
        "comparison": "Collapse K50 — test",
        "primary_metric": (
            "descriptive_combined_score"
        ),
        "value": definition_test_indexed.loc[
            "COLLAPSE_K50",
            "descriptive_combined_score",
        ],
        "interpretation": (
            "No reliable improvement over either "
            "baseline."
        ),
    },
]

publication_overview = pd.DataFrame(
    overview_rows
)


# ----------------------------------------------------------------
# PROTOCOL AND INTERPRETATION TABLE
# ----------------------------------------------------------------

protocol_rows = [
    {
        "component": "Concept-linking split",
        "value": (
            "Grouped document holdout: "
            "training, development, test"
        ),
    },
    {
        "component": "Selected linker",
        "value": (
            "Non-overlapping compositional "
            "containment, minimum atomic length 5"
        ),
    },
    {
        "component": "Linker selection",
        "value": (
            "Selected on development before test "
            "evaluation"
        ),
    },
    {
        "component": "Linker reconstruction",
        "value": (
            "Exact agreement on 102/102 development "
            "inputs"
        ),
    },
    {
        "component": "Linking test",
        "value": (
            "Internal document holdout; no official "
            "hidden Task B reference was provided"
        ),
    },
    {
        "component": "Linking uncertainty",
        "value": (
            "5,000 paired document-bootstrap "
            "replicates"
        ),
    },
    {
        "component": "Definition concept split",
        "value": (
            "Disjoint training, development, and test "
            "concept partitions"
        ),
    },
    {
        "component": "Definition model",
        "value": "gpt-4o-mini; temperature 0",
    },
    {
        "component": "Prompt style",
        "value": "DETECH_ISO_INTENSIONAL_V1",
    },
    {
        "component": "Selected collapse setting",
        "value": (
            "K=50, selected using development only"
        ),
    },
    {
        "component": "Definition test safeguards",
        "value": (
            "Reference-blind prompt construction and "
            "generation"
        ),
    },
    {
        "component": "Identical-prompt control",
        "value": (
            "Exact prompt hashes canonicalized to the "
            "earliest randomized API response"
        ),
    },
    {
        "component": "Definition uncertainty",
        "value": (
            "5,000 paired concept-bootstrap "
            "replicates"
        ),
    },
    {
        "component": "Primary interpretation",
        "value": (
            "Collapse-based context selection did not "
            "reliably improve definition generation."
        ),
    },
    {
        "component": "Compositional interpretation",
        "value": (
            "Improved the small multi-label subgroup "
            "but reduced aggregate held-out accuracy."
        ),
    },
]

protocol_table = pd.DataFrame(
    protocol_rows
)


# ----------------------------------------------------------------
# SAVE WORKBOOK
# ----------------------------------------------------------------

sheets = {
    "Overview": publication_overview,
    "Linking main": linking_main_results,
    "Linking bootstrap": linking_bootstrap,
    "Linking subgroups": linking_subgroups,
    "Linking changes": linking_changes,
    "Definition dev": definition_development,
    "Definition test": definition_test,
    "Definition dev boot": (
        definition_development_bootstrap
    ),
    "Definition test boot": (
        definition_test_bootstrap
    ),
    "Definition intervention": (
        definition_intervention
    ),
    "Protocol": protocol_table,
}

with pd.ExcelWriter(
    PUBLICATION_WORKBOOK_PATH,
    engine="openpyxl",
) as writer:
    for sheet_name, frame in sheets.items():
        frame.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
        )

    workbook = writer.book

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78",
    )

    header_font = Font(
        color="FFFFFF",
        bold=True,
    )

    section_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )

    for sheet_name in sheets:
        worksheet = workbook[sheet_name]

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        worksheet.row_dimensions[1].height = 32

        for column_cells in worksheet.columns:
            values = [
                str(cell.value)
                if cell.value is not None
                else ""
                for cell in column_cells
            ]

            maximum_length = min(
                max(
                    len(value)
                    for value in values
                )
                + 2,
                55,
            )

            column_letter = get_column_letter(
                column_cells[0].column
            )

            worksheet.column_dimensions[
                column_letter
            ].width = max(
                12,
                maximum_length,
            )

        for row in worksheet.iter_rows(
            min_row=2
        ):
            for cell in row:
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True,
                )

                if isinstance(
                    cell.value,
                    float,
                ):
                    cell.number_format = "0.0000"

    overview_sheet = workbook["Overview"]

    for row_number in range(
        2,
        overview_sheet.max_row + 1,
    ):
        overview_sheet.cell(
            row=row_number,
            column=1,
        ).fill = section_fill


# ----------------------------------------------------------------
# MACHINE-READABLE SUMMARY
# ----------------------------------------------------------------

linking_bootstrap_f1 = (
    linking_bootstrap.loc[
        linking_bootstrap["metric"].eq(
            "micro_f1"
        )
    ]
    .iloc[0]
    .to_dict()
)

multi_label_rows = linking_subgroups.loc[
    (
        linking_subgroups["analysis"]
        .eq("gold_cardinality")
    )
    & (
        linking_subgroups["subgroup"]
        .eq("MULTI_LABEL")
    )
]

single_label_rows = linking_subgroups.loc[
    (
        linking_subgroups["analysis"]
        .eq("gold_cardinality")
    )
    & (
        linking_subgroups["subgroup"]
        .eq("SINGLE_LABEL")
    )
]

publication_summary = {
    "task": "DETECH Task B",
    "evaluation_design": {
        "official_hidden_test_reference": False,
        "internal_document_holdout": True,
        "test_tuning_performed": False,
        "concept_linker_development_reconstruction": (
            "102/102"
        ),
        "definition_generation_reference_blind": True,
    },
    "concept_linking": {
        "development_selected_system": (
            "COMPOSITIONAL_MINCHAR_5"
        ),
        "development_selected_micro_f1": (
            development_compositional["micro_f1"]
        ),
        "test_compositional": (
            test_compositional.to_dict()
        ),
        "test_morphological_open": (
            test_morphological.to_dict()
        ),
        "paired_document_bootstrap_micro_f1": (
            linking_bootstrap_f1
        ),
        "multi_label_subgroup": (
            multi_label_rows.to_dict(
                orient="records"
            )
        ),
        "single_label_subgroup": (
            single_label_rows.to_dict(
                orient="records"
            )
        ),
        "changed_test_predictions": 13,
        "improved_exact_predictions": 3,
        "regressed_exact_predictions": 7,
    },
    "definition_generation": {
        "selected_collapse_condition": (
            "COLLAPSE_K50"
        ),
        "development_results": (
            definition_development.to_dict(
                orient="records"
            )
        ),
        "test_results": (
            definition_test.to_dict(
                orient="records"
            )
        ),
        "test_inference": (
            "All paired bootstrap intervals included "
            "zero."
        ),
        "test_evidence_changed_concepts": 14,
        "test_output_changed_concepts": 14,
    },
    "main_conclusions": [
        (
            "The morphological-open concept linker "
            "descriptively outperformed the selected "
            "compositional linker on the held-out test."
        ),
        (
            "Compositional decomposition improved the "
            "small multi-label subgroup but harmed the "
            "predominant single-label subgroup."
        ),
        (
            "Concept-only definition generation was "
            "descriptively strongest on development "
            "and test."
        ),
        (
            "Neither unranked contextual evidence nor "
            "collapse-ranked evidence produced a "
            "reliable definition-generation gain."
        ),
    ],
}

with open(
    PUBLICATION_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        make_json_safe(publication_summary),
        file_handle,
        ensure_ascii=False,
        indent=2,
    )


# ----------------------------------------------------------------
# FINAL AUDIT
# ----------------------------------------------------------------

print("=" * 78)
print("TASK B PUBLICATION RESULTS CONSOLIDATED")
print("=" * 78)

print("\nConcept-linking headline results:")
print(
    linking_main_results.loc[
        (
            linking_main_results["domain"]
            .eq("ALL")
        )
        & (
            linking_main_results["system"]
            .isin(
                [
                    "MORPHOLOGICAL_OPEN",
                    "COMPOSITIONAL_MINCHAR_5",
                ]
            )
        ),
        [
            "split",
            "system",
            "micro_precision",
            "micro_recall",
            "micro_f1",
            "exact_set_accuracy",
            "mean_jaccard",
        ],
    ]
    .round(4)
    .to_string(index=False)
)

print("\nDefinition-generation test results:")
print(
    definition_test[
        [
            "condition",
            "corpus_bleu",
            "rouge1_f1_mean",
            "rouge2_f1_mean",
            "rougeL_f1_mean",
            "biomedbert_f1_mean",
            "descriptive_combined_score",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\nWorkbook sheets:", len(sheets))

print("\nSaved publication workbook to:")
print(PUBLICATION_WORKBOOK_PATH)

print("\nSaved machine-readable summary to:")
print(PUBLICATION_SUMMARY_PATH)

print("\nCell 37 completed successfully.")